In [1]:
import boto3
import pandas as pd
import numpy as np

BUCKET = "construction-payment-risk-dev-gk53"
CURATED_PREFIX = "curated/payment-risk/"

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix=CURATED_PREFIX
)

objects = response.get("Contents", [])

print(f"Objects found: {len(objects)}")

for obj in objects[:20]:
    print(obj["Key"])

Objects found: 21
curated/payment-risk/assessment_year=2025/assessment_month=1/part-00000-669590c8-5ab9-45a5-a037-c4c7a0456a8e.c000.snappy.parquet
curated/payment-risk/assessment_year=2025/assessment_month=10/part-00000-669590c8-5ab9-45a5-a037-c4c7a0456a8e.c000.snappy.parquet
curated/payment-risk/assessment_year=2025/assessment_month=11/part-00000-669590c8-5ab9-45a5-a037-c4c7a0456a8e.c000.snappy.parquet
curated/payment-risk/assessment_year=2025/assessment_month=12/part-00000-669590c8-5ab9-45a5-a037-c4c7a0456a8e.c000.snappy.parquet
curated/payment-risk/assessment_year=2025/assessment_month=2/part-00000-669590c8-5ab9-45a5-a037-c4c7a0456a8e.c000.snappy.parquet
curated/payment-risk/assessment_year=2025/assessment_month=3/part-00000-669590c8-5ab9-45a5-a037-c4c7a0456a8e.c000.snappy.parquet
curated/payment-risk/assessment_year=2025/assessment_month=4/part-00000-669590c8-5ab9-45a5-a037-c4c7a0456a8e.c000.snappy.parquet
curated/payment-risk/assessment_year=2025/assessment_month=5/part-00000-6695

In [2]:
import pyarrow.dataset as ds
import pyarrow.fs as fs
import pandas as pd

REGION = "ap-southeast-2"

s3_fs = fs.S3FileSystem(region=REGION)

dataset = ds.dataset(
    f"{BUCKET}/{CURATED_PREFIX}",
    filesystem=s3_fs,
    format="parquet",
    partitioning="hive",
)

df = dataset.to_table().to_pandas()

print("Dataset loaded successfully")
print("-" * 50)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

print(
    f"Positive cases: "
    f"{df['escalation_required'].sum():,}"
)

print(
    f"Positive rate: "
    f"{df['escalation_required'].mean() * 100:.2f}%"
)

print(
    f"Assessment date range: "
    f"{df['assessment_date'].min()} "
    f"to {df['assessment_date'].max()}"
)

print(
    f"Unique record IDs: "
    f"{df['record_id'].nunique():,}"
)

Dataset loaded successfully
--------------------------------------------------
Rows: 10,000
Columns: 45
Positive cases: 2,389
Positive rate: 23.89%
Assessment date range: 2025-01-01 to 2026-09-03
Unique record IDs: 10,000


In [3]:
# Ensure assessment_date is datetime
df["assessment_date"] = pd.to_datetime(df["assessment_date"])

train_df = df[
    df["assessment_date"] < "2026-03-01"
].copy()

validation_df = df[
    (df["assessment_date"] >= "2026-03-01") &
    (df["assessment_date"] < "2026-06-01")
].copy()

test_df = df[
    (df["assessment_date"] >= "2026-06-01") &
    (df["assessment_date"] < "2026-09-01")
].copy()

partial_holdout_df = df[
    df["assessment_date"] >= "2026-09-01"
].copy()


def summarize_split(name, data):
    positives = int(data["escalation_required"].sum())
    total = len(data)
    rate = 100 * data["escalation_required"].mean()

    print(
        f"{name:<18} "
        f"rows={total:>5,} | "
        f"positives={positives:>4,} | "
        f"rate={rate:>6.2f}%"
    )


summarize_split("TRAIN", train_df)
summarize_split("VALIDATION", validation_df)
summarize_split("TEST", test_df)
summarize_split("PARTIAL_HOLDOUT", partial_holdout_df)

print("-" * 70)
print(
    "Total rows:",
    len(train_df)
    + len(validation_df)
    + len(test_df)
    + len(partial_holdout_df)
)

TRAIN              rows=6,892 | positives=1,666 | rate= 24.17%
VALIDATION         rows=1,581 | positives= 398 | rate= 25.17%
TEST               rows=1,479 | positives= 314 | rate= 21.23%
PARTIAL_HOLDOUT    rows=   48 | positives=  11 | rate= 22.92%
----------------------------------------------------------------------
Total rows: 10000


In [4]:
print(f"Total columns: {len(df.columns)}")
print("-" * 60)

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column:<40} {str(df[column].dtype)}")

Total columns: 45
------------------------------------------------------------
01. record_id                                object
02. project_id                               object
03. customer_id                              object
04. assessment_date                          datetime64[ns]
05. state                                    object
06. project_type                             object
07. public_private                           object
08. customer_role                            object
09. contractual_tier                         int32
10. hiring_party_type                        object
11. distance_from_owner                      int32
12. owner_identified                         int32
13. general_contractor_identified            int32
14. hiring_party_identified                  int32
15. surety_applicable                        int32
16. surety_identified                        int32
17. lender_identified                        int32
18. known_party_count                

Feature Contract v1

Using both raw and derived versions isn't automatically wrong, especially for tree models. But for our first Logistic Regression baseline, reducing redundancy makes the coefficients easier to interpret.
We're also deliberately postponing contractual_tier and distance_from_owner: our clean EDA found very weak individual linear target correlations for them. Later we'll test whether adding them improves validation performance, and XGBoost can investigate nonlinear interactions.


In [5]:
TARGET = "escalation_required"

CATEGORICAL_FEATURES = [
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type",
]

NUMERIC_FEATURES = [
    "payment_chain_completeness_score",
    "research_confidence_score",
    "deadline_days_remaining",
    "prior_escalation_rate",
    "prior_projects_with_hiring_party",
    "expected_party_count",
]

BINARY_FEATURES = [
    "critical_field_missing",
    "multiple_candidate_records",
    "conflicting_project_information",
]

MODEL_FEATURES = (
    CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
    + BINARY_FEATURES
)

print("Feature Contract v1")
print("-" * 50)
print(f"Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"Numeric features:     {len(NUMERIC_FEATURES)}")
print(f"Binary features:      {len(BINARY_FEATURES)}")
print(f"Total model features: {len(MODEL_FEATURES)}")

print("\nFeatures:")
for feature in MODEL_FEATURES:
    print(f" - {feature}")

assert TARGET not in MODEL_FEATURES
assert "assessment_date" not in MODEL_FEATURES
assert "record_id" not in MODEL_FEATURES
assert len(MODEL_FEATURES) == len(set(MODEL_FEATURES))

print("\nFeature contract validation: PASS")

Feature Contract v1
--------------------------------------------------
Categorical features: 5
Numeric features:     6
Binary features:      3
Total model features: 14

Features:
 - state
 - project_type
 - public_private
 - customer_role
 - hiring_party_type
 - payment_chain_completeness_score
 - research_confidence_score
 - deadline_days_remaining
 - prior_escalation_rate
 - prior_projects_with_hiring_party
 - expected_party_count
 - critical_field_missing
 - multiple_candidate_records
 - conflicting_project_information

Feature contract validation: PASS


Build the preprocessing pipeline.


Categorical
state, project_type, etc.
        ↓
OneHotEncoder

Numeric
confidence scores, deadline days, counts...
        ↓
StandardScaler

Binary
0 / 1 indicators
        ↓
Pass through

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Numeric preprocessing
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=None
            )
        ),
    ]
)

# Binary preprocessing
binary_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        ),
        (
            "binary",
            binary_pipeline,
            BINARY_FEATURES
        ),
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")
print(preprocessor)

Preprocessing pipeline created successfully.
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['payment_chain_completeness_score',
                                  'research_confidence_score',
                                  'deadline_days_remaining',
                                  'prior_escalation_rate',
                                  'prior_projects_with_hiring_party',
                                  'expected_party_count']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                        

Build modeling matrices and integrity checks

In [7]:
# Build train/validation matrices from the frozen temporal split

X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df[TARGET].copy()

X_validation = validation_df[MODEL_FEATURES].copy()
y_validation = validation_df[TARGET].copy()

X_test = test_df[MODEL_FEATURES].copy()
y_test = test_df[TARGET].copy()

print("Shapes")
print("-" * 50)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTarget rates")
print("-" * 50)
print(f"Train:      {y_train.mean() * 100:.2f}%")
print(f"Validation: {y_validation.mean() * 100:.2f}%")
print(f"Test:       {y_test.mean() * 100:.2f}%")

# Integrity checks
assert len(X_train) == 6892
assert len(X_validation) == 1581
assert len(X_test) == 1479

assert list(X_train.columns) == MODEL_FEATURES
assert TARGET not in X_train.columns
assert "assessment_date" not in X_train.columns
assert "record_id" not in X_train.columns

assert y_train.isna().sum() == 0
assert y_validation.isna().sum() == 0
assert y_test.isna().sum() == 0

assert set(y_train.unique()).issubset({0, 1})
assert set(y_validation.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})

print("\nModeling dataset integrity checks: PASS")

Shapes
--------------------------------------------------
X_train: (6892, 14)
y_train: (6892,)
X_validation: (1581, 14)
y_validation: (1581,)
X_test: (1479, 14)
y_test: (1479,)

Target rates
--------------------------------------------------
Train:      24.17%
Validation: 25.17%
Test:       21.23%

Modeling dataset integrity checks: PASS


Train Logistic Regression Baseline

In [8]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        ),
    ]
)

# Train only on TRAIN
logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression training complete.")

Logistic Regression training complete.


In [9]:
# Generate validation probabilities and default-threshold predictions

validation_probabilities = logistic_pipeline.predict_proba(
    X_validation
)[:, 1]

validation_predictions = (
    validation_probabilities >= 0.50
).astype(int)

roc_auc = roc_auc_score(
    y_validation,
    validation_probabilities
)

pr_auc = average_precision_score(
    y_validation,
    validation_probabilities
)

precision = precision_score(
    y_validation,
    validation_predictions,
    zero_division=0
)

recall = recall_score(
    y_validation,
    validation_predictions,
    zero_division=0
)

f1 = f1_score(
    y_validation,
    validation_predictions,
    zero_division=0
)

cm = confusion_matrix(
    y_validation,
    validation_predictions
)

print("VALIDATION METRICS")
print("-" * 50)

print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nConfusion Matrix")
print(cm)

print("\nClassification Report")
print(
    classification_report(
        y_validation,
        validation_predictions,
        digits=4,
        zero_division=0
    )
)

VALIDATION METRICS
--------------------------------------------------
ROC-AUC:   0.6746
PR-AUC:    0.4036
Precision: 0.4659
Recall:    0.1030
F1 Score:  0.1687

Confusion Matrix
[[1136   47]
 [ 357   41]]

Classification Report
              precision    recall  f1-score   support

           0     0.7609    0.9603    0.8490      1183
           1     0.4659    0.1030    0.1687       398

    accuracy                         0.7445      1581
   macro avg     0.6134    0.5316    0.5089      1581
weighted avg     0.6866    0.7445    0.6778      1581



Threshold Analysis

In [10]:
threshold_results = []

for threshold in np.arange(0.10, 0.61, 0.05):

    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predictions
    ).ravel()

    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "flagged_workflows": tp + fp
    })

threshold_df = pd.DataFrame(threshold_results)

print(
    threshold_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

 threshold  precision  recall     f1  true_positives  false_positives  false_negatives  true_negatives  flagged_workflows
    0.1000     0.2627  0.9598 0.4125             382             1072               16             111               1454
    0.1500     0.3024  0.8518 0.4463             339              782               59             401               1121
    0.2000     0.3462  0.7211 0.4678             287              542              111             641                829
    0.2500     0.3891  0.5729 0.4634             228              358              170             825                586
    0.3000     0.4014  0.4196 0.4103             167              249              231             934                416
    0.3500     0.4340  0.3141 0.3644             125              163              273            1020                288
    0.4000     0.4877  0.2487 0.3295              99              104              299            1079                203
    0.4500     0.5000  0

Calibration analysis

In [11]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(
    y_validation,
    validation_probabilities
)

prob_true, prob_pred = calibration_curve(
    y_validation,
    validation_probabilities,
    n_bins=10,
    strategy="quantile"
)

calibration_df = pd.DataFrame({
    "mean_predicted_probability": prob_pred,
    "observed_escalation_rate": prob_true
})

calibration_df["calibration_gap"] = (
    calibration_df["observed_escalation_rate"]
    - calibration_df["mean_predicted_probability"]
)

print(f"Brier Score: {brier:.4f}")
print()
print(
    calibration_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

Brier Score: 0.1756

 mean_predicted_probability  observed_escalation_rate  calibration_gap
                     0.0878                    0.1258           0.0380
                     0.1161                    0.1392           0.0232
                     0.1410                    0.1203          -0.0207
                     0.1648                    0.1772           0.0124
                     0.1924                    0.2215           0.0291
                     0.2224                    0.2089          -0.0136
                     0.2593                    0.3228           0.0635
                     0.3078                    0.3291           0.0213
                     0.3822                    0.3861           0.0039
                     0.5379                    0.4873          -0.0505


Compare against a naive baseline

In [12]:
from sklearn.metrics import brier_score_loss

# Probability learned only from TRAIN prevalence
train_prevalence = y_train.mean()

naive_validation_probabilities = np.full(
    len(y_validation),
    train_prevalence
)

naive_brier = brier_score_loss(
    y_validation,
    naive_validation_probabilities
)

model_brier = brier_score_loss(
    y_validation,
    validation_probabilities
)

brier_improvement = (
    (naive_brier - model_brier)
    / naive_brier
) * 100

print(f"Train prevalence:       {train_prevalence:.4f}")
print(f"Naive Brier Score:      {naive_brier:.4f}")
print(f"Logistic Brier Score:   {model_brier:.4f}")
print(f"Relative improvement:   {brier_improvement:.2f}%")

Train prevalence:       0.2417
Naive Brier Score:      0.1885
Logistic Brier Score:   0.1756
Relative improvement:   6.80%


Interpret Logistic Regression coefficients

In [13]:
# Extract fitted preprocessing and model components

fitted_preprocessor = logistic_pipeline.named_steps["preprocessor"]
fitted_model = logistic_pipeline.named_steps["model"]

# Get transformed feature names
feature_names = fitted_preprocessor.get_feature_names_out()

coefficients = fitted_model.coef_[0]

coefficient_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coefficient_df["odds_ratio"] = np.exp(
    coefficient_df["coefficient"]
)

coefficient_df["abs_coefficient"] = (
    coefficient_df["coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    "abs_coefficient",
    ascending=False
)

print("TOP LOGISTIC REGRESSION COEFFICIENTS")
print("-" * 90)

print(
    coefficient_df[
        ["feature", "coefficient", "odds_ratio"]
    ]
    .head(25)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

TOP LOGISTIC REGRESSION COEFFICIENTS
------------------------------------------------------------------------------------------
                                          feature  coefficient  odds_ratio
          binary__conflicting_project_information       0.8024      2.2309
                   binary__critical_field_missing       0.6749      1.9639
               binary__multiple_candidate_records       0.6440      1.9040
               categorical__public_private_public      -0.3517      0.7035
                   numeric__prior_escalation_rate       0.3498      1.4188
              categorical__public_private_private      -0.3200      0.7261
        numeric__payment_chain_completeness_score      -0.2697      0.7636
categorical__hiring_party_type_general_contractor      -0.2695      0.7638
                 numeric__deadline_days_remaining      -0.2386      0.7878
     categorical__hiring_party_type_subcontractor      -0.2316      0.7932
         categorical__customer_role_subcontract

Save the baseline artifact

In [14]:
import joblib
import json
from pathlib import Path

ARTIFACT_DIR = Path("artifacts/logistic-regression-v1")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Complete inference pipeline
joblib.dump(
    logistic_pipeline,
    ARTIFACT_DIR / "logistic_pipeline.joblib"
)

# 2. Feature contract
feature_contract = {
    "version": "1.0",
    "target": TARGET,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "binary_features": BINARY_FEATURES,
    "model_features": MODEL_FEATURES,
    "split_variable": "assessment_date"
}

with open(
    ARTIFACT_DIR / "feature_contract.json",
    "w"
) as f:
    json.dump(feature_contract, f, indent=2)

# 3. Validation metrics
validation_metrics = {
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "brier_score": float(model_brier),
    "naive_brier_score": float(naive_brier),
    "candidate_threshold": 0.20,
    "precision_at_020": 0.3462,
    "recall_at_020": 0.7211,
    "f1_at_020": 0.4678
}

with open(
    ARTIFACT_DIR / "validation_metrics.json",
    "w"
) as f:
    json.dump(validation_metrics, f, indent=2)

# 4. Coefficients
coefficient_df.to_csv(
    ARTIFACT_DIR / "coefficients.csv",
    index=False
)

print("Baseline artifacts created:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path)

Baseline artifacts created:
 - artifacts/logistic-regression-v1/coefficients.csv
 - artifacts/logistic-regression-v1/feature_contract.json
 - artifacts/logistic-regression-v1/logistic_pipeline.joblib
 - artifacts/logistic-regression-v1/validation_metrics.json


In [15]:
import joblib
import numpy as np

MODEL_PATH = (
    "artifacts/logistic-regression-v1/"
    "logistic_pipeline.joblib"
)

# Reload model from disk
reloaded_pipeline = joblib.load(MODEL_PATH)

# Small deterministic sample from validation data
smoke_test_X = X_validation.head(25).copy()

# Predictions from model currently in memory
original_probabilities = logistic_pipeline.predict_proba(
    smoke_test_X
)[:, 1]

# Predictions from serialized + reloaded model
reloaded_probabilities = reloaded_pipeline.predict_proba(
    smoke_test_X
)[:, 1]

# Maximum numerical difference
max_probability_difference = np.max(
    np.abs(
        original_probabilities
        - reloaded_probabilities
    )
)

print("MODEL ARTIFACT SMOKE TEST")
print("-" * 50)
print(
    f"Records tested: {len(smoke_test_X)}"
)
print(
    "Maximum probability difference:",
    f"{max_probability_difference:.12f}"
)

assert np.allclose(
    original_probabilities,
    reloaded_probabilities,
    rtol=1e-10,
    atol=1e-12
)

print("\nSerialization/reload test: PASS")

MODEL ARTIFACT SMOKE TEST
--------------------------------------------------
Records tested: 25
Maximum probability difference: 0.000000000000

Serialization/reload test: PASS


full validation reproducibility check

In [16]:
reloaded_validation_probabilities = (
    reloaded_pipeline.predict_proba(
        X_validation
    )[:, 1]
)

reloaded_roc_auc = roc_auc_score(
    y_validation,
    reloaded_validation_probabilities
)

reloaded_pr_auc = average_precision_score(
    y_validation,
    reloaded_validation_probabilities
)

reloaded_brier = brier_score_loss(
    y_validation,
    reloaded_validation_probabilities
)

print("RELOADED MODEL VALIDATION")
print("-" * 50)
print(f"ROC-AUC: {reloaded_roc_auc:.4f}")
print(f"PR-AUC:  {reloaded_pr_auc:.4f}")
print(f"Brier:   {reloaded_brier:.4f}")

assert np.allclose(
    validation_probabilities,
    reloaded_validation_probabilities
)

print("\nFull validation reproducibility: PASS")

RELOADED MODEL VALIDATION
--------------------------------------------------
ROC-AUC: 0.6746
PR-AUC:  0.4036
Brier:   0.1756

Full validation reproducibility: PASS


Versioned S3 artifact promotion.

In [18]:
import boto3
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

BUCKET = "construction-payment-risk-dev-gk53"
S3_PREFIX = "artifacts/payment-risk/logistic-regression/v1"

ARTIFACT_DIR = Path("artifacts/logistic-regression-v1")

s3 = boto3.client("s3")

artifact_files = [
    "logistic_pipeline.joblib",
    "feature_contract.json",
    "validation_metrics.json",
    "coefficients.csv",
]

uploaded_files = []

for filename in artifact_files:
    local_path = ARTIFACT_DIR / filename
    s3_key = f"{S3_PREFIX}/{filename}"

    s3.upload_file(
        str(local_path),
        BUCKET,
        s3_key
    )

    sha256 = hashlib.sha256(
        local_path.read_bytes()
    ).hexdigest()

    uploaded_files.append({
        "filename": filename,
        "s3_key": s3_key,
        "sha256": sha256,
        "size_bytes": local_path.stat().st_size
    })

print("Uploaded model artifacts successfully.")

Uploaded model artifacts successfully.


In [19]:
manifest = {
    "model_name": "construction-payment-risk-logistic-regression",
    "model_version": "v1",
    "model_type": "LogisticRegression",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "target": TARGET,

    "candidate_threshold": 0.20,

    "validation_metrics": {
        "roc_auc": float(reloaded_roc_auc),
        "pr_auc": float(reloaded_pr_auc),
        "brier_score": float(reloaded_brier),
        "precision_at_020": 0.3462,
        "recall_at_020": 0.7211,
        "f1_at_020": 0.4678
    },

    "training_split": {
        "rows": len(X_train),
        "positive_rate": float(y_train.mean()),
        "date_end_exclusive": "2026-03-01"
    },

    "validation_split": {
        "rows": len(X_validation),
        "positive_rate": float(y_validation.mean()),
        "date_start": "2026-03-01",
        "date_end_exclusive": "2026-06-01"
    },

    "test_status": "LOCKED_NOT_EVALUATED",

    "feature_contract_version": "1.0",

    "artifacts": uploaded_files
}

manifest_path = ARTIFACT_DIR / "manifest.json"

with open(
    manifest_path,
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

manifest_s3_key = (
    f"{S3_PREFIX}/manifest.json"
)

s3.upload_file(
    str(manifest_path),
    BUCKET,
    manifest_s3_key
)

print(
    f"Manifest uploaded to:\n"
    f"s3://{BUCKET}/{manifest_s3_key}"
)

Manifest uploaded to:
s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/logistic-regression/v1/manifest.json


In [20]:
response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix=S3_PREFIX
)

print("S3 MODEL ARTIFACT PACKAGE")
print("-" * 70)

for obj in response.get("Contents", []):
    print(
        obj["Key"],
        obj["Size"],
        "bytes"
    )

S3 MODEL ARTIFACT PACKAGE
----------------------------------------------------------------------
artifacts/payment-risk/logistic-regression/v1/coefficients.csv 3172 bytes
artifacts/payment-risk/logistic-regression/v1/feature_contract.json 1019 bytes
artifacts/payment-risk/logistic-regression/v1/logistic_pipeline.joblib 6906 bytes
artifacts/payment-risk/logistic-regression/v1/manifest.json 1867 bytes
artifacts/payment-risk/logistic-regression/v1/validation_metrics.json 259 bytes


In [21]:
import json
from pathlib import Path
from datetime import datetime, timezone

EXPERIMENT_DIR = Path("artifacts/experiments")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

experiment_record = {
    "experiment_id": "EXP-001",
    "experiment_name": "logistic-regression-baseline-v1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "model": {
        "type": "LogisticRegression",
        "version": "v1",
        "random_state": 42,
        "max_iter": 1000
    },

    "feature_contract": {
        "version": "1.0",
        "feature_count": len(MODEL_FEATURES),
        "categorical_count": len(CATEGORICAL_FEATURES),
        "numeric_count": len(NUMERIC_FEATURES),
        "binary_count": len(BINARY_FEATURES)
    },

    "dataset": {
        "total_records": 10000,
        "target": TARGET,

        "train_rows": len(X_train),
        "validation_rows": len(X_validation),
        "test_rows": len(X_test),

        "train_positive_rate": float(y_train.mean()),
        "validation_positive_rate": float(y_validation.mean()),
        "test_status": "LOCKED_NOT_EVALUATED"
    },

    "validation_metrics": {
        "roc_auc": float(reloaded_roc_auc),
        "pr_auc": float(reloaded_pr_auc),
        "brier_score": float(reloaded_brier),

        "candidate_threshold": 0.20,

        "precision": 0.3462,
        "recall": 0.7211,
        "f1": 0.4678
    },

    "artifact_location": (
        "s3://construction-payment-risk-dev-gk53/"
        "artifacts/payment-risk/logistic-regression/v1/"
    ),

    "status": "BASELINE_REGISTERED"
}

experiment_path = (
    EXPERIMENT_DIR /
    "EXP-001-logistic-regression-baseline-v1.json"
)

with open(experiment_path, "w") as f:
    json.dump(
        experiment_record,
        f,
        indent=2
    )

print("Experiment record created:")
print(experiment_path)

Experiment record created:
artifacts/experiments/EXP-001-logistic-regression-baseline-v1.json


Check XGBoost environment

In [22]:
import xgboost as xgb
import sklearn

print("XGBoost version:", xgb.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("XGBoost environment: READY")

XGBoost version: 2.1.4
Scikit-learn version: 1.7.2
XGBoost environment: READY


XGBoost Feature Contract v1.

In [23]:
XGB_CATEGORICAL_FEATURES = [
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type",
]

XGB_NUMERIC_FEATURES = [
    "payment_chain_completeness_score",
    "research_confidence_score",
    "deadline_days_remaining",
    "prior_escalation_rate",
    "prior_projects_with_hiring_party",
    "expected_party_count",

    "contractual_tier",
    "distance_from_owner",
    "prior_escalation_count",
    "prior_projects_with_gc",
    "prior_missing_info_rate",
    "prior_manual_review_rate",
    "known_party_count",
]

XGB_BINARY_FEATURES = [
    "critical_field_missing",
    "multiple_candidate_records",
    "conflicting_project_information",

    "owner_identified",
    "general_contractor_identified",
    "hiring_party_identified",
    "surety_applicable",
    "surety_identified",
    "lender_identified",
    "noc_found",
    "first_furnishing_date_known",
    "notice_required_flag",
]

XGB_MODEL_FEATURES = (
    XGB_CATEGORICAL_FEATURES
    + XGB_NUMERIC_FEATURES
    + XGB_BINARY_FEATURES
)

print("XGBoost Feature Contract v1")
print("-" * 50)
print(f"Categorical: {len(XGB_CATEGORICAL_FEATURES)}")
print(f"Numeric:     {len(XGB_NUMERIC_FEATURES)}")
print(f"Binary:      {len(XGB_BINARY_FEATURES)}")
print(f"Total:       {len(XGB_MODEL_FEATURES)}")

assert TARGET not in XGB_MODEL_FEATURES
assert "assessment_date" not in XGB_MODEL_FEATURES
assert "record_id" not in XGB_MODEL_FEATURES
assert len(XGB_MODEL_FEATURES) == len(set(XGB_MODEL_FEATURES))

print("\nXGBoost feature contract validation: PASS")

XGBoost Feature Contract v1
--------------------------------------------------
Categorical: 5
Numeric:     13
Binary:      12
Total:       30

XGBoost feature contract validation: PASS


XGBoost preprocessing

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Numeric preprocessing
xgb_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

# Categorical preprocessing
xgb_categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
    ]
)

# Binary preprocessing
xgb_binary_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)

xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            xgb_numeric_pipeline,
            XGB_NUMERIC_FEATURES
        ),
        (
            "categorical",
            xgb_categorical_pipeline,
            XGB_CATEGORICAL_FEATURES
        ),
        (
            "binary",
            xgb_binary_pipeline,
            XGB_BINARY_FEATURES
        ),
    ],
    remainder="drop"
)

print("XGBoost preprocessing pipeline created successfully.")
print(xgb_preprocessor)

XGBoost preprocessing pipeline created successfully.
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['payment_chain_completeness_score',
                                  'research_confidence_score',
                                  'deadline_days_remaining',
                                  'prior_escalation_rate',
                                  'prior_projects_with_hiring_party',
                                  'expected_party_count', 'contractual_tier',
                                  'distance_from_owner',
                                  'prior_escalation_count',
                                  'prior...
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['cri

Then create the train/validation matrices:

In [25]:
XGB_X_train = train_df[XGB_MODEL_FEATURES].copy()
XGB_y_train = train_df[TARGET].copy()

XGB_X_validation = validation_df[
    XGB_MODEL_FEATURES
].copy()

XGB_y_validation = validation_df[TARGET].copy()

print("XGBoost dataset shapes")
print("-" * 50)
print("Train:", XGB_X_train.shape)
print("Validation:", XGB_X_validation.shape)

assert XGB_X_train.shape == (6892, 30)
assert XGB_X_validation.shape == (1581, 30)

print("\nXGBoost modeling dataset validation: PASS")

XGBoost dataset shapes
--------------------------------------------------
Train: (6892, 30)
Validation: (1581, 30)

XGBoost modeling dataset validation: PASS


XGBoost preprocessing + dataset preparation is complete.

Now we build EXP-002 — XGBoost Baseline v1.

In [26]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("model", xgb_model),
    ]
)

xgb_pipeline.fit(
    XGB_X_train,
    XGB_y_train
)

print("XGBoost baseline training complete.")

XGBoost baseline training complete.


Then evaluate on VALIDATION only:

In [27]:
xgb_validation_probabilities = (
    xgb_pipeline.predict_proba(
        XGB_X_validation
    )[:, 1]
)

xgb_validation_predictions = (
    xgb_validation_probabilities >= 0.50
).astype(int)

xgb_roc_auc = roc_auc_score(
    XGB_y_validation,
    xgb_validation_probabilities
)

xgb_pr_auc = average_precision_score(
    XGB_y_validation,
    xgb_validation_probabilities
)

xgb_brier = brier_score_loss(
    XGB_y_validation,
    xgb_validation_probabilities
)

xgb_precision = precision_score(
    XGB_y_validation,
    xgb_validation_predictions,
    zero_division=0
)

xgb_recall = recall_score(
    XGB_y_validation,
    xgb_validation_predictions,
    zero_division=0
)

xgb_f1 = f1_score(
    XGB_y_validation,
    xgb_validation_predictions,
    zero_division=0
)

print("XGBOOST BASELINE — VALIDATION")
print("-" * 50)
print(f"ROC-AUC:   {xgb_roc_auc:.4f}")
print(f"PR-AUC:    {xgb_pr_auc:.4f}")
print(f"Brier:     {xgb_brier:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall:    {xgb_recall:.4f}")
print(f"F1 Score:  {xgb_f1:.4f}")

XGBOOST BASELINE — VALIDATION
--------------------------------------------------
ROC-AUC:   0.6610
PR-AUC:    0.3917
Brier:     0.1781
Precision: 0.5104
Recall:    0.1231
F1 Score:  0.1984


And immediately compare against Logistic Regression:

In [28]:
comparison_df = pd.DataFrame({
    "model": [
        "Logistic Regression v1",
        "XGBoost Baseline v1"
    ],
    "roc_auc": [
        reloaded_roc_auc,
        xgb_roc_auc
    ],
    "pr_auc": [
        reloaded_pr_auc,
        xgb_pr_auc
    ],
    "brier": [
        reloaded_brier,
        xgb_brier
    ]
})

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

                 model  roc_auc  pr_auc  brier
Logistic Regression v1   0.6746  0.4036 0.1756
   XGBoost Baseline v1   0.6610  0.3917 0.1781


Controlled XGBoost tuning

In [29]:
xgb_candidates = [
    {
        "name": "XGB-A-conservative",
        "n_estimators": 200,
        "max_depth": 2,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 2.0,
        "reg_alpha": 0.0,
    },
    {
        "name": "XGB-B-balanced",
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 3.0,
        "reg_alpha": 0.1,
    },
    {
        "name": "XGB-C-regularized",
        "n_estimators": 400,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 10,
        "subsample": 0.75,
        "colsample_bytree": 0.75,
        "reg_lambda": 5.0,
        "reg_alpha": 0.25,
    }
]

xgb_tuning_results = []

for params in xgb_candidates:

    model = XGBClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        min_child_weight=params["min_child_weight"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_lambda=params["reg_lambda"],
        reg_alpha=params["reg_alpha"],
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", xgb_preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(
        XGB_X_train,
        XGB_y_train
    )

    probabilities = pipeline.predict_proba(
        XGB_X_validation
    )[:, 1]

    xgb_tuning_results.append({
        "model": params["name"],
        "roc_auc": roc_auc_score(
            XGB_y_validation,
            probabilities
        ),
        "pr_auc": average_precision_score(
            XGB_y_validation,
            probabilities
        ),
        "brier": brier_score_loss(
            XGB_y_validation,
            probabilities
        )
    })

xgb_tuning_df = pd.DataFrame(
    xgb_tuning_results
)

# Add our champion for direct comparison
champion_row = pd.DataFrame([{
    "model": "Logistic Regression v1",
    "roc_auc": reloaded_roc_auc,
    "pr_auc": reloaded_pr_auc,
    "brier": reloaded_brier
}])

model_comparison = pd.concat(
    [champion_row, xgb_tuning_df],
    ignore_index=True
)

print(
    model_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

                 model  roc_auc  pr_auc  brier
Logistic Regression v1   0.6746  0.4036 0.1756
    XGB-A-conservative   0.6725  0.4027 0.1756
        XGB-B-balanced   0.6683  0.4003 0.1763
     XGB-C-regularized   0.6728  0.4040 0.1754


Register EXP-002

In [30]:
xgb_experiment_record = {
    "experiment_id": "EXP-002",
    "experiment_name": "xgboost-controlled-comparison-v1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "feature_contract": {
        "version": "xgb-v1",
        "feature_count": len(XGB_MODEL_FEATURES),
        "categorical_count": len(XGB_CATEGORICAL_FEATURES),
        "numeric_count": len(XGB_NUMERIC_FEATURES),
        "binary_count": len(XGB_BINARY_FEATURES)
    },

    "validation_results": {
        "xgb_baseline": {
            "roc_auc": 0.6610,
            "pr_auc": 0.3917,
            "brier": 0.1781
        },

        "xgb_a_conservative": {
            "roc_auc": 0.6725,
            "pr_auc": 0.4027,
            "brier": 0.1756
        },

        "xgb_b_balanced": {
            "roc_auc": 0.6683,
            "pr_auc": 0.4003,
            "brier": 0.1763
        },

        "xgb_c_regularized": {
            "roc_auc": 0.6728,
            "pr_auc": 0.4040,
            "brier": 0.1754
        }
    },

    "champion": "Logistic Regression v1",
    "challenger": "XGB-C-regularized",

    "decision": (
        "Retain Logistic Regression v1 as champion. "
        "XGB-C produced marginally better PR-AUC and Brier "
        "but lower ROC-AUC. Improvements were too small to "
        "justify replacing the simpler interpretable model."
    ),

    "test_status": "LOCKED_NOT_EVALUATED"
}

xgb_experiment_path = (
    EXPERIMENT_DIR /
    "EXP-002-xgboost-controlled-comparison-v1.json"
)

with open(xgb_experiment_path, "w") as f:
    json.dump(
        xgb_experiment_record,
        f,
        indent=2
    )

print("EXP-002 created:")
print(xgb_experiment_path)

EXP-002 created:
artifacts/experiments/EXP-002-xgboost-controlled-comparison-v1.json


Freeze the model-selection protocol

In [31]:
selection_protocol = {
    "protocol_version": "1.0",

    "champion": {
        "model": "Logistic Regression v1",
        "feature_contract": "1.0",
        "candidate_threshold": 0.20
    },

    "challenger": {
        "model": "XGB-C-regularized",
        "feature_contract": "xgb-v1"
    },

    "primary_metric": "roc_auc",

    "secondary_metrics": [
        "pr_auc",
        "brier_score"
    ],

    "operational_metrics": [
        "precision",
        "recall",
        "f1",
        "flagged_workflow_rate"
    ],

    "test_period": {
        "start": "2026-06-01",
        "end_exclusive": "2026-09-01",
        "expected_rows": 1479
    },

    "rules": [
        "No additional feature engineering before test evaluation.",
        "No additional hyperparameter tuning before test evaluation.",
        "Do not select a new threshold using test labels.",
        "Test metrics are final generalization estimates, not tuning inputs.",
        "Model complexity must be justified by material performance improvement."
    ],

    "status": "FROZEN_BEFORE_TEST"
}

protocol_path = (
    EXPERIMENT_DIR /
    "model-selection-protocol-v1.json"
)

with open(protocol_path, "w") as f:
    json.dump(selection_protocol, f, indent=2)

print("Model-selection protocol frozen.")
print(protocol_path)

Model-selection protocol frozen.
artifacts/experiments/model-selection-protocol-v1.json


Model-selection protocol is officially frozen.

In [32]:
# ============================================================
# FINAL LOCKED TEST EVALUATION
# Logistic Regression v1
# ============================================================

FINAL_THRESHOLD = 0.20

# Test probabilities
test_probabilities = reloaded_pipeline.predict_proba(
    X_test
)[:, 1]

# Predictions using the threshold selected on VALIDATION
test_predictions = (
    test_probabilities >= FINAL_THRESHOLD
).astype(int)

# Threshold-independent metrics
test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

test_pr_auc = average_precision_score(
    y_test,
    test_probabilities
)

test_brier = brier_score_loss(
    y_test,
    test_probabilities
)

# Operational metrics at frozen threshold
test_precision = precision_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_predictions,
    zero_division=0
)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions
).ravel()

flagged_workflows = tp + fp

flagged_rate = (
    flagged_workflows / len(y_test)
)

print("FINAL TEST — LOGISTIC REGRESSION v1")
print("=" * 60)

print("\nRanking / Probability Metrics")
print("-" * 60)

print(f"ROC-AUC:        {test_roc_auc:.4f}")
print(f"PR-AUC:         {test_pr_auc:.4f}")
print(f"Brier Score:    {test_brier:.4f}")

print("\nOperational Metrics @ Threshold 0.20")
print("-" * 60)

print(f"Precision:      {test_precision:.4f}")
print(f"Recall:         {test_recall:.4f}")
print(f"F1 Score:       {test_f1:.4f}")

print("\nConfusion Matrix")
print("-" * 60)

print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")

print("\nOperational Volume")
print("-" * 60)

print(f"Total workflows:   {len(y_test)}")
print(f"Actual positives:  {int(y_test.sum())}")
print(f"Flagged workflows: {flagged_workflows}")
print(f"Flagged rate:      {flagged_rate:.2%}")

FINAL TEST — LOGISTIC REGRESSION v1

Ranking / Probability Metrics
------------------------------------------------------------
ROC-AUC:        0.6787
PR-AUC:         0.3989
Brier Score:    0.1549

Operational Metrics @ Threshold 0.20
------------------------------------------------------------
Precision:      0.2959
Recall:         0.7038
F1 Score:       0.4166

Confusion Matrix
------------------------------------------------------------
True Negatives:  639
False Positives: 526
False Negatives: 93
True Positives:  221

Operational Volume
------------------------------------------------------------
Total workflows:   1479
Actual positives:  314
Flagged workflows: 747
Flagged rate:      50.51%


Record the immutable final test result

In [33]:
final_test_record = {
    "evaluation_id": "FINAL-TEST-001",
    "model": "Logistic Regression v1",
    "feature_contract_version": "1.0",

    "evaluation_type": "locked_out_of_time_test",

    "test_period": {
        "start": "2026-06-01",
        "end_exclusive": "2026-09-01",
        "rows": 1479,
        "positives": 314,
        "positive_rate": float(y_test.mean())
    },

    "threshold": {
        "value": 0.20,
        "selection_source": "validation_set",
        "retuned_on_test": False
    },

    "ranking_probability_metrics": {
        "roc_auc": float(test_roc_auc),
        "pr_auc": float(test_pr_auc),
        "brier_score": float(test_brier)
    },

    "operational_metrics": {
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
        "flagged_workflows": int(flagged_workflows),
        "flagged_rate": float(flagged_rate)
    },

    "confusion_matrix": {
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    },

    "decision": {
        "gate": "PASS",
        "status": "ACCEPTED_BASELINE",
        "reason": (
            "Logistic Regression v1 maintained stable ranking "
            "performance on the untouched out-of-time test period. "
            "The model remains the selected interpretable baseline."
        )
    },

    "data_disclaimer": (
        "Evaluation uses synthetic construction payment-protection "
        "workflow data and must not be represented as real SunRay "
        "customer or production performance."
    )
}

final_test_path = (
    EXPERIMENT_DIR /
    "FINAL-TEST-001-logistic-regression-v1.json"
)

with open(final_test_path, "w") as f:
    json.dump(
        final_test_record,
        f,
        indent=2
    )

print("Final test record created:")
print(final_test_path)

Final test record created:
artifacts/experiments/FINAL-TEST-001-logistic-regression-v1.json


Test Brier vs prevalence-only baseline

In [34]:
# ============================================================
# TEST PROBABILITY QUALITY
# Model vs prevalence-only baseline
# ============================================================

import numpy as np

# IMPORTANT:
# Use TRAIN prevalence, not TEST prevalence, for the naive
# probability predictor. This avoids learning from test labels.
train_prevalence = float(y_train.mean())

naive_test_probabilities = np.full(
    shape=len(y_test),
    fill_value=train_prevalence
)

naive_test_brier = brier_score_loss(
    y_test,
    naive_test_probabilities
)

model_test_brier = test_brier

absolute_improvement = (
    naive_test_brier - model_test_brier
)

relative_improvement = (
    absolute_improvement / naive_test_brier
)

print("TEST BRIER SCORE ANALYSIS")
print("=" * 55)

print(f"Train prevalence:        {train_prevalence:.4f}")
print(f"Test prevalence:         {y_test.mean():.4f}")

print("\nProbability Quality")
print("-" * 55)

print(f"Naive Brier:             {naive_test_brier:.4f}")
print(f"Logistic Brier:          {model_test_brier:.4f}")

print("\nImprovement")
print("-" * 55)

print(f"Absolute improvement:    {absolute_improvement:.4f}")
print(f"Relative improvement:    {relative_improvement:.2%}")

TEST BRIER SCORE ANALYSIS
Train prevalence:        0.2417
Test prevalence:         0.2123

Probability Quality
-------------------------------------------------------
Naive Brier:             0.1681
Logistic Brier:          0.1549

Improvement
-------------------------------------------------------
Absolute improvement:    0.0132
Relative improvement:    7.84%


Create Explainability Contract v1

Explainability Contract v1
Before SHAP or explanation code, we define what an explanation actually means.
For our system:
An explanation describes which model inputs contributed to a predicted operational escalation-risk score. It does not establish causation, determine legal rights, or provide legal advice.

We will support three explanation layers:
Layer	Question answered	Example
Global	What generally influences the model?	Conflicting project information is associated with higher predicted escalation risk
Local	Why did this workflow receive this score?	Critical information missing increased this workflow's predicted risk
Business-readable	How do we communicate it?	“Missing or conflicting project information contributed to this workflow being prioritized for review.”

In [35]:
from pathlib import Path
import json

EXPLAINABILITY_DIR = Path(
    "artifacts/explainability/logistic-regression-v1"
)

EXPLAINABILITY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

explainability_contract = {
    "contract_version": "1.0",

    "model": "Logistic Regression v1",

    "prediction_target": "escalation_required",

    "purpose": (
        "Explain model-generated operational escalation-risk "
        "scores for construction payment-protection workflows."
    ),

    "supported_explanation_levels": [
        "global",
        "local",
        "business_readable"
    ],

    "allowed_language": [
        "contributed to a higher predicted risk score",
        "contributed to a lower predicted risk score",
        "associated with higher model risk",
        "associated with lower model risk",
        "workflow may deserve additional operational review"
    ],

    "prohibited_language": [
        "caused the escalation",
        "will result in nonpayment",
        "will result in a lien",
        "legal rights are preserved",
        "legal rights are lost",
        "customer will not pay",
        "project will default"
    ],

    "interpretation_rules": {
        "association_not_causation": True,
        "not_legal_advice": True,
        "not_payment_guarantee": True,
        "human_review_required_for_high_impact_actions": True
    },

    "global_explanation_method": (
        "logistic_regression_coefficients"
    ),

    "local_explanation_method": (
        "feature_contribution_to_log_odds"
    ),

    "secondary_explanation_method": (
        "SHAP"
    ),

    "data_disclaimer": (
        "Model was developed and evaluated using synthetic "
        "construction payment-protection workflow data."
    )
}

contract_path = (
    EXPLAINABILITY_DIR /
    "explainability-contract-v1.json"
)

with open(contract_path, "w") as f:
    json.dump(
        explainability_contract,
        f,
        indent=2
    )

print("Explainability Contract v1 created.")
print(contract_path)

Explainability Contract v1 created.
artifacts/explainability/logistic-regression-v1/explainability-contract-v1.json


Global Logistic Regression Explainability
For Logistic Regression, the model learns:
\[
\text{log-odds} = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_nx_n
\]Each feature contributes approximately:
feature contribution = transformed feature value × coefficient
A positive coefficient pushes predicted escalation risk upward; a negative coefficient pushes it downward. But remember that our numeric features were standardized, categorical variables were one-hot encoded, and binary features were passed through. So we need to interpret each type correctly.
First, let's extract the transformed feature names and coefficients directly from our saved champion pipeline rather than manually copying our earlier coefficient table.

In [37]:
import numpy as np
import pandas as pd

# ============================================================
# STEP 16B — GLOBAL LOGISTIC REGRESSION EXPLAINABILITY
# ============================================================

preprocessor = reloaded_pipeline.named_steps["preprocessor"]
logistic_model = reloaded_pipeline.named_steps["model"]

# Get feature names after preprocessing
transformed_feature_names = (
    preprocessor.get_feature_names_out()
)

coefficients = logistic_model.coef_[0]

assert len(transformed_feature_names) == len(coefficients)

global_explanations = pd.DataFrame({
    "feature": transformed_feature_names,
    "coefficient": coefficients
})

# Direction of contribution
global_explanations["direction"] = np.where(
    global_explanations["coefficient"] > 0,
    "higher_predicted_risk",
    "lower_predicted_risk"
)

# Odds ratio
global_explanations["odds_ratio"] = np.exp(
    global_explanations["coefficient"]
)

# Magnitude — useful for ranking
global_explanations["absolute_coefficient"] = (
    global_explanations["coefficient"].abs()
)

global_explanations = (
    global_explanations
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)

print("GLOBAL MODEL EXPLANATIONS")
print("=" * 80)

print(
    global_explanations[
        [
            "feature",
            "coefficient",
            "odds_ratio",
            "direction"
        ]
    ]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

GLOBAL MODEL EXPLANATIONS
                                          feature  coefficient  odds_ratio             direction
          binary__conflicting_project_information       0.8024      2.2309 higher_predicted_risk
                   binary__critical_field_missing       0.6749      1.9639 higher_predicted_risk
               binary__multiple_candidate_records       0.6440      1.9040 higher_predicted_risk
               categorical__public_private_public      -0.3517      0.7035  lower_predicted_risk
                   numeric__prior_escalation_rate       0.3498      1.4188 higher_predicted_risk
              categorical__public_private_private      -0.3200      0.7261  lower_predicted_risk
        numeric__payment_chain_completeness_score      -0.2697      0.7636  lower_predicted_risk
categorical__hiring_party_type_general_contractor      -0.2695      0.7638  lower_predicted_risk
                 numeric__deadline_days_remaining      -0.2386      0.7878  lower_predicted_risk
    

Business Explanation Mapping

In [38]:
# ============================================================
# STEP 16C — BUSINESS EXPLANATION MAPPING
# ============================================================

BUSINESS_EXPLANATION_MAP = {

    "conflicting_project_information": {
        "label": "Conflicting project information",
        "higher": (
            "Conflicting project information contributed "
            "to a higher predicted operational risk score."
        ),
        "lower": (
            "No conflicting project information contributed "
            "to a lower predicted operational risk score."
        )
    },

    "critical_field_missing": {
        "label": "Missing critical project information",
        "higher": (
            "Missing critical project information contributed "
            "to a higher predicted operational risk score."
        ),
        "lower": (
            "Availability of critical project information contributed "
            "to a lower predicted operational risk score."
        )
    },

    "multiple_candidate_records": {
        "label": "Multiple candidate project records",
        "higher": (
            "Multiple candidate project records contributed "
            "to a higher predicted operational risk score."
        ),
        "lower": (
            "A more clearly identified project record contributed "
            "to a lower predicted operational risk score."
        )
    },

    "prior_escalation_rate": {
        "label": "Historical escalation pattern",
        "higher": (
            "Historical workflow escalation patterns contributed "
            "to a higher predicted operational risk score."
        ),
        "lower": (
            "Historical workflow escalation patterns contributed "
            "to a lower predicted operational risk score."
        )
    },

    "payment_chain_completeness_score": {
        "label": "Payment-chain information completeness",
        "higher": (
            "Lower payment-chain information completeness contributed "
            "to a higher predicted operational risk score."
        ),
        "lower": (
            "More complete payment-chain information contributed "
            "to a lower predicted operational risk score."
        )
    },

    "deadline_days_remaining": {
        "label": "Deadline proximity",
        "higher": (
            "Greater deadline urgency contributed "
            "to a higher predicted operational risk score."
        ),
        "lower": (
            "More time remaining before the relevant workflow deadline "
            "contributed to a lower predicted operational risk score."
        )
    },

    "research_confidence_score": {
        "label": "Project research confidence",
        "higher": (
            "Lower project research confidence contributed "
            "to a higher predicted operational risk score."
        ),
        "lower": (
            "Higher project research confidence contributed "
            "to a lower predicted operational risk score."
        )
    }
}

print("Business explanation mappings created:")
print(len(BUSINESS_EXPLANATION_MAP))

for feature, config in BUSINESS_EXPLANATION_MAP.items():
    print(f"\n{feature}")
    print(f"  Label: {config['label']}")

Business explanation mappings created:
7

conflicting_project_information
  Label: Conflicting project information

critical_field_missing
  Label: Missing critical project information

multiple_candidate_records
  Label: Multiple candidate project records

prior_escalation_rate
  Label: Historical escalation pattern

payment_chain_completeness_score
  Label: Payment-chain information completeness

deadline_days_remaining
  Label: Deadline proximity

research_confidence_score
  Label: Project research confidence


Local Prediction Explanation Engine

In [39]:
# ============================================================
# STEP 16D — LOCAL LOGISTIC REGRESSION EXPLANATION
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Select one TEST workflow
#    Explanation only — no tuning or model changes.
# ------------------------------------------------------------

workflow_position = 0

workflow = X_test.iloc[[workflow_position]]

actual_target = int(
    y_test.iloc[workflow_position]
)

# ------------------------------------------------------------
# 2. Get model probability
# ------------------------------------------------------------

model_probability = float(
    reloaded_pipeline.predict_proba(workflow)[0, 1]
)

# ------------------------------------------------------------
# 3. Transform workflow using fitted preprocessor
# ------------------------------------------------------------

transformed_workflow = (
    preprocessor.transform(workflow)
)

# Handle sparse/dense output safely
if hasattr(transformed_workflow, "toarray"):
    transformed_values = (
        transformed_workflow.toarray()[0]
    )
else:
    transformed_values = np.asarray(
        transformed_workflow
    )[0]

# ------------------------------------------------------------
# 4. Extract coefficients and intercept
# ------------------------------------------------------------

coefficients = logistic_model.coef_[0]

intercept = float(
    logistic_model.intercept_[0]
)

# ------------------------------------------------------------
# 5. Calculate individual contributions
# ------------------------------------------------------------

feature_contributions = (
    transformed_values * coefficients
)

local_explanation = pd.DataFrame({
    "feature": transformed_feature_names,
    "transformed_value": transformed_values,
    "coefficient": coefficients,
    "contribution_log_odds": feature_contributions
})

local_explanation["absolute_contribution"] = (
    local_explanation[
        "contribution_log_odds"
    ].abs()
)

local_explanation = (
    local_explanation
    .sort_values(
        "absolute_contribution",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Reconstruct model prediction
# ------------------------------------------------------------

reconstructed_log_odds = (
    intercept +
    feature_contributions.sum()
)

reconstructed_probability = (
    1 /
    (
        1 +
        np.exp(-reconstructed_log_odds)
    )
)

probability_difference = abs(
    model_probability -
    reconstructed_probability
)

# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

print("LOCAL PREDICTION EXPLANATION")
print("=" * 70)

print(f"Workflow position:        {workflow_position}")
print(f"Actual synthetic target:  {actual_target}")

print(
    f"Model probability:        "
    f"{model_probability:.6f}"
)

print(
    f"Reconstructed probability:"
    f" {reconstructed_probability:.6f}"
)

print(
    f"Probability difference:   "
    f"{probability_difference:.12f}"
)

print(f"Intercept:                {intercept:.6f}")

print("\nTOP LOCAL CONTRIBUTIONS")
print("-" * 70)

print(
    local_explanation[
        [
            "feature",
            "transformed_value",
            "coefficient",
            "contribution_log_odds"
        ]
    ]
    .head(15)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

LOCAL PREDICTION EXPLANATION
Workflow position:        0
Actual synthetic target:  1
Model probability:        0.121493
Reconstructed probability: 0.121493
Probability difference:   0.000000000000
Intercept:                -0.701368

TOP LOCAL CONTRIBUTIONS
----------------------------------------------------------------------
                                     feature  transformed_value  coefficient  contribution_log_odds
         categorical__public_private_private             1.0000      -0.3200                -0.3200
              numeric__prior_escalation_rate            -0.6815       0.3498                -0.2384
categorical__hiring_party_type_subcontractor             1.0000      -0.2316                -0.2316
                       categorical__state_AZ             1.0000      -0.1714                -0.1714
            numeric__deadline_days_remaining             0.6498      -0.2386                -0.1550
   numeric__payment_chain_completeness_score            -0.5608      -0

Safe Business-Readable Local Explanation

In [40]:
# ============================================================
# STEP 16E — BUSINESS-READABLE LOCAL EXPLANATION
# ============================================================

def extract_base_feature(transformed_feature_name):
    """
    Convert transformed sklearn names such as:
        numeric__deadline_days_remaining
        binary__critical_field_missing

    into their original feature names.
    """

    for prefix in [
        "numeric__",
        "binary__"
    ]:
        if transformed_feature_name.startswith(prefix):
            return transformed_feature_name.replace(prefix, "", 1)

    return None


business_local_explanations = []

for _, row in local_explanation.iterrows():

    base_feature = extract_base_feature(
        row["feature"]
    )

    # Only expose approved explanation features
    if base_feature not in BUSINESS_EXPLANATION_MAP:
        continue

    contribution = float(
        row["contribution_log_odds"]
    )

    # Ignore effectively zero contributions
    if abs(contribution) < 1e-8:
        continue

    config = BUSINESS_EXPLANATION_MAP[
        base_feature
    ]

    contribution_direction = (
        "higher"
        if contribution > 0
        else "lower"
    )

    business_local_explanations.append({
        "feature": base_feature,
        "label": config["label"],
        "direction": contribution_direction,
        "contribution_log_odds": contribution,
        "absolute_contribution": abs(contribution),
        "explanation": config[
            contribution_direction
        ]
    })


business_local_explanations = sorted(
    business_local_explanations,
    key=lambda x: x["absolute_contribution"],
    reverse=True
)


print("BUSINESS-READABLE LOCAL EXPLANATION")
print("=" * 75)

print(
    f"Predicted operational risk: "
    f"{model_probability:.2%}"
)

print(
    f"Frozen decision threshold:  "
    f"{FINAL_THRESHOLD:.0%}"
)

decision = (
    "REVIEW"
    if model_probability >= FINAL_THRESHOLD
    else "NO REVIEW"
)

print(f"Model decision:             {decision}")

print("\nApproved explanation factors")
print("-" * 75)

for item in business_local_explanations:

    arrow = (
        "↑"
        if item["direction"] == "higher"
        else "↓"
    )

    print(
        f"{arrow} {item['label']}"
    )

    print(
        f"  Contribution: "
        f"{item['contribution_log_odds']:+.4f}"
    )

    print(
        f"  {item['explanation']}"
    )

BUSINESS-READABLE LOCAL EXPLANATION
Predicted operational risk: 12.15%
Frozen decision threshold:  20%
Model decision:             NO REVIEW

Approved explanation factors
---------------------------------------------------------------------------
↓ Historical escalation pattern
  Contribution: -0.2384
  Historical workflow escalation patterns contributed to a lower predicted operational risk score.
↓ Deadline proximity
  Contribution: -0.1550
  More time remaining before the relevant workflow deadline contributed to a lower predicted operational risk score.
↑ Payment-chain information completeness
  Contribution: +0.1512
  Lower payment-chain information completeness contributed to a higher predicted operational risk score.
↓ Project research confidence
  Contribution: -0.0794
  Higher project research confidence contributed to a lower predicted operational risk score.


Make the explanation semantics production-safe

In [41]:
BUSINESS_EXPLANATION_MAP["prior_escalation_rate"].update({
    "higher": (
        "The workflow's historical escalation pattern, relative "
        "to the model's training reference, contributed to a "
        "higher predicted operational risk score."
    ),
    "lower": (
        "The workflow's historical escalation pattern, relative "
        "to the model's training reference, contributed to a "
        "lower predicted operational risk score."
    )
})

BUSINESS_EXPLANATION_MAP["payment_chain_completeness_score"].update({
    "higher": (
        "Lower payment-chain information completeness relative "
        "to the model's training reference contributed to a "
        "higher predicted operational risk score."
    ),
    "lower": (
        "Higher payment-chain information completeness relative "
        "to the model's training reference contributed to a "
        "lower predicted operational risk score."
    )
})

BUSINESS_EXPLANATION_MAP["deadline_days_remaining"].update({
    "higher": (
        "Greater deadline urgency relative to the model's "
        "training reference contributed to a higher predicted "
        "operational risk score."
    ),
    "lower": (
        "More time remaining relative to the model's training "
        "reference contributed to a lower predicted operational "
        "risk score."
    )
})

BUSINESS_EXPLANATION_MAP["research_confidence_score"].update({
    "higher": (
        "Lower project research confidence relative to the "
        "model's training reference contributed to a higher "
        "predicted operational risk score."
    ),
    "lower": (
        "Higher project research confidence relative to the "
        "model's training reference contributed to a lower "
        "predicted operational risk score."
    )
})

print("Numeric explanation semantics updated.")

Numeric explanation semantics updated.


In [42]:
decision = (
    "FLAGGED_BY_MODEL"
    if model_probability >= FINAL_THRESHOLD
    else "NOT_FLAGGED_BY_MODEL"
)

print(f"Model decision: {decision}")

Model decision: NOT_FLAGGED_BY_MODEL


reusable local explanation function.

In [43]:
def explain_workflow(
    workflow_row,
    pipeline,
    business_map,
    threshold=0.20,
    top_n=5
):
    """
    Generate a structured local explanation for one workflow.

    Parameters
    ----------
    workflow_row : pandas.DataFrame
        Exactly one workflow row with model input features.

    pipeline : sklearn Pipeline
        Trained Logistic Regression pipeline.

    business_map : dict
        Approved business explanation mappings.

    threshold : float
        Frozen operational decision threshold.

    top_n : int
        Maximum number of approved business factors to return.

    Returns
    -------
    dict
        Structured prediction and explanation payload.
    """

    if len(workflow_row) != 1:
        raise ValueError(
            "workflow_row must contain exactly one row."
        )

    preprocessor = pipeline.named_steps["preprocessor"]
    model = pipeline.named_steps["model"]

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    probability = float(
        pipeline.predict_proba(workflow_row)[0, 1]
    )

    decision = (
        "FLAGGED_BY_MODEL"
        if probability >= threshold
        else "NOT_FLAGGED_BY_MODEL"
    )

    # --------------------------------------------------------
    # Transform features
    # --------------------------------------------------------

    transformed = preprocessor.transform(
        workflow_row
    )

    if hasattr(transformed, "toarray"):
        transformed_values = transformed.toarray()[0]
    else:
        transformed_values = np.asarray(transformed)[0]

    feature_names = (
        preprocessor.get_feature_names_out()
    )

    coefficients = model.coef_[0]
    intercept = float(model.intercept_[0])

    contributions = (
        transformed_values * coefficients
    )

    # --------------------------------------------------------
    # Mathematical verification
    # --------------------------------------------------------

    reconstructed_log_odds = (
        intercept + contributions.sum()
    )

    reconstructed_probability = float(
        1 / (
            1 +
            np.exp(-reconstructed_log_odds)
        )
    )

    reconstruction_error = abs(
        probability -
        reconstructed_probability
    )

    # --------------------------------------------------------
    # Raw contribution table
    # --------------------------------------------------------

    raw_factors = []

    approved_factors = []

    for feature_name, value, coefficient, contribution in zip(
        feature_names,
        transformed_values,
        coefficients,
        contributions
    ):

        contribution = float(contribution)

        raw_factors.append({
            "feature": feature_name,
            "transformed_value": float(value),
            "coefficient": float(coefficient),
            "contribution_log_odds": contribution,
            "absolute_contribution": abs(contribution)
        })

        base_feature = None

        for prefix in [
            "numeric__",
            "binary__"
        ]:
            if feature_name.startswith(prefix):
                base_feature = feature_name.replace(
                    prefix,
                    "",
                    1
                )
                break

        if base_feature not in business_map:
            continue

        if abs(contribution) < 1e-8:
            continue

        direction = (
            "higher"
            if contribution > 0
            else "lower"
        )

        config = business_map[
            base_feature
        ]

        approved_factors.append({
            "feature": base_feature,
            "label": config["label"],
            "direction": direction,
            "contribution_log_odds": contribution,
            "absolute_contribution": abs(contribution),
            "explanation": config[direction]
        })

    raw_factors = sorted(
        raw_factors,
        key=lambda x: x[
            "absolute_contribution"
        ],
        reverse=True
    )

    approved_factors = sorted(
        approved_factors,
        key=lambda x: x[
            "absolute_contribution"
        ],
        reverse=True
    )[:top_n]

    # --------------------------------------------------------
    # API-ready payload
    # --------------------------------------------------------

    return {
        "model": "Logistic Regression v1",
        "predicted_operational_risk": probability,
        "threshold": threshold,
        "model_decision": decision,

        "explanation": {
            "top_factors": approved_factors,
            "method": (
                "exact_logistic_regression_"
                "log_odds_decomposition"
            ),
            "reconstruction_error": (
                reconstruction_error
            )
        },

        "raw_model_details": {
            "intercept": intercept,
            "reconstructed_probability": (
                reconstructed_probability
            ),
            "top_raw_factors": raw_factors[:10]
        },

        "disclaimers": [
            (
                "Explanation describes model "
                "associations, not causation."
            ),
            (
                "Prediction does not determine "
                "legal rights or provide legal advice."
            ),
            (
                "Model was evaluated using synthetic "
                "construction payment-protection data."
            )
        ]
    }

In [44]:
explanation_result = explain_workflow(
    workflow_row=X_test.iloc[[0]],
    pipeline=reloaded_pipeline,
    business_map=BUSINESS_EXPLANATION_MAP,
    threshold=FINAL_THRESHOLD,
    top_n=5
)

print(
    json.dumps(
        explanation_result,
        indent=2
    )
)

{
  "model": "Logistic Regression v1",
  "predicted_operational_risk": 0.12149310253594506,
  "threshold": 0.2,
  "model_decision": "NOT_FLAGGED_BY_MODEL",
  "explanation": {
    "top_factors": [
      {
        "feature": "prior_escalation_rate",
        "label": "Historical escalation pattern",
        "direction": "lower",
        "contribution_log_odds": -0.23840813573923836,
        "absolute_contribution": 0.23840813573923836,
        "explanation": "The workflow's historical escalation pattern, relative to the model's training reference, contributed to a lower predicted operational risk score."
      },
      {
        "feature": "deadline_days_remaining",
        "label": "Deadline proximity",
        "direction": "lower",
        "contribution_log_odds": -0.1550229000078173,
        "absolute_contribution": 0.1550229000078173,
        "explanation": "More time remaining relative to the model's training reference contributed to a lower predicted operational risk score."
      }

Explain a high-risk flagged workflow

In [45]:
# ============================================================
# STEP 16H — HIGH-RISK WORKFLOW EXPLANATION
# ============================================================

highest_risk_position = int(
    np.argmax(test_probabilities)
)

highest_risk_probability = float(
    test_probabilities[highest_risk_position]
)

high_risk_workflow = X_test.iloc[
    [highest_risk_position]
]

high_risk_actual_target = int(
    y_test.iloc[highest_risk_position]
)

high_risk_explanation = explain_workflow(
    workflow_row=high_risk_workflow,
    pipeline=reloaded_pipeline,
    business_map=BUSINESS_EXPLANATION_MAP,
    threshold=FINAL_THRESHOLD,
    top_n=5
)

print("HIGH-RISK WORKFLOW")
print("=" * 70)

print(
    f"Workflow position:       "
    f"{highest_risk_position}"
)

print(
    f"Predicted risk:          "
    f"{highest_risk_probability:.2%}"
)

print(
    f"Model decision:          "
    f"{high_risk_explanation['model_decision']}"
)

print(
    f"Actual synthetic target: "
    f"{high_risk_actual_target}"
)

print(
    f"Reconstruction error:    "
    f"{high_risk_explanation['explanation']['reconstruction_error']:.12f}"
)

print("\nAPPROVED EXPLANATION FACTORS")
print("-" * 70)

for factor in high_risk_explanation[
    "explanation"
]["top_factors"]:

    arrow = (
        "↑"
        if factor["direction"] == "higher"
        else "↓"
    )

    print(
        f"{arrow} {factor['label']}"
    )

    print(
        f"  Contribution: "
        f"{factor['contribution_log_odds']:+.4f}"
    )

    print(
        f"  {factor['explanation']}"
    )

HIGH-RISK WORKFLOW
Workflow position:       916
Predicted risk:          82.37%
Model decision:          FLAGGED_BY_MODEL
Actual synthetic target: 0
Reconstruction error:    0.000000000000

APPROVED EXPLANATION FACTORS
----------------------------------------------------------------------
↑ Conflicting project information
  Contribution: +0.8024
  Conflicting project information contributed to a higher predicted operational risk score.
↑ Missing critical project information
  Contribution: +0.6749
  Missing critical project information contributed to a higher predicted operational risk score.
↑ Multiple candidate project records
  Contribution: +0.6440
  Multiple candidate project records contributed to a higher predicted operational risk score.
↑ Historical escalation pattern
  Contribution: +0.4287
  The workflow's historical escalation pattern, relative to the model's training reference, contributed to a higher predicted operational risk score.
↑ Project research confidence
  Contri

Package Explainability v1

In [46]:
# ============================================================
# STEP 16I — PACKAGE EXPLAINABILITY v1
# ============================================================

import json
from pathlib import Path

EXPLAINABILITY_DIR = Path(
    "artifacts/explainability/logistic-regression-v1"
)

EXPLAINABILITY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Save global explanations
# ------------------------------------------------------------

global_explanation_path = (
    EXPLAINABILITY_DIR /
    "global_explanations.csv"
)

global_explanations.to_csv(
    global_explanation_path,
    index=False
)

# ------------------------------------------------------------
# 2. Save business explanation policy
# ------------------------------------------------------------

business_mapping_path = (
    EXPLAINABILITY_DIR /
    "business-explanation-map-v1.json"
)

with open(business_mapping_path, "w") as f:
    json.dump(
        BUSINESS_EXPLANATION_MAP,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 3. Save low-risk false-negative example
# ------------------------------------------------------------

low_risk_example = explain_workflow(
    workflow_row=X_test.iloc[[0]],
    pipeline=reloaded_pipeline,
    business_map=BUSINESS_EXPLANATION_MAP,
    threshold=FINAL_THRESHOLD,
    top_n=5
)

low_risk_example["evaluation_context"] = {
    "workflow_position": 0,
    "actual_synthetic_target": int(
        y_test.iloc[0]
    ),
    "case_type": "false_negative"
}

low_risk_path = (
    EXPLAINABILITY_DIR /
    "example-false-negative.json"
)

with open(low_risk_path, "w") as f:
    json.dump(
        low_risk_example,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 4. Save high-risk false-positive example
# ------------------------------------------------------------

high_risk_explanation[
    "evaluation_context"
] = {
    "workflow_position": int(
        highest_risk_position
    ),
    "actual_synthetic_target": int(
        high_risk_actual_target
    ),
    "case_type": "false_positive"
}

high_risk_path = (
    EXPLAINABILITY_DIR /
    "example-false-positive.json"
)

with open(high_risk_path, "w") as f:
    json.dump(
        high_risk_explanation,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 5. Manifest
# ------------------------------------------------------------

explainability_manifest = {
    "artifact": "Explainability v1",
    "model": "Logistic Regression v1",

    "methods": {
        "global": "logistic_regression_coefficients",
        "local": (
            "exact_log_odds_contribution_decomposition"
        )
    },

    "business_explanation_allowlist_count": len(
        BUSINESS_EXPLANATION_MAP
    ),

    "decision_threshold": FINAL_THRESHOLD,

    "local_validation_cases": [
        {
            "type": "false_negative",
            "workflow_position": 0,
            "predicted_risk": float(
                low_risk_example[
                    "predicted_operational_risk"
                ]
            )
        },
        {
            "type": "false_positive",
            "workflow_position": int(
                highest_risk_position
            ),
            "predicted_risk": float(
                high_risk_explanation[
                    "predicted_operational_risk"
                ]
            )
        }
    ],

    "safety": {
        "association_not_causation": True,
        "legal_advice": False,
        "customer_facing_raw_features": False,
        "human_review_required_for_high_impact_actions": True
    },

    "data_disclaimer": (
        "Model and explanations use synthetic construction "
        "payment-protection workflow data."
    )
}

manifest_path = (
    EXPLAINABILITY_DIR /
    "manifest.json"
)

with open(manifest_path, "w") as f:
    json.dump(
        explainability_manifest,
        f,
        indent=2
    )

print("Explainability v1 packaged.")
print(EXPLAINABILITY_DIR)

print("\nFiles:")
for path in sorted(EXPLAINABILITY_DIR.iterdir()):
    print("-", path.name)

Explainability v1 packaged.
artifacts/explainability/logistic-regression-v1

Files:
- business-explanation-map-v1.json
- example-false-negative.json
- example-false-positive.json
- explainability-contract-v1.json
- global_explanations.csv
- manifest.json


Upload Explainability v1 to S3

In [47]:
import boto3
from pathlib import Path

# ============================================================
# STEP 16J — PROMOTE EXPLAINABILITY v1 TO S3
# ============================================================

BUCKET = "construction-payment-risk-dev-gk53"

S3_PREFIX = (
    "artifacts/payment-risk/"
    "explainability/logistic-regression/v1"
)

LOCAL_DIR = Path(
    "artifacts/explainability/logistic-regression-v1"
)

s3 = boto3.client(
    "s3",
    region_name="ap-southeast-2"
)

uploaded_files = []

for local_path in sorted(LOCAL_DIR.iterdir()):

    if not local_path.is_file():
        continue

    s3_key = (
        f"{S3_PREFIX}/{local_path.name}"
    )

    s3.upload_file(
        str(local_path),
        BUCKET,
        s3_key
    )

    uploaded_files.append(
        {
            "file": local_path.name,
            "s3_uri": (
                f"s3://{BUCKET}/{s3_key}"
            )
        }
    )

print("Explainability v1 uploaded to S3.")
print("=" * 70)

for item in uploaded_files:
    print(item["s3_uri"])

Explainability v1 uploaded to S3.
s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/explainability/logistic-regression/v1/business-explanation-map-v1.json
s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/explainability/logistic-regression/v1/example-false-negative.json
s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/explainability/logistic-regression/v1/example-false-positive.json
s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/explainability/logistic-regression/v1/explainability-contract-v1.json
s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/explainability/logistic-regression/v1/global_explanations.csv
s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/explainability/logistic-regression/v1/manifest.json


In [48]:
response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix=f"{S3_PREFIX}/"
)

objects = response.get("Contents", [])

print("S3 VERIFICATION")
print("=" * 70)

for obj in objects:
    print(
        f"{obj['Key']} "
        f"({obj['Size']} bytes)"
    )

print()
print(f"Verified object count: {len(objects)}")

assert len(objects) == 6, (
    "Expected exactly 6 Explainability v1 artifacts."
)

print("Explainability v1 S3 verification: PASS")

S3 VERIFICATION
artifacts/payment-risk/explainability/logistic-regression/v1/business-explanation-map-v1.json (2502 bytes)
artifacts/payment-risk/explainability/logistic-regression/v1/example-false-negative.json (5172 bytes)
artifacts/payment-risk/explainability/logistic-regression/v1/example-false-positive.json (5428 bytes)
artifacts/payment-risk/explainability/logistic-regression/v1/explainability-contract-v1.json (1331 bytes)
artifacts/payment-risk/explainability/logistic-regression/v1/global_explanations.csv (3887 bytes)
artifacts/payment-risk/explainability/logistic-regression/v1/manifest.json (842 bytes)

Verified object count: 6
Explainability v1 S3 verification: PASS


You verified all six Explainability v1 artifacts in S3, including object sizes, so this phase is now genuinely closed—not just “uploaded.”
At this point we have a strong chain of evidence:
- Champion model selected and frozen
- Untouched test evaluation completed
- Final test record created
- Probability-quality benchmark passed
- Global explainability completed
- Local exact log-odds explanations completed
- Safe business explanation policy created
- False-negative and false-positive cases preserved
- Explainability package versioned locally
- Explainability package promoted and verified in S3
So Step 16 — Explainable Payment Risk Intelligence v1 = COMPLETE.
The next best step is Step 17 — ML Experiment Registry & Reproducibility cleanup. We should consolidate EXP-001, EXP-002, the selection protocol, final test record, model package, and explainability package into one clean experiment registry structure before building the prediction API.

ML Experiment Registry & Reproducibility.

Create the master experiment registry

In [49]:
from pathlib import Path
import json
from datetime import datetime, timezone

REGISTRY_DIR = Path("artifacts/registry")
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

registry = {
    "registry_version": "1.0",
    "project": "Construction Payment Risk Intelligence",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "dataset": {
        "canonical_batch":
            "batch-20260911T023612Z-6638eb0b",
        "records": 10000,
        "generator_version": "1.0.2",
        "synthetic_data": True
    },

    "experiments": [
        {
            "experiment_id": "EXP-001",
            "model": "Logistic Regression v1",
            "status": "CHAMPION"
        },
        {
            "experiment_id": "EXP-002",
            "model": "XGBoost",
            "status": "CHALLENGER_EVALUATED"
        }
    ],

    "champion": {
        "model": "Logistic Regression v1",
        "feature_contract_version": "1.0",
        "decision_threshold": 0.20,

        "final_test_metrics": {
            "roc_auc": 0.6787,
            "pr_auc": 0.3989,
            "brier": 0.1549,
            "precision": 0.2959,
            "recall": 0.7038,
            "f1": 0.4166
        },

        "test_status": "CONSUMED_FINAL_EVALUATION",

        "selection_reason": (
            "Stable out-of-time generalization with "
            "comparable or better performance than the "
            "more complex XGBoost challenger, while "
            "retaining greater interpretability."
        )
    },

    "artifacts": {
        "model": (
            "s3://construction-payment-risk-dev-gk53/"
            "artifacts/payment-risk/"
            "logistic-regression/v1/"
        ),

        "explainability": (
            "s3://construction-payment-risk-dev-gk53/"
            "artifacts/payment-risk/explainability/"
            "logistic-regression/v1/"
        )
    },

    "governance": {
        "selection_protocol_frozen": True,
        "test_set_reusable_for_tuning": False,
        "human_review_required": True,
        "legal_decision_system": False
    },

    "data_disclaimer": (
        "Model development and evaluation use synthetic "
        "construction payment-protection workflow data."
    )
}

registry_path = (
    REGISTRY_DIR /
    "model-registry-v1.json"
)

with open(registry_path, "w") as f:
    json.dump(
        registry,
        f,
        indent=2
    )

print("Master model registry created.")
print(registry_path)

Master model registry created.
artifacts/registry/model-registry-v1.json


Validate the registry against the actual local artifacts.

In [50]:
# ============================================================
# STEP 17B — REGISTRY VALIDATION
# ============================================================

from pathlib import Path
import json

REGISTRY_PATH = Path(
    "artifacts/registry/model-registry-v1.json"
)

with open(REGISTRY_PATH, "r") as f:
    registry = json.load(f)

required_local_artifacts = {
    "registry": REGISTRY_PATH,

    "model_manifest": Path(
        "artifacts/logistic-regression-v1/manifest.json"
    ),

    "feature_contract": Path(
        "artifacts/logistic-regression-v1/feature_contract.json"
    ),

    "validation_metrics": Path(
        "artifacts/logistic-regression-v1/validation_metrics.json"
    ),

    "experiment_001": Path(
        "artifacts/experiments/"
        "EXP-001-logistic-regression-baseline-v1.json"
    ),

    "experiment_002": Path(
        "artifacts/experiments/"
        "EXP-002-xgboost-controlled-comparison-v1.json"
    ),

    "selection_protocol": Path(
        "artifacts/experiments/"
        "model-selection-protocol-v1.json"
    ),

    "final_test": Path(
        "artifacts/experiments/"
        "FINAL-TEST-001-logistic-regression-v1.json"
    ),

    "explainability_manifest": Path(
        "artifacts/explainability/"
        "logistic-regression-v1/manifest.json"
    )
}

print("REGISTRY VALIDATION")
print("=" * 70)

all_exist = True

for name, path in required_local_artifacts.items():

    exists = path.exists()

    status = "PASS" if exists else "MISSING"

    print(
        f"{status:<8} {name:<25} {path}"
    )

    if not exists:
        all_exist = False

print()
print("-" * 70)

assert all_exist, (
    "Registry validation failed: "
    "one or more referenced artifacts are missing."
)

assert (
    registry["champion"]["model"]
    == "Logistic Regression v1"
)

assert (
    registry["champion"]["decision_threshold"]
    == 0.20
)

assert (
    registry["champion"]["test_status"]
    == "CONSUMED_FINAL_EVALUATION"
)

assert (
    registry["governance"][
        "test_set_reusable_for_tuning"
    ]
    is False
)

print("Registry artifact validation: PASS")
print("Champion governance validation: PASS")

REGISTRY VALIDATION
PASS     registry                  artifacts/registry/model-registry-v1.json
PASS     model_manifest            artifacts/logistic-regression-v1/manifest.json
PASS     feature_contract          artifacts/logistic-regression-v1/feature_contract.json
PASS     validation_metrics        artifacts/logistic-regression-v1/validation_metrics.json
PASS     experiment_001            artifacts/experiments/EXP-001-logistic-regression-baseline-v1.json
PASS     experiment_002            artifacts/experiments/EXP-002-xgboost-controlled-comparison-v1.json
PASS     selection_protocol        artifacts/experiments/model-selection-protocol-v1.json
PASS     final_test                artifacts/experiments/FINAL-TEST-001-logistic-regression-v1.json
PASS     explainability_manifest   artifacts/explainability/logistic-regression-v1/manifest.json

----------------------------------------------------------------------
Registry artifact validation: PASS
Champion governance validation: PASS


Promote registry + experiment evidence to S3

In [51]:
# ============================================================
# STEP 17C — PROMOTE REGISTRY + EXPERIMENT EVIDENCE TO S3
# ============================================================

import boto3
from pathlib import Path

BUCKET = "construction-payment-risk-dev-gk53"

s3 = boto3.client(
    "s3",
    region_name="ap-southeast-2"
)

uploads = {
    Path(
        "artifacts/registry/model-registry-v1.json"
    ): (
        "artifacts/payment-risk/"
        "registry/v1/model-registry-v1.json"
    ),

    Path(
        "artifacts/experiments/"
        "EXP-001-logistic-regression-baseline-v1.json"
    ): (
        "artifacts/payment-risk/"
        "experiments/v1/"
        "EXP-001-logistic-regression-baseline-v1.json"
    ),

    Path(
        "artifacts/experiments/"
        "EXP-002-xgboost-controlled-comparison-v1.json"
    ): (
        "artifacts/payment-risk/"
        "experiments/v1/"
        "EXP-002-xgboost-controlled-comparison-v1.json"
    ),

    Path(
        "artifacts/experiments/"
        "model-selection-protocol-v1.json"
    ): (
        "artifacts/payment-risk/"
        "experiments/v1/"
        "model-selection-protocol-v1.json"
    ),

    Path(
        "artifacts/experiments/"
        "FINAL-TEST-001-logistic-regression-v1.json"
    ): (
        "artifacts/payment-risk/"
        "experiments/v1/"
        "FINAL-TEST-001-logistic-regression-v1.json"
    )
}

print("S3 PROMOTION")
print("=" * 70)

for local_path, s3_key in uploads.items():

    assert local_path.exists(), (
        f"Missing local artifact: {local_path}"
    )

    s3.upload_file(
        str(local_path),
        BUCKET,
        s3_key
    )

    print(
        f"UPLOADED  "
        f"s3://{BUCKET}/{s3_key}"
    )

print()
print("Registry and experiment evidence uploaded.")

S3 PROMOTION
UPLOADED  s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/registry/v1/model-registry-v1.json
UPLOADED  s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/experiments/v1/EXP-001-logistic-regression-baseline-v1.json
UPLOADED  s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/experiments/v1/EXP-002-xgboost-controlled-comparison-v1.json
UPLOADED  s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/experiments/v1/model-selection-protocol-v1.json
UPLOADED  s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/experiments/v1/FINAL-TEST-001-logistic-regression-v1.json

Registry and experiment evidence uploaded.


In [52]:
# ============================================================
# STEP 17C — S3 VERIFICATION
# ============================================================

prefixes = [
    "artifacts/payment-risk/registry/v1/",
    "artifacts/payment-risk/experiments/v1/"
]

verified_objects = []

print("S3 GOVERNANCE VERIFICATION")
print("=" * 70)

for prefix in prefixes:

    response = s3.list_objects_v2(
        Bucket=BUCKET,
        Prefix=prefix
    )

    for obj in response.get("Contents", []):
        verified_objects.append(obj)

        print(
            f"{obj['Key']} "
            f"({obj['Size']} bytes)"
        )

print()
print(
    f"Verified governance object count: "
    f"{len(verified_objects)}"
)

assert len(verified_objects) == 5, (
    "Expected exactly 5 governance artifacts."
)

print(
    "Registry + experiment S3 verification: PASS"
)

S3 GOVERNANCE VERIFICATION
artifacts/payment-risk/registry/v1/model-registry-v1.json (1634 bytes)
artifacts/payment-risk/experiments/v1/EXP-001-logistic-regression-baseline-v1.json (1099 bytes)
artifacts/payment-risk/experiments/v1/EXP-002-xgboost-controlled-comparison-v1.json (1070 bytes)
artifacts/payment-risk/experiments/v1/FINAL-TEST-001-logistic-regression-v1.json (1346 bytes)
artifacts/payment-risk/experiments/v1/model-selection-protocol-v1.json (942 bytes)

Verified governance object count: 5
Registry + experiment S3 verification: PASS


Prediction API Contract

create the API contract artifact

In [53]:
import json
from pathlib import Path

API_DIR = Path("artifacts/api")
API_DIR.mkdir(parents=True, exist_ok=True)

api_contract = {
    "api_version": "v1",
    "service": "Construction Payment Risk Prediction API",

    "purpose": (
        "Score construction payment-protection workflows "
        "for operational review prioritization."
    ),

    "endpoints": {
        "health": {
            "method": "GET",
            "path": "/health"
        },

        "model_metadata": {
            "method": "GET",
            "path": "/v1/model"
        },

        "prediction": {
            "method": "POST",
            "path": "/v1/predict"
        }
    },

    "prediction_model": {
        "model": "Logistic Regression v1",
        "feature_contract_version": "1.0",
        "threshold": 0.20
    },

    "request_features": {
        "categorical": [
            "state",
            "project_type",
            "public_private",
            "customer_role",
            "hiring_party_type"
        ],

        "numeric": [
            "payment_chain_completeness_score",
            "research_confidence_score",
            "deadline_days_remaining",
            "prior_escalation_rate",
            "prior_projects_with_hiring_party",
            "expected_party_count"
        ],

        "binary": [
            "critical_field_missing",
            "multiple_candidate_records",
            "conflicting_project_information"
        ]
    },

    "response_fields": [
        "model_version",
        "predicted_operational_risk",
        "threshold",
        "model_decision",
        "explanation",
        "disclaimers"
    ],

    "internal_only_fields": [
        "raw_model_details",
        "transformed_feature_values",
        "raw_coefficients"
    ],

    "governance": {
        "legal_decision_system": False,
        "human_review_required_for_high_impact_actions": True,
        "synthetic_model": True
    }
}

contract_path = API_DIR / "prediction-api-contract-v1.json"

with open(contract_path, "w") as f:
    json.dump(
        api_contract,
        f,
        indent=2
    )

print("Prediction API Contract v1 created.")
print(contract_path)

Prediction API Contract v1 created.
artifacts/api/prediction-api-contract-v1.json


Pydantic request validation rules before writing FastAPI.

state → 2-letter code \
payment_chain_completeness_score → 0.0 to 1.0 \
research_confidence_score → 0.0 to 1.0 \ 
prior_escalation_rate → 0.0 to 1.0 \
deadline_days_remaining → integer, can be negative if already past deadline \
prior_projects_with_hiring_party → integer >= 0 \ 
expected_party_count → integer >= 1 \
binary fields → only 0 or 1

In [54]:
# ============================================================
# STEP 18A.2 — API INPUT VALIDATION CONTRACT v1
# ============================================================

import json
from pathlib import Path

API_DIR = Path("artifacts/api")
API_DIR.mkdir(parents=True, exist_ok=True)

validation_contract = {
    "validation_contract_version": "1.0",
    "api_version": "v1",
    "model": "Logistic Regression v1",

    "unknown_fields_policy": "reject",

    "categorical_fields": {
        "state": {
            "type": "string",
            "required": True,
            "rule": "two_letter_uppercase_code"
        },

        "project_type": {
            "type": "string",
            "required": True,
            "allowed_values": [
                "commercial",
                "residential",
                "infrastructure",
                "mixed_use"
            ]
        },

        "public_private": {
            "type": "string",
            "required": True,
            "allowed_values": [
                "private",
                "public"
            ]
        },

        "customer_role": {
            "type": "string",
            "required": True,
            "rule": "must_match_training_feature_contract"
        },

        "hiring_party_type": {
            "type": "string",
            "required": True,
            "rule": "must_match_training_feature_contract"
        }
    },

    "numeric_fields": {
        "payment_chain_completeness_score": {
            "type": "float",
            "required": True,
            "minimum": 0.0,
            "maximum": 1.0
        },

        "research_confidence_score": {
            "type": "float",
            "required": True,
            "minimum": 0.0,
            "maximum": 1.0
        },

        "deadline_days_remaining": {
            "type": "integer",
            "required": True,
            "allow_negative": True
        },

        "prior_escalation_rate": {
            "type": "float",
            "required": True,
            "minimum": 0.0,
            "maximum": 1.0
        },

        "prior_projects_with_hiring_party": {
            "type": "integer",
            "required": True,
            "minimum": 0
        },

        "expected_party_count": {
            "type": "integer",
            "required": True,
            "minimum": 1
        }
    },

    "binary_fields": {
        "critical_field_missing": {
            "type": "integer",
            "required": True,
            "allowed_values": [0, 1]
        },

        "multiple_candidate_records": {
            "type": "integer",
            "required": True,
            "allowed_values": [0, 1]
        },

        "conflicting_project_information": {
            "type": "integer",
            "required": True,
            "allowed_values": [0, 1]
        }
    },

    "semantic_rules": [
        {
            "rule_id": "SEM-001",
            "description": (
                "Scores representing proportions or confidence "
                "must remain between 0 and 1."
            )
        },
        {
            "rule_id": "SEM-002",
            "description": (
                "Binary indicators must contain exactly 0 or 1."
            )
        },
        {
            "rule_id": "SEM-003",
            "description": (
                "Negative deadline_days_remaining is permitted "
                "because an assessment may occur after a relevant "
                "operational deadline."
            )
        }
    ],

    "rejection_policy": {
        "missing_required_field": "HTTP_422",
        "invalid_type": "HTTP_422",
        "out_of_range": "HTTP_422",
        "unknown_field": "HTTP_422",
        "unsupported_category": "HTTP_422"
    },

    "governance": {
        "validation_occurs_before_inference": True,
        "target_field_accepted": False,
        "future_outcome_fields_accepted": False
    }
}

validation_path = (
    API_DIR /
    "prediction-api-validation-contract-v1.json"
)

with open(validation_path, "w") as f:
    json.dump(
        validation_contract,
        f,
        indent=2
    )

print("Prediction API Validation Contract v1 created.")
print(validation_path)

Prediction API Validation Contract v1 created.
artifacts/api/prediction-api-validation-contract-v1.json


artifacts/api/\
├── prediction-api-contract-v1.json\
└── prediction-api-validation-contract-v1.json

In [55]:
# ============================================================
# STEP 18A.3 — DISCOVER TRAINED CATEGORICAL VOCABULARY
# ============================================================

preprocessor = reloaded_pipeline.named_steps["preprocessor"]

categorical_pipeline = (
    preprocessor.named_transformers_["categorical"]
)

one_hot_encoder = (
    categorical_pipeline.named_steps["onehot"]
)

categorical_features = [
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type"
]

trained_categories = {}

print("TRAINED CATEGORICAL VOCABULARY")
print("=" * 70)

for feature, categories in zip(
    categorical_features,
    one_hot_encoder.categories_
):
    values = [
        str(value)
        for value in categories.tolist()
    ]

    trained_categories[feature] = values

    print(f"\n{feature}")
    print("-" * 40)

    for value in values:
        print(f"  - {value}")

TRAINED CATEGORICAL VOCABULARY

state
----------------------------------------
  - AZ
  - CA
  - FL
  - GA
  - NC
  - OTHER
  - TX

project_type
----------------------------------------
  - commercial
  - industrial
  - infrastructure
  - mixed_use
  - other
  - residential

public_private
----------------------------------------
  - private
  - public

customer_role
----------------------------------------
  - equipment_supplier
  - general_contractor
  - labor_provider
  - material_supplier
  - sub_subcontractor
  - subcontractor

hiring_party_type
----------------------------------------
  - general_contractor
  - owner
  - subcontractor


update the validation contract from the discovered training vocabulary:

In [56]:
# ============================================================
# STEP 18A.4 — ALIGN SERVING CONTRACT WITH TRAINING VOCABULARY
# ============================================================

import json
from pathlib import Path

validation_path = Path(
    "artifacts/api/"
    "prediction-api-validation-contract-v1.json"
)

with open(validation_path, "r") as f:
    validation_contract = json.load(f)

# Replace categorical rules with exact trained values
validation_contract["categorical_fields"]["state"] = {
    "type": "string",
    "required": True,
    "allowed_values": trained_categories["state"]
}

validation_contract["categorical_fields"]["project_type"] = {
    "type": "string",
    "required": True,
    "allowed_values": trained_categories["project_type"]
}

validation_contract["categorical_fields"]["public_private"] = {
    "type": "string",
    "required": True,
    "allowed_values": trained_categories["public_private"]
}

validation_contract["categorical_fields"]["customer_role"] = {
    "type": "string",
    "required": True,
    "allowed_values": trained_categories["customer_role"]
}

validation_contract["categorical_fields"]["hiring_party_type"] = {
    "type": "string",
    "required": True,
    "allowed_values": trained_categories["hiring_party_type"]
}

validation_contract["training_vocabulary_source"] = {
    "source": "serialized Logistic Regression v1 pipeline",
    "method": "OneHotEncoder.categories_",
    "status": "VERIFIED"
}

with open(validation_path, "w") as f:
    json.dump(
        validation_contract,
        f,
        indent=2
    )

print("Serving validation contract aligned with trained vocabulary.")
print(validation_path)

print("\nVERIFIED CATEGORIES")
print("=" * 70)

for field, values in trained_categories.items():
    print(f"{field}: {values}")

Serving validation contract aligned with trained vocabulary.
artifacts/api/prediction-api-validation-contract-v1.json

VERIFIED CATEGORIES
state: ['AZ', 'CA', 'FL', 'GA', 'NC', 'OTHER', 'TX']
project_type: ['commercial', 'industrial', 'infrastructure', 'mixed_use', 'other', 'residential']
public_private: ['private', 'public']
customer_role: ['equipment_supplier', 'general_contractor', 'labor_provider', 'material_supplier', 'sub_subcontractor', 'subcontractor']
hiring_party_type: ['general_contractor', 'owner', 'subcontractor']


Pydantic request/response schemas.

In [58]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field


# ============================================================
# ALLOWED CATEGORICAL VALUES
# ============================================================

State = Literal[
    "AZ",
    "CA",
    "FL",
    "GA",
    "NC",
    "OTHER",
    "TX"
]

ProjectType = Literal[
    "commercial",
    "industrial",
    "infrastructure",
    "mixed_use",
    "other",
    "residential"
]

PublicPrivate = Literal[
    "private",
    "public"
]

CustomerRole = Literal[
    "equipment_supplier",
    "general_contractor",
    "labor_provider",
    "material_supplier",
    "sub_subcontractor",
    "subcontractor"
]

HiringPartyType = Literal[
    "general_contractor",
    "owner",
    "subcontractor"
]


# ============================================================
# REQUEST SCHEMA
# ============================================================

class PredictionRequest(BaseModel):

    model_config = ConfigDict(
        extra="forbid"
    )

    state: State
    project_type: ProjectType
    public_private: PublicPrivate
    customer_role: CustomerRole
    hiring_party_type: HiringPartyType

    payment_chain_completeness_score: float = Field(
        ge=0.0,
        le=1.0
    )

    research_confidence_score: float = Field(
        ge=0.0,
        le=1.0
    )

    deadline_days_remaining: int

    prior_escalation_rate: float = Field(
        ge=0.0,
        le=1.0
    )

    prior_projects_with_hiring_party: int = Field(
        ge=0
    )

    expected_party_count: int = Field(
        ge=1
    )

    critical_field_missing: Literal[0, 1]

    multiple_candidate_records: Literal[0, 1]

    conflicting_project_information: Literal[0, 1]


# ============================================================
# RESPONSE SCHEMAS
# ============================================================

class ExplanationFactor(BaseModel):
    feature: str
    label: str

    direction: Literal[
        "higher",
        "lower"
    ]

    contribution_log_odds: float
    absolute_contribution: float
    explanation: str


class PredictionExplanation(BaseModel):
    top_factors: list[ExplanationFactor]
    method: str


class PredictionResponse(BaseModel):
    model_version: str

    predicted_operational_risk: float = Field(
        ge=0.0,
        le=1.0
    )

    threshold: float = Field(
        ge=0.0,
        le=1.0
    )

    model_decision: Literal[
        "FLAGGED_BY_MODEL",
        "NOT_FLAGGED_BY_MODEL"
    ]

    explanation: PredictionExplanation

    disclaimers: list[str]


class HealthResponse(BaseModel):
    status: Literal["ok"]


class ModelMetadataResponse(BaseModel):
    model_version: str
    feature_contract_version: str
    threshold: float
    status: str


print("Pydantic API schemas loaded successfully.")

Pydantic API schemas loaded successfully.


In [59]:
valid_request = PredictionRequest(
    state="FL",
    project_type="commercial",
    public_private="private",
    customer_role="material_supplier",
    hiring_party_type="general_contractor",
    payment_chain_completeness_score=0.72,
    research_confidence_score=0.81,
    deadline_days_remaining=24,
    prior_escalation_rate=0.18,
    prior_projects_with_hiring_party=3,
    expected_party_count=5,
    critical_field_missing=0,
    multiple_candidate_records=1,
    conflicting_project_information=0
)

print("VALID REQUEST")
print(valid_request.model_dump())

VALID REQUEST
{'state': 'FL', 'project_type': 'commercial', 'public_private': 'private', 'customer_role': 'material_supplier', 'hiring_party_type': 'general_contractor', 'payment_chain_completeness_score': 0.72, 'research_confidence_score': 0.81, 'deadline_days_remaining': 24, 'prior_escalation_rate': 0.18, 'prior_projects_with_hiring_party': 3, 'expected_party_count': 5, 'critical_field_missing': 0, 'multiple_candidate_records': 1, 'conflicting_project_information': 0}


In [60]:
from pydantic import ValidationError

try:
    PredictionRequest(
        state="FL",
        project_type="commercial_project",   # invalid category
        public_private="private",
        customer_role="material_supplier",
        hiring_party_type="general_contractor",

        payment_chain_completeness_score=1.20,  # invalid > 1
        research_confidence_score=0.81,
        deadline_days_remaining=24,
        prior_escalation_rate=0.18,
        prior_projects_with_hiring_party=3,
        expected_party_count=5,

        critical_field_missing=2,             # invalid binary
        multiple_candidate_records=1,
        conflicting_project_information=0,

        escalation_required=1                 # forbidden extra field
    )

except ValidationError as exc:
    print("INVALID REQUEST REJECTED")
    print("=" * 70)
    print(exc)

INVALID REQUEST REJECTED
4 validation errors for PredictionRequest
project_type
  Input should be 'commercial', 'industrial', 'infrastructure', 'mixed_use', 'other' or 'residential' [type=literal_error, input_value='commercial_project', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
payment_chain_completeness_score
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.2, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
critical_field_missing
  Input should be 0 or 1 [type=literal_error, input_value=2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
escalation_required
  Extra inputs are not permitted [type=extra_forbidden, input_value=1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


FastAPI service implementation. We’ll keep it minimal and production-oriented: GET /health, GET /v1/model, and POST /v1/predict, with the prediction endpoint wired to the approved Logistic Regression v1 and the safe explanation layer.

Build the inference function

In [61]:
# ============================================================
# STEP 18C.1 — PRODUCTION INFERENCE FUNCTION
# ============================================================

import pandas as pd

MODEL_VERSION = "logistic-regression-v1"
FINAL_THRESHOLD = 0.20

MODEL_FEATURES = [
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type",
    "payment_chain_completeness_score",
    "research_confidence_score",
    "deadline_days_remaining",
    "prior_escalation_rate",
    "prior_projects_with_hiring_party",
    "expected_party_count",
    "critical_field_missing",
    "multiple_candidate_records",
    "conflicting_project_information",
]


def predict_workflow(request: PredictionRequest) -> dict:
    """
    Generate an operational risk prediction and safe explanation
    for one construction payment-protection workflow.
    """

    # --------------------------------------------------------
    # 1. Convert validated Pydantic request to model input
    # --------------------------------------------------------

    payload = request.model_dump()

    workflow_df = pd.DataFrame(
        [payload],
        columns=MODEL_FEATURES
    )

    # Defensive contract check
    if workflow_df.columns.tolist() != MODEL_FEATURES:
        raise RuntimeError(
            "Model feature contract mismatch."
        )

    # --------------------------------------------------------
    # 2. Generate probability
    # --------------------------------------------------------

    probability = float(
        reloaded_pipeline.predict_proba(
            workflow_df
        )[0, 1]
    )

    # --------------------------------------------------------
    # 3. Apply frozen threshold
    # --------------------------------------------------------

    decision = (
        "FLAGGED_BY_MODEL"
        if probability >= FINAL_THRESHOLD
        else "NOT_FLAGGED_BY_MODEL"
    )

    # --------------------------------------------------------
    # 4. Generate exact safe explanation
    # --------------------------------------------------------

    explanation_result = explain_workflow(
        workflow_row=workflow_df,
        pipeline=reloaded_pipeline,
        business_map=BUSINESS_EXPLANATION_MAP,
        threshold=FINAL_THRESHOLD,
        top_n=5
    )

    # --------------------------------------------------------
    # 5. Build CUSTOMER-SAFE response
    # --------------------------------------------------------

    response = {
        "model_version": MODEL_VERSION,

        "predicted_operational_risk":
            probability,

        "threshold":
            FINAL_THRESHOLD,

        "model_decision":
            decision,

        "explanation": {
            "top_factors":
                explanation_result[
                    "explanation"
                ]["top_factors"],

            "method":
                explanation_result[
                    "explanation"
                ]["method"]
        },

        "disclaimers":
            explanation_result[
                "disclaimers"
            ]
    }

    # --------------------------------------------------------
    # 6. Validate our own output
    # --------------------------------------------------------

    validated_response = PredictionResponse(
        **response
    )

    return validated_response.model_dump()


print("Production inference function loaded.")

Production inference function loaded.


Now run the test cell so we verify the full inference path actually works, not just that Python accepted the function

In [62]:
result = predict_workflow(
    valid_request
)

print("PREDICTION SERVICE TEST")
print("=" * 70)

print(
    f"Risk:     "
    f"{result['predicted_operational_risk']:.2%}"
)

print(
    f"Decision: "
    f"{result['model_decision']}"
)

print(
    f"Threshold: "
    f"{result['threshold']:.2f}"
)

print(
    f"Explanation factors: "
    f"{len(result['explanation']['top_factors'])}"
)

print("\nCUSTOMER-SAFE RESPONSE")
print("=" * 70)

import json

print(
    json.dumps(
        result,
        indent=2
    )
)

PREDICTION SERVICE TEST
Risk:     33.20%
Decision: FLAGGED_BY_MODEL
Threshold: 0.20
Explanation factors: 5

CUSTOMER-SAFE RESPONSE
{
  "model_version": "logistic-regression-v1",
  "predicted_operational_risk": 0.3319541555119854,
  "threshold": 0.2,
  "model_decision": "FLAGGED_BY_MODEL",
  "explanation": {
    "top_factors": [
      {
        "feature": "multiple_candidate_records",
        "label": "Multiple candidate project records",
        "direction": "higher",
        "contribution_log_odds": 0.6439512926781611,
        "absolute_contribution": 0.6439512926781611,
        "explanation": "Multiple candidate project records contributed to a higher predicted operational risk score."
      },
      {
        "feature": "payment_chain_completeness_score",
        "label": "Payment-chain information completeness",
        "direction": "higher",
        "contribution_log_odds": 0.08857918077863886,
        "absolute_contribution": 0.08857918077863886,
        "explanation": "Lower pay

Create the actual FastAPI application

In [63]:
# ============================================================
# STEP 18C.2 — FASTAPI APPLICATION
# ============================================================

from fastapi import FastAPI
from fastapi import HTTPException

app = FastAPI(
    title="Construction Payment Risk Prediction API",
    description=(
        "Operational risk scoring service for construction "
        "payment-protection workflows."
    ),
    version="1.0.0"
)


# ============================================================
# HEALTH ENDPOINT
# ============================================================

@app.get(
    "/health",
    response_model=HealthResponse
)
def health():
    return {
        "status": "ok"
    }


# ============================================================
# MODEL METADATA ENDPOINT
# ============================================================

@app.get(
    "/v1/model",
    response_model=ModelMetadataResponse
)
def model_metadata():

    return {
        "model_version":
            "logistic-regression-v1",

        "feature_contract_version":
            "1.0",

        "threshold":
            FINAL_THRESHOLD,

        "status":
            "APPROVED_CHAMPION"
    }


# ============================================================
# PREDICTION ENDPOINT
# ============================================================

@app.post(
    "/v1/predict",
    response_model=PredictionResponse
)
def predict(
    request: PredictionRequest
):

    try:

        return predict_workflow(
            request
        )

    except Exception as exc:

        # Do not expose internal model details
        # or stack traces to API clients.

        raise HTTPException(
            status_code=500,
            detail=(
                "Prediction service encountered "
                "an internal error."
            )
        ) from exc


print("FastAPI application created.")

FastAPI application created.


In [64]:
from fastapi.testclient import TestClient

client = TestClient(app)

print("TestClient created.")

TestClient created.


In [65]:
# ============================================================
# 1. HEALTH
# ============================================================

response = client.get("/health")

print("HEALTH")
print("STATUS:", response.status_code)
print("BODY:", response.json())


# ============================================================
# 2. MODEL METADATA
# ============================================================

response = client.get("/v1/model")

print("\nMODEL METADATA")
print("STATUS:", response.status_code)
print("BODY:", response.json())


# ============================================================
# 3. PREDICTION
# ============================================================

prediction_payload = valid_request.model_dump()

response = client.post(
    "/v1/predict",
    json=prediction_payload
)

print("\nPREDICTION")
print("STATUS:", response.status_code)

print(
    json.dumps(
        response.json(),
        indent=2
    )
)

HEALTH
STATUS: 200
BODY: {'status': 'ok'}

MODEL METADATA
STATUS: 200
BODY: {'model_version': 'logistic-regression-v1', 'feature_contract_version': '1.0', 'threshold': 0.2, 'status': 'APPROVED_CHAMPION'}

PREDICTION
STATUS: 200
{
  "model_version": "logistic-regression-v1",
  "predicted_operational_risk": 0.3319541555119854,
  "threshold": 0.2,
  "model_decision": "FLAGGED_BY_MODEL",
  "explanation": {
    "top_factors": [
      {
        "feature": "multiple_candidate_records",
        "label": "Multiple candidate project records",
        "direction": "higher",
        "contribution_log_odds": 0.6439512926781611,
        "absolute_contribution": 0.6439512926781611,
        "explanation": "Multiple candidate project records contributed to a higher predicted operational risk score."
      },
      {
        "feature": "payment_chain_completeness_score",
        "label": "Payment-chain information completeness",
        "direction": "higher",
        "contribution_log_odds": 0.088579180

API failure-path testing.

In [66]:
# ============================================================
# STEP 18C.4 — INVALID API REQUEST TESTS
# ============================================================

invalid_payload = {
    "state": "FL",

    # invalid trained category
    "project_type": "commercial_project",

    "public_private": "private",
    "customer_role": "material_supplier",
    "hiring_party_type": "general_contractor",

    # invalid > 1
    "payment_chain_completeness_score": 1.20,

    "research_confidence_score": 0.81,
    "deadline_days_remaining": 24,
    "prior_escalation_rate": 0.18,
    "prior_projects_with_hiring_party": 3,
    "expected_party_count": 5,

    # invalid binary
    "critical_field_missing": 2,

    "multiple_candidate_records": 1,
    "conflicting_project_information": 0,

    # forbidden target leakage field
    "escalation_required": 1
}

response = client.post(
    "/v1/predict",
    json=invalid_payload
)

print("INVALID REQUEST TEST")
print("=" * 70)

print("STATUS:", response.status_code)

print(
    json.dumps(
        response.json(),
        indent=2
    )
)

assert response.status_code == 422

print()
print("Invalid request rejection: PASS")

INVALID REQUEST TEST
STATUS: 422
{
  "detail": [
    {
      "type": "literal_error",
      "loc": [
        "body",
        "project_type"
      ],
      "msg": "Input should be 'commercial', 'industrial', 'infrastructure', 'mixed_use', 'other' or 'residential'",
      "input": "commercial_project",
      "ctx": {
        "expected": "'commercial', 'industrial', 'infrastructure', 'mixed_use', 'other' or 'residential'"
      }
    },
    {
      "type": "less_than_equal",
      "loc": [
        "body",
        "payment_chain_completeness_score"
      ],
      "msg": "Input should be less than or equal to 1",
      "input": 1.2,
      "ctx": {
        "le": 1.0
      }
    },
    {
      "type": "literal_error",
      "loc": [
        "body",
        "critical_field_missing"
      ],
      "msg": "Input should be 0 or 1",
      "input": 2,
      "ctx": {
        "expected": "0 or 1"
      }
    },
    {
      "type": "extra_forbidden",
      "loc": [
        "body",
        "escalation_

We still have one final validation-path check for this phase: missing required field.

In [67]:
missing_field_payload = valid_request.model_dump()

del missing_field_payload[
    "research_confidence_score"
]

response = client.post(
    "/v1/predict",
    json=missing_field_payload
)

print("MISSING FIELD TEST")
print("=" * 70)

print("STATUS:", response.status_code)

print(
    json.dumps(
        response.json(),
        indent=2
    )
)

assert response.status_code == 422

print()
print("Missing required field rejection: PASS")

MISSING FIELD TEST
STATUS: 422
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "research_confidence_score"
      ],
      "msg": "Field required",
      "input": {
        "state": "FL",
        "project_type": "commercial",
        "public_private": "private",
        "customer_role": "material_supplier",
        "hiring_party_type": "general_contractor",
        "payment_chain_completeness_score": 0.72,
        "deadline_days_remaining": 24,
        "prior_escalation_rate": 0.18,
        "prior_projects_with_hiring_party": 3,
        "expected_party_count": 5,
        "critical_field_missing": 0,
        "multiple_candidate_records": 1,
        "conflicting_project_information": 0
      }
    }
  ]
}

Missing required field rejection: PASS


Create the package

payment-delay-predictor/\
└── app/\
        ├── __init__.py\
        ├── main.py\
        ├── schemas.py\
        ├── inference.py\
        ├── explainability.py\
        └── config.py\


tests/\
├── test_health.py\
├── test_model_metadata.py\
├── test_prediction.py\
└── test_validation.py\

In [68]:
from pathlib import Path

APP_DIR = Path("app")
APP_DIR.mkdir(parents=True, exist_ok=True)

(APP_DIR / "__init__.py").touch()

print("Application package created:")
print(APP_DIR.resolve())

for path in sorted(APP_DIR.iterdir()):
    print("-", path.name)

Application package created:
/home/sagemaker-user/app
- __init__.py


In [69]:
schemas_code = '''
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field


# ============================================================
# TRAINED CATEGORICAL VOCABULARY
# Logistic Regression v1
# ============================================================

State = Literal[
    "AZ",
    "CA",
    "FL",
    "GA",
    "NC",
    "OTHER",
    "TX",
]

ProjectType = Literal[
    "commercial",
    "industrial",
    "infrastructure",
    "mixed_use",
    "other",
    "residential",
]

PublicPrivate = Literal[
    "private",
    "public",
]

CustomerRole = Literal[
    "equipment_supplier",
    "general_contractor",
    "labor_provider",
    "material_supplier",
    "sub_subcontractor",
    "subcontractor",
]

HiringPartyType = Literal[
    "general_contractor",
    "owner",
    "subcontractor",
]


# ============================================================
# REQUEST
# ============================================================

class PredictionRequest(BaseModel):
    """Validated input for one payment-protection workflow."""

    model_config = ConfigDict(
        extra="forbid"
    )

    state: State
    project_type: ProjectType
    public_private: PublicPrivate
    customer_role: CustomerRole
    hiring_party_type: HiringPartyType

    payment_chain_completeness_score: float = Field(
        ge=0.0,
        le=1.0,
    )

    research_confidence_score: float = Field(
        ge=0.0,
        le=1.0,
    )

    deadline_days_remaining: int

    prior_escalation_rate: float = Field(
        ge=0.0,
        le=1.0,
    )

    prior_projects_with_hiring_party: int = Field(
        ge=0
    )

    expected_party_count: int = Field(
        ge=1
    )

    critical_field_missing: Literal[0, 1]
    multiple_candidate_records: Literal[0, 1]
    conflicting_project_information: Literal[0, 1]


# ============================================================
# EXPLANATION RESPONSE
# ============================================================

class ExplanationFactor(BaseModel):
    feature: str
    label: str

    direction: Literal[
        "higher",
        "lower",
    ]

    contribution_log_odds: float
    absolute_contribution: float
    explanation: str


class PredictionExplanation(BaseModel):
    top_factors: list[ExplanationFactor]
    method: str


# ============================================================
# API RESPONSES
# ============================================================

class PredictionResponse(BaseModel):
    model_version: str

    predicted_operational_risk: float = Field(
        ge=0.0,
        le=1.0,
    )

    threshold: float = Field(
        ge=0.0,
        le=1.0,
    )

    model_decision: Literal[
        "FLAGGED_BY_MODEL",
        "NOT_FLAGGED_BY_MODEL",
    ]

    explanation: PredictionExplanation
    disclaimers: list[str]


class HealthResponse(BaseModel):
    status: Literal["ok"]


class ModelMetadataResponse(BaseModel):
    model_version: str
    feature_contract_version: str
    threshold: float
    status: str
'''

schemas_path = APP_DIR / "schemas.py"

schemas_path.write_text(
    schemas_code.strip() + "\n"
)

print("Created:", schemas_path)

Created: app/schemas.py


In [70]:
from app.schemas import PredictionRequest as FilePredictionRequest


file_request = FilePredictionRequest(
    state="FL",
    project_type="commercial",
    public_private="private",
    customer_role="material_supplier",
    hiring_party_type="general_contractor",

    payment_chain_completeness_score=0.72,
    research_confidence_score=0.81,
    deadline_days_remaining=24,
    prior_escalation_rate=0.18,
    prior_projects_with_hiring_party=3,
    expected_party_count=5,

    critical_field_missing=0,
    multiple_candidate_records=1,
    conflicting_project_information=0,
)


print("SCHEMA MODULE TEST")
print("=" * 70)

print(
    file_request.model_dump()
)

print()
print("app.schemas import: PASS")

SCHEMA MODULE TEST
{'state': 'FL', 'project_type': 'commercial', 'public_private': 'private', 'customer_role': 'material_supplier', 'hiring_party_type': 'general_contractor', 'payment_chain_completeness_score': 0.72, 'research_confidence_score': 0.81, 'deadline_days_remaining': 24, 'prior_escalation_rate': 0.18, 'prior_projects_with_hiring_party': 3, 'expected_party_count': 5, 'critical_field_missing': 0, 'multiple_candidate_records': 1, 'conflicting_project_information': 0}

app.schemas import: PASS


create app/config.py -- So model constants are no longer scattered across notebook cells.

In [71]:
from pathlib import Path

APP_DIR = Path("app")

config_code = '''
MODEL_VERSION = "logistic-regression-v1"
FEATURE_CONTRACT_VERSION = "1.0"
FINAL_THRESHOLD = 0.20

MODEL_FEATURES = [
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type",
    "payment_chain_completeness_score",
    "research_confidence_score",
    "deadline_days_remaining",
    "prior_escalation_rate",
    "prior_projects_with_hiring_party",
    "expected_party_count",
    "critical_field_missing",
    "multiple_candidate_records",
    "conflicting_project_information",
]

MODEL_ARTIFACT_PATH = (
    "artifacts/logistic-regression-v1/"
    "logistic_pipeline.joblib"
)

EXPLAINABILITY_MAP_PATH = (
    "artifacts/explainability/"
    "logistic-regression-v1/"
    "business-explanation-map-v1.json"
)

MODEL_REGISTRY_PATH = (
    "artifacts/registry/"
    "model-registry-v1.json"
)
'''

config_path = APP_DIR / "config.py"

config_path.write_text(
    config_code.strip() + "\n"
)

print("Created:", config_path)

Created: app/config.py


In [72]:
from app.config import (
    MODEL_VERSION,
    FEATURE_CONTRACT_VERSION,
    FINAL_THRESHOLD,
    MODEL_FEATURES,
    MODEL_ARTIFACT_PATH,
)

print("CONFIG MODULE TEST")
print("=" * 70)

print("Model version:", MODEL_VERSION)
print("Feature contract:", FEATURE_CONTRACT_VERSION)
print("Threshold:", FINAL_THRESHOLD)
print("Feature count:", len(MODEL_FEATURES))
print("Model artifact:", MODEL_ARTIFACT_PATH)

assert MODEL_VERSION == "logistic-regression-v1"
assert FEATURE_CONTRACT_VERSION == "1.0"
assert FINAL_THRESHOLD == 0.20
assert len(MODEL_FEATURES) == 14

print()
print("app.config import: PASS")

CONFIG MODULE TEST
Model version: logistic-regression-v1
Feature contract: 1.0
Threshold: 0.2
Feature count: 14
Model artifact: artifacts/logistic-regression-v1/logistic_pipeline.joblib

app.config import: PASS


move the explanation engine into app/explainability.py. This is especially important because our API must not depend on the notebook-defined explain_workflow() function.

In [73]:
from pathlib import Path

APP_DIR = Path("app")

explainability_code = '''
import json
from pathlib import Path

import numpy as np


def load_business_explanation_map(path: str) -> dict:
    """Load the approved business explanation allowlist."""

    map_path = Path(path)

    if not map_path.exists():
        raise FileNotFoundError(
            f"Business explanation map not found: {map_path}"
        )

    with open(map_path, "r") as f:
        return json.load(f)


def explain_workflow(
    workflow_row,
    pipeline,
    business_map: dict,
    threshold: float,
    top_n: int = 5,
) -> dict:
    """
    Generate an exact local explanation for one
    Logistic Regression workflow prediction.

    Contributions are calculated in log-odds space.
    Only approved business-facing features are returned
    in the customer-safe explanation.
    """

    if len(workflow_row) != 1:
        raise ValueError(
            "workflow_row must contain exactly one row."
        )

    preprocessor = pipeline.named_steps["preprocessor"]
    model = pipeline.named_steps["model"]

    probability = float(
        pipeline.predict_proba(workflow_row)[0, 1]
    )

    decision = (
        "FLAGGED_BY_MODEL"
        if probability >= threshold
        else "NOT_FLAGGED_BY_MODEL"
    )

    transformed = preprocessor.transform(
        workflow_row
    )

    if hasattr(transformed, "toarray"):
        transformed_values = transformed.toarray()[0]
    else:
        transformed_values = np.asarray(
            transformed
        )[0]

    feature_names = (
        preprocessor.get_feature_names_out()
    )

    coefficients = model.coef_[0]
    intercept = float(model.intercept_[0])

    contributions = (
        transformed_values * coefficients
    )

    reconstructed_log_odds = (
        intercept + contributions.sum()
    )

    reconstructed_probability = float(
        1 / (
            1 + np.exp(
                -reconstructed_log_odds
            )
        )
    )

    reconstruction_error = abs(
        probability -
        reconstructed_probability
    )

    approved_factors = []

    for (
        feature_name,
        coefficient,
        contribution,
    ) in zip(
        feature_names,
        coefficients,
        contributions,
    ):

        contribution = float(
            contribution
        )

        base_feature = None

        for prefix in [
            "numeric__",
            "binary__",
        ]:
            if feature_name.startswith(prefix):
                base_feature = feature_name.replace(
                    prefix,
                    "",
                    1,
                )
                break

        if base_feature not in business_map:
            continue

        if abs(contribution) < 1e-8:
            continue

        direction = (
            "higher"
            if contribution > 0
            else "lower"
        )

        config = business_map[
            base_feature
        ]

        approved_factors.append(
            {
                "feature": base_feature,
                "label": config["label"],
                "direction": direction,
                "contribution_log_odds":
                    contribution,
                "absolute_contribution":
                    abs(contribution),
                "explanation":
                    config[direction],
            }
        )

    approved_factors = sorted(
        approved_factors,
        key=lambda item:
            item["absolute_contribution"],
        reverse=True,
    )[:top_n]

    return {
        "predicted_operational_risk":
            probability,

        "threshold":
            threshold,

        "model_decision":
            decision,

        "explanation": {
            "top_factors":
                approved_factors,

            "method":
                "exact_logistic_regression_"
                "log_odds_decomposition",

            "reconstruction_error":
                reconstruction_error,
        },

        "disclaimers": [
            (
                "Explanation describes model "
                "associations, not causation."
            ),
            (
                "Prediction does not determine "
                "legal rights or provide legal advice."
            ),
            (
                "Model was evaluated using synthetic "
                "construction payment-protection data."
            ),
        ],
    }
'''

explainability_path = (
    APP_DIR / "explainability.py"
)

explainability_path.write_text(
    explainability_code.strip() + "\n"
)

print("Created:", explainability_path)

Created: app/explainability.py


In [74]:
from app.explainability import (
    explain_workflow as file_explain_workflow,
    load_business_explanation_map,
)

from app.config import (
    EXPLAINABILITY_MAP_PATH,
    FINAL_THRESHOLD,
)

file_business_map = (
    load_business_explanation_map(
        EXPLAINABILITY_MAP_PATH
    )
)

file_explanation = file_explain_workflow(
    workflow_row=X_test.iloc[[0]],
    pipeline=reloaded_pipeline,
    business_map=file_business_map,
    threshold=FINAL_THRESHOLD,
    top_n=5,
)

print("EXPLAINABILITY MODULE TEST")
print("=" * 70)

print(
    "Risk:",
    file_explanation[
        "predicted_operational_risk"
    ],
)

print(
    "Decision:",
    file_explanation[
        "model_decision"
    ],
)

print(
    "Reconstruction error:",
    file_explanation[
        "explanation"
    ]["reconstruction_error"],
)

print(
    "Approved factors:",
    len(
        file_explanation[
            "explanation"
        ]["top_factors"]
    ),
)

assert (
    file_explanation[
        "model_decision"
    ]
    == "NOT_FLAGGED_BY_MODEL"
)

assert (
    file_explanation[
        "explanation"
    ]["reconstruction_error"]
    < 1e-10
)

print()
print("app.explainability import: PASS")

EXPLAINABILITY MODULE TEST
Risk: 0.12149310253594506
Decision: NOT_FLAGGED_BY_MODEL
Reconstruction error: 0.0
Approved factors: 4

app.explainability import: PASS


build app/inference.py. This is a major step because we will stop depending on the notebook variable reloaded_pipeline. The module itself will load the approved model artifact from disk.

In [75]:
from pathlib import Path

APP_DIR = Path("app")

inference_code = '''
import joblib
import pandas as pd

from app.config import (
    MODEL_VERSION,
    FINAL_THRESHOLD,
    MODEL_FEATURES,
    MODEL_ARTIFACT_PATH,
    EXPLAINABILITY_MAP_PATH,
)

from app.schemas import (
    PredictionRequest,
    PredictionResponse,
)

from app.explainability import (
    explain_workflow,
    load_business_explanation_map,
)


# ============================================================
# LOAD APPROVED MODEL ARTIFACTS
# ============================================================

_model_pipeline = joblib.load(
    MODEL_ARTIFACT_PATH
)

_business_map = load_business_explanation_map(
    EXPLAINABILITY_MAP_PATH
)


# ============================================================
# PRODUCTION INFERENCE
# ============================================================

def predict_workflow(
    request: PredictionRequest,
) -> dict:
    """
    Run one validated workflow through the approved
    Logistic Regression v1 serving pipeline.
    """

    request_data = request.model_dump()

    workflow_row = pd.DataFrame(
        [request_data],
        columns=MODEL_FEATURES,
    )

    # Defensive serving contract check
    if list(workflow_row.columns) != MODEL_FEATURES:
        raise ValueError(
            "Prediction feature order does not match "
            "the approved model feature contract."
        )

    explanation_result = explain_workflow(
        workflow_row=workflow_row,
        pipeline=_model_pipeline,
        business_map=_business_map,
        threshold=FINAL_THRESHOLD,
        top_n=5,
    )

    response_payload = {
        "model_version":
            MODEL_VERSION,

        "predicted_operational_risk":
            explanation_result[
                "predicted_operational_risk"
            ],

        "threshold":
            explanation_result[
                "threshold"
            ],

        "model_decision":
            explanation_result[
                "model_decision"
            ],

        "explanation": {
            "top_factors":
                explanation_result[
                    "explanation"
                ]["top_factors"],

            "method":
                explanation_result[
                    "explanation"
                ]["method"],
        },

        "disclaimers":
            explanation_result[
                "disclaimers"
            ],
    }

    validated_response = (
        PredictionResponse(
            **response_payload
        )
    )

    return validated_response.model_dump()
'''

inference_path = (
    APP_DIR / "inference.py"
)

inference_path.write_text(
    inference_code.strip() + "\n"
)

print("Created:", inference_path)

Created: app/inference.py


In [76]:
from app.inference import (
    predict_workflow as file_predict_workflow,
)

from app.schemas import PredictionRequest


module_test_request = PredictionRequest(
    state="FL",
    project_type="commercial",
    public_private="private",
    customer_role="material_supplier",
    hiring_party_type="general_contractor",

    payment_chain_completeness_score=0.72,
    research_confidence_score=0.81,
    deadline_days_remaining=24,
    prior_escalation_rate=0.18,
    prior_projects_with_hiring_party=3,
    expected_party_count=5,

    critical_field_missing=0,
    multiple_candidate_records=1,
    conflicting_project_information=0,
)


module_prediction = file_predict_workflow(
    module_test_request
)

print("INFERENCE MODULE TEST")
print("=" * 70)

print(
    "Risk:",
    module_prediction[
        "predicted_operational_risk"
    ],
)

print(
    "Decision:",
    module_prediction[
        "model_decision"
    ],
)

print(
    "Threshold:",
    module_prediction[
        "threshold"
    ],
)

print(
    "Explanation factors:",
    len(
        module_prediction[
            "explanation"
        ]["top_factors"]
    ),
)

assert (
    abs(
        module_prediction[
            "predicted_operational_risk"
        ]
        - 0.3319541555119854
    )
    < 1e-10
)

assert (
    module_prediction[
        "model_decision"
    ]
    == "FLAGGED_BY_MODEL"
)

assert (
    module_prediction[
        "threshold"
    ]
    == 0.20
)

print()
print("app.inference import: PASS")

INFERENCE MODULE TEST
Risk: 0.3319541555119854
Decision: FLAGGED_BY_MODEL
Threshold: 0.2
Explanation factors: 5

app.inference import: PASS


create app/main.py and rebuild the FastAPI application entirely from the source modules.

In [77]:
from pathlib import Path

APP_DIR = Path("app")

main_code = '''
from fastapi import FastAPI, HTTPException

from app.config import (
    MODEL_VERSION,
    FEATURE_CONTRACT_VERSION,
    FINAL_THRESHOLD,
)

from app.schemas import (
    PredictionRequest,
    PredictionResponse,
    HealthResponse,
    ModelMetadataResponse,
)

from app.inference import (
    predict_workflow,
)


app = FastAPI(
    title="Construction Payment Risk Prediction API",
    description=(
        "Operational risk scoring service for "
        "construction payment-protection workflows."
    ),
    version="1.0.0",
)


@app.get(
    "/health",
    response_model=HealthResponse,
)
def health():
    return {
        "status": "ok"
    }


@app.get(
    "/v1/model",
    response_model=ModelMetadataResponse,
)
def model_metadata():
    return {
        "model_version":
            MODEL_VERSION,

        "feature_contract_version":
            FEATURE_CONTRACT_VERSION,

        "threshold":
            FINAL_THRESHOLD,

        "status":
            "APPROVED_CHAMPION",
    }


@app.post(
    "/v1/predict",
    response_model=PredictionResponse,
)
def predict(
    request: PredictionRequest,
):
    try:
        return predict_workflow(
            request
        )

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=(
                "Prediction service encountered "
                "an internal error."
            ),
        ) from exc
'''

main_path = APP_DIR / "main.py"

main_path.write_text(
    main_code.strip() + "\n"
)

print("Created:", main_path)

Created: app/main.py


In [78]:
from app.main import app as file_app

print("MAIN MODULE TEST")
print("=" * 70)

print("Title:", file_app.title)
print("Version:", file_app.version)

routes = sorted(
    route.path
    for route in file_app.routes
)

print("Routes:")

for route in routes:
    print("-", route)

assert "/health" in routes
assert "/v1/model" in routes
assert "/v1/predict" in routes

print()
print("app.main import: PASS")

MAIN MODULE TEST
Title: Construction Payment Risk Prediction API
Version: 1.0.0
Routes:
- /docs
- /docs/oauth2-redirect
- /health
- /openapi.json
- /redoc
- /v1/model
- /v1/predict

app.main import: PASS


Then do one more HTTP test, but this time against the file-based application, not the notebook app object:

In [79]:
from fastapi.testclient import TestClient

file_client = TestClient(
    file_app
)

response = file_client.get(
    "/health"
)

print("FILE-BASED API TEST")
print("=" * 70)

print(
    "Health:",
    response.status_code,
    response.json(),
)

prediction_payload = {
    "state": "FL",
    "project_type": "commercial",
    "public_private": "private",
    "customer_role": "material_supplier",
    "hiring_party_type": "general_contractor",

    "payment_chain_completeness_score": 0.72,
    "research_confidence_score": 0.81,
    "deadline_days_remaining": 24,
    "prior_escalation_rate": 0.18,
    "prior_projects_with_hiring_party": 3,
    "expected_party_count": 5,

    "critical_field_missing": 0,
    "multiple_candidate_records": 1,
    "conflicting_project_information": 0,
}

response = file_client.post(
    "/v1/predict",
    json=prediction_payload,
)

print(
    "Prediction:",
    response.status_code,
)

print(
    "Risk:",
    response.json()[
        "predicted_operational_risk"
    ],
)

print(
    "Decision:",
    response.json()[
        "model_decision"
    ],
)

assert response.status_code == 200

assert (
    abs(
        response.json()[
            "predicted_operational_risk"
        ]
        - 0.3319541555119854
    )
    < 1e-10
)

print()
print(
    "File-based FastAPI serving: PASS"
)

FILE-BASED API TEST
Health: 200 {'status': 'ok'}
Prediction: 200
Risk: 0.3319541555119854
Decision: FLAGGED_BY_MODEL

File-based FastAPI serving: PASS


create automated API tests

In [80]:
from pathlib import Path

TESTS_DIR = Path("tests")
TESTS_DIR.mkdir(parents=True, exist_ok=True)

print("Created:", TESTS_DIR)

Created: tests


In [81]:
test_api_code = '''
from fastapi.testclient import TestClient

from app.main import app


client = TestClient(app)


VALID_PAYLOAD = {
    "state": "FL",
    "project_type": "commercial",
    "public_private": "private",
    "customer_role": "material_supplier",
    "hiring_party_type": "general_contractor",
    "payment_chain_completeness_score": 0.72,
    "research_confidence_score": 0.81,
    "deadline_days_remaining": 24,
    "prior_escalation_rate": 0.18,
    "prior_projects_with_hiring_party": 3,
    "expected_party_count": 5,
    "critical_field_missing": 0,
    "multiple_candidate_records": 1,
    "conflicting_project_information": 0,
}


def test_health():
    response = client.get("/health")

    assert response.status_code == 200
    assert response.json() == {
        "status": "ok"
    }


def test_model_metadata():
    response = client.get("/v1/model")

    assert response.status_code == 200

    body = response.json()

    assert body["model_version"] == (
        "logistic-regression-v1"
    )

    assert body[
        "feature_contract_version"
    ] == "1.0"

    assert body["threshold"] == 0.20

    assert body["status"] == (
        "APPROVED_CHAMPION"
    )


def test_valid_prediction():
    response = client.post(
        "/v1/predict",
        json=VALID_PAYLOAD,
    )

    assert response.status_code == 200

    body = response.json()

    assert (
        abs(
            body[
                "predicted_operational_risk"
            ]
            - 0.3319541555119854
        )
        < 1e-10
    )

    assert body[
        "model_decision"
    ] == "FLAGGED_BY_MODEL"

    assert body["threshold"] == 0.20

    assert (
        len(
            body[
                "explanation"
            ]["top_factors"]
        )
        > 0
    )


def test_invalid_category_returns_422():
    payload = VALID_PAYLOAD.copy()

    payload["project_type"] = (
        "commercial_project"
    )

    response = client.post(
        "/v1/predict",
        json=payload,
    )

    assert response.status_code == 422


def test_invalid_numeric_range_returns_422():
    payload = VALID_PAYLOAD.copy()

    payload[
        "payment_chain_completeness_score"
    ] = 1.2

    response = client.post(
        "/v1/predict",
        json=payload,
    )

    assert response.status_code == 422


def test_forbidden_extra_field_returns_422():
    payload = VALID_PAYLOAD.copy()

    payload["escalation_required"] = 1

    response = client.post(
        "/v1/predict",
        json=payload,
    )

    assert response.status_code == 422


def test_missing_required_field_returns_422():
    payload = VALID_PAYLOAD.copy()

    del payload[
        "research_confidence_score"
    ]

    response = client.post(
        "/v1/predict",
        json=payload,
    )

    assert response.status_code == 422
'''

test_path = TESTS_DIR / "test_api.py"

test_path.write_text(
    test_api_code.strip() + "\n"
)

print("Created:", test_path)

Created: tests/test_api.py


In [82]:
import subprocess

result = subprocess.run(
    [
        "python",
        "-m",
        "pytest",
        "tests/test_api.py",
        "-v",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0

print()
print("Automated API test suite: PASS")

============================= test session starts ==============================
platform linux -- Python 3.12.14, pytest-9.1.1, pluggy-1.6.0 -- /opt/conda/bin/python
cachedir: .pytest_cache
Fugue tests will be initialized with options:
rootdir: /home/sagemaker-user
plugins: anyio-4.14.2, dash-2.18.1, fugue-0.9.7, langsmith-0.11.1, cov-7.1.0
collecting ... collected 7 items

tests/test_api.py::test_health PASSED                                    [ 14%]
tests/test_api.py::test_model_metadata PASSED                            [ 28%]
tests/test_api.py::test_valid_prediction PASSED                          [ 42%]
tests/test_api.py::test_invalid_category_returns_422 PASSED              [ 57%]
tests/test_api.py::test_invalid_numeric_range_returns_422 PASSED         [ 71%]
tests/test_api.py::test_forbidden_extra_field_returns_422 PASSED         [ 85%]
tests/test_api.py::test_missing_required_field_returns_422 PASSED        [100%]

============================== 7 passed in 1.95s ============

API Artifact Manifest.

The purpose is governance and reproducibility. Months later, we should be able to answer: Which API version served which model, at what threshold, with which files, and did its tests pass?

In [83]:
import json
from pathlib import Path
from datetime import datetime, timezone

API_ARTIFACT_DIR = Path(
    "artifacts/api/payment-risk-prediction-api/v1"
)

API_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

manifest = {
    "artifact_type": "prediction_api",
    "api_name": "Construction Payment Risk Prediction API",
    "api_version": "1.0.0",

    "model": {
        "model_version": "logistic-regression-v1",
        "feature_contract_version": "1.0",
        "decision_threshold": 0.20,
        "status": "APPROVED_CHAMPION",
    },

    "purpose": (
        "Operational risk scoring and review prioritization "
        "for construction payment-protection workflows."
    ),

    "endpoints": {
        "health": {
            "method": "GET",
            "path": "/health",
        },
        "model_metadata": {
            "method": "GET",
            "path": "/v1/model",
        },
        "prediction": {
            "method": "POST",
            "path": "/v1/predict",
        },
    },

    "application_files": [
        "app/__init__.py",
        "app/config.py",
        "app/schemas.py",
        "app/explainability.py",
        "app/inference.py",
        "app/main.py",
    ],

    "test_files": [
        "tests/test_api.py",
    ],

    "test_status": {
        "status": "PASS",
        "tests_passed": 7,
        "tests_failed": 0,
        "framework": "pytest",
    },

    "serving_contract": {
        "input_feature_count": 14,
        "unknown_fields": "REJECT",
        "unsupported_categories": "REJECT",
        "invalid_requests_http_status": 422,
        "internal_errors_http_status": 500,
    },

    "explainability": {
        "enabled": True,
        "method": (
            "exact_logistic_regression_log_odds_decomposition"
        ),
        "business_safe_allowlist": True,
        "raw_model_details_exposed": False,
    },

    "governance": {
        "synthetic_model": True,
        "legal_decision_system": False,
        "human_review_required_for_high_impact_actions": True,
        "test_dataset_status": "CONSUMED_FINAL_EVALUATION",
    },

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

manifest_path = (
    API_ARTIFACT_DIR /
    "api-manifest-v1.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    )
    + "\n"
)

print("Created:", manifest_path)

Created: artifacts/api/payment-risk-prediction-api/v1/api-manifest-v1.json


In [84]:
with open(manifest_path, "r") as f:
    loaded_manifest = json.load(f)

print("API MANIFEST VALIDATION")
print("=" * 70)

print(
    "API version:",
    loaded_manifest["api_version"]
)

print(
    "Model:",
    loaded_manifest[
        "model"
    ]["model_version"]
)

print(
    "Threshold:",
    loaded_manifest[
        "model"
    ]["decision_threshold"]
)

print(
    "Tests passed:",
    loaded_manifest[
        "test_status"
    ]["tests_passed"]
)

print(
    "Raw model details exposed:",
    loaded_manifest[
        "explainability"
    ]["raw_model_details_exposed"]
)

assert loaded_manifest["api_version"] == "1.0.0"

assert (
    loaded_manifest["model"]["model_version"]
    == "logistic-regression-v1"
)

assert (
    loaded_manifest["model"]["decision_threshold"]
    == 0.20
)

assert (
    loaded_manifest["test_status"]["status"]
    == "PASS"
)

assert (
    loaded_manifest[
        "test_status"
    ]["tests_passed"]
    == 7
)

assert (
    loaded_manifest[
        "explainability"
    ]["raw_model_details_exposed"]
    is False
)

print()
print("API manifest validation: PASS")

API MANIFEST VALIDATION
API version: 1.0.0
Model: logistic-regression-v1
Threshold: 0.2
Tests passed: 7
Raw model details exposed: False

API manifest validation: PASS


promote the API manifest to S3 and verify it.

In [85]:
import boto3
from pathlib import Path

BUCKET = "construction-payment-risk-dev-gk53"

LOCAL_MANIFEST = (
    Path("artifacts/api/payment-risk-prediction-api/v1/")
    / "api-manifest-v1.json"
)

S3_KEY = (
    "artifacts/payment-risk/api/"
    "prediction-api/v1/"
    "api-manifest-v1.json"
)

s3 = boto3.client("s3")

s3.upload_file(
    str(LOCAL_MANIFEST),
    BUCKET,
    S3_KEY,
)

print("Uploaded:")
print(f"s3://{BUCKET}/{S3_KEY}")

Uploaded:
s3://construction-payment-risk-dev-gk53/artifacts/payment-risk/api/prediction-api/v1/api-manifest-v1.json


In [86]:
response = s3.head_object(
    Bucket=BUCKET,
    Key=S3_KEY,
)

print("API MANIFEST S3 VERIFICATION")
print("=" * 70)

print("Bucket:", BUCKET)
print("Key:", S3_KEY)
print("Size:", response["ContentLength"])
print("Content type:", response.get("ContentType"))
print("ETag:", response["ETag"])

assert response["ContentLength"] > 0

print()
print("API manifest S3 verification: PASS")

API MANIFEST S3 VERIFICATION
Bucket: construction-payment-risk-dev-gk53
Key: artifacts/payment-risk/api/prediction-api/v1/api-manifest-v1.json
Size: 1649
Content type: binary/octet-stream
ETag: "5ac8d612162524231c8e4f6998e76462"

API manifest S3 verification: PASS


In [87]:
import json
from io import BytesIO

obj = s3.get_object(
    Bucket=BUCKET,
    Key=S3_KEY,
)

s3_manifest = json.loads(
    obj["Body"].read().decode("utf-8")
)

assert s3_manifest["api_version"] == "1.0.0"

assert (
    s3_manifest["model"]["model_version"]
    == "logistic-regression-v1"
)

assert (
    s3_manifest["model"]["decision_threshold"]
    == 0.20
)

assert (
    s3_manifest["test_status"]["tests_passed"]
    == 7
)

assert (
    s3_manifest["governance"]["legal_decision_system"]
    is False
)

print("S3 manifest content validation: PASS")

S3 manifest content validation: PASS


Containerized ML Serving System.

define the runtime dependency contract before writing a Dockerfile. We want the container to include only what the API actually needs.

In [94]:
from pathlib import Path

requirements = """
fastapi==0.141.1
uvicorn[standard]==0.52.4
pydantic==2.13.4
pandas==2.3.3
numpy==1.26.4
scikit-learn==1.7.2
joblib==1.5.3
"""

requirements_path = Path("requirements-api.txt")

requirements_path.write_text(
    requirements.strip() + "\n"
)

print("FINAL API RUNTIME CONTRACT")
print("=" * 70)
print(requirements_path.read_text())

FINAL API RUNTIME CONTRACT
fastapi==0.141.1
uvicorn[standard]==0.52.4
pydantic==2.13.4
pandas==2.3.3
numpy==1.26.4
scikit-learn==1.7.2
joblib==1.5.3



Now verify the versions currently running in SageMaker so we know whether the container contract matches the environment that successfully served the model:

In [95]:
import fastapi
import pydantic
import pandas
import numpy
import sklearn
import joblib
import uvicorn


print("RUNTIME VERSION CHECK")
print("=" * 70)

print("FastAPI:      ", fastapi.__version__)
print("Pydantic:     ", pydantic.__version__)
print("Pandas:       ", pandas.__version__)
print("NumPy:        ", numpy.__version__)
print("scikit-learn: ", sklearn.__version__)
print("Joblib:       ", joblib.__version__)
print("Uvicorn:", uvicorn.__version__)

RUNTIME VERSION CHECK
FastAPI:       0.141.1
Pydantic:      2.13.4
Pandas:        2.3.3
NumPy:         1.26.4
scikit-learn:  1.7.2
Joblib:        1.5.3
Uvicorn: 0.52.4


Dockerfile creation for the Containerized ML Serving System.
We are not deploying yet. First we create the container definition, inspect it, and then verify that all files Docker needs actually exist.

In [96]:
from pathlib import Path

dockerfile_content = """
FROM python:3.12-slim

# ------------------------------------------------------------
# Runtime configuration
# ------------------------------------------------------------

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PIP_NO_CACHE_DIR=1

WORKDIR /app

# ------------------------------------------------------------
# Install Python dependencies first.
# Keeping this separate improves Docker layer caching.
# ------------------------------------------------------------

COPY requirements-api.txt .

RUN pip install --upgrade pip && \
    pip install -r requirements-api.txt

# ------------------------------------------------------------
# Copy application source
# ------------------------------------------------------------

COPY app ./app

# ------------------------------------------------------------
# Copy approved model artifact
# ------------------------------------------------------------

COPY artifacts/logistic-regression-v1 \
     ./artifacts/logistic-regression-v1

# ------------------------------------------------------------
# Copy approved business explanation artifact
# ------------------------------------------------------------

COPY artifacts/explainability/logistic-regression-v1 \
     ./artifacts/explainability/logistic-regression-v1

# ------------------------------------------------------------
# Container networking
# ------------------------------------------------------------

EXPOSE 8000

# ------------------------------------------------------------
# Start FastAPI through Uvicorn
# ------------------------------------------------------------

CMD [
    "uvicorn",
    "app.main:app",
    "--host",
    "0.0.0.0",
    "--port",
    "8000"
]
"""

dockerfile_path = Path("Dockerfile")

dockerfile_path.write_text(
    dockerfile_content.strip() + "\n"
)

print("Created:", dockerfile_path)
print()
print(dockerfile_path.read_text())

Created: Dockerfile

FROM python:3.12-slim

# ------------------------------------------------------------
# Runtime configuration
# ------------------------------------------------------------

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PIP_NO_CACHE_DIR=1

WORKDIR /app

# ------------------------------------------------------------
# Install Python dependencies first.
# Keeping this separate improves Docker layer caching.
# ------------------------------------------------------------

COPY requirements-api.txt .

RUN pip install --upgrade pip &&     pip install -r requirements-api.txt

# ------------------------------------------------------------
# Copy application source
# ------------------------------------------------------------

COPY app ./app

# ------------------------------------------------------------
# Copy approved model artifact
# ------------------------------------------------------------

COPY artifacts/logistic-regression-v1      ./artifacts/logistic-r

Why python:3.12-slim?
Your successful SageMaker environment is running Python 3.12, so we're keeping the container on the same major/minor Python version.
We're also using slim rather than a full Python image because this API doesn't need Jupyter, SageMaker SDKs, visualization packages, training libraries, Glue tooling, etc.
Our container is for inference, not training.
Now run a Docker build-context validation before trying to build anything:

In [97]:
from pathlib import Path

required_paths = [
    Path("Dockerfile"),
    Path("requirements-api.txt"),

    Path("app/__init__.py"),
    Path("app/config.py"),
    Path("app/schemas.py"),
    Path("app/explainability.py"),
    Path("app/inference.py"),
    Path("app/main.py"),

    Path(
        "artifacts/logistic-regression-v1/"
        "logistic_pipeline.joblib"
    ),

    Path(
        "artifacts/explainability/"
        "logistic-regression-v1/"
        "business-explanation-map-v1.json"
    ),
]

print("DOCKER BUILD CONTEXT CHECK")
print("=" * 70)

all_present = True

for path in required_paths:
    exists = path.exists()

    print(
        f"{'PASS' if exists else 'FAIL'}  {path}"
    )

    if not exists:
        all_present = False

print()

assert all_present, (
    "Docker build context is incomplete."
)

print("Docker build context: PASS")

DOCKER BUILD CONTEXT CHECK
PASS  Dockerfile
PASS  requirements-api.txt
PASS  app/__init__.py
PASS  app/config.py
PASS  app/schemas.py
PASS  app/explainability.py
PASS  app/inference.py
PASS  app/main.py
PASS  artifacts/logistic-regression-v1/logistic_pipeline.joblib
PASS  artifacts/explainability/logistic-regression-v1/business-explanation-map-v1.json

Docker build context: PASS


determine whether this SageMaker environment can actually build Docker images.

In [98]:
import subprocess

commands = [
    ["docker", "--version"],
    ["docker", "info"],
]

for command in commands:
    print("=" * 70)
    print("COMMAND:", " ".join(command))
    print("=" * 70)

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=30,
        )

        print("RETURN CODE:", result.returncode)

        if result.stdout:
            print("\nSTDOUT:")
            print(result.stdout)

        if result.stderr:
            print("\nSTDERR:")
            print(result.stderr)

    except FileNotFoundError:
        print("Docker CLI: NOT INSTALLED")

    except subprocess.TimeoutExpired:
        print("Docker command timed out")

COMMAND: docker --version
RETURN CODE: 0

STDOUT:
Docker version 29.7.1, build unknown-commit

COMMAND: docker info
RETURN CODE: 1

STDOUT:
Client:
 Version:    29.7.1
 Context:    default
 Debug Mode: false

Server:


STDERR:
failed to connect to the docker API at unix:///var/run/docker.sock; check if the path is correct and if the daemon is running: dial unix /var/run/docker.sock: connect: no such file or directory



Create the ECR repository

In [99]:
import boto3
from botocore.exceptions import ClientError

REGION = "ap-southeast-2"

ECR_REPOSITORY = (
    "construction-payment-risk-api-dev"
)

ecr = boto3.client(
    "ecr",
    region_name=REGION,
)

try:
    response = ecr.describe_repositories(
        repositoryNames=[
            ECR_REPOSITORY
        ]
    )

    repository = (
        response[
            "repositories"
        ][0]
    )

    print(
        "ECR repository already exists."
    )

except ecr.exceptions.RepositoryNotFoundException:

    response = ecr.create_repository(
        repositoryName=ECR_REPOSITORY,

        imageScanningConfiguration={
            "scanOnPush": True
        },

        imageTagMutability="IMMUTABLE",
    )

    repository = response[
        "repository"
    ]

    print(
        "ECR repository created."
    )


print()
print("ECR REPOSITORY")
print("=" * 70)

print(
    "Name:",
    repository[
        "repositoryName"
    ]
)

print(
    "URI:",
    repository[
        "repositoryUri"
    ]
)

print(
    "Registry ID:",
    repository[
        "registryId"
    ]
)

print(
    "Image tag mutability:",
    repository[
        "imageTagMutability"
    ]
)

print(
    "Scan on push:",
    repository[
        "imageScanningConfiguration"
    ]["scanOnPush"]
)

ECR repository created.

ECR REPOSITORY
Name: construction-payment-risk-api-dev
URI: 911797457166.dkr.ecr.ap-southeast-2.amazonaws.com/construction-payment-risk-api-dev
Registry ID: 911797457166
Image tag mutability: IMMUTABLE
Scan on push: True


create the CodeBuild build specification.

authenticate to ECR\
        ↓\
build Docker image\
        ↓\
tag api-v1.0.0\
        ↓\
push image to ECR

In [101]:
from pathlib import Path

buildspec_content = """
version: 0.2

phases:
  pre_build:
    commands:
      - echo Logging in to Amazon ECR...
      - aws --version
      - ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
      - REGION=ap-southeast-2
      - REPOSITORY=construction-payment-risk-api-dev
      - IMAGE_TAG=api-v1.0.0
      - ECR_URI=$ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/$REPOSITORY
      - aws ecr get-login-password --region $REGION | docker login --username AWS --password-stdin $ECR_URI

  build:
    commands:
      - echo Build started on `date`
      - echo Building Docker image...
      - docker build -t $REPOSITORY:$IMAGE_TAG .
      - docker tag $REPOSITORY:$IMAGE_TAG $ECR_URI:$IMAGE_TAG

  post_build:
    commands:
      - echo Build completed on `date`
      - echo Pushing Docker image...
      - docker push $ECR_URI:$IMAGE_TAG
      - echo Writing image metadata...
      - printf '{"imageUri":"%s"}' "$ECR_URI:$IMAGE_TAG" > imageDetail.json
      - cat imageDetail.json

artifacts:
  files:
    - imageDetail.json
"""

buildspec_path = Path("buildspec.yml")

buildspec_path.write_text(
    buildspec_content.strip() + "\n"
)

print("Created:", buildspec_path)
print()
print(buildspec_path.read_text())

Created: buildspec.yml

version: 0.2

phases:
  pre_build:
    commands:
      - echo Logging in to Amazon ECR...
      - aws --version
      - ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
      - REGION=ap-southeast-2
      - REPOSITORY=construction-payment-risk-api-dev
      - IMAGE_TAG=api-v1.0.0
      - ECR_URI=$ACCOUNT_ID.dkr.ecr.$REGION.amazonaws.com/$REPOSITORY
      - aws ecr get-login-password --region $REGION | docker login --username AWS --password-stdin $ECR_URI

  build:
    commands:
      - echo Build started on `date`
      - echo Building Docker image...
      - docker build -t $REPOSITORY:$IMAGE_TAG .
      - docker tag $REPOSITORY:$IMAGE_TAG $ECR_URI:$IMAGE_TAG

  post_build:
    commands:
      - echo Build completed on `date`
      - echo Pushing Docker image...
      - docker push $ECR_URI:$IMAGE_TAG
      - echo Writing image metadata...
      - printf '{"imageUri":"%s"}' "$ECR_URI:$IMAGE_TAG" > imageDetail.json
      - cat imageDetail.

Now verify the build package has everything CodeBuild will need:

In [102]:
from pathlib import Path

required_paths = [
    Path("Dockerfile"),
    Path("buildspec.yml"),
    Path("requirements-api.txt"),

    Path("app/main.py"),
    Path("app/config.py"),
    Path("app/schemas.py"),
    Path("app/inference.py"),
    Path("app/explainability.py"),

    Path(
        "artifacts/logistic-regression-v1/"
        "logistic_pipeline.joblib"
    ),

    Path(
        "artifacts/explainability/"
        "logistic-regression-v1/"
        "business-explanation-map-v1.json"
    ),
]

print("CODEBUILD CONTEXT CHECK")
print("=" * 70)

for path in required_paths:
    print(
        f"{'PASS' if path.exists() else 'FAIL'}  {path}"
    )

assert all(
    path.exists()
    for path in required_paths
)

print()
print("CodeBuild context: PASS")

CODEBUILD CONTEXT CHECK
PASS  Dockerfile
PASS  buildspec.yml
PASS  requirements-api.txt
PASS  app/main.py
PASS  app/config.py
PASS  app/schemas.py
PASS  app/inference.py
PASS  app/explainability.py
PASS  artifacts/logistic-regression-v1/logistic_pipeline.joblib
PASS  artifacts/explainability/logistic-regression-v1/business-explanation-map-v1.json

CodeBuild context: PASS


The CodeBuild context contains every required application, model, explanation, dependency, Docker, and buildspec file.


Before creating the CodeBuild project, there is one important cloud detail: CodeBuild cannot see files sitting inside your SageMaker filesystem. We first need to package this exact build context and place it in S3.

Package CodeBuild source and upload to S3

We will create a clean ZIP containing only what the container build needs.

Package CodeBuild source and upload to S3

In [103]:
from pathlib import Path
import zipfile

SOURCE_ZIP = Path(
    "artifacts/build/payment-risk-api/"
    "payment-risk-api-v1-source.zip"
)

SOURCE_ZIP.parent.mkdir(
    parents=True,
    exist_ok=True,
)

files_to_package = [
    Path("Dockerfile"),
    Path("buildspec.yml"),
    Path("requirements-api.txt"),

    Path("app/__init__.py"),
    Path("app/config.py"),
    Path("app/schemas.py"),
    Path("app/explainability.py"),
    Path("app/inference.py"),
    Path("app/main.py"),

    Path(
        "artifacts/logistic-regression-v1/"
        "logistic_pipeline.joblib"
    ),

    Path(
        "artifacts/explainability/"
        "logistic-regression-v1/"
        "business-explanation-map-v1.json"
    ),
]

with zipfile.ZipFile(
    SOURCE_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for file_path in files_to_package:
        zf.write(
            file_path,
            arcname=str(file_path),
        )

print("Created:", SOURCE_ZIP)
print(
    "Size:",
    SOURCE_ZIP.stat().st_size,
    "bytes"
)

Created: artifacts/build/payment-risk-api/payment-risk-api-v1-source.zip
Size: 9644 bytes


In [104]:
with zipfile.ZipFile(
    SOURCE_ZIP,
    "r",
) as zf:

    packaged_files = zf.namelist()

print("CODEBUILD SOURCE PACKAGE")
print("=" * 70)

for name in packaged_files:
    print("PASS ", name)

assert "Dockerfile" in packaged_files
assert "buildspec.yml" in packaged_files
assert "requirements-api.txt" in packaged_files

assert (
    "app/main.py"
    in packaged_files
)

assert (
    "artifacts/logistic-regression-v1/"
    "logistic_pipeline.joblib"
    in packaged_files
)

assert (
    "artifacts/explainability/"
    "logistic-regression-v1/"
    "business-explanation-map-v1.json"
    in packaged_files
)

print()
print("CodeBuild source package validation: PASS")

CODEBUILD SOURCE PACKAGE
PASS  Dockerfile
PASS  buildspec.yml
PASS  requirements-api.txt
PASS  app/__init__.py
PASS  app/config.py
PASS  app/schemas.py
PASS  app/explainability.py
PASS  app/inference.py
PASS  app/main.py
PASS  artifacts/logistic-regression-v1/logistic_pipeline.joblib
PASS  artifacts/explainability/logistic-regression-v1/business-explanation-map-v1.json

CodeBuild source package validation: PASS


In [108]:
import boto3

s3_test = boto3.client(
    "s3",
    region_name="ap-southeast-2"
)

test_key = (
    "builds/payment-risk-api/"
    "iam-test.txt"
)

s3_test.put_object(
    Bucket="construction-payment-risk-dev-gk53",
    Key=test_key,
    Body=b"IAM permission test"
)

print("S3 PutObject permission: PASS")

S3 PutObject permission: PASS


In [109]:
s3.upload_file(
    str(SOURCE_ZIP),
    BUCKET,
    SOURCE_S3_KEY,
)

print(
    f"s3://{BUCKET}/{SOURCE_S3_KEY}"
)

s3://construction-payment-risk-dev-gk53/builds/payment-risk-api/v1/payment-risk-api-v1-source.zip


In [110]:
response = s3.head_object(
    Bucket=BUCKET,
    Key=SOURCE_S3_KEY,
)

print("CODEBUILD SOURCE S3 VERIFICATION")
print("=" * 70)

print("Bucket:", BUCKET)
print("Key:", SOURCE_S3_KEY)
print("Size:", response["ContentLength"])
print("ETag:", response["ETag"])

assert response["ContentLength"] > 0

print()
print("CodeBuild source S3 verification: PASS")

CODEBUILD SOURCE S3 VERIFICATION
Bucket: construction-payment-risk-dev-gk53
Key: builds/payment-risk-api/v1/payment-risk-api-v1-source.zip
Size: 9644
ETag: "c472fb1ff07f7416569eb4ae6dfb82cf"

CodeBuild source S3 verification: PASS


create the CodeBuild IAM role.

verify the ECR image digest and metadata

In [113]:
import boto3

ecr = boto3.client(
    "ecr",
    region_name="ap-southeast-2",
)

response = ecr.list_images(
    repositoryName="construction-payment-risk-api-dev"
)

print("ECR read permission: PASS")
print(response.get("imageIds", []))

ECR read permission: PASS
[]


In [116]:
response = ecr.list_images(
    repositoryName="construction-payment-risk-api-dev",
    filter={
        "tagStatus": "ANY"
    }
)

images = response.get("imageIds", [])

print("ECR IMAGES")
print("=" * 70)

if not images:
    print("No images found in repository.")
else:
    for i, image in enumerate(images, start=1):
        print(f"Image {i}")
        print("  Tag:", image.get("imageTag", "<UNTAGGED>"))
        print("  Digest:", image.get("imageDigest"))
        print()

print("Total image references:", len(images))

ECR IMAGES
No images found in repository.
Total image references: 0


In [117]:
response = ecr.describe_images(
    repositoryName="construction-payment-risk-api-dev"
)

details = response.get("imageDetails", [])

print("ECR IMAGE DETAILS")
print("=" * 70)

if not details:
    print("No image details found.")
else:
    for i, image in enumerate(details, start=1):
        print(f"Image {i}")
        print("  Tags:", image.get("imageTags", ["<UNTAGGED>"]))
        print("  Digest:", image.get("imageDigest"))
        print("  Size:", image.get("imageSizeInBytes"))
        print("  Pushed at:", image.get("imagePushedAt"))
        print()

ECR IMAGE DETAILS
No image details found.


In [118]:
from pathlib import Path

dockerfile_content = """FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PIP_NO_CACHE_DIR=1

WORKDIR /app

COPY requirements-api.txt .

RUN pip install --upgrade pip && pip install -r requirements-api.txt

COPY app ./app

COPY artifacts/logistic-regression-v1 ./artifacts/logistic-regression-v1

COPY artifacts/explainability/logistic-regression-v1 ./artifacts/explainability/logistic-regression-v1

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

Path("Dockerfile").write_text(dockerfile_content)

print(Path("Dockerfile").read_text())

FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PIP_NO_CACHE_DIR=1

WORKDIR /app

COPY requirements-api.txt .

RUN pip install --upgrade pip && pip install -r requirements-api.txt

COPY app ./app

COPY artifacts/logistic-regression-v1 ./artifacts/logistic-regression-v1

COPY artifacts/explainability/logistic-regression-v1 ./artifacts/explainability/logistic-regression-v1

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]



In [119]:
from pathlib import Path
import zipfile

SOURCE_ZIP = Path(
    "artifacts/build/payment-risk-api/"
    "payment-risk-api-v1-source.zip"
)

files_to_package = [
    Path("Dockerfile"),
    Path("buildspec.yml"),
    Path("requirements-api.txt"),
    Path("app/__init__.py"),
    Path("app/config.py"),
    Path("app/schemas.py"),
    Path("app/explainability.py"),
    Path("app/inference.py"),
    Path("app/main.py"),
    Path(
        "artifacts/logistic-regression-v1/"
        "logistic_pipeline.joblib"
    ),
    Path(
        "artifacts/explainability/"
        "logistic-regression-v1/"
        "business-explanation-map-v1.json"
    ),
]

with zipfile.ZipFile(
    SOURCE_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:
    for file_path in files_to_package:
        zf.write(
            file_path,
            arcname=str(file_path)
        )

print("Rebuilt:", SOURCE_ZIP)
print("Size:", SOURCE_ZIP.stat().st_size)

Rebuilt: artifacts/build/payment-risk-api/payment-risk-api-v1-source.zip
Size: 9460


In [120]:
with zipfile.ZipFile(SOURCE_ZIP, "r") as zf:
    dockerfile_in_zip = zf.read(
        "Dockerfile"
    ).decode("utf-8")

print(dockerfile_in_zip)

assert (
    'CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]'
    in dockerfile_in_zip
)

print()
print("Dockerfile ZIP validation: PASS")

FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PIP_NO_CACHE_DIR=1

WORKDIR /app

COPY requirements-api.txt .

RUN pip install --upgrade pip && pip install -r requirements-api.txt

COPY app ./app

COPY artifacts/logistic-regression-v1 ./artifacts/logistic-regression-v1

COPY artifacts/explainability/logistic-regression-v1 ./artifacts/explainability/logistic-regression-v1

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]


Dockerfile ZIP validation: PASS


In [121]:
s3.upload_file(
    str(SOURCE_ZIP),
    BUCKET,
    SOURCE_S3_KEY
)

print(
    f"s3://{BUCKET}/{SOURCE_S3_KEY}"
)

s3://construction-payment-risk-dev-gk53/builds/payment-risk-api/v1/payment-risk-api-v1-source.zip


In [122]:
response = s3.head_object(
    Bucket=BUCKET,
    Key=SOURCE_S3_KEY
)

print("UPDATED BUILD SOURCE")
print("=" * 70)
print("Size:", response["ContentLength"])
print("ETag:", response["ETag"])
print()
print("Updated CodeBuild source: PASS")

UPDATED BUILD SOURCE
Size: 9460
ETag: "85fd797777b2550a1b40f2bb37374b98"

Updated CodeBuild source: PASS


Minor Project #15 — Containerized ML Serving System complete, we should do one final verification: confirm the exact ECR tag and immutable image digest.

In [123]:
import boto3

ecr = boto3.client(
    "ecr",
    region_name="ap-southeast-2",
)

response = ecr.describe_images(
    repositoryName="construction-payment-risk-api-dev",
    imageIds=[
        {
            "imageTag": "api-v1.0.0"
        }
    ],
)

image = response["imageDetails"][0]

print("ECR IMAGE VERIFICATION")
print("=" * 70)
print("Tags:", image.get("imageTags"))
print("Digest:", image["imageDigest"])
print("Size:", image["imageSizeInBytes"])
print("Pushed at:", image["imagePushedAt"])

if "imageScanStatus" in image:
    print(
        "Scan status:",
        image["imageScanStatus"].get("status")
    )

print()
print("ECR image verification: PASS")

ECR IMAGE VERIFICATION
Tags: ['api-v1.0.0']
Digest: sha256:3d1c2ba7ada18a232d836e11285ff1110517895e34746824880ea147342f17b1
Size: 166189542
Pushed at: 2026-09-11 09:55:56.494000+00:00

ECR image verification: PASS


## Minor Project #16 — ML Monitoring & Observability

# The goal is to define what must be observable before we add CloudWatch metrics and alarms. For this API, monitoring should answer four questions: is the service healthy, are inputs changing, are predictions behaving differently, and is the model still safe to use operationally.

In [124]:
from pathlib import Path
import json

monitoring_contract = {
    "contract_version": "1.0",
    "project": "Construction Payment Risk Intelligence",
    "component": "Payment Risk Prediction API",
    "model_version": "logistic-regression-v1",
    "api_version": "v1.0.0",
    "purpose": (
        "Monitor serving reliability, input quality, prediction behavior, "
        "and model-risk signals for operational review prioritization."
    ),

    "monitoring_principles": {
        "not_legal_decision_system": True,
        "human_review_required_for_high_impact_actions": True,
        "synthetic_model": True,
        "test_set_consumed": True,
        "monitoring_does_not_replace_model_revalidation": True
    },

    "service_health": {
        "metrics": [
            "request_count",
            "success_count",
            "client_error_count",
            "server_error_count",
            "latency_ms"
        ],
        "initial_alerts": {
            "server_error_rate": {
                "condition": "> 5%",
                "window": "5 minutes"
            },
            "p95_latency_ms": {
                "condition": "> 1000 ms",
                "window": "5 minutes"
            }
        }
    },

    "input_quality": {
        "metrics": [
            "validation_failure_count",
            "validation_failure_rate",
            "unsupported_category_count",
            "missing_required_field_count"
        ],
        "initial_alerts": {
            "validation_failure_rate": {
                "condition": "> 10%",
                "window": "15 minutes"
            }
        }
    },

    "prediction_behavior": {
        "metrics": [
            "prediction_count",
            "mean_predicted_risk",
            "flagged_count",
            "flagged_rate"
        ],
        "reference": {
            "decision_threshold": 0.20,
            "final_test_flagged_rate": 0.5051,
            "final_test_mean_prediction": None
        },
        "notes": [
            "Flagged-rate movement is a monitoring signal, not proof of model drift.",
            "Changes may reflect real population changes, upstream process changes, or data-quality issues."
        ]
    },

    "feature_monitoring": {
        "numeric_features": [
            "payment_chain_completeness_score",
            "research_confidence_score",
            "deadline_days_remaining",
            "prior_escalation_rate",
            "prior_projects_with_hiring_party",
            "expected_party_count"
        ],
        "binary_features": [
            "critical_field_missing",
            "multiple_candidate_records",
            "conflicting_project_information"
        ],
        "categorical_features": [
            "state",
            "project_type",
            "public_private",
            "customer_role",
            "hiring_party_type"
        ],
        "drift_methods_planned": {
            "numeric": [
                "population_stability_index",
                "distribution_summary"
            ],
            "categorical": [
                "category_frequency_shift",
                "unseen_category_count"
            ],
            "binary": [
                "rate_shift"
            ]
        }
    },

    "model_performance_monitoring": {
        "requires_delayed_labels": True,
        "metrics_when_labels_available": [
            "roc_auc",
            "pr_auc",
            "brier_score",
            "precision",
            "recall",
            "f1",
            "flagged_rate",
            "calibration"
        ],
        "important_rule": (
            "Do not infer true performance degradation from unlabeled production traffic."
        )
    },

    "observability_stack": {
        "logs": "Amazon CloudWatch Logs",
        "custom_metrics": "Amazon CloudWatch Metrics",
        "alarms": "Amazon CloudWatch Alarms",
        "container_registry": "Amazon ECR",
        "future_dashboard": "Amazon CloudWatch Dashboard"
    },

    "privacy_and_logging": {
        "log_raw_request_payloads": False,
        "log_raw_model_details": False,
        "log_transformed_features": False,
        "preferred_logging": [
            "request_id",
            "timestamp",
            "model_version",
            "api_version",
            "latency_ms",
            "http_status",
            "predicted_risk",
            "model_decision"
        ]
    },

    "failure_response": {
        "service_failure": "investigate immediately",
        "input_quality_shift": "inspect upstream data/process changes",
        "prediction_shift": "investigate population and feature drift",
        "confirmed_performance_degradation": (
            "freeze promotion, investigate, retrain/revalidate using a new evaluation strategy"
        )
    }
}

output_path = Path(
    "artifacts/monitoring/payment-risk-api/v1/"
    "monitoring-contract-v1.json"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

output_path.write_text(
    json.dumps(
        monitoring_contract,
        indent=2
    )
)

print("Created:", output_path)
print()
print(
    json.dumps(
        monitoring_contract,
        indent=2
    )
)

Created: artifacts/monitoring/payment-risk-api/v1/monitoring-contract-v1.json

{
  "contract_version": "1.0",
  "project": "Construction Payment Risk Intelligence",
  "component": "Payment Risk Prediction API",
  "model_version": "logistic-regression-v1",
  "api_version": "v1.0.0",
  "purpose": "Monitor serving reliability, input quality, prediction behavior, and model-risk signals for operational review prioritization.",
  "monitoring_principles": {
    "not_legal_decision_system": true,
    "human_review_required_for_high_impact_actions": true,
    "synthetic_model": true,
    "test_set_consumed": true,
    "monitoring_does_not_replace_model_revalidation": true
  },
  "service_health": {
    "metrics": [
      "request_count",
      "success_count",
      "client_error_count",
      "server_error_count",
      "latency_ms"
    ],
    "initial_alerts": {
      "server_error_rate": {
        "condition": "> 5%",
        "window": "5 minutes"
      },
      "p95_latency_ms": {
        "

In [125]:
required_sections = [
    "service_health",
    "input_quality",
    "prediction_behavior",
    "feature_monitoring",
    "model_performance_monitoring",
    "privacy_and_logging",
    "failure_response"
]

loaded = json.loads(
    output_path.read_text()
)

for section in required_sections:
    assert section in loaded
    print("PASS", section)

assert loaded["reference"] if "reference" in loaded else True

assert (
    loaded["prediction_behavior"]
    ["reference"]
    ["decision_threshold"]
    == 0.20
)

assert (
    loaded["privacy_and_logging"]
    ["log_raw_request_payloads"]
    is False
)

print()
print("Monitoring Contract v1: PASS")

PASS service_health
PASS input_quality
PASS prediction_behavior
PASS feature_monitoring
PASS model_performance_monitoring
PASS privacy_and_logging
PASS failure_response

Monitoring Contract v1: PASS


In [126]:
from app.observability import (
    RequestTimer,
    emit_structured_log,
    generate_request_id,
)

request_id = generate_request_id()
timer = RequestTimer()

emit_structured_log(
    event_type="test_observability",
    request_id=request_id,
    http_status=200,
    latency_ms=timer.elapsed_ms(),
    model_version="logistic-regression-v1",
    predicted_risk=0.42,
    model_decision="FLAGGED_BY_MODEL",
)

print("Request ID:", request_id)
print("Structured logging smoke test: PASS")

Request ID: 5c9e7011-985e-4fd7-99b0-a008e6e675e7
Structured logging smoke test: PASS


In [127]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

payload = {
    "state": "FL",
    "project_type": "commercial",
    "public_private": "private",
    "customer_role": "material_supplier",
    "hiring_party_type": "general_contractor",
    "payment_chain_completeness_score": 0.72,
    "research_confidence_score": 0.81,
    "deadline_days_remaining": 24,
    "prior_escalation_rate": 0.18,
    "prior_projects_with_hiring_party": 3,
    "expected_party_count": 5,
    "critical_field_missing": 0,
    "multiple_candidate_records": 1,
    "conflicting_project_information": 0,
}

response = client.post(
    "/v1/predict",
    json=payload,
)

print("HTTP status:", response.status_code)
print("X-Request-ID:", response.headers.get("X-Request-ID"))

body = response.json()

print("Risk:", body["predicted_operational_risk"])
print("Decision:", body["model_decision"])

assert response.status_code == 200
assert response.headers.get("X-Request-ID")
assert body["model_version"] == "logistic-regression-v1"

print()
print("End-to-end observability request test: PASS")

HTTP status: 200
X-Request-ID: None
Risk: 0.3319541555119854
Decision: FLAGGED_BY_MODEL


AssertionError: 

In [128]:
import importlib
import app.main

importlib.reload(app.main)

from fastapi.testclient import TestClient

client = TestClient(app.main.app)

print("FastAPI app reloaded: PASS")

FastAPI app reloaded: PASS


In [129]:
response = client.get("/health")

print("Status:", response.status_code)
print("Headers:", dict(response.headers))
print("X-Request-ID:", response.headers.get("X-Request-ID"))

assert response.status_code == 200
assert response.headers.get("X-Request-ID")

print()
print("Request-ID middleware test: PASS")

Status: 200
Headers: {'content-length': '15', 'content-type': 'application/json', 'x-request-id': '02524e07-662c-4f29-90bf-b674955ea82a'}
X-Request-ID: 02524e07-662c-4f29-90bf-b674955ea82a

Request-ID middleware test: PASS


In [130]:
response = client.post(
    "/v1/predict",
    json=payload,
)

print("HTTP status:", response.status_code)
print(
    "X-Request-ID:",
    response.headers.get("X-Request-ID")
)

body = response.json()

print(
    "Risk:",
    body["predicted_operational_risk"]
)
print(
    "Decision:",
    body["model_decision"]
)

assert response.status_code == 200

request_id = response.headers.get(
    "X-Request-ID"
)

assert request_id
assert body["model_version"] == (
    "logistic-regression-v1"
)

print()
print(
    "End-to-end observability request test: PASS"
)

HTTP status: 200
X-Request-ID: 0057143b-e20b-402b-9f4a-2589a9ff6d81
Risk: 0.3319541555119854
Decision: FLAGGED_BY_MODEL

End-to-end observability request test: PASS


In [134]:
import importlib
import app.metrics

importlib.reload(app.metrics)

print("Metrics module reloaded: PASS")
print("Namespace:", app.metrics.NAMESPACE)
print("Dimensions:", app.metrics.API_DIMENSIONS)

Metrics module reloaded: PASS
Namespace: ConstructionPaymentRisk/API
Dimensions: [{'Name': 'ModelVersion', 'Value': 'logistic-regression-v1'}, {'Name': 'Environment', 'Value': 'dev'}]


In [135]:
from app.metrics import (
    API_DIMENSIONS,
    emit_metric,
)

emit_metric(
    metric_name="RequestCount",
    value=1,
    unit="Count",
    dimensions=API_DIMENSIONS,
)

emit_metric(
    metric_name="LatencyMs",
    value=125.4,
    unit="Milliseconds",
    dimensions=API_DIMENSIONS,
)

emit_metric(
    metric_name="PredictionCount",
    value=1,
    unit="Count",
    dimensions=API_DIMENSIONS,
)

emit_metric(
    metric_name="FlaggedCount",
    value=1,
    unit="Count",
    dimensions=API_DIMENSIONS,
)

print("CloudWatch metric smoke test: PASS")

CloudWatch metric smoke test: PASS


In [137]:
import boto3

cw = boto3.client(
    "cloudwatch",
    region_name="ap-southeast-2",
)

response = cw.put_metric_data(
    Namespace="ConstructionPaymentRisk/API",
    MetricData=[
        {
            "MetricName": "VerificationMetric",
            "Value": 1.0,
            "Unit": "Count",
            "Dimensions": [
                {
                    "Name": "ModelVersion",
                    "Value": "logistic-regression-v1",
                },
                {
                    "Name": "Environment",
                    "Value": "dev",
                },
            ],
        }
    ],
)

print("HTTP status:",
      response["ResponseMetadata"]["HTTPStatusCode"])

print("Direct CloudWatch PutMetricData: PASS")

HTTP status: 200
Direct CloudWatch PutMetricData: PASS


# So the earlier empty list_metrics() result was just discovery/indexing delay, not a permissions issue.
Now verify that the datapoint is actually queryable. Wait about 1–3 minutes, then run:

In [138]:
from datetime import datetime, timedelta, timezone

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(minutes=10)

response = cw.get_metric_statistics(
    Namespace="ConstructionPaymentRisk/API",
    MetricName="VerificationMetric",
    Dimensions=[
        {
            "Name": "ModelVersion",
            "Value": "logistic-regression-v1",
        },
        {
            "Name": "Environment",
            "Value": "dev",
        },
    ],
    StartTime=start_time,
    EndTime=end_time,
    Period=60,
    Statistics=["Sum"],
)

print("CLOUDWATCH DATA VERIFICATION")
print("=" * 70)
print("Datapoints:", response["Datapoints"])

assert len(response["Datapoints"]) > 0

print()
print("CloudWatch datapoint verification: PASS")

CLOUDWATCH DATA VERIFICATION
Datapoints: [{'Timestamp': datetime.datetime(2026, 9, 11, 10, 19, tzinfo=tzlocal()), 'Sum': 1.0, 'Unit': 'Count'}]

CloudWatch datapoint verification: PASS


In [139]:
expected_metrics = [
    "RequestCount",
    "LatencyMs",
    "PredictionCount",
    "FlaggedCount",
]

print("REAL METRIC VERIFICATION")
print("=" * 70)

for metric_name in expected_metrics:
    response = cw.get_metric_statistics(
        Namespace="ConstructionPaymentRisk/API",
        MetricName=metric_name,
        Dimensions=[
            {
                "Name": "ModelVersion",
                "Value": "logistic-regression-v1",
            },
            {
                "Name": "Environment",
                "Value": "dev",
            },
        ],
        StartTime=start_time,
        EndTime=end_time,
        Period=60,
        Statistics=["Sum", "Average"],
    )

    datapoints = response["Datapoints"]

    print(metric_name, "->", datapoints)

    assert datapoints, (
        f"No datapoints found for {metric_name}"
    )

print()
print("CloudWatch real metric verification: PASS")

REAL METRIC VERIFICATION
RequestCount -> [{'Timestamp': datetime.datetime(2026, 9, 11, 10, 18, tzinfo=tzlocal()), 'Average': 1.0, 'Sum': 1.0, 'Unit': 'Count'}]
LatencyMs -> [{'Timestamp': datetime.datetime(2026, 9, 11, 10, 18, tzinfo=tzlocal()), 'Average': 125.4, 'Sum': 125.4, 'Unit': 'Milliseconds'}]
PredictionCount -> [{'Timestamp': datetime.datetime(2026, 9, 11, 10, 18, tzinfo=tzlocal()), 'Average': 1.0, 'Sum': 1.0, 'Unit': 'Count'}]
FlaggedCount -> [{'Timestamp': datetime.datetime(2026, 9, 11, 10, 18, tzinfo=tzlocal()), 'Average': 1.0, 'Sum': 1.0, 'Unit': 'Count'}]

CloudWatch real metric verification: PASS


In [141]:
import importlib
import app.main

importlib.reload(app.main)

from fastapi.testclient import TestClient

client = TestClient(app.main.app)

print("Instrumented FastAPI app reloaded: PASS")

Instrumented FastAPI app reloaded: PASS


In [142]:
responses = []

for i in range(3):
    response = client.post(
        "/v1/predict",
        json=payload,
    )

    responses.append(response)

    print(
        f"Request {i + 1}:",
        "status =", response.status_code,
        "| request_id =",
        response.headers.get("X-Request-ID"),
        "| decision =",
        response.json()["model_decision"],
    )

    assert response.status_code == 200
    assert response.headers.get("X-Request-ID")

print()
print("Instrumented API traffic test: PASS")

Request 1: status = 200 | request_id = f254118a-7216-4478-b77a-54bfc4dac097 | decision = FLAGGED_BY_MODEL
Request 2: status = 200 | request_id = cbe7e0e8-b81a-45fa-bd5c-3e007f64a1d5 | decision = FLAGGED_BY_MODEL
Request 3: status = 200 | request_id = 75cf6aff-952a-4c5e-8010-3124e5d49f92 | decision = FLAGGED_BY_MODEL

Instrumented API traffic test: PASS


In [143]:
from datetime import datetime, timedelta, timezone

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(minutes=10)

print("LIVE CLOUDWATCH API METRICS")
print("=" * 70)

for metric_name in [
    "RequestCount",
    "PredictionCount",
    "FlaggedCount",
    "LatencyMs",
]:
    result = cw.get_metric_statistics(
        Namespace="ConstructionPaymentRisk/API",
        MetricName=metric_name,
        Dimensions=API_DIMENSIONS,
        StartTime=start_time,
        EndTime=end_time,
        Period=60,
        Statistics=[
            "Sum",
            "Average",
            "Maximum",
        ],
    )

    print(f"\n{metric_name}")

    datapoints = sorted(
        result["Datapoints"],
        key=lambda x: x["Timestamp"],
    )

    for datapoint in datapoints:
        print(datapoint)

    assert datapoints, (
        f"No CloudWatch datapoints for {metric_name}"
    )

print()
print(
    "Live API metric integration verification: PASS"
)

LIVE CLOUDWATCH API METRICS

RequestCount
{'Timestamp': datetime.datetime(2026, 9, 11, 10, 18, tzinfo=tzlocal()), 'Average': 1.0, 'Sum': 1.0, 'Maximum': 1.0, 'Unit': 'Count'}

PredictionCount
{'Timestamp': datetime.datetime(2026, 9, 11, 10, 18, tzinfo=tzlocal()), 'Average': 1.0, 'Sum': 1.0, 'Maximum': 1.0, 'Unit': 'Count'}

FlaggedCount
{'Timestamp': datetime.datetime(2026, 9, 11, 10, 18, tzinfo=tzlocal()), 'Average': 1.0, 'Sum': 1.0, 'Maximum': 1.0, 'Unit': 'Count'}

LatencyMs
{'Timestamp': datetime.datetime(2026, 9, 11, 10, 18, tzinfo=tzlocal()), 'Average': 125.4, 'Sum': 125.4, 'Maximum': 125.4, 'Unit': 'Milliseconds'}

Live API metric integration verification: PASS


In [144]:
import time
from datetime import datetime, timedelta, timezone


def get_metric_value(
    metric_name,
    statistic,
    start_time,
    end_time,
):
    response = cw.get_metric_statistics(
        Namespace="ConstructionPaymentRisk/API",
        MetricName=metric_name,
        Dimensions=API_DIMENSIONS,
        StartTime=start_time,
        EndTime=end_time,
        Period=60,
        Statistics=[statistic],
    )

    datapoints = response["Datapoints"]

    return sum(
        point.get(statistic, 0)
        for point in datapoints
    )


# Fixed observation window.
window_start = (
    datetime.now(timezone.utc)
    - timedelta(hours=1)
)

baseline_end = datetime.now(timezone.utc)

baseline = {
    "RequestCount": get_metric_value(
        "RequestCount",
        "Sum",
        window_start,
        baseline_end,
    ),
    "PredictionCount": get_metric_value(
        "PredictionCount",
        "Sum",
        window_start,
        baseline_end,
    ),
    "FlaggedCount": get_metric_value(
        "FlaggedCount",
        "Sum",
        window_start,
        baseline_end,
    ),
    "LatencySamples": get_metric_value(
        "LatencyMs",
        "SampleCount",
        window_start,
        baseline_end,
    ),
}

print("BASELINE")
print(baseline)


# Generate fresh real API traffic.
print("\nGENERATING TRAFFIC")

for i in range(3):
    response = client.post(
        "/v1/predict",
        json=payload,
    )

    assert response.status_code == 200

    print(
        f"Request {i + 1}:",
        response.headers.get("X-Request-ID"),
        response.json()["model_decision"],
    )


print(
    "\nWaiting for CloudWatch ingestion..."
)

time.sleep(90)


verification_end = datetime.now(timezone.utc)

after = {
    "RequestCount": get_metric_value(
        "RequestCount",
        "Sum",
        window_start,
        verification_end,
    ),
    "PredictionCount": get_metric_value(
        "PredictionCount",
        "Sum",
        window_start,
        verification_end,
    ),
    "FlaggedCount": get_metric_value(
        "FlaggedCount",
        "Sum",
        window_start,
        verification_end,
    ),
    "LatencySamples": get_metric_value(
        "LatencyMs",
        "SampleCount",
        window_start,
        verification_end,
    ),
}

print("\nAFTER")
print(after)


delta = {
    key: after[key] - baseline[key]
    for key in baseline
}

print("\nDELTA")
print(delta)

BASELINE
{'RequestCount': 4.0, 'PredictionCount': 4.0, 'FlaggedCount': 4.0, 'LatencySamples': 4.0}

GENERATING TRAFFIC
Request 1: 3c73c780-eafb-4910-80de-bd32d0050c69 FLAGGED_BY_MODEL
Request 2: b3aa32db-7cdb-4870-b59a-335d90afc41c FLAGGED_BY_MODEL
Request 3: ec82879a-ba9e-4f57-aa46-05e9604cab60 FLAGGED_BY_MODEL

Waiting for CloudWatch ingestion...

AFTER
{'RequestCount': 7.0, 'PredictionCount': 7.0, 'FlaggedCount': 7.0, 'LatencySamples': 7.0}

DELTA
{'RequestCount': 3.0, 'PredictionCount': 3.0, 'FlaggedCount': 3.0, 'LatencySamples': 3.0}


In [145]:
assert delta["RequestCount"] >= 3
assert delta["PredictionCount"] >= 3
assert delta["FlaggedCount"] >= 3
assert delta["LatencySamples"] >= 3

print()
print(
    "Live FastAPI -> CloudWatch metric integration: PASS"
)


Live FastAPI -> CloudWatch metric integration: PASS


In [146]:
import importlib
import app.main

importlib.reload(app.main)

from fastapi.testclient import TestClient

client = TestClient(app.main.app)

print("Validation monitoring app reload: PASS")

Validation monitoring app reload: PASS


In [147]:
invalid_payload = payload.copy()

invalid_payload.pop(
    "research_confidence_score"
)

response = client.post(
    "/v1/predict",
    json=invalid_payload,
)

print("HTTP status:", response.status_code)
print(
    "X-Request-ID:",
    response.headers.get("X-Request-ID")
)

body = response.json()

print(
    "Response request_id:",
    body.get("request_id")
)

print(
    "Validation errors:",
    len(body["detail"])
)

assert response.status_code == 422
assert response.headers.get("X-Request-ID")
assert body.get("request_id")
assert len(body["detail"]) >= 1

print()
print(
    "Validation monitoring HTTP test: PASS"
)

HTTP status: 422
X-Request-ID: c3227a8d-2f39-48c3-ad22-fb7aa37154e2
Response request_id: None
Validation errors: 1


AssertionError: 

In [148]:
import app.main

print("STATUS:", response.status_code)
print("HEADERS:", dict(response.headers))
print("BODY:")
print(response.json())

print()
print("REGISTERED EXCEPTION HANDLERS")
print("=" * 70)

for exc_type, handler in app.main.app.exception_handlers.items():
    print(exc_type, "->", handler)

STATUS: 422
HEADERS: {'content-length': '506', 'content-type': 'application/json', 'x-request-id': 'c3227a8d-2f39-48c3-ad22-fb7aa37154e2'}
BODY:
{'detail': [{'type': 'missing', 'loc': ['body', 'research_confidence_score'], 'msg': 'Field required', 'input': {'state': 'FL', 'project_type': 'commercial', 'public_private': 'private', 'customer_role': 'material_supplier', 'hiring_party_type': 'general_contractor', 'payment_chain_completeness_score': 0.72, 'deadline_days_remaining': 24, 'prior_escalation_rate': 0.18, 'prior_projects_with_hiring_party': 3, 'expected_party_count': 5, 'critical_field_missing': 0, 'multiple_candidate_records': 1, 'conflicting_project_information': 0}}]}

REGISTERED EXCEPTION HANDLERS
<class 'starlette.exceptions.HTTPException'> -> <function http_exception_handler at 0x7f51c98a6200>
<class 'fastapi.exceptions.RequestValidationError'> -> <function request_validation_exception_handler at 0x7f51c98ea3e0>
<class 'fastapi.exceptions.WebSocketRequestValidationError'> -

In [150]:
import importlib
import app.main

importlib.reload(app.main)

from fastapi.testclient import TestClient

client = TestClient(app.main.app)

print("Validation handler app reloaded: PASS")

Validation handler app reloaded: PASS


In [151]:
invalid_payload = payload.copy()
invalid_payload.pop("research_confidence_score")

response = client.post(
    "/v1/predict",
    json=invalid_payload,
)

print("HTTP status:", response.status_code)
print(
    "X-Request-ID:",
    response.headers.get("X-Request-ID")
)

body = response.json()

print(
    "Response request_id:",
    body.get("request_id")
)

print(
    "Validation errors:",
    len(body["detail"])
)

assert response.status_code == 422
assert response.headers.get("X-Request-ID")
assert body.get("request_id")

assert (
    response.headers.get("X-Request-ID")
    == body.get("request_id")
)

assert len(body["detail"]) >= 1

print()
print("Validation monitoring HTTP test: PASS")

HTTP status: 422
X-Request-ID: d9883c11-7cf7-4b91-a1c3-24c19b81907a
Response request_id: d9883c11-7cf7-4b91-a1c3-24c19b81907a
Validation errors: 1

Validation monitoring HTTP test: PASS


# Now we need to prove the CloudWatch metrics are actually emitted for validation failures.

In [152]:
import time
from datetime import datetime, timedelta, timezone


def metric_sum(metric_name, start_time, end_time):
    response = cw.get_metric_statistics(
        Namespace="ConstructionPaymentRisk/API",
        MetricName=metric_name,
        Dimensions=API_DIMENSIONS,
        StartTime=start_time,
        EndTime=end_time,
        Period=60,
        Statistics=["Sum"],
    )

    return sum(
        point.get("Sum", 0)
        for point in response["Datapoints"]
    )


window_start = (
    datetime.now(timezone.utc)
    - timedelta(hours=1)
)

baseline_end = datetime.now(timezone.utc)

baseline = {
    "ClientErrorCount": metric_sum(
        "ClientErrorCount",
        window_start,
        baseline_end,
    ),
    "ValidationFailureCount": metric_sum(
        "ValidationFailureCount",
        window_start,
        baseline_end,
    ),
    "MissingRequiredFieldCount": metric_sum(
        "MissingRequiredFieldCount",
        window_start,
        baseline_end,
    ),
}

print("BASELINE")
print(baseline)


# Generate three validation failures.
for i in range(3):
    bad_payload = payload.copy()
    bad_payload.pop("research_confidence_score")

    response = client.post(
        "/v1/predict",
        json=bad_payload,
    )

    assert response.status_code == 422

    print(
        f"Invalid request {i + 1}:",
        response.headers.get("X-Request-ID"),
    )


print("\nWaiting for CloudWatch ingestion...")
time.sleep(90)


verification_end = datetime.now(timezone.utc)

after = {
    "ClientErrorCount": metric_sum(
        "ClientErrorCount",
        window_start,
        verification_end,
    ),
    "ValidationFailureCount": metric_sum(
        "ValidationFailureCount",
        window_start,
        verification_end,
    ),
    "MissingRequiredFieldCount": metric_sum(
        "MissingRequiredFieldCount",
        window_start,
        verification_end,
    ),
}

print("\nAFTER")
print(after)

delta = {
    key: after[key] - baseline[key]
    for key in baseline
}

print("\nDELTA")
print(delta)

BASELINE
{'ClientErrorCount': 1.0, 'ValidationFailureCount': 1.0, 'MissingRequiredFieldCount': 1.0}
Invalid request 1: de91f1c8-1053-417d-a80b-2f44d31b946a
Invalid request 2: 5af8dbc9-fae4-42c9-bb64-9984dc869033
Invalid request 3: 1924fab1-df2d-406a-8426-3c7017b5cdf3

Waiting for CloudWatch ingestion...

AFTER
{'ClientErrorCount': 4.0, 'ValidationFailureCount': 4.0, 'MissingRequiredFieldCount': 4.0}

DELTA
{'ClientErrorCount': 3.0, 'ValidationFailureCount': 3.0, 'MissingRequiredFieldCount': 3.0}


In [154]:
assert delta["ClientErrorCount"] >= 3
assert delta["ValidationFailureCount"] >= 3
assert delta["MissingRequiredFieldCount"] >= 3

print()
print(
    "Validation monitoring CloudWatch integration: PASS"
)


Validation monitoring CloudWatch integration: PASS


# Prediction Distribution Monitoring.

Low:    risk < 0.20
Medium: 0.20 <= risk < 0.50
High:   risk >= 0.50

In [155]:
import importlib
import app.main

importlib.reload(app.main)

from fastapi.testclient import TestClient

client = TestClient(app.main.app)

print(
    "Prediction distribution monitoring app reload: PASS"
)

Prediction distribution monitoring app reload: PASS


In [156]:
response = client.post(
    "/v1/predict",
    json=payload,
)

assert response.status_code == 200

body = response.json()

risk = body[
    "predicted_operational_risk"
]

print("Risk:", risk)
print(
    "Decision:",
    body["model_decision"]
)

if risk < 0.20:
    band = "LOW"

elif risk < 0.50:
    band = "MEDIUM"

else:
    band = "HIGH"

print("Monitoring band:", band)

print()
print(
    "Prediction distribution HTTP test: PASS"
)

Risk: 0.3319541555119854
Decision: FLAGGED_BY_MODEL
Monitoring band: MEDIUM

Prediction distribution HTTP test: PASS


Now we need the stronger proof: generate low, medium, and high-risk traffic and verify CloudWatch receives the corresponding distribution metrics.\


Use this first to create three representative payloads. Keep your known medium case as-is, then vary only a few already-approved model features:

In [157]:
low_payload = payload.copy()
low_payload.update({
    "multiple_candidate_records": 0,
    "critical_field_missing": 0,
    "conflicting_project_information": 0,
    "payment_chain_completeness_score": 0.95,
    "research_confidence_score": 0.95,
    "prior_escalation_rate": 0.02,
    "deadline_days_remaining": 90,
})

medium_payload = payload.copy()

high_payload = payload.copy()
high_payload.update({
    "multiple_candidate_records": 1,
    "critical_field_missing": 1,
    "conflicting_project_information": 1,
    "payment_chain_completeness_score": 0.20,
    "research_confidence_score": 0.20,
    "prior_escalation_rate": 0.80,
    "deadline_days_remaining": 2,
})

In [158]:
test_cases = {
    "low_candidate": low_payload,
    "medium_candidate": medium_payload,
    "high_candidate": high_payload,
}

observed = {}

for name, case_payload in test_cases.items():
    response = client.post(
        "/v1/predict",
        json=case_payload,
    )

    assert response.status_code == 200

    body = response.json()
    risk = body["predicted_operational_risk"]

    if risk < 0.20:
        band = "LOW"
    elif risk < 0.50:
        band = "MEDIUM"
    else:
        band = "HIGH"

    observed[name] = {
        "risk": risk,
        "band": band,
        "decision": body["model_decision"],
    }

    print(
        name,
        "| risk =", round(risk, 6),
        "| band =", band,
        "| decision =", body["model_decision"],
    )

print()
print("Distribution case generation: PASS")

low_candidate | risk = 0.06963 | band = LOW | decision = NOT_FLAGGED_BY_MODEL
medium_candidate | risk = 0.331954 | band = MEDIUM | decision = FLAGGED_BY_MODEL
high_candidate | risk = 0.955042 | band = HIGH | decision = FLAGGED_BY_MODEL

Distribution case generation: PASS


In [159]:
import time
from datetime import datetime, timedelta, timezone


def metric_sum(metric_name, start_time, end_time):
    response = cw.get_metric_statistics(
        Namespace="ConstructionPaymentRisk/API",
        MetricName=metric_name,
        Dimensions=API_DIMENSIONS,
        StartTime=start_time,
        EndTime=end_time,
        Period=60,
        Statistics=["Sum"],
    )

    return sum(
        point.get("Sum", 0)
        for point in response["Datapoints"]
    )


def metric_sample_count(metric_name, start_time, end_time):
    response = cw.get_metric_statistics(
        Namespace="ConstructionPaymentRisk/API",
        MetricName=metric_name,
        Dimensions=API_DIMENSIONS,
        StartTime=start_time,
        EndTime=end_time,
        Period=60,
        Statistics=["SampleCount"],
    )

    return sum(
        point.get("SampleCount", 0)
        for point in response["Datapoints"]
    )


window_start = (
    datetime.now(timezone.utc)
    - timedelta(hours=1)
)

baseline_end = datetime.now(timezone.utc)

baseline = {
    "PredictedRiskSamples": metric_sample_count(
        "PredictedRisk",
        window_start,
        baseline_end,
    ),
    "LowRiskCount": metric_sum(
        "LowRiskCount",
        window_start,
        baseline_end,
    ),
    "MediumRiskCount": metric_sum(
        "MediumRiskCount",
        window_start,
        baseline_end,
    ),
    "HighRiskCount": metric_sum(
        "HighRiskCount",
        window_start,
        baseline_end,
    ),
}

print("BASELINE")
print(baseline)

BASELINE
{'PredictedRiskSamples': 4.0, 'LowRiskCount': 1.0, 'MediumRiskCount': 2.0, 'HighRiskCount': 1.0}


In [160]:
cases = [
    ("LOW", low_payload),
    ("MEDIUM", medium_payload),
    ("HIGH", high_payload),
]

for expected_band, case_payload in cases:
    response = client.post(
        "/v1/predict",
        json=case_payload,
    )

    assert response.status_code == 200

    risk = response.json()[
        "predicted_operational_risk"
    ]

    if risk < 0.20:
        observed_band = "LOW"
    elif risk < 0.50:
        observed_band = "MEDIUM"
    else:
        observed_band = "HIGH"

    print(
        expected_band,
        "->",
        round(risk, 6),
        observed_band,
    )

    assert observed_band == expected_band

print()
print("Three-band live traffic generation: PASS")

LOW -> 0.06963 LOW
MEDIUM -> 0.331954 MEDIUM
HIGH -> 0.955042 HIGH

Three-band live traffic generation: PASS


In [161]:
print("Waiting for CloudWatch ingestion...")
time.sleep(90)

Waiting for CloudWatch ingestion...


In [162]:
verification_end = datetime.now(timezone.utc)

after = {
    "PredictedRiskSamples": metric_sample_count(
        "PredictedRisk",
        window_start,
        verification_end,
    ),
    "LowRiskCount": metric_sum(
        "LowRiskCount",
        window_start,
        verification_end,
    ),
    "MediumRiskCount": metric_sum(
        "MediumRiskCount",
        window_start,
        verification_end,
    ),
    "HighRiskCount": metric_sum(
        "HighRiskCount",
        window_start,
        verification_end,
    ),
}

print("AFTER")
print(after)

delta = {
    key: after[key] - baseline[key]
    for key in baseline
}

print()
print("DELTA")
print(delta)

assert delta["PredictedRiskSamples"] >= 3
assert delta["LowRiskCount"] >= 1
assert delta["MediumRiskCount"] >= 1
assert delta["HighRiskCount"] >= 1

print()
print(
    "Prediction distribution CloudWatch integration: PASS"
)

AFTER
{'PredictedRiskSamples': 7.0, 'LowRiskCount': 2.0, 'MediumRiskCount': 3.0, 'HighRiskCount': 2.0}

DELTA
{'PredictedRiskSamples': 3.0, 'LowRiskCount': 1.0, 'MediumRiskCount': 1.0, 'HighRiskCount': 1.0}

Prediction distribution CloudWatch integration: PASS


In [163]:
from pathlib import Path
import json

reference = {
    "reference_version": "1.0",
    "model_version": "logistic-regression-v1",
    "source": "training_split",
    "numeric_features": {},
    "binary_features": {},
    "categorical_features": {},
}

numeric_features = [
    "payment_chain_completeness_score",
    "research_confidence_score",
    "deadline_days_remaining",
    "prior_escalation_rate",
    "prior_projects_with_hiring_party",
    "expected_party_count",
]

binary_features = [
    "critical_field_missing",
    "multiple_candidate_records",
    "conflicting_project_information",
]

categorical_features = [
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type",
]

# Numeric reference statistics
for col in numeric_features:
    reference["numeric_features"][col] = {
        "mean": float(X_train[col].mean()),
        "std": float(X_train[col].std()),
        "min": float(X_train[col].min()),
        "max": float(X_train[col].max()),
    }

# Binary reference rates
for col in binary_features:
    reference["binary_features"][col] = {
        "positive_rate": float(X_train[col].mean())
    }

# Categorical reference frequencies
for col in categorical_features:
    frequencies = (
        X_train[col]
        .value_counts(normalize=True)
        .sort_index()
        .to_dict()
    )

    reference["categorical_features"][col] = {
        str(category): float(frequency)
        for category, frequency in frequencies.items()
    }

output_path = Path(
    "artifacts/monitoring/payment-risk-api/v1/"
    "feature-reference-v1.json"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

output_path.write_text(
    json.dumps(reference, indent=2)
)

print("Created:", output_path)
print("Feature reference creation: PASS")

Created: artifacts/monitoring/payment-risk-api/v1/feature-reference-v1.json
Feature reference creation: PASS


In [164]:
loaded_reference = json.loads(
    output_path.read_text()
)

assert loaded_reference["model_version"] == "logistic-regression-v1"

assert len(
    loaded_reference["numeric_features"]
) == 6

assert len(
    loaded_reference["binary_features"]
) == 3

assert len(
    loaded_reference["categorical_features"]
) == 5

print("Numeric features:",
      len(loaded_reference["numeric_features"]))

print("Binary features:",
      len(loaded_reference["binary_features"]))

print("Categorical features:",
      len(loaded_reference["categorical_features"]))

print()
print("Feature reference validation: PASS")

Numeric features: 6
Binary features: 3
Categorical features: 5

Feature reference validation: PASS


Build numeric PSI drift detection
We'll start with numeric features. A common monitoring metric is Population Stability Index (PSI):
\[
PSI = \sum (P_i-Q_i)\ln(P_i/Q_i)
\]where \(P_i\) is the reference proportion and \(Q_i\) is the current production proportion within each bin.
Typical operational interpretation:

In [165]:
import numpy as np
import json

for col in numeric_features:
    series = X_train[col].dropna().astype(float)

    # Decile boundaries from training/reference population
    raw_edges = np.quantile(
        series,
        np.linspace(0, 1, 11)
    )

    # Remove duplicate boundaries
    edges = np.unique(raw_edges)

    # Ensure future values outside training range still enter a bin
    edges[0] = -np.inf
    edges[-1] = np.inf

    counts, _ = np.histogram(
        series,
        bins=edges
    )

    proportions = counts / counts.sum()

    reference["numeric_features"][col] = {
        "mean": float(series.mean()),
        "std": float(series.std()),
        "min": float(series.min()),
        "max": float(series.max()),
        "bin_edges": [
            None if np.isneginf(x) else
            None if np.isposinf(x) else
            float(x)
            for x in edges
        ],
        "bin_proportions": [
            float(x)
            for x in proportions
        ],
    }

output_path.write_text(
    json.dumps(reference, indent=2)
)

print("Numeric PSI reference bins added: PASS")

Numeric PSI reference bins added: PASS


In [166]:
loaded_reference = json.loads(
    output_path.read_text()
)

for feature in numeric_features:

    profile = loaded_reference[
        "numeric_features"
    ][feature]

    assert "bin_edges" in profile
    assert "bin_proportions" in profile

    assert (
        len(profile["bin_edges"])
        ==
        len(profile["bin_proportions"]) + 1
    )

    assert abs(
        sum(profile["bin_proportions"]) - 1.0
    ) < 1e-6

    print(
        feature,
        "| bins =",
        len(profile["bin_proportions"]),
        "| proportion sum =",
        round(
            sum(profile["bin_proportions"]),
            6
        )
    )

print()
print("Numeric PSI reference validation: PASS")

payment_chain_completeness_score | bins = 6 | proportion sum = 1.0
research_confidence_score | bins = 10 | proportion sum = 1.0
deadline_days_remaining | bins = 10 | proportion sum = 1.0
prior_escalation_rate | bins = 5 | proportion sum = 1.0
prior_projects_with_hiring_party | bins = 8 | proportion sum = 1.0
expected_party_count | bins = 3 | proportion sum = 1.0

Numeric PSI reference validation: PASS


Create reusable PSI calculation

In [167]:
import numpy as np


def calculate_psi(
    current_values,
    reference_profile,
    epsilon=1e-6,
):
    """
    Compare a current numeric feature distribution against
    the stored training/reference distribution.

    Returns:
        psi_score
        drift_status
        current_proportions
    """

    current = np.asarray(
        current_values,
        dtype=float,
    )

    current = current[
        ~np.isnan(current)
    ]

    if len(current) == 0:
        raise ValueError(
            "Current population contains no valid numeric values."
        )

    # Restore JSON-safe None boundaries back to infinities
    stored_edges = reference_profile["bin_edges"]

    edges = np.array([
        -np.inf
        if i == 0 and value is None
        else np.inf
        if i == len(stored_edges) - 1 and value is None
        else float(value)
        for i, value in enumerate(stored_edges)
    ])

    reference_proportions = np.asarray(
        reference_profile["bin_proportions"],
        dtype=float,
    )

    current_counts, _ = np.histogram(
        current,
        bins=edges,
    )

    current_proportions = (
        current_counts / current_counts.sum()
    )

    # Prevent division/log(0)
    ref_safe = np.clip(
        reference_proportions,
        epsilon,
        None,
    )

    current_safe = np.clip(
        current_proportions,
        epsilon,
        None,
    )

    psi_components = (
        current_safe - ref_safe
    ) * np.log(
        current_safe / ref_safe
    )

    psi_score = float(
        np.sum(psi_components)
    )

    if psi_score < 0.10:
        drift_status = "STABLE"

    elif psi_score < 0.25:
        drift_status = "INVESTIGATE"

    else:
        drift_status = "SIGNIFICANT_DRIFT"

    return {
        "psi": psi_score,
        "status": drift_status,
        "current_proportions": [
            float(x)
            for x in current_proportions
        ],
    }


print("PSI function creation: PASS")

PSI function creation: PASS


In [168]:
self_check_results = {}

for feature in numeric_features:

    profile = loaded_reference[
        "numeric_features"
    ][feature]

    result = calculate_psi(
        X_train[feature],
        profile,
    )

    self_check_results[feature] = result

    print(
        feature,
        "| PSI =",
        round(result["psi"], 8),
        "|",
        result["status"],
    )

print()

assert all(
    result["psi"] < 1e-6
    for result in self_check_results.values()
)

print(
    "Training-reference PSI self-check: PASS"
)

payment_chain_completeness_score | PSI = 0.0 | STABLE
research_confidence_score | PSI = 0.0 | STABLE
deadline_days_remaining | PSI = 0.0 | STABLE
prior_escalation_rate | PSI = 0.0 | STABLE
prior_projects_with_hiring_party | PSI = 0.0 | STABLE
expected_party_count | PSI = 0.0 | STABLE

Training-reference PSI self-check: PASS


Then test a realistic unchanged population
Testing training against itself only validates our implementation. It doesn't prove PSI behaves properly with a different sample.
We'll therefore compare the validation population against the training reference:

In [176]:
validation_psi_results = {}

for feature in numeric_features:

    result = calculate_psi(
        X_val[feature],
        loaded_reference[
            "numeric_features"
        ][feature],
    )

    validation_psi_results[feature] = result

    print(
        feature,
        "| PSI =",
        round(result["psi"], 4),
        "|",
        result["status"],
    )

print()
print(
    "Validation population PSI test: PASS"
)

payment_chain_completeness_score | PSI = 0.0006 | STABLE
research_confidence_score | PSI = 0.0033 | STABLE
deadline_days_remaining | PSI = 0.008 | STABLE
prior_escalation_rate | PSI = 0.0004 | STABLE
prior_projects_with_hiring_party | PSI = 0.0056 | STABLE
expected_party_count | PSI = 0.0016 | STABLE

Validation population PSI test: PASS


In [171]:
candidate_names = [
    "X_train",
    "X_val",
    "X_test",
    "train_df",
    "val_df",
    "test_df",
    "train",
    "val",
    "test",
    "df",
    "curated_df",
]

print("Variables currently available:")
print()

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        try:
            print(
                f"{name:15} | shape = {obj.shape}"
            )
        except Exception:
            print(
                f"{name:15} | exists"
            )

Variables currently available:

X_train         | shape = (6892, 14)
X_test          | shape = (1479, 14)
train_df        | shape = (6892, 45)
test_df         | shape = (1479, 45)
df              | shape = (10000, 45)


In [172]:
print(
    "X_train exists:",
    "X_train" in globals()
)

print(
    "X_val exists:",
    "X_val" in globals()
)

print(
    "calculate_psi exists:",
    "calculate_psi" in globals()
)

print(
    "loaded_reference exists:",
    "loaded_reference" in globals()
)

X_train exists: True
X_val exists: False
calculate_psi exists: True
loaded_reference exists: True


In [173]:
import pandas as pd

df["assessment_date"] = pd.to_datetime(
    df["assessment_date"]
)

val_df = df[
    (df["assessment_date"] >= "2026-03-01")
    &
    (df["assessment_date"] < "2026-06-01")
].copy()

feature_columns = [
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type",
    "payment_chain_completeness_score",
    "research_confidence_score",
    "deadline_days_remaining",
    "prior_escalation_rate",
    "prior_projects_with_hiring_party",
    "expected_party_count",
    "critical_field_missing",
    "multiple_candidate_records",
    "conflicting_project_information",
]

X_val = val_df[
    feature_columns
].copy()

print(
    "val_df shape:",
    val_df.shape
)

print(
    "X_val shape:",
    X_val.shape
)

print(
    "Validation date range:",
    val_df["assessment_date"].min(),
    "->",
    val_df["assessment_date"].max()
)

print()
print(
    "Temporal validation reconstruction: PASS"
)

val_df shape: (1581, 45)
X_val shape: (1581, 14)
Validation date range: 2026-03-01 00:00:00 -> 2026-05-31 00:00:00

Temporal validation reconstruction: PASS


In [174]:
assert val_df.shape[0] == 1581
assert X_val.shape == (1581, 14)

assert (
    val_df["assessment_date"].min()
    >= pd.Timestamp("2026-03-01")
)

assert (
    val_df["assessment_date"].max()
    < pd.Timestamp("2026-06-01")
)

assert list(X_val.columns) == feature_columns

print(
    "Validation split integrity: PASS"
)

Validation split integrity: PASS


In [175]:
validation_psi_results = {}

for feature in numeric_features:

    result = calculate_psi(
        X_val[feature],
        loaded_reference[
            "numeric_features"
        ][feature],
    )

    validation_psi_results[
        feature
    ] = result

    print(
        feature,
        "| PSI =",
        round(result["psi"], 4),
        "|",
        result["status"],
    )

print()
print(
    "Validation population PSI test: PASS"
)

payment_chain_completeness_score | PSI = 0.0006 | STABLE
research_confidence_score | PSI = 0.0033 | STABLE
deadline_days_remaining | PSI = 0.008 | STABLE
prior_escalation_rate | PSI = 0.0004 | STABLE
prior_projects_with_hiring_party | PSI = 0.0056 | STABLE
expected_party_count | PSI = 0.0016 | STABLE

Validation population PSI test: PASS


# Deliberately create numeric drift
We'll create a copy of the validation population and modify only the six monitored numeric features. This is purely a monitoring test; we're not changing the model dataset.

In [177]:
drifted_val = X_val.copy()

drifted_val[
    "payment_chain_completeness_score"
] = 0.15

drifted_val[
    "research_confidence_score"
] = 0.20

drifted_val[
    "deadline_days_remaining"
] = 2

drifted_val[
    "prior_escalation_rate"
] = 0.85

drifted_val[
    "prior_projects_with_hiring_party"
] = 0

drifted_val[
    "expected_party_count"
] = 8

print(
    "Synthetic drift population:",
    drifted_val.shape
)

print(
    "Synthetic drift population creation: PASS"
)

Synthetic drift population: (1581, 14)
Synthetic drift population creation: PASS


In [178]:
drift_test_results = {}

for feature in numeric_features:

    result = calculate_psi(
        drifted_val[feature],
        loaded_reference[
            "numeric_features"
        ][feature],
    )

    drift_test_results[feature] = result

    print(
        feature,
        "| PSI =",
        round(result["psi"], 4),
        "|",
        result["status"],
    )

print()
print(
    "Synthetic numeric drift test executed: PASS"
)

payment_chain_completeness_score | PSI = 13.7829 | SIGNIFICANT_DRIFT
research_confidence_score | PSI = 12.4347 | SIGNIFICANT_DRIFT
deadline_days_remaining | PSI = 12.1416 | SIGNIFICANT_DRIFT
prior_escalation_rate | PSI = 12.352 | SIGNIFICANT_DRIFT
prior_projects_with_hiring_party | PSI = 11.1069 | SIGNIFICANT_DRIFT
expected_party_count | PSI = 6.5355 | SIGNIFICANT_DRIFT

Synthetic numeric drift test executed: PASS


In [179]:
significant_count = sum(
    result["status"] == "SIGNIFICANT_DRIFT"
    for result in drift_test_results.values()
)

print(
    "Significant drift features:",
    significant_count,
    "/",
    len(numeric_features)
)

assert significant_count >= 5

print()
print(
    "Numeric PSI drift detection acceptance test: PASS"
)

Significant drift features: 6 / 6

Numeric PSI drift detection acceptance test: PASS


# Binary feature drift

For binary variables, an interpretable metric is the absolute positive-rate shift:
\[
RateShift = |ProductionRate - ReferenceRate|
\]For our v1 monitoring policy, we'll use:

In [180]:
def calculate_binary_drift(
    current_values,
    reference_profile,
):
    current = np.asarray(
        current_values,
        dtype=float,
    )

    current = current[
        ~np.isnan(current)
    ]

    if len(current) == 0:
        raise ValueError(
            "Current population contains no valid binary values."
        )

    current_rate = float(
        np.mean(current)
    )

    reference_rate = float(
        reference_profile["positive_rate"]
    )

    absolute_shift = abs(
        current_rate - reference_rate
    )

    if absolute_shift < 0.10:
        status = "STABLE"

    elif absolute_shift < 0.20:
        status = "INVESTIGATE"

    else:
        status = "SIGNIFICANT_DRIFT"

    return {
        "reference_rate": reference_rate,
        "current_rate": current_rate,
        "absolute_shift": absolute_shift,
        "status": status,
    }


print(
    "Binary drift function creation: PASS"
)

Binary drift function creation: PASS


In [181]:
validation_binary_results = {}

for feature in binary_features:

    result = calculate_binary_drift(
        X_val[feature],
        loaded_reference[
            "binary_features"
        ][feature],
    )

    validation_binary_results[
        feature
    ] = result

    print(
        feature,
        "| reference =",
        round(result["reference_rate"], 4),
        "| validation =",
        round(result["current_rate"], 4),
        "| shift =",
        round(result["absolute_shift"], 4),
        "|",
        result["status"],
    )

print()
print(
    "Validation binary drift test: PASS"
)

critical_field_missing | reference = 0.1811 | validation = 0.1809 | shift = 0.0002 | STABLE
multiple_candidate_records | reference = 0.1608 | validation = 0.1594 | shift = 0.0014 | STABLE
conflicting_project_information | reference = 0.1132 | validation = 0.1151 | shift = 0.0019 | STABLE

Validation binary drift test: PASS


In [182]:
drifted_binary = X_val.copy()

for feature in binary_features:

    reference_rate = loaded_reference[
        "binary_features"
    ][feature]["positive_rate"]

    # Force population toward the opposite side
    # of its reference distribution.
    drifted_binary[feature] = (
        1
        if reference_rate < 0.50
        else 0
    )

print(
    "Synthetic binary drift population creation: PASS"
)

Synthetic binary drift population creation: PASS


In [183]:
synthetic_binary_results = {}

for feature in binary_features:

    result = calculate_binary_drift(
        drifted_binary[feature],
        loaded_reference[
            "binary_features"
        ][feature],
    )

    synthetic_binary_results[
        feature
    ] = result

    print(
        feature,
        "| reference =",
        round(result["reference_rate"], 4),
        "| synthetic =",
        round(result["current_rate"], 4),
        "| shift =",
        round(result["absolute_shift"], 4),
        "|",
        result["status"],
    )

print()

significant_binary_count = sum(
    result["status"] == "SIGNIFICANT_DRIFT"
    for result
    in synthetic_binary_results.values()
)

print(
    "Significant binary drift features:",
    significant_binary_count,
    "/",
    len(binary_features)
)

assert significant_binary_count == 3

print()
print(
    "Binary drift detection acceptance test: PASS"
)

critical_field_missing | reference = 0.1811 | synthetic = 1.0 | shift = 0.8189 | SIGNIFICANT_DRIFT
multiple_candidate_records | reference = 0.1608 | synthetic = 1.0 | shift = 0.8392 | SIGNIFICANT_DRIFT
conflicting_project_information | reference = 0.1132 | synthetic = 1.0 | shift = 0.8868 | SIGNIFICANT_DRIFT

Significant binary drift features: 3 / 3

Binary drift detection acceptance test: PASS


# Categorical drift detection

Distribution shift: known categories are arriving in very different proportions.
Unseen category: production sends a value that never existed in the training contract. Our API already rejects unsupported categories, but the drift layer should still be capable of identifying them when evaluating batches.
We'll use Total Variation Distance (TVD):
\[
TVD = \frac{1}{2}\sum_i |P_i-Q_i|
\]

In [184]:
def calculate_categorical_drift(
    current_values,
    reference_profile,
):
    current = (
        current_values
        .dropna()
        .astype(str)
    )

    if len(current) == 0:
        raise ValueError(
            "Current population contains no valid categorical values."
        )

    reference_categories = set(
        reference_profile.keys()
    )

    current_frequencies = (
        current
        .value_counts(normalize=True)
        .to_dict()
    )

    current_categories = set(
        current_frequencies.keys()
    )

    unseen_categories = sorted(
        current_categories
        - reference_categories
    )

    all_categories = (
        reference_categories
        | current_categories
    )

    total_difference = 0.0

    for category in all_categories:

        reference_frequency = float(
            reference_profile.get(
                category,
                0.0,
            )
        )

        current_frequency = float(
            current_frequencies.get(
                category,
                0.0,
            )
        )

        total_difference += abs(
            reference_frequency
            - current_frequency
        )

    tvd = 0.5 * total_difference

    if tvd < 0.10:
        status = "STABLE"

    elif tvd < 0.20:
        status = "INVESTIGATE"

    else:
        status = "SIGNIFICANT_DRIFT"

    return {
        "tvd": float(tvd),
        "status": status,
        "unseen_categories": unseen_categories,
        "unseen_category_count": len(
            unseen_categories
        ),
        "current_frequencies": {
            str(k): float(v)
            for k, v
            in current_frequencies.items()
        },
    }


print(
    "Categorical drift function creation: PASS"
)

Categorical drift function creation: PASS


In [185]:
validation_categorical_results = {}

for feature in categorical_features:

    result = calculate_categorical_drift(
        X_val[feature],
        loaded_reference[
            "categorical_features"
        ][feature],
    )

    validation_categorical_results[
        feature
    ] = result

    print(
        feature,
        "| TVD =",
        round(result["tvd"], 4),
        "| unseen =",
        result["unseen_category_count"],
        "|",
        result["status"],
    )

print()
print(
    "Validation categorical drift test: PASS"
)

state | TVD = 0.0311 | unseen = 0 | STABLE
project_type | TVD = 0.0156 | unseen = 0 | STABLE
public_private | TVD = 0.0037 | unseen = 0 | STABLE
customer_role | TVD = 0.0255 | unseen = 0 | STABLE
hiring_party_type | TVD = 0.0229 | unseen = 0 | STABLE

Validation categorical drift test: PASS


In [186]:
drifted_categorical = X_val.copy()

selected_categories = {}

for feature in categorical_features:

    reference_profile = loaded_reference[
        "categorical_features"
    ][feature]

    # Choose the least common legitimate training category.
    selected_category = min(
        reference_profile,
        key=reference_profile.get,
    )

    selected_categories[
        feature
    ] = selected_category

    drifted_categorical[
        feature
    ] = selected_category


print("Selected valid categories:")

for feature, category in selected_categories.items():

    print(
        feature,
        "->",
        category,
    )

print()
print(
    "Synthetic categorical drift population creation: PASS"
)

Selected valid categories:
state -> OTHER
project_type -> other
public_private -> public
customer_role -> labor_provider
hiring_party_type -> owner

Synthetic categorical drift population creation: PASS


In [187]:
synthetic_categorical_results = {}

for feature in categorical_features:

    result = calculate_categorical_drift(
        drifted_categorical[feature],
        loaded_reference[
            "categorical_features"
        ][feature],
    )

    synthetic_categorical_results[
        feature
    ] = result

    print(
        feature,
        "| TVD =",
        round(result["tvd"], 4),
        "| unseen =",
        result["unseen_category_count"],
        "|",
        result["status"],
    )

print()

significant_categorical_count = sum(
    result["status"]
    == "SIGNIFICANT_DRIFT"

    for result
    in synthetic_categorical_results.values()
)

print(
    "Significant categorical drift features:",
    significant_categorical_count,
    "/",
    len(categorical_features),
)

assert significant_categorical_count == 5

print()
print(
    "Categorical drift detection acceptance test: PASS"
)

state | TVD = 0.9482 | unseen = 0 | SIGNIFICANT_DRIFT
project_type | TVD = 0.924 | unseen = 0 | SIGNIFICANT_DRIFT
public_private | TVD = 0.7774 | unseen = 0 | SIGNIFICANT_DRIFT
customer_role | TVD = 0.9228 | unseen = 0 | SIGNIFICANT_DRIFT
hiring_party_type | TVD = 0.9005 | unseen = 0 | SIGNIFICANT_DRIFT

Significant categorical drift features: 5 / 5

Categorical drift detection acceptance test: PASS


In [188]:
unseen_test = X_val[
    "project_type"
].copy()

unseen_test.iloc[:100] = (
    "__UNSEEN_TEST_CATEGORY__"
)

unseen_result = (
    calculate_categorical_drift(
        unseen_test,
        loaded_reference[
            "categorical_features"
        ]["project_type"],
    )
)

print(
    "TVD:",
    round(
        unseen_result["tvd"],
        4,
    )
)

print(
    "Unseen categories:",
    unseen_result[
        "unseen_categories"
    ],
)

assert (
    "__UNSEEN_TEST_CATEGORY__"
    in unseen_result[
        "unseen_categories"
    ]
)

assert (
    unseen_result[
        "unseen_category_count"
    ] >= 1
)

print()
print(
    "Unseen category detection: PASS"
)

TVD: 0.065
Unseen categories: ['__UNSEEN_TEST_CATEGORY__']

Unseen category detection: PASS


# Build one unified drift report

In [189]:
from datetime import datetime, timezone


def generate_drift_report(
    current_df,
    reference,
):
    report = {
        "generated_at": datetime.now(
            timezone.utc
        ).isoformat(),
        "reference_version": reference[
            "reference_version"
        ],
        "model_version": reference[
            "model_version"
        ],
        "numeric_features": {},
        "binary_features": {},
        "categorical_features": {},
    }

    # Numeric
    for feature in numeric_features:

        report["numeric_features"][feature] = (
            calculate_psi(
                current_df[feature],
                reference[
                    "numeric_features"
                ][feature],
            )
        )

    # Binary
    for feature in binary_features:

        report["binary_features"][feature] = (
            calculate_binary_drift(
                current_df[feature],
                reference[
                    "binary_features"
                ][feature],
            )
        )

    # Categorical
    for feature in categorical_features:

        report[
            "categorical_features"
        ][feature] = (
            calculate_categorical_drift(
                current_df[feature],
                reference[
                    "categorical_features"
                ][feature],
            )
        )

    all_results = (
        list(
            report[
                "numeric_features"
            ].values()
        )
        +
        list(
            report[
                "binary_features"
            ].values()
        )
        +
        list(
            report[
                "categorical_features"
            ].values()
        )
    )

    status_counts = {
        "STABLE": 0,
        "INVESTIGATE": 0,
        "SIGNIFICANT_DRIFT": 0,
    }

    for result in all_results:
        status_counts[
            result["status"]
        ] += 1

    unseen_count = sum(
        result.get(
            "unseen_category_count",
            0,
        )
        for result in report[
            "categorical_features"
        ].values()
    )

    report["summary"] = {
        "total_features": len(
            all_results
        ),
        "stable_features": status_counts[
            "STABLE"
        ],
        "investigate_features": status_counts[
            "INVESTIGATE"
        ],
        "significant_drift_features":
            status_counts[
                "SIGNIFICANT_DRIFT"
            ],
        "unseen_category_count":
            unseen_count,
    }

    return report


print(
    "Unified drift report function: PASS"
)

Unified drift report function: PASS


In [190]:
validation_drift_report = (
    generate_drift_report(
        X_val,
        loaded_reference,
    )
)

print(
    json.dumps(
        validation_drift_report[
            "summary"
        ],
        indent=2,
    )
)

assert (
    validation_drift_report[
        "summary"
    ]["total_features"]
    == 14
)

print()
print(
    "Validation unified drift report: PASS"
)

{
  "total_features": 14,
  "stable_features": 14,
  "investigate_features": 0,
  "significant_drift_features": 0,
  "unseen_category_count": 0
}

Validation unified drift report: PASS


In [194]:
full_drift_population = X_val.copy()

# Numeric drift
for feature in numeric_features:
    full_drift_population[feature] = (
        drifted_val[feature]
    )

# Binary drift
for feature in binary_features:
    full_drift_population[feature] = (
        drifted_binary[feature]
    )

# Categorical drift
for feature in categorical_features:
    full_drift_population[feature] = (
        drifted_categorical[feature]
    )


full_drift_report = generate_drift_report(
    full_drift_population,
    loaded_reference,
)

print(
    json.dumps(
        full_drift_report["summary"],
        indent=2,
    )
)

assert (
    full_drift_report[
        "summary"
    ]["total_features"]
    == 14
)

assert (
    full_drift_report[
        "summary"
    ]["significant_drift_features"]
    >= 13
)

print()
print(
    "Synthetic unified drift report acceptance test: PASS"
)

{
  "total_features": 14,
  "stable_features": 0,
  "investigate_features": 0,
  "significant_drift_features": 14,
  "unseen_category_count": 0
}

Synthetic unified drift report acceptance test: PASS


In [191]:
def emit_drift_metrics(
    drift_report,
):
    summary = drift_report["summary"]

    numeric_results = drift_report[
        "numeric_features"
    ]

    max_numeric_psi = max(
        result["psi"]
        for result in numeric_results.values()
    )

    emit_metric(
        metric_name="DriftedFeatureCount",
        value=summary[
            "significant_drift_features"
        ],
        unit="Count",
        dimensions=API_DIMENSIONS,
    )

    emit_metric(
        metric_name="InvestigateFeatureCount",
        value=summary[
            "investigate_features"
        ],
        unit="Count",
        dimensions=API_DIMENSIONS,
    )

    emit_metric(
        metric_name="UnseenCategoryCount",
        value=summary[
            "unseen_category_count"
        ],
        unit="Count",
        dimensions=API_DIMENSIONS,
    )

    emit_metric(
        metric_name="MaxNumericPSI",
        value=max_numeric_psi,
        unit="None",
        dimensions=API_DIMENSIONS,
    )

    return {
        "DriftedFeatureCount":
            summary[
                "significant_drift_features"
            ],
        "InvestigateFeatureCount":
            summary[
                "investigate_features"
            ],
        "UnseenCategoryCount":
            summary[
                "unseen_category_count"
            ],
        "MaxNumericPSI":
            max_numeric_psi,
    }


print(
    "Drift metric emitter creation: PASS"
)

Drift metric emitter creation: PASS


In [192]:
from datetime import datetime, timedelta, timezone


def metric_sum(
    metric_name,
    start_time,
    end_time,
):
    response = cw.get_metric_statistics(
        Namespace="ConstructionPaymentRisk/API",
        MetricName=metric_name,
        Dimensions=API_DIMENSIONS,
        StartTime=start_time,
        EndTime=end_time,
        Period=60,
        Statistics=["Sum"],
    )

    return sum(
        point.get("Sum", 0)
        for point in response["Datapoints"]
    )


window_start = (
    datetime.now(timezone.utc)
    - timedelta(hours=1)
)

baseline_end = datetime.now(timezone.utc)

baseline = {
    "DriftedFeatureCount":
        metric_sum(
            "DriftedFeatureCount",
            window_start,
            baseline_end,
        ),
    "InvestigateFeatureCount":
        metric_sum(
            "InvestigateFeatureCount",
            window_start,
            baseline_end,
        ),
    "UnseenCategoryCount":
        metric_sum(
            "UnseenCategoryCount",
            window_start,
            baseline_end,
        ),
}

print("BASELINE")
print(baseline)

BASELINE
{'DriftedFeatureCount': 0, 'InvestigateFeatureCount': 0, 'UnseenCategoryCount': 0}


In [195]:
emitted = emit_drift_metrics(
    full_drift_report
)

print(
    "EMITTED"
)

print(
    emitted
)

print()
print(
    "Synthetic drift metrics emitted: PASS"
)

EMITTED
{'DriftedFeatureCount': 14, 'InvestigateFeatureCount': 0, 'UnseenCategoryCount': 0, 'MaxNumericPSI': 13.782864450559414}

Synthetic drift metrics emitted: PASS


In [196]:
import time

print(
    "Waiting for CloudWatch ingestion..."
)

time.sleep(90)

Waiting for CloudWatch ingestion...


In [197]:
verification_end = datetime.now(
    timezone.utc
)

after = {
    "DriftedFeatureCount":
        metric_sum(
            "DriftedFeatureCount",
            window_start,
            verification_end,
        ),
    "InvestigateFeatureCount":
        metric_sum(
            "InvestigateFeatureCount",
            window_start,
            verification_end,
        ),
    "UnseenCategoryCount":
        metric_sum(
            "UnseenCategoryCount",
            window_start,
            verification_end,
        ),
}

print("AFTER")
print(after)

delta = {
    key: after[key] - baseline[key]
    for key in baseline
}

print()
print("DELTA")
print(delta)

AFTER
{'DriftedFeatureCount': 14.0, 'InvestigateFeatureCount': 0.0, 'UnseenCategoryCount': 0.0}

DELTA
{'DriftedFeatureCount': 14.0, 'InvestigateFeatureCount': 0.0, 'UnseenCategoryCount': 0.0}


In [198]:
expected_drifted = (
    full_drift_report[
        "summary"
    ]["significant_drift_features"]
)

expected_investigate = (
    full_drift_report[
        "summary"
    ]["investigate_features"]
)

expected_unseen = (
    full_drift_report[
        "summary"
    ]["unseen_category_count"]
)

assert (
    delta["DriftedFeatureCount"]
    >= expected_drifted
)

assert (
    delta["InvestigateFeatureCount"]
    >= expected_investigate
)

assert (
    delta["UnseenCategoryCount"]
    >= expected_unseen
)

print()
print(
    "Drift monitoring CloudWatch integration: PASS"
)


Drift monitoring CloudWatch integration: PASS


# Drifted feature alarm

This means: if any monitored production feature reaches our SIGNIFICANT_DRIFT state during a monitoring run, CloudWatch moves the alarm into ALARM.

In [199]:
DRIFT_ALARM_NAME = (
    "construction-payment-risk-"
    "significant-feature-drift-dev"
)

response = cw.put_metric_alarm(
    AlarmName=DRIFT_ALARM_NAME,

    AlarmDescription=(
        "Alerts when one or more monitored model input "
        "features show significant distribution drift."
    ),

    Namespace="ConstructionPaymentRisk/API",

    MetricName="DriftedFeatureCount",

    Dimensions=API_DIMENSIONS,

    Statistic="Maximum",

    Period=300,

    EvaluationPeriods=1,

    DatapointsToAlarm=1,

    Threshold=1,

    ComparisonOperator=(
        "GreaterThanOrEqualToThreshold"
    ),

    TreatMissingData="notBreaching",

    ActionsEnabled=False,
)

print(
    "Create alarm HTTP status:",
    response[
        "ResponseMetadata"
    ]["HTTPStatusCode"]
)

assert (
    response[
        "ResponseMetadata"
    ]["HTTPStatusCode"]
    == 200
)

print()
print(
    "Feature drift alarm creation: PASS"
)

Create alarm HTTP status: 200

Feature drift alarm creation: PASS


In [200]:
ActionsEnabled=False

That prevents emails/SNS notifications while we're engineering and testing the monitoring system. Later we can attach SNS after the alarms themselves are trustworthy.
Now verify CloudWatch actually stored the configuration:

In [201]:
alarm_response = cw.describe_alarms(
    AlarmNames=[
        DRIFT_ALARM_NAME
    ]
)

alarms = alarm_response[
    "MetricAlarms"
]

assert len(alarms) == 1

alarm = alarms[0]

print(
    "Alarm:",
    alarm["AlarmName"]
)

print(
    "Metric:",
    alarm["MetricName"]
)

print(
    "Statistic:",
    alarm["Statistic"]
)

print(
    "Threshold:",
    alarm["Threshold"]
)

print(
    "Period:",
    alarm["Period"]
)

print(
    "EvaluationPeriods:",
    alarm["EvaluationPeriods"]
)

print(
    "TreatMissingData:",
    alarm["TreatMissingData"]
)

print(
    "Current state:",
    alarm["StateValue"]
)

Alarm: construction-payment-risk-significant-feature-drift-dev
Metric: DriftedFeatureCount
Statistic: Maximum
Threshold: 1.0
Period: 300
EvaluationPeriods: 1
TreatMissingData: notBreaching
Current state: INSUFFICIENT_DATA


In [202]:
assert (
    alarm["MetricName"]
    == "DriftedFeatureCount"
)

assert (
    alarm["Statistic"]
    == "Maximum"
)

assert (
    alarm["Threshold"]
    == 1
)

assert (
    alarm["Period"]
    == 300
)

assert (
    alarm["EvaluationPeriods"]
    == 1
)

assert (
    alarm["TreatMissingData"]
    == "notBreaching"
)

print()
print(
    "Feature drift alarm configuration: PASS"
)


Feature drift alarm configuration: PASS


Using Sum could work for counts, but for a snapshot-style monitoring metric like “how many features are currently drifted?”, Maximum represents the alarm semantics more cleanly.
Why missing data is not breaching
Our drift evaluator is a periodic monitoring process, not something emitted on every API request.

API server-error-rate alarm, which is more interesting because we'll use CloudWatch metric math:
\[
\text{Error Rate} =
\frac{\text{ServerErrorCount}}
{\text{RequestCount}}
\times 100
\]and alarm when the 5-minute server error rate exceeds our monitoring-contract threshold of 5%.




  









ChatGPT can make mistakes. Check important info.

# API Server Error Rate Alarm.

In [203]:
SERVER_ERROR_ALARM_NAME = (
    "construction-payment-risk-"
    "server-error-rate-dev"
)

response = cw.put_metric_alarm(
    AlarmName=SERVER_ERROR_ALARM_NAME,

    AlarmDescription=(
        "Alerts when the API server error rate exceeds "
        "5 percent during a 5-minute evaluation period."
    ),

    Metrics=[
        {
            "Id": "servererrors",
            "MetricStat": {
                "Metric": {
                    "Namespace":
                        "ConstructionPaymentRisk/API",
                    "MetricName":
                        "ServerErrorCount",
                    "Dimensions":
                        API_DIMENSIONS,
                },
                "Period": 300,
                "Stat": "Sum",
            },
            "ReturnData": False,
        },
        {
            "Id": "requests",
            "MetricStat": {
                "Metric": {
                    "Namespace":
                        "ConstructionPaymentRisk/API",
                    "MetricName":
                        "RequestCount",
                    "Dimensions":
                        API_DIMENSIONS,
                },
                "Period": 300,
                "Stat": "Sum",
            },
            "ReturnData": False,
        },
        {
            "Id": "errorrate",
            "Expression": (
                "IF(requests>0,"
                "(servererrors/requests)*100,"
                "0)"
            ),
            "Label": "Server Error Rate %",
            "ReturnData": True,
        },
    ],

    Threshold=5.0,

    ComparisonOperator=(
        "GreaterThanThreshold"
    ),

    EvaluationPeriods=1,

    DatapointsToAlarm=1,

    TreatMissingData="notBreaching",

    ActionsEnabled=False,
)

print(
    "Create alarm HTTP status:",
    response[
        "ResponseMetadata"
    ]["HTTPStatusCode"]
)

assert (
    response[
        "ResponseMetadata"
    ]["HTTPStatusCode"]
    == 200
)

print()
print(
    "Server error rate alarm creation: PASS"
)

Create alarm HTTP status: 200

Server error rate alarm creation: PASS


In [204]:
response = cw.describe_alarms(
    AlarmNames=[
        SERVER_ERROR_ALARM_NAME
    ]
)

alarms = response["MetricAlarms"]

assert len(alarms) == 1

server_alarm = alarms[0]

print(
    "Alarm:",
    server_alarm["AlarmName"]
)

print(
    "Threshold:",
    server_alarm["Threshold"]
)

print(
    "Comparison:",
    server_alarm[
        "ComparisonOperator"
    ]
)

print(
    "EvaluationPeriods:",
    server_alarm[
        "EvaluationPeriods"
    ]
)

print(
    "DatapointsToAlarm:",
    server_alarm[
        "DatapointsToAlarm"
    ]
)

print(
    "TreatMissingData:",
    server_alarm[
        "TreatMissingData"
    ]
)

print(
    "Current state:",
    server_alarm["StateValue"]
)

print()
print("Metric math queries:")

for metric in server_alarm["Metrics"]:
    print(
        metric["Id"],
        "|",
        metric.get(
            "Expression",
            metric.get(
                "MetricStat",
                {}
            ).get(
                "Metric",
                {}
            ).get(
                "MetricName"
            )
        ),
    )

Alarm: construction-payment-risk-server-error-rate-dev
Threshold: 5.0
Comparison: GreaterThanThreshold
EvaluationPeriods: 1
DatapointsToAlarm: 1
TreatMissingData: notBreaching
Current state: INSUFFICIENT_DATA

Metric math queries:
servererrors | ServerErrorCount
requests | RequestCount
errorrate | IF(requests>0,(servererrors/requests)*100,0)


In [205]:
assert (
    server_alarm["Threshold"]
    == 5.0
)

assert (
    server_alarm[
        "ComparisonOperator"
    ]
    == "GreaterThanThreshold"
)

assert (
    server_alarm[
        "EvaluationPeriods"
    ]
    == 1
)

assert (
    server_alarm[
        "DatapointsToAlarm"
    ]
    == 1
)

assert (
    server_alarm[
        "TreatMissingData"
    ]
    == "notBreaching"
)

metric_ids = {
    metric["Id"]
    for metric
    in server_alarm["Metrics"]
}

assert metric_ids == {
    "servererrors",
    "requests",
    "errorrate",
}

print()
print(
    "Server error rate alarm configuration: PASS"
)


Server error rate alarm configuration: PASS


# p95 Latency Alarm.

This catches cases where the API is technically succeeding but becoming too slow.\

Why p95 instead of average latency? Suppose 95 requests take 100 ms, but 5 take 4 seconds. The average can hide that bad tail behavior. p95 is much more useful for detecting a degraded user experience.

In [206]:
LATENCY_ALARM_NAME = (
    "construction-payment-risk-"
    "p95-latency-dev"
)

response = cw.put_metric_alarm(
    AlarmName=LATENCY_ALARM_NAME,

    AlarmDescription=(
        "Alerts when API p95 latency exceeds "
        "1000 milliseconds during a 5-minute period."
    ),

    Namespace="ConstructionPaymentRisk/API",

    MetricName="LatencyMs",

    Dimensions=API_DIMENSIONS,

    ExtendedStatistic="p95",

    Period=300,

    EvaluationPeriods=1,

    DatapointsToAlarm=1,

    Threshold=1000.0,

    ComparisonOperator="GreaterThanThreshold",

    TreatMissingData="notBreaching",

    ActionsEnabled=False,
)

assert (
    response["ResponseMetadata"]["HTTPStatusCode"]
    == 200
)

print("p95 latency alarm creation: PASS")

p95 latency alarm creation: PASS


In [207]:
response = cw.describe_alarms(
    AlarmNames=[LATENCY_ALARM_NAME]
)

alarms = response["MetricAlarms"]

assert len(alarms) == 1

latency_alarm = alarms[0]

print(
    "Alarm:",
    latency_alarm["AlarmName"]
)

print(
    "Metric:",
    latency_alarm["MetricName"]
)

print(
    "ExtendedStatistic:",
    latency_alarm["ExtendedStatistic"]
)

print(
    "Threshold:",
    latency_alarm["Threshold"]
)

print(
    "Period:",
    latency_alarm["Period"]
)

print(
    "EvaluationPeriods:",
    latency_alarm["EvaluationPeriods"]
)

print(
    "TreatMissingData:",
    latency_alarm["TreatMissingData"]
)

print(
    "Current state:",
    latency_alarm["StateValue"]
)

Alarm: construction-payment-risk-p95-latency-dev
Metric: LatencyMs
ExtendedStatistic: p95
Threshold: 1000.0
Period: 300
EvaluationPeriods: 1
TreatMissingData: notBreaching
Current state: INSUFFICIENT_DATA


In [208]:
assert latency_alarm["MetricName"] == "LatencyMs"

assert (
    latency_alarm["ExtendedStatistic"]
    == "p95"
)

assert latency_alarm["Threshold"] == 1000.0

assert latency_alarm["Period"] == 300

assert (
    latency_alarm["EvaluationPeriods"]
    == 1
)

assert (
    latency_alarm["TreatMissingData"]
    == "notBreaching"
)

print()
print(
    "p95 latency alarm configuration: PASS"
)


p95 latency alarm configuration: PASS


Validation Failure Rate Alarm.

In [209]:
VALIDATION_ALARM_NAME = (
    "construction-payment-risk-"
    "validation-failure-rate-dev"
)

response = cw.put_metric_alarm(
    AlarmName=VALIDATION_ALARM_NAME,

    AlarmDescription=(
        "Alerts when API validation failure rate exceeds "
        "10 percent during a 15-minute evaluation period."
    ),

    Metrics=[
        {
            "Id": "validationfailures",
            "MetricStat": {
                "Metric": {
                    "Namespace":
                        "ConstructionPaymentRisk/API",
                    "MetricName":
                        "ValidationFailureCount",
                    "Dimensions":
                        API_DIMENSIONS,
                },
                "Period": 900,
                "Stat": "Sum",
            },
            "ReturnData": False,
        },
        {
            "Id": "requests",
            "MetricStat": {
                "Metric": {
                    "Namespace":
                        "ConstructionPaymentRisk/API",
                    "MetricName":
                        "RequestCount",
                    "Dimensions":
                        API_DIMENSIONS,
                },
                "Period": 900,
                "Stat": "Sum",
            },
            "ReturnData": False,
        },
        {
            "Id": "validationrate",
            "Expression": (
                "IF(requests>0,"
                "(validationfailures/requests)*100,"
                "0)"
            ),
            "Label": "Validation Failure Rate %",
            "ReturnData": True,
        },
    ],

    Threshold=10.0,

    ComparisonOperator="GreaterThanThreshold",

    EvaluationPeriods=1,

    DatapointsToAlarm=1,

    TreatMissingData="notBreaching",

    ActionsEnabled=False,
)

assert (
    response["ResponseMetadata"]["HTTPStatusCode"]
    == 200
)

print(
    "Validation failure rate alarm creation: PASS"
)

Validation failure rate alarm creation: PASS


In [210]:
response = cw.describe_alarms(
    AlarmNames=[
        VALIDATION_ALARM_NAME
    ]
)

alarms = response["MetricAlarms"]

assert len(alarms) == 1

validation_alarm = alarms[0]

print(
    "Alarm:",
    validation_alarm["AlarmName"]
)

print(
    "Threshold:",
    validation_alarm["Threshold"]
)

print(
    "Comparison:",
    validation_alarm[
        "ComparisonOperator"
    ]
)

print(
    "EvaluationPeriods:",
    validation_alarm[
        "EvaluationPeriods"
    ]
)

print(
    "DatapointsToAlarm:",
    validation_alarm[
        "DatapointsToAlarm"
    ]
)

print(
    "TreatMissingData:",
    validation_alarm[
        "TreatMissingData"
    ]
)

print(
    "Current state:",
    validation_alarm[
        "StateValue"
    ]
)

print("\nMetric math queries:")

for metric in validation_alarm["Metrics"]:
    print(
        metric["Id"],
        "|",
        metric.get(
            "Expression",
            metric.get(
                "MetricStat",
                {}
            ).get(
                "Metric",
                {}
            ).get(
                "MetricName"
            )
        ),
    )

Alarm: construction-payment-risk-validation-failure-rate-dev
Threshold: 10.0
Comparison: GreaterThanThreshold
EvaluationPeriods: 1
DatapointsToAlarm: 1
TreatMissingData: notBreaching
Current state: INSUFFICIENT_DATA

Metric math queries:
validationfailures | ValidationFailureCount
requests | RequestCount
validationrate | IF(requests>0,(validationfailures/requests)*100,0)


In [211]:
assert (
    validation_alarm["Threshold"]
    == 10.0
)

assert (
    validation_alarm[
        "ComparisonOperator"
    ]
    == "GreaterThanThreshold"
)

assert (
    validation_alarm[
        "EvaluationPeriods"
    ]
    == 1
)

assert (
    validation_alarm[
        "DatapointsToAlarm"
    ]
    == 1
)

assert (
    validation_alarm[
        "TreatMissingData"
    ]
    == "notBreaching"
)

metric_ids = {
    metric["Id"]
    for metric
    in validation_alarm["Metrics"]
}

assert metric_ids == {
    "validationfailures",
    "requests",
    "validationrate",
}

print()
print(
    "Validation failure rate alarm configuration: PASS"
)


Validation failure rate alarm configuration: PASS


# Unseen Category Alarm.
This is important because an unseen category can mean the production input distribution has moved outside the categories the model saw during training. That is more actionable than a generic drift signal.

In [212]:
UNSEEN_CATEGORY_ALARM_NAME = (
    "construction-payment-risk-"
    "unseen-category-dev"
)

response = cw.put_metric_alarm(
    AlarmName=UNSEEN_CATEGORY_ALARM_NAME,

    AlarmDescription=(
        "Alerts when unseen categorical values are detected "
        "during feature drift monitoring."
    ),

    Namespace="ConstructionPaymentRisk/API",

    MetricName="UnseenCategoryCount",

    Dimensions=API_DIMENSIONS,

    Statistic="Maximum",

    Period=300,

    EvaluationPeriods=1,

    DatapointsToAlarm=1,

    Threshold=1.0,

    ComparisonOperator="GreaterThanOrEqualToThreshold",

    TreatMissingData="notBreaching",

    ActionsEnabled=False,
)

assert (
    response["ResponseMetadata"]["HTTPStatusCode"]
    == 200
)

print(
    "Unseen category alarm creation: PASS"
)

Unseen category alarm creation: PASS


In [213]:
response = cw.describe_alarms(
    AlarmNames=[
        UNSEEN_CATEGORY_ALARM_NAME
    ]
)

alarms = response["MetricAlarms"]

assert len(alarms) == 1

unseen_alarm = alarms[0]

print(
    "Alarm:",
    unseen_alarm["AlarmName"]
)

print(
    "Metric:",
    unseen_alarm["MetricName"]
)

print(
    "Statistic:",
    unseen_alarm["Statistic"]
)

print(
    "Threshold:",
    unseen_alarm["Threshold"]
)

print(
    "Period:",
    unseen_alarm["Period"]
)

print(
    "EvaluationPeriods:",
    unseen_alarm["EvaluationPeriods"]
)

print(
    "TreatMissingData:",
    unseen_alarm["TreatMissingData"]
)

print(
    "Current state:",
    unseen_alarm["StateValue"]
)

Alarm: construction-payment-risk-unseen-category-dev
Metric: UnseenCategoryCount
Statistic: Maximum
Threshold: 1.0
Period: 300
EvaluationPeriods: 1
TreatMissingData: notBreaching
Current state: INSUFFICIENT_DATA


In [214]:
assert (
    unseen_alarm["MetricName"]
    == "UnseenCategoryCount"
)

assert (
    unseen_alarm["Statistic"]
    == "Maximum"
)

assert (
    unseen_alarm["Threshold"]
    == 1.0
)

assert (
    unseen_alarm["Period"]
    == 300
)

assert (
    unseen_alarm["EvaluationPeriods"]
    == 1
)

assert (
    unseen_alarm["TreatMissingData"]
    == "notBreaching"
)

print()
print(
    "Unseen category alarm configuration: PASS"
)


Unseen category alarm configuration: PASS


# alarm inventory verification. 

We should confirm all five alarms exist before marking the whole step complete.

In [215]:
EXPECTED_ALARMS = {
    "construction-payment-risk-significant-feature-drift-dev",
    "construction-payment-risk-server-error-rate-dev",
    "construction-payment-risk-p95-latency-dev",
    "construction-payment-risk-validation-failure-rate-dev",
    "construction-payment-risk-unseen-category-dev",
}

response = cw.describe_alarms(
    AlarmNamePrefix="construction-payment-risk-"
)

found_alarms = {
    alarm["AlarmName"]
    for alarm in response["MetricAlarms"]
}

print("Expected alarms:")
for name in sorted(EXPECTED_ALARMS):
    print(" -", name)

print("\nFound expected alarms:")
for name in sorted(EXPECTED_ALARMS & found_alarms):
    print(" -", name)

missing = EXPECTED_ALARMS - found_alarms

print("\nMissing alarms:", missing)

assert not missing

print()
print("CloudWatch alarm inventory: PASS")

Expected alarms:
 - construction-payment-risk-p95-latency-dev
 - construction-payment-risk-server-error-rate-dev
 - construction-payment-risk-significant-feature-drift-dev
 - construction-payment-risk-unseen-category-dev
 - construction-payment-risk-validation-failure-rate-dev

Found expected alarms:
 - construction-payment-risk-p95-latency-dev
 - construction-payment-risk-server-error-rate-dev
 - construction-payment-risk-significant-feature-drift-dev
 - construction-payment-risk-unseen-category-dev
 - construction-payment-risk-validation-failure-rate-dev

Missing alarms: set()

CloudWatch alarm inventory: PASS


In [216]:
response = cw.describe_alarms(
    AlarmNames=sorted(EXPECTED_ALARMS)
)

for alarm in response["MetricAlarms"]:
    print(
        alarm["AlarmName"],
        "| State:",
        alarm["StateValue"],
        "| ActionsEnabled:",
        alarm["ActionsEnabled"],
    )

assert len(response["MetricAlarms"]) == 5

print()
print("16G CloudWatch Alarms: PASS")

construction-payment-risk-p95-latency-dev | State: OK | ActionsEnabled: False
construction-payment-risk-server-error-rate-dev | State: OK | ActionsEnabled: False
construction-payment-risk-significant-feature-drift-dev | State: OK | ActionsEnabled: False
construction-payment-risk-unseen-category-dev | State: INSUFFICIENT_DATA | ActionsEnabled: False
construction-payment-risk-validation-failure-rate-dev | State: OK | ActionsEnabled: False

16G CloudWatch Alarms: PASS


## CloudWatch Monitoring Dashboard.

1. SERVICE HEALTH\
   RequestCount\
   ServerErrorCount\
   LatencyMs p95\

2. INPUT QUALITY\
   ValidationFailureCount\
   MissingRequiredFieldCount\
   ClientErrorCount\

3. MODEL BEHAVIOR\
   PredictionCount\
   PredictedRisk\
   FlaggedCount\
   LowRiskCount / MediumRiskCount / HighRiskCount\

4. DATA / MODEL DRIFT\
   DriftedFeatureCount\
   InvestigateFeatureCount\
   UnseenCategoryCount\
   MaxNumericPSI

In [223]:
import json

DASHBOARD_NAME = (
    "construction-payment-risk-monitoring-dev"
)

namespace = "ConstructionPaymentRisk/API"

model_version = "logistic-regression-v1"
environment = "dev"

dims = [
    "ModelVersion", model_version,
    "Environment", environment,
]

dashboard_body = {
    "widgets": [
        # -------------------------------------------------
        # SECTION 1 — SERVICE HEALTH
        # -------------------------------------------------
        {
            "type": "text",
            "x": 0,
            "y": 0,
            "width": 24,
            "height": 2,
            "properties": {
                "markdown": (
                    "# Construction Payment Risk — Monitoring\n"
                    "## 1. Service Health"
                )
            },
        },

        {
            "type": "metric",
            "x": 0,
            "y": 2,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Request Volume",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Sum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "RequestCount",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 8,
            "y": 2,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Server Errors",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Sum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "ServerErrorCount",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 16,
            "y": 2,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "API Latency — p95",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "p95",
                "period": 300,
                "yAxis": {
                    "left": {
                        "label": "Milliseconds",
                        "showUnits": False,
                    }
                },
                "metrics": [
                    [
                        namespace,
                        "LatencyMs",
                        *dims,
                    ]
                ],
            },
        },

        # -------------------------------------------------
        # SECTION 2 — INPUT QUALITY
        # -------------------------------------------------
        {
            "type": "text",
            "x": 0,
            "y": 8,
            "width": 24,
            "height": 2,
            "properties": {
                "markdown": (
                    "## 2. Input Quality"
                )
            },
        },

        {
            "type": "metric",
            "x": 0,
            "y": 10,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Validation Failures",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Sum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "ValidationFailureCount",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 8,
            "y": 10,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Missing Required Fields",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Sum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "MissingRequiredFieldCount",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 16,
            "y": 10,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Client Errors",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Sum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "ClientErrorCount",
                        *dims,
                    ]
                ],
            },
        },

        # -------------------------------------------------
        # SECTION 3 — MODEL BEHAVIOR
        # -------------------------------------------------
        {
            "type": "text",
            "x": 0,
            "y": 16,
            "width": 24,
            "height": 2,
            "properties": {
                "markdown": (
                    "## 3. Model Behaviour"
                )
            },
        },

        {
            "type": "metric",
            "x": 0,
            "y": 18,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Prediction Volume",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Sum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "PredictionCount",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 8,
            "y": 18,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Predicted Risk",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Average",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "PredictedRisk",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 16,
            "y": 18,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Flagged Predictions",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Sum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "FlaggedCount",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 0,
            "y": 24,
            "width": 24,
            "height": 6,
            "properties": {
                "title": "Prediction Risk Bands",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Sum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "LowRiskCount",
                        *dims,
                        {"label": "Low"},
                    ],
                    [
                        namespace,
                        "MediumRiskCount",
                        *dims,
                        {"label": "Medium"},
                    ],
                    [
                        namespace,
                        "HighRiskCount",
                        *dims,
                        {"label": "High"},
                    ],
                ],
            },
        },

        # -------------------------------------------------
        # SECTION 4 — DRIFT
        # -------------------------------------------------
        {
            "type": "text",
            "x": 0,
            "y": 30,
            "width": 24,
            "height": 2,
            "properties": {
                "markdown": (
                    "## 4. Feature & Data Drift"
                )
            },
        },

        {
            "type": "metric",
            "x": 0,
            "y": 32,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Significant Drifted Features",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Maximum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "DriftedFeatureCount",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 8,
            "y": 32,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Features Under Investigation",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Maximum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "InvestigateFeatureCount",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 16,
            "y": 32,
            "width": 8,
            "height": 6,
            "properties": {
                "title": "Unseen Categories",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Maximum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "UnseenCategoryCount",
                        *dims,
                    ]
                ],
            },
        },

        {
            "type": "metric",
            "x": 0,
            "y": 38,
            "width": 24,
            "height": 6,
            "properties": {
                "title": "Maximum Numeric PSI",
                "region": "ap-southeast-2",
                "view": "timeSeries",
                "stat": "Maximum",
                "period": 300,
                "metrics": [
                    [
                        namespace,
                        "MaxNumericPSI",
                        *dims,
                    ]
                ],
                "annotations": {
                    "horizontal": [
                        {
                            "label": "Investigate",
                            "value": 0.10,
                        },
                        {
                            "label": "Significant Drift",
                            "value": 0.25,
                        },
                    ]
                },
            },
        },
    ]
}

response = cw.put_dashboard(
    DashboardName=DASHBOARD_NAME,
    DashboardBody=json.dumps(
        dashboard_body
    ),
)

print(
    "Create dashboard HTTP status:",
    response[
        "ResponseMetadata"
    ]["HTTPStatusCode"]
)

print(
    "Dashboard validation messages:",
    response.get(
        "DashboardValidationMessages",
        []
    )
)

assert (
    response[
        "ResponseMetadata"
    ]["HTTPStatusCode"]
    == 200
)

assert not response.get(
    "DashboardValidationMessages"
)

print()
print(
    "CloudWatch dashboard creation: PASS"
)

Create dashboard HTTP status: 200
Dashboard validation messages: []

CloudWatch dashboard creation: PASS


In [219]:
import boto3

sts = boto3.client("sts")

identity = sts.get_caller_identity()

print("Account:", identity["Account"])
print("ARN:", identity["Arn"])

Account: 911797457166
ARN: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker


In [220]:
import boto3

sts = boto3.client("sts")

identity = sts.get_caller_identity()

print("Account:", identity["Account"])
print("ARN:", identity["Arn"])

Account: 911797457166
ARN: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker


In [221]:
test_response = cw.put_dashboard(
    DashboardName="construction-payment-risk-monitoring-dev",
    DashboardBody='{"widgets":[]}'
)

print(
    "PutDashboard HTTP status:",
    test_response["ResponseMetadata"]["HTTPStatusCode"]
)

assert (
    test_response["ResponseMetadata"]["HTTPStatusCode"] == 200
)

print("CloudWatch dashboard IAM permission: PASS")

PutDashboard HTTP status: 200
CloudWatch dashboard IAM permission: PASS


In [224]:
expected_metrics = {
    "RequestCount",
    "ServerErrorCount",
    "LatencyMs",
    "ValidationFailureCount",
    "MissingRequiredFieldCount",
    "ClientErrorCount",
    "PredictionCount",
    "PredictedRisk",
    "FlaggedCount",
    "LowRiskCount",
    "MediumRiskCount",
    "HighRiskCount",
    "DriftedFeatureCount",
    "InvestigateFeatureCount",
    "UnseenCategoryCount",
    "MaxNumericPSI",
}

referenced_metrics = set()

for widget in saved_dashboard["widgets"]:
    if widget.get("type") != "metric":
        continue

    for metric in widget["properties"].get("metrics", []):
        if (
            isinstance(metric, list)
            and len(metric) >= 2
            and isinstance(metric[1], str)
        ):
            referenced_metrics.add(metric[1])

print("Expected metrics:", len(expected_metrics))
print("Referenced metrics:", len(referenced_metrics))

missing_metrics = (
    expected_metrics - referenced_metrics
)

unexpected_metrics = (
    referenced_metrics - expected_metrics
)

print("Missing metrics:", missing_metrics)
print("Unexpected metrics:", unexpected_metrics)

assert not missing_metrics

print()
print(
    "CloudWatch dashboard metric references: PASS"
)

NameError: name 'saved_dashboard' is not defined

In [225]:
import json

DASHBOARD_NAME = "construction-payment-risk-monitoring-dev"

# Fetch the dashboard directly from CloudWatch
response = cw.get_dashboard(
    DashboardName=DASHBOARD_NAME
)

saved_dashboard = json.loads(
    response["DashboardBody"]
)

print("Dashboard:", response["DashboardName"])
print(
    "Widget count:",
    len(saved_dashboard["widgets"])
)

print("Dashboard fetch: PASS")

Dashboard: construction-payment-risk-monitoring-dev
Widget count: 18
Dashboard fetch: PASS


In [226]:
expected_metrics = {
    "RequestCount",
    "ServerErrorCount",
    "LatencyMs",
    "ValidationFailureCount",
    "MissingRequiredFieldCount",
    "ClientErrorCount",
    "PredictionCount",
    "PredictedRisk",
    "FlaggedCount",
    "LowRiskCount",
    "MediumRiskCount",
    "HighRiskCount",
    "DriftedFeatureCount",
    "InvestigateFeatureCount",
    "UnseenCategoryCount",
    "MaxNumericPSI",
}

referenced_metrics = set()

for widget in saved_dashboard["widgets"]:

    if widget.get("type") != "metric":
        continue

    metrics = (
        widget
        .get("properties", {})
        .get("metrics", [])
    )

    for metric in metrics:

        if (
            isinstance(metric, list)
            and len(metric) >= 2
            and isinstance(metric[1], str)
        ):
            referenced_metrics.add(
                metric[1]
            )


print("\nExpected metrics:")
for metric in sorted(expected_metrics):
    print(" -", metric)

print("\nReferenced metrics:")
for metric in sorted(referenced_metrics):
    print(" -", metric)


missing_metrics = (
    expected_metrics
    - referenced_metrics
)

unexpected_metrics = (
    referenced_metrics
    - expected_metrics
)

print(
    "\nExpected metric count:",
    len(expected_metrics)
)

print(
    "Referenced metric count:",
    len(referenced_metrics)
)

print(
    "Missing metrics:",
    missing_metrics
)

print(
    "Unexpected metrics:",
    unexpected_metrics
)


assert not missing_metrics, (
    f"Dashboard missing metrics: "
    f"{missing_metrics}"
)

print()
print(
    "CloudWatch dashboard metric references: PASS"
)


Expected metrics:
 - ClientErrorCount
 - DriftedFeatureCount
 - FlaggedCount
 - HighRiskCount
 - InvestigateFeatureCount
 - LatencyMs
 - LowRiskCount
 - MaxNumericPSI
 - MediumRiskCount
 - MissingRequiredFieldCount
 - PredictedRisk
 - PredictionCount
 - RequestCount
 - ServerErrorCount
 - UnseenCategoryCount
 - ValidationFailureCount

Referenced metrics:
 - ClientErrorCount
 - DriftedFeatureCount
 - FlaggedCount
 - HighRiskCount
 - InvestigateFeatureCount
 - LatencyMs
 - LowRiskCount
 - MaxNumericPSI
 - MediumRiskCount
 - MissingRequiredFieldCount
 - PredictedRisk
 - PredictionCount
 - RequestCount
 - ServerErrorCount
 - UnseenCategoryCount
 - ValidationFailureCount

Expected metric count: 16
Referenced metric count: 16
Missing metrics: set()
Unexpected metrics: set()

CloudWatch dashboard metric references: PASS


# Automated monitoring infrastructure acceptance test
This test will verify the AWS resources we just built rather than merely checking notebook variables.

In [227]:
import boto3
import json

REGION = "ap-southeast-2"

cw = boto3.client(
    "cloudwatch",
    region_name=REGION,
)

DASHBOARD_NAME = (
    "construction-payment-risk-monitoring-dev"
)

EXPECTED_ALARMS = {
    "construction-payment-risk-significant-feature-drift-dev",
    "construction-payment-risk-server-error-rate-dev",
    "construction-payment-risk-p95-latency-dev",
    "construction-payment-risk-validation-failure-rate-dev",
    "construction-payment-risk-unseen-category-dev",
}

EXPECTED_METRICS = {
    "RequestCount",
    "ServerErrorCount",
    "LatencyMs",

    "ValidationFailureCount",
    "MissingRequiredFieldCount",
    "ClientErrorCount",

    "PredictionCount",
    "PredictedRisk",
    "FlaggedCount",
    "LowRiskCount",
    "MediumRiskCount",
    "HighRiskCount",

    "DriftedFeatureCount",
    "InvestigateFeatureCount",
    "UnseenCategoryCount",
    "MaxNumericPSI",
}

In [228]:
alarm_response = cw.describe_alarms(
    AlarmNames=sorted(EXPECTED_ALARMS)
)

alarms = alarm_response["MetricAlarms"]

found_alarm_names = {
    alarm["AlarmName"]
    for alarm in alarms
}

missing_alarms = (
    EXPECTED_ALARMS
    - found_alarm_names
)

print("Expected alarms:", len(EXPECTED_ALARMS))
print("Found alarms:", len(found_alarm_names))
print("Missing alarms:", missing_alarms)

assert not missing_alarms

print(
    "Monitoring alarm inventory test: PASS"
)

Expected alarms: 5
Found alarms: 5
Missing alarms: set()
Monitoring alarm inventory test: PASS


# Critical alarm configuration

In [230]:
alarms_by_name = {
    alarm["AlarmName"]: alarm
    for alarm in alarms
}


# -------------------------------------
# Significant feature drift
# -------------------------------------

drift_alarm = alarms_by_name[
    "construction-payment-risk-significant-feature-drift-dev"
]

assert (
    drift_alarm["Threshold"]
    == 1.0
)

assert (
    drift_alarm["ComparisonOperator"]
    == "GreaterThanOrEqualToThreshold"
)

assert (
    drift_alarm["Period"]
    == 300
)


# -------------------------------------
# p95 latency
# -------------------------------------

latency_alarm = alarms_by_name[
    "construction-payment-risk-p95-latency-dev"
]

assert (
    latency_alarm["ExtendedStatistic"]
    == "p95"
)

assert (
    latency_alarm["Threshold"]
    == 1000.0
)

assert (
    latency_alarm["Period"]
    == 300
)


# -------------------------------------
# Server error rate
# -------------------------------------

error_alarm = alarms_by_name[
    "construction-payment-risk-server-error-rate-dev"
]

assert (
    error_alarm["Threshold"]
    == 5.0
)

assert (
    error_alarm["ComparisonOperator"]
    == "GreaterThanThreshold"
)


# -------------------------------------
# Validation failure rate
# -------------------------------------

validation_alarm = alarms_by_name[
    "construction-payment-risk-validation-failure-rate-dev"
]

assert (
    validation_alarm["Threshold"]
    == 10.0
)

assert (
    validation_alarm["ComparisonOperator"]
    == "GreaterThanThreshold"
)


# -------------------------------------
# Unseen categories
# -------------------------------------

unseen_alarm = alarms_by_name[
    "construction-payment-risk-unseen-category-dev"
]

assert (
    unseen_alarm["Threshold"]
    == 1.0
)

assert (
    unseen_alarm["ComparisonOperator"]
    == "GreaterThanOrEqualToThreshold"
)


print(
    "Monitoring alarm policy test: PASS"
)

Monitoring alarm policy test: PASS


In [231]:
dashboard_response = cw.get_dashboard(
    DashboardName=DASHBOARD_NAME
)

dashboard = json.loads(
    dashboard_response["DashboardBody"]
)

referenced_metrics = set()

for widget in dashboard["widgets"]:

    if widget.get("type") != "metric":
        continue

    for metric in (
        widget
        .get("properties", {})
        .get("metrics", [])
    ):

        if (
            isinstance(metric, list)
            and len(metric) >= 2
            and isinstance(metric[1], str)
        ):
            referenced_metrics.add(
                metric[1]
            )


missing_metrics = (
    EXPECTED_METRICS
    - referenced_metrics
)

unexpected_metrics = (
    referenced_metrics
    - EXPECTED_METRICS
)

print(
    "Expected metrics:",
    len(EXPECTED_METRICS)
)

print(
    "Referenced metrics:",
    len(referenced_metrics)
)

print(
    "Missing:",
    missing_metrics
)

print(
    "Unexpected:",
    unexpected_metrics
)

assert not missing_metrics

print(
    "Monitoring dashboard coverage test: PASS"
)

Expected metrics: 16
Referenced metrics: 16
Missing: set()
Unexpected: set()
Monitoring dashboard coverage test: PASS


In [232]:
assert len(found_alarm_names) == 5
assert len(EXPECTED_METRICS) == 16
assert not missing_alarms
assert not missing_metrics

print()
print("=" * 60)
print(
    "16I.1 MONITORING INFRASTRUCTURE ACCEPTANCE: PASS"
)
print("=" * 60)


16I.1 MONITORING INFRASTRUCTURE ACCEPTANCE: PASS


# Monitoring Governance Manifest. 

This turns our monitoring setup into a formal production artifact: what we monitor, why, thresholds, interpretation, limitations, and expected response.

In [233]:
import json
from pathlib import Path
from datetime import datetime, timezone

GOVERNANCE_DIR = Path(
    "artifacts/monitoring/payment-risk-api/v1"
)

GOVERNANCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

GOVERNANCE_PATH = (
    GOVERNANCE_DIR
    / "monitoring-governance-v1.json"
)

governance = {
    "manifest_version": "1.0.0",
    "model_version": "logistic-regression-v1",
    "environment": "dev",
    "system": (
        "Construction Payment Risk Intelligence API"
    ),
    "generated_at": (
        datetime.now(timezone.utc)
        .isoformat()
    ),

    "purpose": (
        "Define operational monitoring, model-input monitoring, "
        "prediction-behaviour monitoring, alert interpretation, "
        "limitations, and human-response expectations."
    ),

    "principles": {
        "monitoring_failure_must_not_break_inference": True,
        "raw_request_payload_logging_allowed": False,
        "raw_transformed_features_logging_allowed": False,
        "drift_is_not_proof_of_performance_degradation": True,
        "true_performance_monitoring_requires_labels": True,
        "high_impact_actions_require_human_review": True,
    },

    "service_monitoring": {
        "RequestCount": {
            "purpose": "Track API traffic volume.",
            "aggregation": "Sum",
            "period_seconds": 300,
        },

        "ServerErrorCount": {
            "purpose": "Track HTTP 5xx failures.",
            "aggregation": "Sum",
            "alert": {
                "type": "rate",
                "formula": (
                    "ServerErrorCount / RequestCount * 100"
                ),
                "threshold_percent": 5.0,
                "comparison": ">",
                "period_seconds": 300,
            },
            "response": (
                "Inspect application logs, dependency failures, "
                "recent deployments, model loading, and infrastructure."
            ),
        },

        "LatencyMs": {
            "purpose": "Track request latency.",
            "aggregation": "p95",
            "alert": {
                "threshold_ms": 1000,
                "comparison": ">",
                "period_seconds": 300,
            },
            "response": (
                "Inspect model inference time, downstream AWS calls, "
                "resource saturation, and deployment changes."
            ),
        },
    },

    "input_quality_monitoring": {
        "ValidationFailureCount": {
            "purpose": (
                "Track requests rejected by API validation."
            ),
            "alert": {
                "type": "rate",
                "formula": (
                    "ValidationFailureCount / RequestCount * 100"
                ),
                "threshold_percent": 10.0,
                "comparison": ">",
                "period_seconds": 900,
            },
            "response": (
                "Check upstream schema changes, malformed integration "
                "traffic, missing fields, and client contract mismatch."
            ),
        },

        "MissingRequiredFieldCount": {
            "purpose": (
                "Track required input fields absent from requests."
            ),
        },

        "ClientErrorCount": {
            "purpose": (
                "Track HTTP 4xx client-side failures."
            ),
        },
    },

    "prediction_monitoring": {
        "PredictionCount": {
            "purpose": "Track prediction volume."
        },

        "PredictedRisk": {
            "purpose": (
                "Monitor aggregate predicted risk behaviour."
            ),
            "note": (
                "Prediction distribution changes may reflect "
                "population change and do not alone prove model degradation."
            ),
        },

        "FlaggedCount": {
            "purpose": (
                "Track predictions above the frozen operational "
                "model threshold."
            ),
            "model_threshold": 0.20,
        },

        "risk_bands": {
            "low": {
                "condition": "risk < 0.20"
            },
            "medium": {
                "condition": "0.20 <= risk < 0.50"
            },
            "high": {
                "condition": "risk >= 0.50"
            },
            "note": (
                "0.50 is an observational monitoring boundary, "
                "not the operational model decision threshold."
            ),
        },
    },

    "feature_drift_monitoring": {
        "numeric": {
            "method": "Population Stability Index",
            "thresholds": {
                "stable": "PSI < 0.10",
                "investigate": (
                    "0.10 <= PSI < 0.25"
                ),
                "significant_drift": (
                    "PSI >= 0.25"
                ),
            },
            "note": (
                "Thresholds are project operational heuristics "
                "and are not universal scientific standards."
            ),
        },

        "binary": {
            "method": (
                "Absolute positive-rate shift"
            ),
            "thresholds": {
                "stable": "shift < 0.10",
                "investigate": (
                    "0.10 <= shift < 0.20"
                ),
                "significant_drift": (
                    "shift >= 0.20"
                ),
            },
        },

        "categorical": {
            "method": (
                "Total Variation Distance"
            ),
            "thresholds": {
                "stable": "TVD < 0.10",
                "investigate": (
                    "0.10 <= TVD < 0.20"
                ),
                "significant_drift": (
                    "TVD >= 0.20"
                ),
            },
            "unseen_category_detection": True,
        },

        "alarms": {
            "significant_feature_drift": {
                "metric": "DriftedFeatureCount",
                "condition": ">= 1",
                "period_seconds": 300,
            },

            "unseen_categories": {
                "metric": "UnseenCategoryCount",
                "condition": ">= 1",
                "period_seconds": 300,
            },
        },
    },

    "performance_monitoring": {
        "current_state": (
            "Delayed-label monitoring not yet automated."
        ),

        "required_when_labels_arrive": [
            "ROC AUC",
            "PR AUC",
            "Brier score",
            "precision at operational threshold",
            "recall at operational threshold",
            "F1 at operational threshold",
            "confusion matrix",
            "calibration",
        ],

        "important_limitation": (
            "Without true outcomes, feature drift and prediction "
            "distribution changes cannot prove that model accuracy "
            "has deteriorated."
        ),
    },

    "privacy_and_logging": {
        "allowed": [
            "request_id",
            "timestamp",
            "model_version",
            "api_version",
            "HTTP status",
            "latency",
            "predicted risk",
            "decision",
            "aggregate monitoring metrics",
        ],

        "prohibited": [
            "raw customer request payload",
            "raw transformed feature vector",
            "sensitive project content",
            "raw model internals in customer response",
        ],
    },

    "human_governance": {
        "automatic_legal_decisions_allowed": False,

        "human_review_required_for": [
            "high-impact operational actions",
            "legal or compliance interpretation",
            "low-confidence cases",
            "material drift investigation",
            "production threshold changes",
            "model replacement or retraining approval",
        ],
    },
}

with open(
    GOVERNANCE_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        governance,
        f,
        indent=2,
    )

print(
    "Governance manifest:",
    GOVERNANCE_PATH
)

print(
    "Monitoring governance manifest creation: PASS"
)

Governance manifest: artifacts/monitoring/payment-risk-api/v1/monitoring-governance-v1.json
Monitoring governance manifest creation: PASS


In [234]:
with open(
    GOVERNANCE_PATH,
    "r",
    encoding="utf-8",
) as f:
    saved_governance = json.load(f)

assert (
    saved_governance[
        "principles"
    ][
        "drift_is_not_proof_of_performance_degradation"
    ]
    is True
)

assert (
    saved_governance[
        "principles"
    ][
        "true_performance_monitoring_requires_labels"
    ]
    is True
)

assert (
    saved_governance[
        "principles"
    ][
        "monitoring_failure_must_not_break_inference"
    ]
    is True
)

assert (
    saved_governance[
        "prediction_monitoring"
    ][
        "FlaggedCount"
    ][
        "model_threshold"
    ]
    == 0.20
)

assert (
    saved_governance[
        "feature_drift_monitoring"
    ][
        "numeric"
    ][
        "thresholds"
    ][
        "significant_drift"
    ]
    == "PSI >= 0.25"
)

assert (
    saved_governance[
        "human_governance"
    ][
        "automatic_legal_decisions_allowed"
    ]
    is False
)

print(
    "Monitoring governance validation: PASS"
)

Monitoring governance validation: PASS


# Incident Response Runbook. This answers the production question:
“An alarm fired. What should the engineer actually do?”

We do not want automatic retraining or automatic legal/compliance actions simply because drift was detected. The runbook creates a controlled investigation path.

In [235]:
import json
from pathlib import Path
from datetime import datetime, timezone

RUNBOOK_DIR = Path(
    "artifacts/monitoring/payment-risk-api/v1"
)

RUNBOOK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUNBOOK_PATH = (
    RUNBOOK_DIR
    / "monitoring-incident-runbook-v1.json"
)

runbook = {
    "runbook_version": "1.0.0",
    "system": (
        "Construction Payment Risk Intelligence API"
    ),
    "model_version": "logistic-regression-v1",
    "environment": "dev",
    "generated_at": (
        datetime.now(timezone.utc).isoformat()
    ),

    "general_response_process": [
        "Acknowledge the monitoring signal.",
        "Confirm the signal using CloudWatch metrics and logs.",
        "Determine whether the issue is service, input, drift, or model related.",
        "Correlate using request IDs where applicable.",
        "Assess business and operational impact.",
        "Mitigate without bypassing validation or governance controls.",
        "Document findings and corrective action.",
        "Escalate for human review when required."
    ],

    "incidents": {

        "server_error_rate": {
            "alarm": (
                "construction-payment-risk-"
                "server-error-rate-dev"
            ),
            "severity": "high",

            "trigger": (
                "Server error rate > 5% "
                "during a 5-minute period."
            ),

            "investigate": [
                "Inspect structured application logs.",
                "Check recent deployments or configuration changes.",
                "Verify model artifacts can be loaded.",
                "Check AWS dependency failures.",
                "Check infrastructure/resource saturation.",
                "Correlate failures using request_id."
            ],

            "mitigation": [
                "Rollback a faulty deployment if identified.",
                "Restore unavailable dependencies.",
                "Correct configuration or artifact problems.",
                "Keep monitoring failures isolated from inference."
            ],

            "do_not": [
                "Disable API validation to reduce errors.",
                "Expose raw customer payloads during debugging.",
                "Silently change the model decision threshold."
            ]
        },

        "high_latency": {
            "alarm": (
                "construction-payment-risk-"
                "p95-latency-dev"
            ),
            "severity": "medium",

            "trigger": (
                "p95 API latency > 1000 ms "
                "during a 5-minute period."
            ),

            "investigate": [
                "Inspect latency trend and request volume.",
                "Check inference execution time.",
                "Check AWS service latency.",
                "Check CPU/memory/resource pressure.",
                "Compare against recent deployments."
            ],

            "mitigation": [
                "Remove unnecessary synchronous operations.",
                "Optimize inference path.",
                "Scale serving resources when justified.",
                "Rollback performance-regressing changes."
            ]
        },

        "validation_failure_rate": {
            "alarm": (
                "construction-payment-risk-"
                "validation-failure-rate-dev"
            ),
            "severity": "medium",

            "trigger": (
                "Validation failure rate > 10% "
                "during a 15-minute period."
            ),

            "investigate": [
                "Check ValidationFailureCount.",
                "Check MissingRequiredFieldCount.",
                "Inspect validation error types without logging raw payloads.",
                "Check upstream schema/API contract changes.",
                "Check whether clients are using unsupported values."
            ],

            "mitigation": [
                "Correct upstream integration errors.",
                "Coordinate intentional schema changes through versioning.",
                "Update the API contract only through controlled release."
            ],

            "do_not": [
                "Relax validation solely to suppress the alarm.",
                "Accept unknown categories without model-impact review."
            ]
        },

        "significant_feature_drift": {
            "alarm": (
                "construction-payment-risk-"
                "significant-feature-drift-dev"
            ),
            "severity": "medium",

            "trigger": (
                "At least one monitored feature reaches "
                "SIGNIFICANT_DRIFT."
            ),

            "investigate": [
                "Identify affected features.",
                "Review PSI, binary-rate shift, or TVD.",
                "Compare current population with training reference.",
                "Check upstream data-generation or schema changes.",
                "Check whether the shift is expected business seasonality.",
                "Review prediction-distribution changes.",
                "Obtain labeled outcomes when available."
            ],

            "mitigation": [
                "Continue monitoring if drift is legitimate and low risk.",
                "Correct upstream data defects when confirmed.",
                "Perform model evaluation using labeled outcomes.",
                "Consider retraining only after evidence-based review."
            ],

            "do_not": [
                "Assume drift means accuracy degraded.",
                "Automatically retrain solely because PSI/TVD crossed a threshold.",
                "Automatically change the operational threshold."
            ]
        },

        "unseen_category": {
            "alarm": (
                "construction-payment-risk-"
                "unseen-category-dev"
            ),
            "severity": "high",

            "trigger": (
                "One or more categorical values are absent "
                "from the training reference vocabulary."
            ),

            "investigate": [
                "Identify the affected feature and category.",
                "Determine whether it is a data-quality error.",
                "Determine whether it represents a legitimate new business value.",
                "Review API/schema compatibility.",
                "Assess whether training data must be expanded."
            ],

            "mitigation": [
                "Correct invalid upstream values.",
                "Version schema changes where required.",
                "Evaluate legitimate new categories before model updates."
            ],

            "do_not": [
                "Silently map an unknown category to an arbitrary known value.",
                "Automatically retrain without validation and approval."
            ]
        }
    },

    "model_performance_degradation": {
        "automatic_detection_available": False,

        "reason": (
            "True model performance requires observed outcome labels. "
            "Current online monitoring primarily observes service health, "
            "input quality, prediction behaviour, and population drift."
        ),

        "when_labels_arrive": [
            "Join predictions to observed outcomes using governed identifiers.",
            "Calculate ROC AUC.",
            "Calculate PR AUC.",
            "Calculate Brier score.",
            "Calculate precision, recall, F1 and confusion matrix at threshold 0.20.",
            "Review calibration.",
            "Compare against approved baseline and previous periods.",
            "Perform human review before retraining or model replacement."
        ]
    },

    "governance": {
        "automatic_retraining_allowed": False,
        "automatic_threshold_change_allowed": False,
        "automatic_legal_action_allowed": False,

        "human_approval_required_for": [
            "production model replacement",
            "operational threshold change",
            "retraining promotion",
            "material schema change",
            "legal or compliance interpretation",
            "high-impact external action"
        ]
    }
}

with open(
    RUNBOOK_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        runbook,
        f,
        indent=2
    )

print("Runbook:", RUNBOOK_PATH)
print("Monitoring incident runbook creation: PASS")

Runbook: artifacts/monitoring/payment-risk-api/v1/monitoring-incident-runbook-v1.json
Monitoring incident runbook creation: PASS


In [236]:
with open(
    RUNBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    saved_runbook = json.load(f)


expected_incidents = {
    "server_error_rate",
    "high_latency",
    "validation_failure_rate",
    "significant_feature_drift",
    "unseen_category"
}

actual_incidents = set(
    saved_runbook["incidents"].keys()
)

assert (
    expected_incidents
    == actual_incidents
)

assert (
    saved_runbook[
        "governance"
    ][
        "automatic_retraining_allowed"
    ]
    is False
)

assert (
    saved_runbook[
        "governance"
    ][
        "automatic_threshold_change_allowed"
    ]
    is False
)

assert (
    saved_runbook[
        "governance"
    ][
        "automatic_legal_action_allowed"
    ]
    is False
)

assert (
    saved_runbook[
        "model_performance_degradation"
    ][
        "automatic_detection_available"
    ]
    is False
)

print(
    "Incident types:",
    len(actual_incidents)
)

print(
    "Automatic retraining:",
    saved_runbook[
        "governance"
    ][
        "automatic_retraining_allowed"
    ]
)

print(
    "Automatic threshold change:",
    saved_runbook[
        "governance"
    ][
        "automatic_threshold_change_allowed"
    ]
)

print(
    "Automatic legal action:",
    saved_runbook[
        "governance"
    ][
        "automatic_legal_action_allowed"
    ]
)

print()
print(
    "Monitoring incident runbook validation: PASS"
)

Incident types: 5
Automatic retraining: False
Automatic threshold change: False
Automatic legal action: False

Monitoring incident runbook validation: PASS


# Final Monitoring Package Validation
We want one acceptance test proving that the artifacts and AWS infrastructure agree with each other.

In [237]:
import json
from pathlib import Path
import boto3

# =========================================================
# CONFIGURATION
# =========================================================

REGION = "ap-southeast-2"

ARTIFACT_DIR = Path(
    "artifacts/monitoring/payment-risk-api/v1"
)

DASHBOARD_NAME = (
    "construction-payment-risk-monitoring-dev"
)

EXPECTED_ALARMS = {
    "construction-payment-risk-significant-feature-drift-dev",
    "construction-payment-risk-server-error-rate-dev",
    "construction-payment-risk-p95-latency-dev",
    "construction-payment-risk-validation-failure-rate-dev",
    "construction-payment-risk-unseen-category-dev",
}

EXPECTED_METRICS = {
    "RequestCount",
    "ServerErrorCount",
    "LatencyMs",
    "ValidationFailureCount",
    "MissingRequiredFieldCount",
    "ClientErrorCount",
    "PredictionCount",
    "PredictedRisk",
    "FlaggedCount",
    "LowRiskCount",
    "MediumRiskCount",
    "HighRiskCount",
    "DriftedFeatureCount",
    "InvestigateFeatureCount",
    "UnseenCategoryCount",
    "MaxNumericPSI",
}

cw = boto3.client(
    "cloudwatch",
    region_name=REGION,
)

print("=" * 65)
print("FINAL ML MONITORING PACKAGE VALIDATION")
print("=" * 65)

FINAL ML MONITORING PACKAGE VALIDATION


In [238]:
# =========================================================
# 1. ARTIFACT VALIDATION
# =========================================================

required_artifacts = {
    "monitoring_contract":
        ARTIFACT_DIR
        / "monitoring-contract-v1.json",

    "feature_reference":
        ARTIFACT_DIR
        / "feature-reference-v1.json",

    "governance_manifest":
        ARTIFACT_DIR
        / "monitoring-governance-v1.json",

    "incident_runbook":
        ARTIFACT_DIR
        / "monitoring-incident-runbook-v1.json",
}

print("\n1. Monitoring artifacts")

for name, path in required_artifacts.items():

    exists = path.exists()

    print(
        f"{name:25} | "
        f"{'PASS' if exists else 'MISSING'}"
    )

    assert exists, (
        f"Required monitoring artifact missing: {path}"
    )

print(
    "\nMonitoring artifact inventory: PASS"
)


1. Monitoring artifacts
monitoring_contract       | PASS
feature_reference         | PASS
governance_manifest       | PASS
incident_runbook          | PASS

Monitoring artifact inventory: PASS


In [239]:
loaded_artifacts = {}

for name, path in required_artifacts.items():

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        loaded_artifacts[name] = (
            json.load(f)
        )

print(
    "Monitoring artifact JSON validation: PASS"
)

Monitoring artifact JSON validation: PASS


In [240]:
# =========================================================
# 2. GOVERNANCE VALIDATION
# =========================================================

governance = loaded_artifacts[
    "governance_manifest"
]

runbook = loaded_artifacts[
    "incident_runbook"
]

assert (
    governance[
        "principles"
    ][
        "drift_is_not_proof_of_performance_degradation"
    ]
    is True
)

assert (
    governance[
        "principles"
    ][
        "true_performance_monitoring_requires_labels"
    ]
    is True
)

assert (
    governance[
        "human_governance"
    ][
        "automatic_legal_decisions_allowed"
    ]
    is False
)

assert (
    runbook[
        "governance"
    ][
        "automatic_retraining_allowed"
    ]
    is False
)

assert (
    runbook[
        "governance"
    ][
        "automatic_threshold_change_allowed"
    ]
    is False
)

assert (
    runbook[
        "governance"
    ][
        "automatic_legal_action_allowed"
    ]
    is False
)

print(
    "\nMonitoring governance controls: PASS"
)


Monitoring governance controls: PASS


In [241]:
# =========================================================
# 3. CLOUDWATCH ALARM VALIDATION
# =========================================================

alarm_response = cw.describe_alarms(
    AlarmNames=sorted(
        EXPECTED_ALARMS
    )
)

alarms = alarm_response[
    "MetricAlarms"
]

found_alarms = {
    alarm["AlarmName"]
    for alarm in alarms
}

missing_alarms = (
    EXPECTED_ALARMS
    - found_alarms
)

print(
    "\nExpected alarms:",
    len(EXPECTED_ALARMS)
)

print(
    "Found alarms:",
    len(found_alarms)
)

print(
    "Missing alarms:",
    missing_alarms
)

assert not missing_alarms

assert len(found_alarms) == 5

print(
    "CloudWatch alarm validation: PASS"
)


Expected alarms: 5
Found alarms: 5
Missing alarms: set()
CloudWatch alarm validation: PASS


In [242]:
# =========================================================
# 4. CLOUDWATCH DASHBOARD VALIDATION
# =========================================================

dashboard_response = (
    cw.get_dashboard(
        DashboardName=DASHBOARD_NAME
    )
)

dashboard = json.loads(
    dashboard_response[
        "DashboardBody"
    ]
)

widgets = dashboard.get(
    "widgets",
    []
)

assert widgets

referenced_metrics = set()

for widget in widgets:

    if (
        widget.get("type")
        != "metric"
    ):
        continue

    metrics = (
        widget
        .get("properties", {})
        .get("metrics", [])
    )

    for metric in metrics:

        if (
            isinstance(metric, list)
            and len(metric) >= 2
            and isinstance(
                metric[1],
                str
            )
        ):
            referenced_metrics.add(
                metric[1]
            )

missing_metrics = (
    EXPECTED_METRICS
    - referenced_metrics
)

unexpected_metrics = (
    referenced_metrics
    - EXPECTED_METRICS
)

print(
    "\nDashboard widgets:",
    len(widgets)
)

print(
    "Expected metrics:",
    len(EXPECTED_METRICS)
)

print(
    "Referenced metrics:",
    len(referenced_metrics)
)

print(
    "Missing metrics:",
    missing_metrics
)

print(
    "Unexpected metrics:",
    unexpected_metrics
)

assert not missing_metrics

assert (
    len(referenced_metrics)
    == 16
)

print(
    "CloudWatch dashboard validation: PASS"
)


Dashboard widgets: 18
Expected metrics: 16
Referenced metrics: 16
Missing metrics: set()
Unexpected metrics: set()
CloudWatch dashboard validation: PASS


In [243]:
# =========================================================
# 5. MODEL / MONITORING CONTRACT CONSISTENCY
# =========================================================

reference = loaded_artifacts[
    "feature_reference"
]

assert (
    reference[
        "model_version"
    ]
    == "logistic-regression-v1"
)

assert (
    governance[
        "model_version"
    ]
    == "logistic-regression-v1"
)

assert (
    runbook[
        "model_version"
    ]
    == "logistic-regression-v1"
)

assert (
    governance[
        "prediction_monitoring"
    ][
        "FlaggedCount"
    ][
        "model_threshold"
    ]
    == 0.20
)

print(
    "\nModel/monitoring contract consistency: PASS"
)


Model/monitoring contract consistency: PASS


In [244]:
# =========================================================
# FINAL ACCEPTANCE
# =========================================================

checks = {
    "monitoring_artifacts": True,
    "json_validation": True,
    "governance_controls": True,
    "cloudwatch_alarms": (
        len(found_alarms) == 5
    ),
    "dashboard_metrics": (
        len(referenced_metrics) == 16
        and not missing_metrics
    ),
    "model_contract_consistency": True,
}

print("\nFinal checks:")

for check, passed in checks.items():

    print(
        f"{check:30} | "
        f"{'PASS' if passed else 'FAIL'}"
    )

assert all(
    checks.values()
)

print()
print("=" * 65)
print(
    "16I.4 FINAL MONITORING PACKAGE ACCEPTANCE: PASS"
)
print("=" * 65)


Final checks:
monitoring_artifacts           | PASS
json_validation                | PASS
governance_controls            | PASS
cloudwatch_alarms              | PASS
dashboard_metrics              | PASS
model_contract_consistency     | PASS

16I.4 FINAL MONITORING PACKAGE ACCEPTANCE: PASS


```text
Logistic Regression v1
Threshold = 0.20
        │
        ▼
Prediction API
        │
        ├── structured logs
        ├── service metrics
        ├── input-quality metrics
        └── prediction metrics
                │
                ▼
        Feature Drift Engine
                │
      ┌─────────┴─────────┐
      ▼                   ▼
CloudWatch Alarms    CloudWatch Dashboard
      │
      ▼
Incident Runbook
      │
      ▼
Human-governed response
```

```text
Our target architecture will be:

                     CONSTRUCTION PAYMENT
                    PROTECTION INTELLIGENCE
                              │
          ┌───────────────────┴───────────────────┐
          │                                       │
          ▼                                       ▼
   Historical / Portfolio                    Single Workflow
       Intelligence                           Assessment
          │                                       │
          ▼                                       ▼
      Athena / S3                         Prediction API
          │                                       │
          └───────────────────┬───────────────────┘
                              ▼
                      Business Dashboard
                              │
             ┌────────────────┼────────────────┐
             ▼                ▼                ▼
         Risk Queue        Explanation       Review
                                             Status
```

In [249]:
import json
from pathlib import Path
from datetime import datetime, timezone


DASHBOARD_ARTIFACT_DIR = Path(
    "artifacts/dashboard/payment-risk-dashboard/v1"
)

DASHBOARD_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DASHBOARD_CONTRACT_PATH = (
    DASHBOARD_ARTIFACT_DIR
    / "dashboard-product-contract-v1.json"
)


dashboard_contract = {

    "contract_version": "1.0.0",

    "product_name": (
        "Construction Payment Protection "
        "Intelligence Dashboard"
    ),

    "flagship": (
        "Construction Payment Protection "
        "Intelligence Engine"
    ),

    "environment": "dev",

    "generated_at": (
        datetime.now(timezone.utc)
        .isoformat()
    ),

    # -------------------------------------------------
    # PURPOSE
    # -------------------------------------------------

    "purpose": (
        "Help operations users identify construction "
        "payment-protection workflows that deserve "
        "additional operational attention and understand "
        "the model signals contributing to prioritization."
    ),

    # -------------------------------------------------
    # USERS
    # -------------------------------------------------

    "primary_users": [
        "payment-protection operations analyst",
        "research analyst",
        "operations supervisor"
    ],

    # -------------------------------------------------
    # BUSINESS QUESTIONS
    # -------------------------------------------------

    "business_questions": [

        (
            "How many assessed workflows are currently "
            "flagged for operational attention?"
        ),

        (
            "How is model risk distributed across "
            "the assessed workflow population?"
        ),

        (
            "Which workflows should analysts review first?"
        ),

        (
            "Which operational factors contributed most "
            "to a workflow's risk score?"
        ),

        (
            "Are missing or conflicting project details "
            "contributing to elevated operational risk?"
        ),

        (
            "How does risk vary across project and "
            "workflow characteristics?"
        )
    ],

    # -------------------------------------------------
    # DASHBOARD PAGES
    # -------------------------------------------------

    "pages": {

        "portfolio_overview": {

            "purpose": (
                "Provide aggregate operational risk "
                "visibility across assessed workflows."
            ),

            "components": [
                "total assessed workflows",
                "flagged workflow count",
                "flagged workflow rate",
                "average predicted risk",
                "risk-band distribution",
                "risk trends",
                "selected business segment breakdowns"
            ]
        },

        "risk_queue": {

            "purpose": (
                "Prioritize workflows for human "
                "operational review."
            ),

            "components": [
                "workflow identifier",
                "assessment date",
                "state",
                "project type",
                "customer role",
                "predicted risk",
                "model decision",
                "deadline context",
                "critical information indicators",
                "review action"
            ]
        },

        "workflow_assessment": {

            "purpose": (
                "Score a single workflow using the "
                "approved prediction API."
            ),

            "components": [
                "14-feature input form",
                "predicted risk",
                "operational threshold",
                "model decision",
                "customer-safe explanation",
                "governance disclaimer"
            ]
        },

        "explanation": {

            "purpose": (
                "Explain important model signals without "
                "exposing raw model internals."
            ),

            "components": [
                "risk-supporting factors",
                "risk-reducing factors",
                "operational context",
                "model limitations"
            ]
        }
    },

    # -------------------------------------------------
    # MODEL CONTRACT
    # -------------------------------------------------

    "model": {

        "model_version":
            "logistic-regression-v1",

        "operational_threshold":
            0.20,

        "decision_values": [
            "FLAGGED_BY_MODEL",
            "NOT_FLAGGED_BY_MODEL"
        ],

        "risk_bands": {

            "low": "risk < 0.20",

            "medium":
                "0.20 <= risk < 0.50",

            "high":
                "risk >= 0.50"
        },

        "risk_band_note": (
            "The 0.50 boundary is used only for "
            "observational dashboard grouping. "
            "The approved operational model threshold "
            "remains 0.20."
        )
    },

    # -------------------------------------------------
    # GOVERNANCE
    # -------------------------------------------------

    "governance": {

        "model_is_legal_decision_engine": False,

        "model_is_payment_guarantee": False,

        "automatic_external_action_allowed": False,

        "human_review_required": True,

        "approved_use": (
            "Operational prioritization and workflow "
            "attention support."
        ),

        "prohibited_interpretations": [
            "legal rights determination",
            "automatic lien eligibility decision",
            "automatic bond-claim eligibility decision",
            "payment guarantee",
            "automatic customer action"
        ]
    },

    # -------------------------------------------------
    # EXPLANATION POLICY
    # -------------------------------------------------

    "explanation_policy": {

        "customer_safe_only": True,

        "expose_raw_coefficients": False,

        "expose_transformed_features": False,

        "expose_internal_log_odds": False,

        "approved_business_features": [

            "conflicting_project_information",

            "critical_field_missing",

            "multiple_candidate_records",

            "prior_escalation_rate",

            "payment_chain_completeness_score",

            "deadline_days_remaining",

            "research_confidence_score"
        ]
    },

    # -------------------------------------------------
    # DATA SOURCES
    # -------------------------------------------------

    "data_sources": {

        "portfolio_analytics": {
            "source": "AWS Athena",
            "database":
                "construction_payment_risk_dev",
            "table":
                "payment_risk"
        },

        "live_prediction": {
            "source":
                "Prediction API",
            "model_version":
                "logistic-regression-v1"
        },

        "monitoring": {
            "source":
                "Amazon CloudWatch",
            "note": (
                "System monitoring remains separate "
                "from business dashboard analytics."
            )
        }
    },

    # -------------------------------------------------
    # PRIVACY
    # -------------------------------------------------

    "privacy": {

        "production_raw_payload_logging": False,

        "sensitive_customer_data_required_for_demo":
            False,

        "portfolio_data_policy":
            "Synthetic/public-safe data only",

        "display_internal_model_details":
            False
    },

    # -------------------------------------------------
    # NON-GOALS
    # -------------------------------------------------

    "non_goals": [

        "replace human operational review",

        "provide legal advice",

        "determine statutory rights",

        "automatically send notices",

        "automatically file liens",

        "automatically initiate bond claims",

        "replace the CloudWatch monitoring dashboard"
    ]
}


with open(
    DASHBOARD_CONTRACT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dashboard_contract,
        f,
        indent=2
    )


print(
    "Dashboard contract:",
    DASHBOARD_CONTRACT_PATH
)

print(
    "17A dashboard product contract creation: PASS"
)

Dashboard contract: artifacts/dashboard/payment-risk-dashboard/v1/dashboard-product-contract-v1.json
17A dashboard product contract creation: PASS


In [250]:
with open(
    DASHBOARD_CONTRACT_PATH,
    "r",
    encoding="utf-8"
) as f:

    saved_contract = json.load(f)


assert (
    saved_contract[
        "model"
    ][
        "model_version"
    ]
    == "logistic-regression-v1"
)


assert (
    saved_contract[
        "model"
    ][
        "operational_threshold"
    ]
    == 0.20
)


assert (
    saved_contract[
        "governance"
    ][
        "model_is_legal_decision_engine"
    ]
    is False
)


assert (
    saved_contract[
        "governance"
    ][
        "automatic_external_action_allowed"
    ]
    is False
)


assert (
    saved_contract[
        "governance"
    ][
        "human_review_required"
    ]
    is True
)


assert (
    saved_contract[
        "privacy"
    ][
        "portfolio_data_policy"
    ]
    == "Synthetic/public-safe data only"
)


assert (
    saved_contract[
        "explanation_policy"
    ][
        "expose_raw_coefficients"
    ]
    is False
)


assert (
    len(
        saved_contract[
            "pages"
        ]
    )
    == 4
)


print(
    "Dashboard pages:",
    len(
        saved_contract[
            "pages"
        ]
    )
)

print(
    "Operational threshold:",
    saved_contract[
        "model"
    ][
        "operational_threshold"
    ]
)

print(
    "Human review required:",
    saved_contract[
        "governance"
    ][
        "human_review_required"
    ]
)

print()

print(
    "17A dashboard product contract validation: PASS"
)

Dashboard pages: 4
Operational threshold: 0.2
Human review required: True

17A dashboard product contract validation: PASS


In [251]:
import json
from pathlib import Path
from datetime import datetime, timezone


ARCH_PATH = (
    Path("artifacts/dashboard/payment-risk-dashboard/v1")
    / "dashboard-architecture-v1.json"
)

architecture = {
    "architecture_version": "1.0.0",
    "generated_at": datetime.now(timezone.utc).isoformat(),

    "dashboard": {
        "name": (
            "Construction Payment Protection "
            "Intelligence Dashboard"
        ),
        "framework": "Streamlit",
        "deployment_target": "AWS container deployment"
    },

    "analytics_path": {
        "purpose": (
            "Historical and aggregate business analytics"
        ),
        "source": "Amazon Athena",
        "database": "construction_payment_risk_dev",
        "table": "payment_risk",
        "storage": "Amazon S3 curated Parquet",
        "access_pattern": "read-only analytical queries"
    },

    "prediction_path": {
        "purpose": "Live single-workflow assessment",
        "source": "Prediction API",
        "endpoint": "/v1/predict",
        "health_endpoint": "/health",
        "model_endpoint": "/v1/model",
        "model_version": "logistic-regression-v1",
        "operational_threshold": 0.20
    },

    "feature_contract": [
        "state",
        "project_type",
        "public_private",
        "customer_role",
        "hiring_party_type",
        "payment_chain_completeness_score",
        "research_confidence_score",
        "deadline_days_remaining",
        "prior_escalation_rate",
        "prior_projects_with_hiring_party",
        "expected_party_count",
        "critical_field_missing",
        "multiple_candidate_records",
        "conflicting_project_information"
    ],

    "responsibility_boundaries": {
        "dashboard": [
            "collect user input",
            "call approved services",
            "query governed analytics",
            "render customer-safe explanations",
            "support human review"
        ],

        "prediction_api": [
            "validate request contract",
            "run model inference",
            "apply approved threshold",
            "return customer-safe explanation",
            "emit monitoring telemetry"
        ],

        "athena": [
            "serve curated historical analytics",
            "support portfolio-level aggregation",
            "avoid model inference logic"
        ],

        "cloudwatch": [
            "monitor service health",
            "monitor prediction behavior",
            "monitor drift and input quality",
            "remain separate from business analytics"
        ]
    },

    "security": {
        "dashboard_iam": "least privilege",
        "athena_access": "read-only",
        "s3_access": "curated + query results only",
        "api_secret_in_source_code": False,
        "raw_payload_logging": False
    },

    "failure_handling": {
        "athena_failure": (
            "Show analytics unavailable message; "
            "do not break live scoring."
        ),

        "prediction_api_failure": (
            "Show scoring unavailable message; "
            "do not fabricate a prediction."
        ),

        "partial_system_availability_supported": True
    }
}

with open(
    ARCH_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        architecture,
        f,
        indent=2
    )

print(
    "Architecture contract:",
    ARCH_PATH
)

print(
    "17B architecture contract creation: PASS"
)

Architecture contract: artifacts/dashboard/payment-risk-dashboard/v1/dashboard-architecture-v1.json
17B architecture contract creation: PASS


In [252]:
with open(
    ARCH_PATH,
    "r",
    encoding="utf-8"
) as f:
    saved_arch = json.load(f)


assert (
    saved_arch[
        "analytics_path"
    ][
        "source"
    ]
    == "Amazon Athena"
)

assert (
    saved_arch[
        "prediction_path"
    ][
        "source"
    ]
    == "Prediction API"
)

assert (
    saved_arch[
        "prediction_path"
    ][
        "model_version"
    ]
    == "logistic-regression-v1"
)

assert (
    saved_arch[
        "prediction_path"
    ][
        "operational_threshold"
    ]
    == 0.20
)

assert (
    len(
        saved_arch[
            "feature_contract"
        ]
    )
    == 14
)

assert (
    saved_arch[
        "failure_handling"
    ][
        "partial_system_availability_supported"
    ]
    is True
)

assert (
    saved_arch[
        "security"
    ][
        "raw_payload_logging"
    ]
    is False
)

print(
    "Feature count:",
    len(
        saved_arch[
            "feature_contract"
        ]
    )
)

print(
    "Analytics source:",
    saved_arch[
        "analytics_path"
    ][
        "source"
    ]
)

print(
    "Prediction source:",
    saved_arch[
        "prediction_path"
    ][
        "source"
    ]
)

print()

print(
    "17B architecture contract validation: PASS"
)

Feature count: 14
Analytics source: Amazon Athena
Prediction source: Prediction API

17B architecture contract validation: PASS


# Verify the Athena analytics source

In [253]:
import boto3
import time

REGION = "ap-southeast-2"

ATHENA_DATABASE = (
    "construction_payment_risk_dev"
)

ATHENA_TABLE = "payment_risk"

ATHENA_OUTPUT = (
    "s3://construction-payment-risk-dev-gk53/"
    "athena-results/dashboard/"
)

athena = boto3.client(
    "athena",
    region_name=REGION
)


def run_athena_query(sql):
    response = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={
            "Database": ATHENA_DATABASE
        },
        ResultConfiguration={
            "OutputLocation": ATHENA_OUTPUT
        }
    )

    execution_id = response[
        "QueryExecutionId"
    ]

    while True:
        status = athena.get_query_execution(
            QueryExecutionId=execution_id
        )

        state = status[
            "QueryExecution"
        ][
            "Status"
        ][
            "State"
        ]

        if state in {
            "SUCCEEDED",
            "FAILED",
            "CANCELLED"
        }:
            break

        time.sleep(1)

    if state != "SUCCEEDED":
        reason = status[
            "QueryExecution"
        ][
            "Status"
        ].get(
            "StateChangeReason",
            "Unknown Athena failure"
        )

        raise RuntimeError(
            f"Athena query {state}: {reason}"
        )

    return athena.get_query_results(
        QueryExecutionId=execution_id
    )


result = run_athena_query(
    f"""
    SELECT COUNT(*) AS total_workflows
    FROM {ATHENA_TABLE}
    """
)

rows = result["ResultSet"]["Rows"]

total_workflows = int(
    rows[1]["Data"][0]["VarCharValue"]
)

print(
    "Athena total workflows:",
    total_workflows
)

assert total_workflows == 10000

print(
    "17C Athena connectivity: PASS"
)

RuntimeError: Athena query FAILED: Insufficient permissions to execute the query.  User: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker is not authorized to perform: glue:GetPartitions on resource: arn:aws:glue:ap-southeast-2:911797457166:catalog because no identity-based policy allows the glue:GetPartitions action 

In [257]:
result = run_athena_query(
    f"""
    SELECT COUNT(*) AS total_workflows
    FROM {ATHENA_TABLE}
    """
)

rows = result["ResultSet"]["Rows"]

total_workflows = int(
    rows[1]["Data"][0]["VarCharValue"]
)

print("Athena total workflows:", total_workflows)

assert total_workflows == 10000

print("17C Athena connectivity: PASS")

RuntimeError: Athena query FAILED: Insufficient permissions to execute the query.  User: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker is not authorized to perform: glue:GetPartitions on resource: arn:aws:glue:ap-southeast-2:911797457166:catalog because no identity-based policy allows the glue:GetPartitions action 

In [256]:
import boto3

glue = boto3.client(
    "glue",
    region_name="ap-southeast-2"
)

response = glue.get_partitions(
    DatabaseName="construction_payment_risk_dev",
    TableName="payment_risk",
    MaxResults=1
)

print(
    "Partitions returned:",
    len(response.get("Partitions", []))
)

print(
    "Glue GetPartitions permission: PASS"
)

Partitions returned: 1
Glue GetPartitions permission: PASS


In [258]:
result = run_athena_query(
    f"""
    SELECT COUNT(*) AS total_workflows
    FROM {ATHENA_TABLE}
    """
)

rows = result[
    "ResultSet"
]["Rows"]

total_workflows = int(
    rows[1][
        "Data"
    ][0][
        "VarCharValue"
    ]
)

print(
    "Athena total workflows:",
    total_workflows
)

assert total_workflows == 10000

print(
    "17C Athena connectivity: PASS"
)

RuntimeError: Athena query FAILED: Access denied when writing output to url: s3://construction-payment-risk-dev-gk53/athena-results/dashboard/dbf817cc-ad5e-40d6-a837-06568a523314.csv . Please ensure you are allowed to access the S3 bucket. If specifying an expected bucket owner, confirm the bucket is owned by the expected account. If you are encrypting query results with KMS key, please ensure you are allowed to access your KMS key

In [259]:
import boto3

s3 = boto3.client(
    "s3",
    region_name="ap-southeast-2"
)

bucket = "construction-payment-risk-dev-gk53"
key = "athena-results/dashboard/permission-test.txt"

s3.put_object(
    Bucket=bucket,
    Key=key,
    Body=b"permission-test"
)

print("Athena results S3 write permission: PASS")

Athena results S3 write permission: PASS


In [260]:
s3.delete_object(
    Bucket=bucket,
    Key=key
)

print("Athena results S3 cleanup: PASS")

AccessDenied: An error occurred (AccessDenied) when calling the DeleteObject operation: User: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker is not authorized to perform: s3:DeleteObject on resource: "arn:aws:s3:::construction-payment-risk-dev-gk53/athena-results/dashboard/permission-test.txt" because no identity-based policy allows the s3:DeleteObject action

In [261]:
s3.delete_object(
    Bucket="construction-payment-risk-dev-gk53",
    Key="athena-results/dashboard/permission-test.txt"
)

print("Athena results S3 cleanup: PASS")

AccessDenied: An error occurred (AccessDenied) when calling the DeleteObject operation: User: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker is not authorized to perform: s3:DeleteObject on resource: "arn:aws:s3:::construction-payment-risk-dev-gk53/athena-results/dashboard/permission-test.txt" because no identity-based policy allows the s3:DeleteObject action

In [262]:
result = run_athena_query(
    f"""
    SELECT COUNT(*) AS total_workflows
    FROM {ATHENA_TABLE}
    """
)

rows = result["ResultSet"]["Rows"]

total_workflows = int(
    rows[1]["Data"][0]["VarCharValue"]
)

print(
    "Athena total workflows:",
    total_workflows
)

assert total_workflows == 10000

print(
    "17C Athena connectivity: PASS"
)

Athena total workflows: 10000
17C Athena connectivity: PASS


In [263]:
kpi_result = run_athena_query(
    f"""
    SELECT
        COUNT(*) AS total_workflows,

        ROUND(
            AVG(payment_chain_completeness_score),
            4
        ) AS avg_chain_completeness,

        ROUND(
            AVG(research_confidence_score),
            4
        ) AS avg_research_confidence,

        SUM(
            CASE
                WHEN critical_field_missing = 1
                THEN 1
                ELSE 0
            END
        ) AS critical_missing_count,

        SUM(
            CASE
                WHEN multiple_candidate_records = 1
                THEN 1
                ELSE 0
            END
        ) AS multiple_candidate_count,

        SUM(
            CASE
                WHEN conflicting_project_information = 1
                THEN 1
                ELSE 0
            END
        ) AS conflicting_info_count,

        SUM(
            CASE
                WHEN deadline_days_remaining <= 15
                THEN 1
                ELSE 0
            END
        ) AS deadline_15d_count

    FROM {ATHENA_TABLE}
    """
)

data = (
    kpi_result[
        "ResultSet"
    ][
        "Rows"
    ][1][
        "Data"
    ]
)

values = [
    item.get("VarCharValue")
    for item in data
]

print("Total workflows:", values[0])
print("Average chain completeness:", values[1])
print("Average research confidence:", values[2])
print("Critical-field missing:", values[3])
print("Multiple candidate records:", values[4])
print("Conflicting project information:", values[5])
print("Deadline <= 15 days:", values[6])

assert int(values[0]) == 10000

print()
print("17C governed business KPI query: PASS")

Total workflows: 10000
Average chain completeness: 0.7956
Average research confidence: 0.7741
Critical-field missing: 1814
Multiple candidate records: 1590
Conflicting project information: 1132
Deadline <= 15 days: 3150

17C governed business KPI query: PASS


In [264]:
segment_result = run_athena_query(
    f"""
    SELECT
        state,

        COUNT(*) AS workflow_count,

        ROUND(
            AVG(payment_chain_completeness_score),
            4
        ) AS avg_chain_completeness,

        ROUND(
            AVG(research_confidence_score),
            4
        ) AS avg_research_confidence,

        SUM(
            CASE
                WHEN critical_field_missing = 1
                THEN 1
                ELSE 0
            END
        ) AS critical_missing_count

    FROM {ATHENA_TABLE}

    GROUP BY state

    ORDER BY workflow_count DESC
    """
)

rows = segment_result[
    "ResultSet"
]["Rows"]

headers = [
    item["VarCharValue"]
    for item in rows[0]["Data"]
]

records = []

for row in rows[1:]:

    row_values = [
        item.get("VarCharValue")
        for item in row["Data"]
    ]

    records.append(
        dict(
            zip(
                headers,
                row_values
            )
        )
    )

print(
    "State segments:",
    len(records)
)

for record in records[:5]:
    print(record)

segment_total = sum(
    int(
        record[
            "workflow_count"
        ]
    )
    for record in records
)

print(
    "Segment workflow total:",
    segment_total
)

assert records
assert segment_total == 10000

print()
print("17C segment analytics: PASS")

State segments: 7
{'state': 'FL', 'workflow_count': '3028', 'avg_chain_completeness': '0.7902', 'avg_research_confidence': '0.7729', 'critical_missing_count': '535'}
{'state': 'TX', 'workflow_count': '1964', 'avg_chain_completeness': '0.7975', 'avg_research_confidence': '0.7756', 'critical_missing_count': '361'}
{'state': 'CA', 'workflow_count': '1580', 'avg_chain_completeness': '0.79', 'avg_research_confidence': '0.7748', 'critical_missing_count': '310'}
{'state': 'GA', 'workflow_count': '1161', 'avg_chain_completeness': '0.7977', 'avg_research_confidence': '0.7762', 'critical_missing_count': '196'}
{'state': 'NC', 'workflow_count': '970', 'avg_chain_completeness': '0.8052', 'avg_research_confidence': '0.7751', 'critical_missing_count': '178'}
Segment workflow total: 10000

17C segment analytics: PASS


```text

Create a dashboard application structure like this:

Streamlit UI
    ↓
Dashboard Service
    ↓
Athena Repository
    ↓
Athena
    ↓
Glue + S3 curated Parquet


dashboard/
├── __init__.py
├── config.py
├── repositories/
│   ├── __init__.py
│   └── athena_repository.py
└── services/
    ├── __init__.py
    └── analytics_service.py

```

In [266]:
from pathlib import Path

BASE = Path("dashboard")

paths = [
    BASE / "__init__.py",
    BASE / "config.py",
    BASE / "repositories" / "__init__.py",
    BASE / "repositories" / "athena_repository.py",
    BASE / "services" / "__init__.py",
    BASE / "services" / "analytics_service.py",
]

for path in paths:
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    path.touch(
        exist_ok=True
    )

print("Dashboard application structure: PASS")

Dashboard application structure: PASS


In [267]:
from pathlib import Path

config_code = '''
AWS_REGION = "ap-southeast-2"

ATHENA_DATABASE = (
    "construction_payment_risk_dev"
)

ATHENA_TABLE = "payment_risk"

ATHENA_OUTPUT = (
    "s3://construction-payment-risk-dev-gk53/"
    "athena-results/dashboard/"
)

ATHENA_POLL_INTERVAL_SECONDS = 1
'''

Path(
    "dashboard/config.py"
).write_text(
    config_code.strip() + "\\n",
    encoding="utf-8"
)

print("Dashboard configuration module: PASS")

Dashboard configuration module: PASS


In [268]:
from pathlib import Path

repository_code = '''
import time
from typing import Any

import boto3

from dashboard.config import (
    AWS_REGION,
    ATHENA_DATABASE,
    ATHENA_OUTPUT,
    ATHENA_POLL_INTERVAL_SECONDS,
    ATHENA_TABLE,
)


class AthenaRepository:

    def __init__(self):
        self.client = boto3.client(
            "athena",
            region_name=AWS_REGION,
        )

    def execute_query(
        self,
        sql: str,
    ) -> list[dict[str, Any]]:

        response = (
            self.client.start_query_execution(
                QueryString=sql,
                QueryExecutionContext={
                    "Database":
                        ATHENA_DATABASE
                },
                ResultConfiguration={
                    "OutputLocation":
                        ATHENA_OUTPUT
                },
            )
        )

        query_id = response[
            "QueryExecutionId"
        ]

        while True:

            execution = (
                self.client
                .get_query_execution(
                    QueryExecutionId=query_id
                )
            )

            status = execution[
                "QueryExecution"
            ][
                "Status"
            ]

            state = status[
                "State"
            ]

            if state in {
                "SUCCEEDED",
                "FAILED",
                "CANCELLED",
            }:
                break

            time.sleep(
                ATHENA_POLL_INTERVAL_SECONDS
            )

        if state != "SUCCEEDED":

            reason = status.get(
                "StateChangeReason",
                "Unknown Athena error",
            )

            raise RuntimeError(
                f"Athena query {state}: "
                f"{reason}"
            )

        result = (
            self.client
            .get_query_results(
                QueryExecutionId=query_id
            )
        )

        rows = result[
            "ResultSet"
        ][
            "Rows"
        ]

        if not rows:
            return []

        headers = [
            column.get(
                "VarCharValue",
                ""
            )
            for column
            in rows[0]["Data"]
        ]

        records = []

        for row in rows[1:]:

            values = [
                column.get(
                    "VarCharValue"
                )
                for column
                in row["Data"]
            ]

            records.append(
                dict(
                    zip(
                        headers,
                        values
                    )
                )
            )

        return records

    def get_portfolio_kpis(
        self,
    ) -> dict[str, Any]:

        sql = f"""
        SELECT
            COUNT(*) AS total_workflows,

            ROUND(
                AVG(
                    payment_chain_completeness_score
                ),
                4
            ) AS avg_chain_completeness,

            ROUND(
                AVG(
                    research_confidence_score
                ),
                4
            ) AS avg_research_confidence,

            SUM(
                CASE
                    WHEN critical_field_missing = 1
                    THEN 1
                    ELSE 0
                END
            ) AS critical_missing_count,

            SUM(
                CASE
                    WHEN multiple_candidate_records = 1
                    THEN 1
                    ELSE 0
                END
            ) AS multiple_candidate_count,

            SUM(
                CASE
                    WHEN conflicting_project_information = 1
                    THEN 1
                    ELSE 0
                END
            ) AS conflicting_info_count,

            SUM(
                CASE
                    WHEN deadline_days_remaining <= 15
                    THEN 1
                    ELSE 0
                END
            ) AS deadline_15d_count

        FROM {ATHENA_TABLE}
        """

        records = self.execute_query(sql)

        if not records:
            return {}

        return records[0]

    def get_state_segments(
        self,
    ) -> list[dict[str, Any]]:

        sql = f"""
        SELECT
            state,

            COUNT(*) AS workflow_count,

            ROUND(
                AVG(
                    payment_chain_completeness_score
                ),
                4
            ) AS avg_chain_completeness,

            ROUND(
                AVG(
                    research_confidence_score
                ),
                4
            ) AS avg_research_confidence,

            SUM(
                CASE
                    WHEN critical_field_missing = 1
                    THEN 1
                    ELSE 0
                END
            ) AS critical_missing_count

        FROM {ATHENA_TABLE}

        GROUP BY state

        ORDER BY workflow_count DESC
        """

        return self.execute_query(sql)
'''

Path(
    "dashboard/repositories/"
    "athena_repository.py"
).write_text(
    repository_code.strip() + "\\n",
    encoding="utf-8"
)

print("Athena repository module: PASS")

Athena repository module: PASS


Now create a service layer. The service will convert Athena’s string-based results into proper Python types so the UI doesn’t need to know anything about Athena formatting.

In [269]:
from pathlib import Path

service_code = '''
from dashboard.repositories.athena_repository import (
    AthenaRepository,
)


class AnalyticsService:

    def __init__(
        self,
        repository=None,
    ):
        self.repository = (
            repository
            or AthenaRepository()
        )

    def get_portfolio_summary(
        self,
    ) -> dict:

        raw = (
            self.repository
            .get_portfolio_kpis()
        )

        if not raw:
            return {}

        total = int(
            raw["total_workflows"]
        )

        critical = int(
            raw[
                "critical_missing_count"
            ]
        )

        multiple = int(
            raw[
                "multiple_candidate_count"
            ]
        )

        conflicting = int(
            raw[
                "conflicting_info_count"
            ]
        )

        deadline = int(
            raw[
                "deadline_15d_count"
            ]
        )

        return {
            "total_workflows":
                total,

            "avg_chain_completeness":
                float(
                    raw[
                        "avg_chain_completeness"
                    ]
                ),

            "avg_research_confidence":
                float(
                    raw[
                        "avg_research_confidence"
                    ]
                ),

            "critical_missing_count":
                critical,

            "critical_missing_rate":
                critical / total
                if total
                else 0.0,

            "multiple_candidate_count":
                multiple,

            "multiple_candidate_rate":
                multiple / total
                if total
                else 0.0,

            "conflicting_info_count":
                conflicting,

            "conflicting_info_rate":
                conflicting / total
                if total
                else 0.0,

            "deadline_15d_count":
                deadline,

            "deadline_15d_rate":
                deadline / total
                if total
                else 0.0,
        }

    def get_state_breakdown(
        self,
    ) -> list[dict]:

        rows = (
            self.repository
            .get_state_segments()
        )

        result = []

        for row in rows:

            count = int(
                row[
                    "workflow_count"
                ]
            )

            critical = int(
                row[
                    "critical_missing_count"
                ]
            )

            result.append({
                "state":
                    row["state"],

                "workflow_count":
                    count,

                "avg_chain_completeness":
                    float(
                        row[
                            "avg_chain_completeness"
                        ]
                    ),

                "avg_research_confidence":
                    float(
                        row[
                            "avg_research_confidence"
                        ]
                    ),

                "critical_missing_count":
                    critical,

                "critical_missing_rate":
                    (
                        critical / count
                        if count
                        else 0.0
                    ),
            })

        return result
'''

Path(
    "dashboard/services/"
    "analytics_service.py"
).write_text(
    service_code.strip() + "\\n",
    encoding="utf-8"
)

print("Analytics service module: PASS")

Analytics service module: PASS


In [274]:
import importlib

import dashboard.repositories.athena_repository as repo_module
import dashboard.services.analytics_service as service_module

importlib.reload(repo_module)
importlib.reload(service_module)

AnalyticsService = (
    service_module.AnalyticsService
)

service = AnalyticsService()

summary = (
    service.get_portfolio_summary()
)

print("Portfolio summary:")
print(summary)

assert (
    summary["total_workflows"]
    == 10000
)

assert (
    0
    <= summary[
        "avg_chain_completeness"
    ]
    <= 1
)

assert (
    0
    <= summary[
        "avg_research_confidence"
    ]
    <= 1
)

assert (
    0
    <= summary[
        "critical_missing_rate"
    ]
    <= 1
)

print()
print(
    "17C.4 analytics repository "
    "portfolio test: PASS"
)

SyntaxError: unexpected character after line continuation character (config.py, line 14)

In [273]:
states = (
    service.get_state_breakdown()
)

print(
    "State rows:",
    len(states)
)

print()

for state in states[:5]:
    print(state)

total_from_states = sum(
    row[
        "workflow_count"
    ]
    for row in states
)

assert states

assert (
    total_from_states
    == 10000
)

assert all(
    0
    <= row[
        "critical_missing_rate"
    ]
    <= 1
    for row in states
)

print()
print(
    "17C.4 analytics repository "
    "segment test: PASS"
)

NameError: name 'service' is not defined

In [272]:
from pathlib import Path

repository_code = '''import time
from typing import Any

import boto3

from dashboard.config import (
    AWS_REGION,
    ATHENA_DATABASE,
    ATHENA_OUTPUT,
    ATHENA_POLL_INTERVAL_SECONDS,
    ATHENA_TABLE,
)


class AthenaRepository:
    """Read-only repository for dashboard analytics in Amazon Athena."""

    def __init__(self):
        self.client = boto3.client(
            "athena",
            region_name=AWS_REGION,
        )

    def execute_query(
        self,
        sql: str,
    ) -> list[dict[str, Any]]:

        response = self.client.start_query_execution(
            QueryString=sql,
            QueryExecutionContext={
                "Database": ATHENA_DATABASE,
            },
            ResultConfiguration={
                "OutputLocation": ATHENA_OUTPUT,
            },
        )

        query_id = response["QueryExecutionId"]

        while True:
            execution = self.client.get_query_execution(
                QueryExecutionId=query_id
            )

            status = execution[
                "QueryExecution"
            ]["Status"]

            state = status["State"]

            if state in {
                "SUCCEEDED",
                "FAILED",
                "CANCELLED",
            }:
                break

            time.sleep(
                ATHENA_POLL_INTERVAL_SECONDS
            )

        if state != "SUCCEEDED":
            reason = status.get(
                "StateChangeReason",
                "Unknown Athena error",
            )

            raise RuntimeError(
                f"Athena query {state}: {reason}"
            )

        result = self.client.get_query_results(
            QueryExecutionId=query_id
        )

        rows = result[
            "ResultSet"
        ]["Rows"]

        if not rows:
            return []

        headers = [
            column.get(
                "VarCharValue",
                ""
            )
            for column in rows[0]["Data"]
        ]

        records = []

        for row in rows[1:]:
            values = [
                column.get("VarCharValue")
                for column in row["Data"]
            ]

            records.append(
                dict(zip(headers, values))
            )

        return records

    def get_portfolio_kpis(
        self,
    ) -> dict[str, Any]:

        sql = f"""
        SELECT
            COUNT(*) AS total_workflows,

            ROUND(
                AVG(payment_chain_completeness_score),
                4
            ) AS avg_chain_completeness,

            ROUND(
                AVG(research_confidence_score),
                4
            ) AS avg_research_confidence,

            SUM(
                CASE
                    WHEN critical_field_missing = 1
                    THEN 1
                    ELSE 0
                END
            ) AS critical_missing_count,

            SUM(
                CASE
                    WHEN multiple_candidate_records = 1
                    THEN 1
                    ELSE 0
                END
            ) AS multiple_candidate_count,

            SUM(
                CASE
                    WHEN conflicting_project_information = 1
                    THEN 1
                    ELSE 0
                END
            ) AS conflicting_info_count,

            SUM(
                CASE
                    WHEN deadline_days_remaining <= 15
                    THEN 1
                    ELSE 0
                END
            ) AS deadline_15d_count

        FROM {ATHENA_TABLE}
        """

        records = self.execute_query(sql)

        if not records:
            return {}

        return records[0]

    def get_state_segments(
        self,
    ) -> list[dict[str, Any]]:

        sql = f"""
        SELECT
            state,

            COUNT(*) AS workflow_count,

            ROUND(
                AVG(payment_chain_completeness_score),
                4
            ) AS avg_chain_completeness,

            ROUND(
                AVG(research_confidence_score),
                4
            ) AS avg_research_confidence,

            SUM(
                CASE
                    WHEN critical_field_missing = 1
                    THEN 1
                    ELSE 0
                END
            ) AS critical_missing_count

        FROM {ATHENA_TABLE}

        GROUP BY state

        ORDER BY workflow_count DESC
        """

        return self.execute_query(sql)
'''

repo_path = Path(
    "dashboard/repositories/athena_repository.py"
)

repo_path.write_text(
    repository_code,
    encoding="utf-8"
)

print(
    "Repository rewritten:",
    repo_path
)

print(
    "Athena repository syntax repair: PASS"
)

Repository rewritten: dashboard/repositories/athena_repository.py
Athena repository syntax repair: PASS


In [275]:
import py_compile

py_compile.compile(
    "dashboard/repositories/athena_repository.py",
    doraise=True,
)

print(
    "Athena repository Python compile: PASS"
)

Athena repository Python compile: PASS


In [276]:
import sys
import importlib

importlib.invalidate_caches()

sys.modules.pop(
    "dashboard.repositories.athena_repository",
    None
)

sys.modules.pop(
    "dashboard.services.analytics_service",
    None
)

from dashboard.repositories.athena_repository import (
    AthenaRepository,
)

from dashboard.services.analytics_service import (
    AnalyticsService,
)

print(
    "Dashboard analytics modules import: PASS"
)

SyntaxError: unexpected character after line continuation character (config.py, line 14)

In [277]:
from pathlib import Path

config_code = '''AWS_REGION = "ap-southeast-2"

ATHENA_DATABASE = "construction_payment_risk_dev"

ATHENA_TABLE = "payment_risk"

ATHENA_OUTPUT = (
    "s3://construction-payment-risk-dev-gk53/"
    "athena-results/dashboard/"
)

ATHENA_POLL_INTERVAL_SECONDS = 1
'''

Path("dashboard/config.py").write_text(
    config_code,
    encoding="utf-8"
)

print("Dashboard config syntax repair: PASS")

Dashboard config syntax repair: PASS


In [278]:
import py_compile
from pathlib import Path

python_files = list(
    Path("dashboard").rglob("*.py")
)

print("Dashboard Python files:")

for file in python_files:
    print(" -", file)

for file in python_files:
    py_compile.compile(
        str(file),
        doraise=True,
    )

print()
print("All dashboard Python modules compile: PASS")

Dashboard Python files:
 - dashboard/__init__.py
 - dashboard/config.py
 - dashboard/repositories/__init__.py
 - dashboard/repositories/athena_repository.py
 - dashboard/services/__init__.py
 - dashboard/services/analytics_service.py


PyCompileError:   File "dashboard/services/analytics_service.py", line 165
    return result\n
                  ^
SyntaxError: unexpected character after line continuation character


In [279]:
from pathlib import Path

service_code = '''from dashboard.repositories.athena_repository import AthenaRepository


class AnalyticsService:
    """Business analytics layer for the payment-risk dashboard."""

    def __init__(
        self,
        repository=None,
    ):
        self.repository = (
            repository
            or AthenaRepository()
        )

    def get_portfolio_summary(
        self,
    ) -> dict:

        raw = self.repository.get_portfolio_kpis()

        if not raw:
            return {}

        total = int(
            raw["total_workflows"]
        )

        critical = int(
            raw["critical_missing_count"]
        )

        multiple = int(
            raw["multiple_candidate_count"]
        )

        conflicting = int(
            raw["conflicting_info_count"]
        )

        deadline = int(
            raw["deadline_15d_count"]
        )

        return {
            "total_workflows": total,

            "avg_chain_completeness": float(
                raw[
                    "avg_chain_completeness"
                ]
            ),

            "avg_research_confidence": float(
                raw[
                    "avg_research_confidence"
                ]
            ),

            "critical_missing_count": critical,

            "critical_missing_rate": (
                critical / total
                if total
                else 0.0
            ),

            "multiple_candidate_count": multiple,

            "multiple_candidate_rate": (
                multiple / total
                if total
                else 0.0
            ),

            "conflicting_info_count": conflicting,

            "conflicting_info_rate": (
                conflicting / total
                if total
                else 0.0
            ),

            "deadline_15d_count": deadline,

            "deadline_15d_rate": (
                deadline / total
                if total
                else 0.0
            ),
        }

    def get_state_breakdown(
        self,
    ) -> list[dict]:

        rows = (
            self.repository
            .get_state_segments()
        )

        result = []

        for row in rows:

            count = int(
                row["workflow_count"]
            )

            critical = int(
                row[
                    "critical_missing_count"
                ]
            )

            result.append(
                {
                    "state": row["state"],

                    "workflow_count": count,

                    "avg_chain_completeness": float(
                        row[
                            "avg_chain_completeness"
                        ]
                    ),

                    "avg_research_confidence": float(
                        row[
                            "avg_research_confidence"
                        ]
                    ),

                    "critical_missing_count": critical,

                    "critical_missing_rate": (
                        critical / count
                        if count
                        else 0.0
                    ),
                }
            )

        return result
'''

Path(
    "dashboard/services/analytics_service.py"
).write_text(
    service_code,
    encoding="utf-8"
)

print("Analytics service syntax repair: PASS")

Analytics service syntax repair: PASS


In [280]:
import py_compile
from pathlib import Path

python_files = list(
    Path("dashboard").rglob("*.py")
)

for file in python_files:
    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "All dashboard Python modules compile: PASS"
)

All dashboard Python modules compile: PASS


In [281]:
import sys
import importlib

importlib.invalidate_caches()

for module_name in [
    "dashboard.config",
    "dashboard.repositories.athena_repository",
    "dashboard.services.analytics_service",
]:
    sys.modules.pop(
        module_name,
        None,
    )

from dashboard.repositories.athena_repository import (
    AthenaRepository,
)

from dashboard.services.analytics_service import (
    AnalyticsService,
)

print(
    "Dashboard analytics modules import: PASS"
)

Dashboard analytics modules import: PASS


In [282]:
service = AnalyticsService()

summary = service.get_portfolio_summary()

print("Portfolio summary:")
print(summary)

assert summary["total_workflows"] == 10000
assert 0 <= summary["avg_chain_completeness"] <= 1
assert 0 <= summary["avg_research_confidence"] <= 1
assert 0 <= summary["critical_missing_rate"] <= 1

print()
print("17C.4 analytics repository portfolio test: PASS")

Portfolio summary:
{'total_workflows': 10000, 'avg_chain_completeness': 0.7956, 'avg_research_confidence': 0.7741, 'critical_missing_count': 1814, 'critical_missing_rate': 0.1814, 'multiple_candidate_count': 1590, 'multiple_candidate_rate': 0.159, 'conflicting_info_count': 1132, 'conflicting_info_rate': 0.1132, 'deadline_15d_count': 3150, 'deadline_15d_rate': 0.315}

17C.4 analytics repository portfolio test: PASS


In [283]:
states = service.get_state_breakdown()

print("State rows:", len(states))

for state in states[:5]:
    print(state)

total_from_states = sum(
    row["workflow_count"]
    for row in states
)

print(
    "Total workflows from state segments:",
    total_from_states
)

assert states
assert total_from_states == 10000

assert all(
    0 <= row["critical_missing_rate"] <= 1
    for row in states
)

print()
print("17C.4 analytics repository segment test: PASS")

State rows: 7
{'state': 'FL', 'workflow_count': 3028, 'avg_chain_completeness': 0.7902, 'avg_research_confidence': 0.7729, 'critical_missing_count': 535, 'critical_missing_rate': 0.17668428005284015}
{'state': 'TX', 'workflow_count': 1964, 'avg_chain_completeness': 0.7975, 'avg_research_confidence': 0.7756, 'critical_missing_count': 361, 'critical_missing_rate': 0.18380855397148677}
{'state': 'CA', 'workflow_count': 1580, 'avg_chain_completeness': 0.79, 'avg_research_confidence': 0.7748, 'critical_missing_count': 310, 'critical_missing_rate': 0.1962025316455696}
{'state': 'GA', 'workflow_count': 1161, 'avg_chain_completeness': 0.7977, 'avg_research_confidence': 0.7762, 'critical_missing_count': 196, 'critical_missing_rate': 0.16881998277347116}
{'state': 'NC', 'workflow_count': 970, 'avg_chain_completeness': 0.8052, 'avg_research_confidence': 0.7751, 'critical_missing_count': 178, 'critical_missing_rate': 0.18350515463917524}
Total workflows from state segments: 10000

17C.4 analytics 

In [284]:
from pathlib import Path

app_code = '''import pandas as pd
import streamlit as st

from dashboard.services.analytics_service import AnalyticsService


st.set_page_config(
    page_title="Construction Payment Protection Intelligence",
    page_icon="🏗️",
    layout="wide",
)


@st.cache_resource
def get_analytics_service():
    return AnalyticsService()


@st.cache_data(ttl=300)
def load_portfolio_summary():
    service = get_analytics_service()
    return service.get_portfolio_summary()


@st.cache_data(ttl=300)
def load_state_breakdown():
    service = get_analytics_service()
    return service.get_state_breakdown()


st.title(
    "Construction Payment Protection Intelligence"
)

st.caption(
    "Operational portfolio intelligence for "
    "construction payment-protection workflows."
)


try:
    summary = load_portfolio_summary()
    states = load_state_breakdown()

except Exception as exc:
    st.error(
        "Portfolio analytics are temporarily unavailable."
    )

    st.exception(exc)

    st.stop()


# ---------------------------------------------------------
# KPI SECTION
# ---------------------------------------------------------

st.subheader("Portfolio Overview")

row1 = st.columns(4)

row1[0].metric(
    "Total Workflows",
    f"{summary['total_workflows']:,}",
)

row1[1].metric(
    "Avg Chain Completeness",
    f"{summary['avg_chain_completeness']:.1%}",
)

row1[2].metric(
    "Avg Research Confidence",
    f"{summary['avg_research_confidence']:.1%}",
)

row1[3].metric(
    "Deadline ≤ 15 Days",
    f"{summary['deadline_15d_count']:,}",
    delta=(
        f"{summary['deadline_15d_rate']:.1%} "
        "of portfolio"
    ),
    delta_color="off",
)


row2 = st.columns(3)

row2[0].metric(
    "Critical Information Missing",
    f"{summary['critical_missing_count']:,}",
    delta=(
        f"{summary['critical_missing_rate']:.1%}"
    ),
    delta_color="off",
)

row2[1].metric(
    "Multiple Candidate Records",
    f"{summary['multiple_candidate_count']:,}",
    delta=(
        f"{summary['multiple_candidate_rate']:.1%}"
    ),
    delta_color="off",
)

row2[2].metric(
    "Conflicting Project Information",
    f"{summary['conflicting_info_count']:,}",
    delta=(
        f"{summary['conflicting_info_rate']:.1%}"
    ),
    delta_color="off",
)


# ---------------------------------------------------------
# STATE ANALYTICS
# ---------------------------------------------------------

st.divider()

st.subheader("State-Level Operational Context")

state_df = pd.DataFrame(states)

if state_df.empty:
    st.info(
        "No state-level analytics are available."
    )

else:
    display_df = state_df.copy()

    display_df[
        "avg_chain_completeness"
    ] = (
        display_df[
            "avg_chain_completeness"
        ] * 100
    )

    display_df[
        "avg_research_confidence"
    ] = (
        display_df[
            "avg_research_confidence"
        ] * 100
    )

    display_df[
        "critical_missing_rate"
    ] = (
        display_df[
            "critical_missing_rate"
        ] * 100
    )

    st.bar_chart(
        state_df.set_index(
            "state"
        )[
            "workflow_count"
        ],
        height=350,
    )

    st.dataframe(
        display_df,
        use_container_width=True,
        hide_index=True,
        column_config={
            "state":
                "State",

            "workflow_count":
                st.column_config.NumberColumn(
                    "Workflows",
                    format="%d",
                ),

            "avg_chain_completeness":
                st.column_config.NumberColumn(
                    "Chain Completeness %",
                    format="%.1f",
                ),

            "avg_research_confidence":
                st.column_config.NumberColumn(
                    "Research Confidence %",
                    format="%.1f",
                ),

            "critical_missing_count":
                st.column_config.NumberColumn(
                    "Critical Missing",
                    format="%d",
                ),

            "critical_missing_rate":
                st.column_config.NumberColumn(
                    "Critical Missing %",
                    format="%.1f",
                ),
        },
    )


# ---------------------------------------------------------
# GOVERNANCE
# ---------------------------------------------------------

st.divider()

st.info(
    "This dashboard supports operational prioritization "
    "and workflow review. It does not determine legal "
    "rights, lien eligibility, bond-claim eligibility, "
    "or guarantee payment outcomes."
)
'''

Path(
    "dashboard/app.py"
).write_text(
    app_code,
    encoding="utf-8"
)

print(
    "17D Streamlit portfolio overview source: PASS"
)

17D Streamlit portfolio overview source: PASS


In [285]:
import py_compile
from pathlib import Path

python_files = list(
    Path("dashboard").rglob("*.py")
)

for file in python_files:
    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17D dashboard package compile: PASS"
)

17D dashboard package compile: PASS


In [286]:
import importlib.util

streamlit_available = (
    importlib.util.find_spec(
        "streamlit"
    )
    is not None
)

print(
    "Streamlit installed:",
    streamlit_available
)

Streamlit installed: False


In [287]:
%pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 60.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 47.3 MB/s  0:00:00m0:00:01
  Attempting uninstall: websockets
    Found existing installation: websockets 17.0.1
    Uninstalling websockets-17.0.1:
      Successfully uninstalled websockets-17.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [streamlit]/4 [streamlit]
Note: you may need to restart the kernel to use updated packages.


In [288]:
import streamlit

print(
    "Streamlit version:",
    streamlit.__version__
)

print(
    "17D Streamlit dependency: PASS"
)

Streamlit version: 1.63.0
17D Streamlit dependency: PASS


SyntaxError: invalid syntax (1953132705.py, line 1)

In [2]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "dashboard/app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Streamlit process started")
print("PID:", process.pid)

Streamlit process started
PID: 43670


In [3]:
import time

time.sleep(3)

print(
    "Process running:",
    process.poll() is None
)

Process running: True


In [4]:
for _ in range(15):
    line = process.stdout.readline()

    if not line:
        break

    print(line.rstrip())



2026-09-14 06:47:25.412 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://169.255.255.2:8501
  External URL: http://54.153.159.231:8501

2026-09-14 06:50:51.351 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.



# Candidate Workflow Repository Query
First we need Athena to return individual workflows with exactly the fields needed for later Prediction API scoring. We are not using escalation_required as a prediction.

In [5]:
from pathlib import Path

path = Path(
    "dashboard/repositories/athena_repository.py"
)

source = path.read_text(encoding="utf-8")

method = '''
    def get_candidate_workflows(
        self,
        limit: int = 100,
    ) -> list[dict]:
        """
        Return candidate workflows for operational review.

        These are curated workflow records, not model predictions.
        """

        if not 1 <= limit <= 500:
            raise ValueError(
                "limit must be between 1 and 500"
            )

        sql = f"""
        SELECT
            record_id,
            project_id,
            assessment_date,

            state,
            project_type,
            public_private,
            customer_role,
            hiring_party_type,

            payment_chain_completeness_score,
            research_confidence_score,
            deadline_days_remaining,
            prior_escalation_rate,
            prior_projects_with_hiring_party,
            expected_party_count,

            critical_field_missing,
            multiple_candidate_records,
            conflicting_project_information

        FROM {ATHENA_TABLE}

        ORDER BY
            critical_field_missing DESC,
            conflicting_project_information DESC,
            multiple_candidate_records DESC,
            deadline_days_remaining ASC

        LIMIT {limit}
        """

        return self.execute_query(sql)
'''

if "def get_candidate_workflows(" not in source:
    source = source.rstrip() + "\\n" + method + "\\n"

    path.write_text(
        source,
        encoding="utf-8",
    )

    print(
        "17E.1 candidate workflow method added: PASS"
    )

else:
    print(
        "17E.1 candidate workflow method already exists"
    )

17E.1 candidate workflow method added: PASS


In [8]:
import py_compile

py_compile.compile(
    "dashboard/repositories/athena_repository.py",
    doraise=True,
)

print(
    "17E.1 repository compile: PASS"
)

17E.1 repository compile: PASS


In [12]:
from pathlib import Path

repository_code = '''import time

import boto3

from dashboard.config import (
    AWS_REGION,
    ATHENA_DATABASE,
    ATHENA_TABLE,
    ATHENA_OUTPUT,
    ATHENA_POLL_INTERVAL_SECONDS,
)


class AthenaRepository:
    """Read-only Athena repository for dashboard analytics."""

    def __init__(self):
        self.client = boto3.client(
            "athena",
            region_name=AWS_REGION,
        )

    def execute_query(
        self,
        sql: str,
    ) -> list[dict]:
        """
        Execute an Athena query and return rows as dictionaries.
        """

        response = self.client.start_query_execution(
            QueryString=sql,
            QueryExecutionContext={
                "Database": ATHENA_DATABASE,
            },
            ResultConfiguration={
                "OutputLocation": ATHENA_OUTPUT,
            },
        )

        execution_id = response[
            "QueryExecutionId"
        ]

        while True:
            execution = (
                self.client.get_query_execution(
                    QueryExecutionId=execution_id
                )
            )

            status = execution[
                "QueryExecution"
            ]["Status"]["State"]

            if status == "SUCCEEDED":
                break

            if status in {
                "FAILED",
                "CANCELLED",
            }:
                reason = execution[
                    "QueryExecution"
                ]["Status"].get(
                    "StateChangeReason",
                    "Unknown Athena error",
                )

                raise RuntimeError(
                    f"Athena query {status}: "
                    f"{reason}"
                )

            time.sleep(
                ATHENA_POLL_INTERVAL_SECONDS
            )

        paginator = (
            self.client.get_paginator(
                "get_query_results"
            )
        )

        raw_rows = []

        for page in paginator.paginate(
            QueryExecutionId=execution_id
        ):
            raw_rows.extend(
                page["ResultSet"]["Rows"]
            )

        if not raw_rows:
            return []

        headers = [
            column.get(
                "VarCharValue",
                "",
            )
            for column in raw_rows[0]["Data"]
        ]

        results = []

        for row in raw_rows[1:]:
            values = [
                column.get(
                    "VarCharValue"
                )
                for column in row["Data"]
            ]

            values += [
                None
            ] * (
                len(headers) - len(values)
            )

            results.append(
                dict(
                    zip(
                        headers,
                        values,
                    )
                )
            )

        return results

    def get_portfolio_kpis(
        self,
    ) -> dict:
        """Return governed portfolio-level operational KPIs."""

        sql = f"""
        SELECT
            COUNT(*) AS total_workflows,

            AVG(
                payment_chain_completeness_score
            ) AS avg_chain_completeness,

            AVG(
                research_confidence_score
            ) AS avg_research_confidence,

            SUM(
                CASE
                    WHEN critical_field_missing = 1
                    THEN 1
                    ELSE 0
                END
            ) AS critical_missing_count,

            SUM(
                CASE
                    WHEN multiple_candidate_records = 1
                    THEN 1
                    ELSE 0
                END
            ) AS multiple_candidate_count,

            SUM(
                CASE
                    WHEN conflicting_project_information = 1
                    THEN 1
                    ELSE 0
                END
            ) AS conflicting_info_count,

            SUM(
                CASE
                    WHEN deadline_days_remaining <= 15
                    THEN 1
                    ELSE 0
                END
            ) AS deadline_15d_count

        FROM {ATHENA_TABLE}
        """

        rows = self.execute_query(sql)

        return rows[0] if rows else {}

    def get_state_segments(
        self,
    ) -> list[dict]:
        """Return state-level operational analytics."""

        sql = f"""
        SELECT
            state,

            COUNT(*) AS workflow_count,

            AVG(
                payment_chain_completeness_score
            ) AS avg_chain_completeness,

            AVG(
                research_confidence_score
            ) AS avg_research_confidence,

            SUM(
                CASE
                    WHEN critical_field_missing = 1
                    THEN 1
                    ELSE 0
                END
            ) AS critical_missing_count

        FROM {ATHENA_TABLE}

        GROUP BY state

        ORDER BY workflow_count DESC
        """

        return self.execute_query(sql)

    def get_candidate_workflows(
        self,
        limit: int = 100,
    ) -> list[dict]:
        """
        Return candidate workflows for operational review.

        Candidate selection is operational filtering,
        not model inference.
        """

        if not 1 <= limit <= 500:
            raise ValueError(
                "limit must be between 1 and 500"
            )

        sql = f"""
        SELECT
            record_id,
            project_id,
            assessment_date,

            state,
            project_type,
            public_private,
            customer_role,
            hiring_party_type,

            payment_chain_completeness_score,
            research_confidence_score,
            deadline_days_remaining,
            prior_escalation_rate,
            prior_projects_with_hiring_party,
            expected_party_count,

            critical_field_missing,
            multiple_candidate_records,
            conflicting_project_information

        FROM {ATHENA_TABLE}

        ORDER BY
            critical_field_missing DESC,
            conflicting_project_information DESC,
            multiple_candidate_records DESC,
            deadline_days_remaining ASC

        LIMIT {limit}
        """

        return self.execute_query(sql)
'''

path = Path(
    "dashboard/repositories/athena_repository.py"
)

path.write_text(
    repository_code,
    encoding="utf-8",
)

print(
    "17E.1 Athena repository clean rewrite: PASS"
)

17E.1 Athena repository clean rewrite: PASS


In [15]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17E.1 full dashboard compile: PASS"
)

17E.1 full dashboard compile: PASS


In [16]:
import sys
import importlib

importlib.invalidate_caches()

for module_name in [
    "dashboard.config",
    "dashboard.repositories.athena_repository",
    "dashboard.services.analytics_service",
]:
    sys.modules.pop(
        module_name,
        None,
    )

from dashboard.repositories.athena_repository import (
    AthenaRepository,
)

print(
    "17E.1 repository import: PASS"
)

17E.1 repository import: PASS


In [17]:
repository = AthenaRepository()

candidates = (
    repository.get_candidate_workflows(
        limit=25
    )
)

print(
    "Candidate workflows returned:",
    len(candidates)
)

assert len(candidates) == 25

required_fields = {
    "record_id",
    "project_id",
    "assessment_date",
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type",
    "payment_chain_completeness_score",
    "research_confidence_score",
    "deadline_days_remaining",
    "prior_escalation_rate",
    "prior_projects_with_hiring_party",
    "expected_party_count",
    "critical_field_missing",
    "multiple_candidate_records",
    "conflicting_project_information",
}

assert required_fields.issubset(
    candidates[0].keys()
)

print(
    "17E.1 candidate workflow Athena test: PASS"
)

Candidate workflows returned: 25
17E.1 candidate workflow Athena test: PASS


# Prediction API Client. 

This is the bridge that takes candidate workflows from Athena and sends only the approved 14 model features to our production prediction API.

In [18]:
from pathlib import Path

api_client_code = '''import requests


MODEL_FEATURES = [
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type",
    "payment_chain_completeness_score",
    "research_confidence_score",
    "deadline_days_remaining",
    "prior_escalation_rate",
    "prior_projects_with_hiring_party",
    "expected_party_count",
    "critical_field_missing",
    "multiple_candidate_records",
    "conflicting_project_information",
]


class PredictionApiClient:
    """Client for the payment-risk prediction API."""

    def __init__(
        self,
        base_url: str,
        timeout_seconds: int = 10,
    ):
        self.base_url = base_url.rstrip("/")
        self.timeout_seconds = timeout_seconds

    def build_payload(
        self,
        workflow: dict,
    ) -> dict:

        missing = [
            feature
            for feature in MODEL_FEATURES
            if feature not in workflow
        ]

        if missing:
            raise ValueError(
                f"Missing model features: {missing}"
            )

        payload = {
            feature: workflow[feature]
            for feature in MODEL_FEATURES
        }

        return payload

    def predict(
        self,
        workflow: dict,
    ) -> dict:

        payload = self.build_payload(
            workflow
        )

        response = requests.post(
            f"{self.base_url}/v1/predict",
            json=payload,
            timeout=self.timeout_seconds,
        )

        response.raise_for_status()

        return response.json()

    def health(
        self,
    ) -> dict:

        response = requests.get(
            f"{self.base_url}/health",
            timeout=self.timeout_seconds,
        )

        response.raise_for_status()

        return response.json()
'''

Path(
    "dashboard/clients"
).mkdir(
    parents=True,
    exist_ok=True,
)

Path(
    "dashboard/clients/__init__.py"
).touch()

Path(
    "dashboard/clients/prediction_api_client.py"
).write_text(
    api_client_code,
    encoding="utf-8",
)

print(
    "17E.2 prediction API client source: PASS"
)

17E.2 prediction API client source: PASS


In [19]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17E.2 dashboard package compile: PASS"
)

17E.2 dashboard package compile: PASS


In [20]:
from dashboard.clients.prediction_api_client import (
    PredictionApiClient,
    MODEL_FEATURES,
)

client = PredictionApiClient(
    base_url="http://localhost:8000"
)

sample_workflow = candidates[0]

payload = client.build_payload(
    sample_workflow
)

print(
    "Payload feature count:",
    len(payload)
)

assert set(payload.keys()) == set(
    MODEL_FEATURES
)

assert len(payload) == 14

print(
    "17E.2 prediction payload contract: PASS"
)

Payload feature count: 14
17E.2 prediction payload contract: PASS


In [21]:
for key, value in payload.items():
    print(
        key,
        "=>",
        value,
        type(value).__name__,
    )

state => CA str
project_type => industrial str
public_private => private str
customer_role => subcontractor str
hiring_party_type => general_contractor str
payment_chain_completeness_score => 0.667 str
research_confidence_score => 0.534 str
deadline_days_remaining => -4 str
prior_escalation_rate => 0.25 str
prior_projects_with_hiring_party => 8 str
expected_party_count => 3 str
critical_field_missing => 1 str
multiple_candidate_records => 1 str
conflicting_project_information => 1 str


That confirms the exact issue: Athena’s get_query_results returns values as strings, so our API client needs an explicit normalization layer before inference.
We should fix this in the client, not in Streamlit and not in the API. That keeps the boundary clean:

In [22]:
from pathlib import Path

api_client_code = '''import requests


CATEGORICAL_FEATURES = [
    "state",
    "project_type",
    "public_private",
    "customer_role",
    "hiring_party_type",
]

FLOAT_FEATURES = [
    "payment_chain_completeness_score",
    "research_confidence_score",
    "prior_escalation_rate",
]

INTEGER_FEATURES = [
    "deadline_days_remaining",
    "prior_projects_with_hiring_party",
    "expected_party_count",
    "critical_field_missing",
    "multiple_candidate_records",
    "conflicting_project_information",
]

MODEL_FEATURES = (
    CATEGORICAL_FEATURES
    + FLOAT_FEATURES
    + INTEGER_FEATURES
)


class PredictionApiClient:
    """Client for the payment-risk prediction API."""

    def __init__(
        self,
        base_url: str,
        timeout_seconds: int = 10,
    ):
        self.base_url = base_url.rstrip("/")
        self.timeout_seconds = timeout_seconds

    def normalize_workflow(
        self,
        workflow: dict,
    ) -> dict:

        missing = [
            feature
            for feature in MODEL_FEATURES
            if feature not in workflow
        ]

        if missing:
            raise ValueError(
                f"Missing model features: {missing}"
            )

        normalized = {}

        for feature in CATEGORICAL_FEATURES:
            value = workflow[feature]

            if value is None:
                raise ValueError(
                    f"{feature} cannot be None"
                )

            normalized[feature] = str(value)

        for feature in FLOAT_FEATURES:
            value = workflow[feature]

            if value is None:
                raise ValueError(
                    f"{feature} cannot be None"
                )

            normalized[feature] = float(value)

        for feature in INTEGER_FEATURES:
            value = workflow[feature]

            if value is None:
                raise ValueError(
                    f"{feature} cannot be None"
                )

            normalized[feature] = int(value)

        return normalized

    def build_payload(
        self,
        workflow: dict,
    ) -> dict:

        return self.normalize_workflow(
            workflow
        )

    def predict(
        self,
        workflow: dict,
    ) -> dict:

        payload = self.build_payload(
            workflow
        )

        response = requests.post(
            f"{self.base_url}/v1/predict",
            json=payload,
            timeout=self.timeout_seconds,
        )

        response.raise_for_status()

        return response.json()

    def health(
        self,
    ) -> dict:

        response = requests.get(
            f"{self.base_url}/health",
            timeout=self.timeout_seconds,
        )

        response.raise_for_status()

        return response.json()
'''

Path(
    "dashboard/clients/prediction_api_client.py"
).write_text(
    api_client_code,
    encoding="utf-8",
)

print(
    "17E.2 API client type normalization: PASS"
)

17E.2 API client type normalization: PASS


In [23]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17E.2 dashboard package compile: PASS"
)

17E.2 dashboard package compile: PASS


In [24]:
import sys
import importlib

importlib.invalidate_caches()

sys.modules.pop(
    "dashboard.clients.prediction_api_client",
    None,
)

from dashboard.clients.prediction_api_client import (
    PredictionApiClient,
    MODEL_FEATURES,
)

client = PredictionApiClient(
    base_url="http://localhost:8000"
)

payload = client.build_payload(
    candidates[0]
)

assert len(payload) == 14
assert set(payload.keys()) == set(MODEL_FEATURES)

print(
    "17E.2 normalized payload contract: PASS"
)

for key, value in payload.items():
    print(
        key,
        "=>",
        value,
        type(value).__name__,
    )

17E.2 normalized payload contract: PASS
state => CA str
project_type => industrial str
public_private => private str
customer_role => subcontractor str
hiring_party_type => general_contractor str
payment_chain_completeness_score => 0.667 float
research_confidence_score => 0.534 float
prior_escalation_rate => 0.25 float
deadline_days_remaining => -4 int
prior_projects_with_hiring_party => 8 int
expected_party_count => 3 int
critical_field_missing => 1 int
multiple_candidate_records => 1 int
conflicting_project_information => 1 int


# Live Prediction API Integration.

In [25]:
import requests

try:
    response = requests.get(
        "http://localhost:8000/health",
        timeout=5,
    )

    print("Status:", response.status_code)
    print("Response:", response.json())

except Exception as exc:
    print(
        "Prediction API not reachable:",
        repr(exc),
    )

Prediction API not reachable: ConnectionError(MaxRetryError("HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f3420a616d0>: Failed to establish a new connection: [Errno 111] Connection refused'))"))


In [26]:
client = PredictionApiClient(
    base_url="http://localhost:8000"
)

health = client.health()

print("Health response:")
print(health)

assert health

print(
    "17E.3 Prediction API health check: PASS"
)

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f340b620560>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [27]:
import subprocess
import sys
import time

api_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "app.main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd="/home/sagemaker-user",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Prediction API process started")
print("PID:", api_process.pid)

time.sleep(3)

print(
    "Process running:",
    api_process.poll() is None
)

Prediction API process started
PID: 44927
Process running: True


In [ ]:
for _ in range(20):
    line = api_process.stdout.readline()

    if not line:
        break

    print(line.rstrip())

INFO:     Started server process [44927]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
import requests

response = requests.get(
    "http://localhost:8000/health",
    timeout=5,
)

print("Status:", response.status_code)
print("Response:", response.json())

In [ ]:
client = PredictionApiClient(
    base_url="http://localhost:800|0"
)

prediction = client.predict(
    candidates[0]
)

print("Prediction response:")

for key, value in prediction.items():
    print(key, "=>", value)

In [1]:
import subprocess
import sys
import time
from pathlib import Path

log_path = Path("/home/sagemaker-user/api_startup.log")

log_file = open(log_path, "w")

api_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "app.main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd="/home/sagemaker-user",
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)

print("PID:", api_process.pid)

time.sleep(4)

print("Return code:", api_process.poll())
print("Running:", api_process.poll() is None)

PID: 45129
Return code: None
Running: True


In [2]:
log_file.flush()

print(
    Path(
        "/home/sagemaker-user/api_startup.log"
    ).read_text(
        encoding="utf-8",
        errors="replace",
    )
)

INFO:     Started server process [45129]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



In [3]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=5,
)

print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'status': 'ok'}


In [6]:
client = PredictionApiClient(
    base_url="http://127.0.0.1:8000"
)

prediction = client.predict(
    candidates[0]
)

print("Prediction response:")

for key, value in prediction.items():
    print(key, "=>", value)

NameError: name 'candidates' is not defined

In [5]:
import sys
import importlib

importlib.invalidate_caches()

sys.modules.pop(
    "dashboard.clients.prediction_api_client",
    None,
)

from dashboard.clients.prediction_api_client import (
    PredictionApiClient,
    MODEL_FEATURES,
)

print("PredictionApiClient import: PASS")

PredictionApiClient import: PASS


In [7]:
client = PredictionApiClient(
    base_url="http://127.0.0.1:8000"
)

print("Prediction API client initialized: PASS")

Prediction API client initialized: PASS


In [8]:
print(
    "candidates available:",
    "candidates" in globals()
)

candidates available: False


In [9]:
from dashboard.repositories.athena_repository import (
    AthenaRepository,
)

repository = AthenaRepository()

candidates = repository.get_candidate_workflows(
    limit=25
)

print(
    "Candidate workflows:",
    len(candidates)
)

assert len(candidates) == 25

print("Candidate workflows reload: PASS")

Candidate workflows: 25
Candidate workflows reload: PASS


In [10]:
prediction = client.predict(
    candidates[0]
)

print("Prediction response:")
print()

for key, value in prediction.items():
    print(
        key,
        "=>",
        value
    )

print()
print(
    "Response keys:",
    list(prediction.keys())
)

Prediction response:

model_version => logistic-regression-v1
predicted_operational_risk => 0.7583439955818435
threshold => 0.2
model_decision => FLAGGED_BY_MODEL
explanation => {'top_factors': [{'feature': 'conflicting_project_information', 'label': 'Conflicting project information', 'direction': 'higher', 'contribution_log_odds': 0.8024216081788677, 'absolute_contribution': 0.8024216081788677, 'explanation': 'Conflicting project information contributed to a higher predicted operational risk score.'}, {'feature': 'critical_field_missing', 'label': 'Missing critical project information', 'direction': 'higher', 'contribution_log_odds': 0.6749125611290474, 'absolute_contribution': 0.6749125611290474, 'explanation': 'Missing critical project information contributed to a higher predicted operational risk score.'}, {'feature': 'multiple_candidate_records', 'label': 'Multiple candidate project records', 'direction': 'higher', 'contribution_log_odds': 0.6439512926781611, 'absolute_contributio

# Create the risk queue service

In [11]:
from pathlib import Path

risk_queue_code = '''from dashboard.clients.prediction_api_client import (
    PredictionApiClient,
)
from dashboard.repositories.athena_repository import (
    AthenaRepository,
)


class RiskQueueService:
    """
    Build a ranked operational review queue.

    Athena selects candidate workflows.
    Prediction API supplies model risk and explanation.
    """

    def __init__(
        self,
        repository=None,
        prediction_client=None,
    ):
        self.repository = (
            repository
            or AthenaRepository()
        )

        self.prediction_client = (
            prediction_client
            or PredictionApiClient(
                base_url="http://127.0.0.1:8000"
            )
        )

    @staticmethod
    def risk_band(
        risk_score: float,
    ) -> str:
        """
        Observational dashboard bands.

        Operational model threshold remains 0.20.
        """

        if risk_score < 0.20:
            return "LOW"

        if risk_score < 0.50:
            return "MEDIUM"

        return "HIGH"

    def build_queue(
        self,
        limit: int = 25,
    ) -> list[dict]:

        candidates = (
            self.repository
            .get_candidate_workflows(
                limit=limit
            )
        )

        queue = []

        for workflow in candidates:

            try:
                prediction = (
                    self.prediction_client
                    .predict(
                        workflow
                    )
                )

            except Exception as exc:
                queue.append(
                    {
                        "record_id":
                            workflow.get(
                                "record_id"
                            ),

                        "project_id":
                            workflow.get(
                                "project_id"
                            ),

                        "state":
                            workflow.get(
                                "state"
                            ),

                        "scoring_status":
                            "FAILED",

                        "scoring_error":
                            str(exc),
                    }
                )

                continue

            risk = float(
                prediction[
                    "predicted_operational_risk"
                ]
            )

            explanation = (
                prediction.get(
                    "explanation",
                    {}
                )
            )

            top_factors = (
                explanation.get(
                    "top_factors",
                    []
                )
            )

            top_factor = (
                top_factors[0]
                if top_factors
                else {}
            )

            queue.append(
                {
                    "record_id":
                        workflow[
                            "record_id"
                        ],

                    "project_id":
                        workflow[
                            "project_id"
                        ],

                    "assessment_date":
                        workflow[
                            "assessment_date"
                        ],

                    "state":
                        workflow[
                            "state"
                        ],

                    "project_type":
                        workflow[
                            "project_type"
                        ],

                    "customer_role":
                        workflow[
                            "customer_role"
                        ],

                    "deadline_days_remaining":
                        int(
                            workflow[
                                "deadline_days_remaining"
                            ]
                        ),

                    "critical_field_missing":
                        int(
                            workflow[
                                "critical_field_missing"
                            ]
                        ),

                    "multiple_candidate_records":
                        int(
                            workflow[
                                "multiple_candidate_records"
                            ]
                        ),

                    "conflicting_project_information":
                        int(
                            workflow[
                                "conflicting_project_information"
                            ]
                        ),

                    "predicted_operational_risk":
                        risk,

                    "risk_band":
                        self.risk_band(
                            risk
                        ),

                    "threshold":
                        float(
                            prediction[
                                "threshold"
                            ]
                        ),

                    "model_decision":
                        prediction[
                            "model_decision"
                        ],

                    "model_version":
                        prediction[
                            "model_version"
                        ],

                    "top_factor":
                        top_factor.get(
                            "label"
                        ),

                    "top_factor_direction":
                        top_factor.get(
                            "direction"
                        ),

                    "top_factor_explanation":
                        top_factor.get(
                            "explanation"
                        ),

                    "scoring_status":
                        "SUCCESS",

                    "scoring_error":
                        None,
                }
            )

        successful = [
            item
            for item in queue
            if item[
                "scoring_status"
            ] == "SUCCESS"
        ]

        failed = [
            item
            for item in queue
            if item[
                "scoring_status"
            ] != "SUCCESS"
        ]

        successful.sort(
            key=lambda item: (
                item[
                    "predicted_operational_risk"
                ]
            ),
            reverse=True,
        )

        return successful + failed
'''

Path(
    "dashboard/services/risk_queue_service.py"
).write_text(
    risk_queue_code,
    encoding="utf-8",
)

print(
    "17E.4 risk queue service source: PASS"
)

17E.4 risk queue service source: PASS


In [12]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17E.4 dashboard package compile: PASS"
)

17E.4 dashboard package compile: PASS


In [13]:
import sys
import importlib

importlib.invalidate_caches()

sys.modules.pop(
    "dashboard.services.risk_queue_service",
    None,
)

from dashboard.services.risk_queue_service import (
    RiskQueueService,
)

queue_service = RiskQueueService()

queue = queue_service.build_queue(
    limit=5
)

print(
    "Queue records:",
    len(queue)
)

for item in queue:
    print(
        item["record_id"],
        item.get(
            "predicted_operational_risk"
        ),
        item.get(
            "risk_band"
        ),
        item.get(
            "model_decision"
        ),
        item[
            "scoring_status"
        ],
    )

Queue records: 5
REC-007661 0.866567588579171 HIGH FLAGGED_BY_MODEL SUCCESS
REC-009624 0.7726465864485356 HIGH FLAGGED_BY_MODEL SUCCESS
REC-005265 0.7694289204700501 HIGH FLAGGED_BY_MODEL SUCCESS
REC-008999 0.7583439955818435 HIGH FLAGGED_BY_MODEL SUCCESS
REC-006421 0.7309091246814261 HIGH FLAGGED_BY_MODEL SUCCESS


In [14]:
assert len(queue) == 5

successful = [
    row
    for row in queue
    if row[
        "scoring_status"
    ] == "SUCCESS"
]

failed = [
    row
    for row in queue
    if row[
        "scoring_status"
    ] == "FAILED"
]

print(
    "Successful:",
    len(successful)
)

print(
    "Failed:",
    len(failed)
)

assert len(successful) == 5

risks = [
    row[
        "predicted_operational_risk"
    ]
    for row in successful
]

assert all(
    0 <= risk <= 1
    for risk in risks
)

assert risks == sorted(
    risks,
    reverse=True,
)

assert all(
    row["risk_band"]
    in {
        "LOW",
        "MEDIUM",
        "HIGH",
    }
    for row in successful
)

print(
    "17E.4 ranked queue integration test: PASS"
)

Successful: 5
Failed: 0
17E.4 ranked queue integration test: PASS


# Add the ranked Operational Risk Queue to Streamlit.

In [15]:
from pathlib import Path

app_code = '''import pandas as pd
import streamlit as st

from dashboard.services.analytics_service import AnalyticsService
from dashboard.services.risk_queue_service import RiskQueueService


st.set_page_config(
    page_title="Construction Payment Protection Intelligence",
    page_icon="🏗️",
    layout="wide",
)


@st.cache_resource
def get_analytics_service():
    return AnalyticsService()


@st.cache_resource
def get_risk_queue_service():
    return RiskQueueService()


@st.cache_data(ttl=300)
def load_portfolio_summary():
    service = get_analytics_service()
    return service.get_portfolio_summary()


@st.cache_data(ttl=300)
def load_state_breakdown():
    service = get_analytics_service()
    return service.get_state_breakdown()


@st.cache_data(ttl=300)
def load_risk_queue(limit=25):
    service = get_risk_queue_service()
    return service.build_queue(
        limit=limit
    )


st.title(
    "Construction Payment Protection Intelligence"
)

st.caption(
    "Operational portfolio intelligence for "
    "construction payment-protection workflows."
)


# ---------------------------------------------------------
# PORTFOLIO ANALYTICS
# ---------------------------------------------------------

try:
    summary = load_portfolio_summary()
    states = load_state_breakdown()

except Exception as exc:
    st.error(
        "Portfolio analytics are temporarily unavailable."
    )

    st.exception(exc)
    st.stop()


st.subheader("Portfolio Overview")

row1 = st.columns(4)

row1[0].metric(
    "Total Workflows",
    f"{summary['total_workflows']:,}",
)

row1[1].metric(
    "Avg Chain Completeness",
    f"{summary['avg_chain_completeness']:.1%}",
)

row1[2].metric(
    "Avg Research Confidence",
    f"{summary['avg_research_confidence']:.1%}",
)

row1[3].metric(
    "Deadline ≤ 15 Days",
    f"{summary['deadline_15d_count']:,}",
    delta=(
        f"{summary['deadline_15d_rate']:.1%} "
        "of portfolio"
    ),
    delta_color="off",
)


row2 = st.columns(3)

row2[0].metric(
    "Critical Information Missing",
    f"{summary['critical_missing_count']:,}",
    delta=(
        f"{summary['critical_missing_rate']:.1%}"
    ),
    delta_color="off",
)

row2[1].metric(
    "Multiple Candidate Records",
    f"{summary['multiple_candidate_count']:,}",
    delta=(
        f"{summary['multiple_candidate_rate']:.1%}"
    ),
    delta_color="off",
)

row2[2].metric(
    "Conflicting Project Information",
    f"{summary['conflicting_info_count']:,}",
    delta=(
        f"{summary['conflicting_info_rate']:.1%}"
    ),
    delta_color="off",
)


# ---------------------------------------------------------
# STATE ANALYTICS
# ---------------------------------------------------------

st.divider()

st.subheader(
    "State-Level Operational Context"
)

state_df = pd.DataFrame(states)

if state_df.empty:
    st.info(
        "No state-level analytics are available."
    )

else:
    display_df = state_df.copy()

    display_df[
        "avg_chain_completeness"
    ] = (
        display_df[
            "avg_chain_completeness"
        ] * 100
    )

    display_df[
        "avg_research_confidence"
    ] = (
        display_df[
            "avg_research_confidence"
        ] * 100
    )

    display_df[
        "critical_missing_rate"
    ] = (
        display_df[
            "critical_missing_rate"
        ] * 100
    )

    st.bar_chart(
        state_df.set_index(
            "state"
        )[
            "workflow_count"
        ],
        height=350,
    )

    st.dataframe(
        display_df,
        use_container_width=True,
        hide_index=True,
    )


# ---------------------------------------------------------
# OPERATIONAL RISK QUEUE
# ---------------------------------------------------------

st.divider()

st.subheader(
    "Operational Risk Queue"
)

st.caption(
    "Candidate workflows are selected from Athena "
    "and ranked using the production prediction API."
)

try:
    queue = load_risk_queue(
        limit=25
    )

except Exception as exc:
    st.error(
        "The operational risk queue is temporarily unavailable."
    )

    st.exception(exc)
    queue = []


if queue:
    queue_df = pd.DataFrame(queue)

    successful_df = queue_df[
        queue_df[
            "scoring_status"
        ] == "SUCCESS"
    ].copy()

    failed_df = queue_df[
        queue_df[
            "scoring_status"
        ] != "SUCCESS"
    ].copy()

    if not successful_df.empty:

        state_options = sorted(
            successful_df[
                "state"
            ].dropna().unique()
        )

        selected_states = st.multiselect(
            "Filter by state",
            options=state_options,
            default=[],
        )

        risk_options = [
            "LOW",
            "MEDIUM",
            "HIGH",
        ]

        selected_risk_bands = st.multiselect(
            "Filter by risk band",
            options=risk_options,
            default=[],
        )

        filtered_df = (
            successful_df.copy()
        )

        if selected_states:
            filtered_df = filtered_df[
                filtered_df[
                    "state"
                ].isin(
                    selected_states
                )
            ]

        if selected_risk_bands:
            filtered_df = filtered_df[
                filtered_df[
                    "risk_band"
                ].isin(
                    selected_risk_bands
                )
            ]

        flagged_count = int(
            (
                filtered_df[
                    "model_decision"
                ]
                == "FLAGGED_BY_MODEL"
            ).sum()
        )

        high_risk_count = int(
            (
                filtered_df[
                    "risk_band"
                ]
                == "HIGH"
            ).sum()
        )

        queue_metrics = st.columns(3)

        queue_metrics[0].metric(
            "Queue Workflows",
            len(filtered_df),
        )

        queue_metrics[1].metric(
            "Flagged by Model",
            flagged_count,
        )

        queue_metrics[2].metric(
            "High Risk Band",
            high_risk_count,
        )

        table_df = filtered_df[
            [
                "record_id",
                "project_id",
                "state",
                "project_type",
                "customer_role",
                "deadline_days_remaining",
                "predicted_operational_risk",
                "risk_band",
                "model_decision",
                "top_factor",
            ]
        ].copy()

        table_df[
            "predicted_operational_risk"
        ] = (
            table_df[
                "predicted_operational_risk"
            ] * 100
        )

        st.dataframe(
            table_df,
            use_container_width=True,
            hide_index=True,
            column_config={
                "record_id":
                    "Workflow ID",

                "project_id":
                    "Project ID",

                "state":
                    "State",

                "project_type":
                    "Project Type",

                "customer_role":
                    "Customer Role",

                "deadline_days_remaining":
                    st.column_config.NumberColumn(
                        "Deadline Days",
                        format="%d",
                    ),

                "predicted_operational_risk":
                    st.column_config.NumberColumn(
                        "Predicted Risk %",
                        format="%.1f",
                    ),

                "risk_band":
                    "Risk Band",

                "model_decision":
                    "Model Decision",

                "top_factor":
                    "Top Model Factor",
            },
        )

    if not failed_df.empty:
        st.warning(
            f"{len(failed_df)} workflow(s) "
            "could not be scored."
        )

else:
    st.info(
        "No operational risk queue data is available."
    )


# ---------------------------------------------------------
# GOVERNANCE
# ---------------------------------------------------------

st.divider()

st.info(
    "This dashboard supports operational prioritization "
    "and workflow review. Model scores describe learned "
    "associations in synthetic training data. They do not "
    "determine legal rights, lien eligibility, bond-claim "
    "eligibility, or guarantee payment outcomes. Human "
    "review is required before any consequential action."
)
'''

Path(
    "dashboard/app.py"
).write_text(
    app_code,
    encoding="utf-8"
)

print(
    "17E.5 risk queue UI source: PASS"
)

17E.5 risk queue UI source: PASS


In [16]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):
    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17E.5 dashboard package compile: PASS"
)

17E.5 dashboard package compile: PASS


# Single Workflow Assessment.
The purpose is simple: an analyst selects one workflow from the queue and sees a focused assessment with the workflow context, model risk, decision, top explanation factors, and governance notes.

In [17]:
from pathlib import Path

workflow_service_code = '''class WorkflowAssessmentService:
    """Prepare one scored workflow for dashboard review."""

    @staticmethod
    def build_assessment(
        queue_item: dict,
    ) -> dict:

        required = {
            "record_id",
            "project_id",
            "state",
            "project_type",
            "customer_role",
            "deadline_days_remaining",
            "predicted_operational_risk",
            "risk_band",
            "threshold",
            "model_decision",
            "model_version",
            "top_factor",
            "top_factor_explanation",
            "scoring_status",
        }

        missing = (
            required
            - set(queue_item.keys())
        )

        if missing:
            raise ValueError(
                f"Missing assessment fields: "
                f"{sorted(missing)}"
            )

        if (
            queue_item["scoring_status"]
            != "SUCCESS"
        ):
            raise ValueError(
                "Cannot build assessment "
                "for failed scoring result."
            )

        return {
            "workflow_id":
                queue_item["record_id"],

            "project_id":
                queue_item["project_id"],

            "state":
                queue_item["state"],

            "project_type":
                queue_item["project_type"],

            "customer_role":
                queue_item["customer_role"],

            "deadline_days_remaining":
                queue_item[
                    "deadline_days_remaining"
                ],

            "predicted_operational_risk":
                float(
                    queue_item[
                        "predicted_operational_risk"
                    ]
                ),

            "risk_band":
                queue_item["risk_band"],

            "threshold":
                float(
                    queue_item["threshold"]
                ),

            "model_decision":
                queue_item[
                    "model_decision"
                ],

            "model_version":
                queue_item[
                    "model_version"
                ],

            "top_factor":
                queue_item[
                    "top_factor"
                ],

            "top_factor_explanation":
                queue_item[
                    "top_factor_explanation"
                ],
        }
'''

Path(
    "dashboard/services/workflow_assessment_service.py"
).write_text(
    workflow_service_code,
    encoding="utf-8",
)

print(
    "17F.1 workflow assessment service source: PASS"
)

17F.1 workflow assessment service source: PASS


In [18]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):
    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17F.1 dashboard package compile: PASS"
)

17F.1 dashboard package compile: PASS


In [19]:
from dashboard.services.workflow_assessment_service import (
    WorkflowAssessmentService,
)

assessment = (
    WorkflowAssessmentService
    .build_assessment(
        queue[0]
    )
)

print(assessment)

assert (
    0
    <= assessment[
        "predicted_operational_risk"
    ]
    <= 1
)

assert assessment[
    "model_decision"
] in {
    "FLAGGED_BY_MODEL",
    "NOT_FLAGGED_BY_MODEL",
}

print(
    "17F.1 workflow assessment service test: PASS"
)

{'workflow_id': 'REC-007661', 'project_id': 'PRJ-02090', 'state': 'CA', 'project_type': 'commercial', 'customer_role': 'material_supplier', 'deadline_days_remaining': 3, 'predicted_operational_risk': 0.866567588579171, 'risk_band': 'HIGH', 'threshold': 0.2, 'model_decision': 'FLAGGED_BY_MODEL', 'model_version': 'logistic-regression-v1', 'top_factor': 'Historical escalation pattern', 'top_factor_explanation': "The workflow's historical escalation pattern, relative to the model's training reference, contributed to a higher predicted operational risk score."}
17F.1 workflow assessment service test: PASS


# add the workflow selector + assessment panel to Streamlit.
Run this cell to replace dashboard/app.py with a version that keeps your existing overview + risk queue and adds a single-workflow review panel underneath:

In [20]:
from pathlib import Path

app_code = '''import pandas as pd
import streamlit as st

from dashboard.services.analytics_service import AnalyticsService
from dashboard.services.risk_queue_service import RiskQueueService
from dashboard.services.workflow_assessment_service import (
    WorkflowAssessmentService,
)


st.set_page_config(
    page_title="Construction Payment Protection Intelligence",
    page_icon="🏗️",
    layout="wide",
)


@st.cache_resource
def get_analytics_service():
    return AnalyticsService()


@st.cache_resource
def get_risk_queue_service():
    return RiskQueueService()


@st.cache_data(ttl=300)
def load_portfolio_summary():
    return get_analytics_service().get_portfolio_summary()


@st.cache_data(ttl=300)
def load_state_breakdown():
    return get_analytics_service().get_state_breakdown()


@st.cache_data(ttl=300)
def load_risk_queue(limit=25):
    return get_risk_queue_service().build_queue(
        limit=limit
    )


st.title(
    "Construction Payment Protection Intelligence"
)

st.caption(
    "Operational portfolio intelligence for "
    "construction payment-protection workflows."
)


# ---------------------------------------------------------
# PORTFOLIO OVERVIEW
# ---------------------------------------------------------

try:
    summary = load_portfolio_summary()
    states = load_state_breakdown()

except Exception as exc:
    st.error(
        "Portfolio analytics are temporarily unavailable."
    )
    st.exception(exc)
    st.stop()


st.subheader("Portfolio Overview")

row1 = st.columns(4)

row1[0].metric(
    "Total Workflows",
    f"{summary['total_workflows']:,}",
)

row1[1].metric(
    "Avg Chain Completeness",
    f"{summary['avg_chain_completeness']:.1%}",
)

row1[2].metric(
    "Avg Research Confidence",
    f"{summary['avg_research_confidence']:.1%}",
)

row1[3].metric(
    "Deadline ≤ 15 Days",
    f"{summary['deadline_15d_count']:,}",
    delta=(
        f"{summary['deadline_15d_rate']:.1%} "
        "of portfolio"
    ),
    delta_color="off",
)


row2 = st.columns(3)

row2[0].metric(
    "Critical Information Missing",
    f"{summary['critical_missing_count']:,}",
    delta=f"{summary['critical_missing_rate']:.1%}",
    delta_color="off",
)

row2[1].metric(
    "Multiple Candidate Records",
    f"{summary['multiple_candidate_count']:,}",
    delta=f"{summary['multiple_candidate_rate']:.1%}",
    delta_color="off",
)

row2[2].metric(
    "Conflicting Project Information",
    f"{summary['conflicting_info_count']:,}",
    delta=f"{summary['conflicting_info_rate']:.1%}",
    delta_color="off",
)


# ---------------------------------------------------------
# STATE CONTEXT
# ---------------------------------------------------------

st.divider()

st.subheader("State-Level Operational Context")

state_df = pd.DataFrame(states)

if not state_df.empty:
    st.bar_chart(
        state_df.set_index("state")[
            "workflow_count"
        ],
        height=350,
    )

    st.dataframe(
        state_df,
        use_container_width=True,
        hide_index=True,
    )


# ---------------------------------------------------------
# RISK QUEUE
# ---------------------------------------------------------

st.divider()

st.subheader("Operational Risk Queue")

st.caption(
    "Candidate workflows are selected from Athena "
    "and ranked using the production prediction API."
)

try:
    queue = load_risk_queue(
        limit=25
    )

except Exception as exc:
    st.error(
        "The operational risk queue is temporarily unavailable."
    )
    st.exception(exc)
    queue = []


successful_queue = [
    item
    for item in queue
    if item.get("scoring_status") == "SUCCESS"
]


if successful_queue:

    queue_df = pd.DataFrame(
        successful_queue
    )

    state_options = sorted(
        queue_df["state"]
        .dropna()
        .unique()
    )

    selected_states = st.multiselect(
        "Filter by state",
        options=state_options,
    )

    risk_options = [
        "LOW",
        "MEDIUM",
        "HIGH",
    ]

    selected_bands = st.multiselect(
        "Filter by risk band",
        options=risk_options,
    )

    filtered_df = queue_df.copy()

    if selected_states:
        filtered_df = filtered_df[
            filtered_df["state"].isin(
                selected_states
            )
        ]

    if selected_bands:
        filtered_df = filtered_df[
            filtered_df["risk_band"].isin(
                selected_bands
            )
        ]

    metrics = st.columns(3)

    metrics[0].metric(
        "Queue Workflows",
        len(filtered_df),
    )

    metrics[1].metric(
        "Flagged by Model",
        int(
            (
                filtered_df["model_decision"]
                == "FLAGGED_BY_MODEL"
            ).sum()
        ),
    )

    metrics[2].metric(
        "High Risk Band",
        int(
            (
                filtered_df["risk_band"]
                == "HIGH"
            ).sum()
        ),
    )

    display_queue = filtered_df[
        [
            "record_id",
            "project_id",
            "state",
            "project_type",
            "customer_role",
            "deadline_days_remaining",
            "predicted_operational_risk",
            "risk_band",
            "model_decision",
            "top_factor",
        ]
    ].copy()

    display_queue[
        "predicted_operational_risk"
    ] *= 100

    st.dataframe(
        display_queue,
        use_container_width=True,
        hide_index=True,
    )


    # -----------------------------------------------------
    # SINGLE WORKFLOW ASSESSMENT
    # -----------------------------------------------------

    st.divider()

    st.subheader(
        "Single Workflow Assessment"
    )

    workflow_options = {
        (
            f"{item['record_id']} | "
            f"{item['project_id']} | "
            f"{item['state']} | "
            f"{item['predicted_operational_risk']:.1%}"
        ):
        item
        for item in successful_queue
    }

    selected_label = st.selectbox(
        "Select workflow for review",
        options=list(
            workflow_options.keys()
        ),
    )

    selected_item = (
        workflow_options[
            selected_label
        ]
    )

    assessment = (
        WorkflowAssessmentService
        .build_assessment(
            selected_item
        )
    )

    risk = assessment[
        "predicted_operational_risk"
    ]

    review_metrics = st.columns(4)

    review_metrics[0].metric(
        "Predicted Risk",
        f"{risk:.1%}",
    )

    review_metrics[1].metric(
        "Risk Band",
        assessment[
            "risk_band"
        ],
    )

    review_metrics[2].metric(
        "Model Threshold",
        f"{assessment['threshold']:.0%}",
    )

    review_metrics[3].metric(
        "Model Decision",
        assessment[
            "model_decision"
        ],
    )


    context_left, context_right = (
        st.columns(2)
    )

    with context_left:

        st.markdown(
            "#### Workflow Context"
        )

        st.write(
            "**Workflow ID:**",
            assessment[
                "workflow_id"
            ],
        )

        st.write(
            "**Project ID:**",
            assessment[
                "project_id"
            ],
        )

        st.write(
            "**State:**",
            assessment[
                "state"
            ],
        )

        st.write(
            "**Project Type:**",
            assessment[
                "project_type"
            ],
        )

        st.write(
            "**Customer Role:**",
            assessment[
                "customer_role"
            ],
        )

        st.write(
            "**Deadline Days Remaining:**",
            assessment[
                "deadline_days_remaining"
            ],
        )


    with context_right:

        st.markdown(
            "#### Model Context"
        )

        st.write(
            "**Model Version:**",
            assessment[
                "model_version"
            ],
        )

        st.write(
            "**Top Model Factor:**",
            assessment[
                "top_factor"
            ],
        )

        st.write(
            "**Factor Explanation:**"
        )

        st.write(
            assessment[
                "top_factor_explanation"
            ]
        )


    st.warning(
        "The model score supports operational review "
        "and prioritization only. It does not determine "
        "legal rights, notice requirements, lien rights, "
        "bond-claim rights, or payment outcomes."
    )

else:
    st.info(
        "No successfully scored workflows "
        "are currently available."
    )


# ---------------------------------------------------------
# GOVERNANCE
# ---------------------------------------------------------

st.divider()

st.info(
    "This dashboard supports operational prioritization "
    "and workflow review. Model scores describe learned "
    "associations in synthetic training data. Human review "
    "is required before consequential action."
)
'''

Path(
    "dashboard/app.py"
).write_text(
    app_code,
    encoding="utf-8",
)

print(
    "17F.2 single workflow UI source: PASS"
)

17F.2 single workflow UI source: PASS


In [21]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17F.2 dashboard package compile: PASS"
)

17F.2 dashboard package compile: PASS


In [22]:
from pathlib import Path

log_path = Path("/home/sagemaker-user/streamlit.log")

if log_path.exists():
    print(log_path.read_text()[-8000:])
else:
    print("streamlit.log not found")

streamlit.log not found


In [23]:
import subprocess

result = subprocess.run(
    ["ps", "-ef"],
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "streamlit" in line.lower():
        print(line)

In [24]:
import subprocess
import time

subprocess.run(
    ["pkill", "-f", "streamlit"],
    capture_output=True,
)

log_file = open(
    "/home/sagemaker-user/streamlit.log",
    "w",
)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "dashboard/app.py",
        "--server.address",
        "0.0.0.0",
        "--server.port",
        "8501",
        "--server.headless",
        "true",
    ],
    cwd="/home/sagemaker-user",
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(5)

print("Streamlit PID:", process.pid)
print("Return code:", process.poll())

print(
    Path(
        "/home/sagemaker-user/streamlit.log"
    ).read_text()
)

Streamlit PID: 45334
Return code: None


2026-09-14 09:28:08.425 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://169.255.255.2:8501
  External URL: http://54.153.159.231:8501




In [25]:
from pathlib import Path

print(
    Path(
        "/home/sagemaker-user/streamlit.log"
    ).read_text()[-10000:]
)



2026-09-14 09:28:08.425 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://169.255.255.2:8501
  External URL: http://54.153.159.231:8501




In [26]:
import subprocess
import os
import time
from pathlib import Path

subprocess.run(
    ["pkill", "-f", "streamlit"],
    capture_output=True,
)

env = os.environ.copy()
env["PYTHONPATH"] = "/home/sagemaker-user"

log_path = "/home/sagemaker-user/streamlit.log"
log_file = open(log_path, "w")

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "dashboard/app.py",
        "--server.address",
        "0.0.0.0",
        "--server.port",
        "8501",
        "--server.headless",
        "true",
    ],
    cwd="/home/sagemaker-user",
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(5)

print("Streamlit PID:", process.pid)
print("Return code:", process.poll())

print(
    Path(log_path).read_text()
)

Streamlit PID: 45361
Return code: None


2026-09-14 09:29:02.719 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://169.255.255.2:8501
  External URL: http://54.153.159.231:8501




In [27]:
import sys

sys.path.insert(
    0,
    "/home/sagemaker-user"
)

from dashboard.services.analytics_service import AnalyticsService
from dashboard.services.risk_queue_service import RiskQueueService
from dashboard.services.workflow_assessment_service import WorkflowAssessmentService

print("Dashboard package imports: PASS")

Dashboard package imports: PASS


In [28]:
from pathlib import Path

path = Path(
    "dashboard/services/risk_queue_service.py"
)

text = path.read_text(
    encoding="utf-8"
)

old = '''                    "top_factor_explanation":
                        top_factor.get(
                            "explanation"
                        ),

                    "scoring_status":
                        "SUCCESS",'''

new = '''                    "top_factor_explanation":
                        top_factor.get(
                            "explanation"
                        ),

                    "explanation":
                        prediction.get(
                            "explanation",
                            {}
                        ),

                    "disclaimers":
                        prediction.get(
                            "disclaimers",
                            []
                        ),

                    "scoring_status":
                        "SUCCESS",'''

if old not in text:
    raise RuntimeError(
        "Expected insertion point not found. "
        "No file changes made."
    )

text = text.replace(
    old,
    new,
    1,
)

path.write_text(
    text,
    encoding="utf-8",
)

print(
    "17G.1 full explanation preservation source: PASS"
)

17G.1 full explanation preservation source: PASS


In [29]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17G.1 dashboard package compile: PASS"
)

17G.1 dashboard package compile: PASS


In [30]:
import sys
import importlib

importlib.invalidate_caches()

sys.modules.pop(
    "dashboard.services.risk_queue_service",
    None,
)

from dashboard.services.risk_queue_service import (
    RiskQueueService,
)

service = RiskQueueService()

test_queue = service.build_queue(
    limit=1
)

item = test_queue[0]

print(
    "Scoring status:",
    item["scoring_status"]
)

print(
    "Explanation method:",
    item["explanation"].get(
        "method"
    )
)

print(
    "Top factor count:",
    len(
        item["explanation"].get(
            "top_factors",
            []
        )
    )
)

print(
    "Disclaimers:",
    len(
        item.get(
            "disclaimers",
            []
        )
    )
)

Scoring status: SUCCESS
Explanation method: exact_logistic_regression_log_odds_decomposition
Top factor count: 5
Disclaimers: 3


In [31]:
assert item[
    "scoring_status"
] == "SUCCESS"

assert isinstance(
    item["explanation"],
    dict,
)

assert (
    item["explanation"].get(
        "method"
    )
    == "exact_logistic_regression_log_odds_decomposition"
)

assert len(
    item["explanation"].get(
        "top_factors",
        []
    )
) > 0

assert isinstance(
    item["disclaimers"],
    list,
)

print(
    "17G.1 explanation contract integration test: PASS"
)

17G.1 explanation contract integration test: PASS


# Extend the assessment service
First, update workflow_assessment_service.py so the complete explanation reaches the UI:

In [32]:
from pathlib import Path

path = Path(
    "dashboard/services/workflow_assessment_service.py"
)

text = path.read_text(
    encoding="utf-8"
)

old = '''            "top_factor_explanation":
                queue_item[
                    "top_factor_explanation"
                ],
        }'''

new = '''            "top_factor_explanation":
                queue_item[
                    "top_factor_explanation"
                ],

            "explanation":
                queue_item.get(
                    "explanation",
                    {}
                ),

            "disclaimers":
                queue_item.get(
                    "disclaimers",
                    []
                ),
        }'''

if old not in text:
    raise RuntimeError(
        "Expected insertion point not found. "
        "No changes made."
    )

text = text.replace(
    old,
    new,
    1,
)

path.write_text(
    text,
    encoding="utf-8",
)

print(
    "17G.2 assessment explanation source: PASS"
)

17G.2 assessment explanation source: PASS


In [33]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):
    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17G.2 dashboard package compile: PASS"
)

17G.2 dashboard package compile: PASS


In [34]:
import sys
import importlib

importlib.invalidate_caches()

sys.modules.pop(
    "dashboard.services.workflow_assessment_service",
    None,
)

from dashboard.services.workflow_assessment_service import (
    WorkflowAssessmentService,
)

assessment = (
    WorkflowAssessmentService
    .build_assessment(
        test_queue[0]
    )
)

factors = assessment[
    "explanation"
].get(
    "top_factors",
    []
)

print(
    "Explanation method:",
    assessment[
        "explanation"
    ].get("method")
)

print(
    "Factor count:",
    len(factors)
)

for rank, factor in enumerate(
    factors,
    start=1,
):
    print(
        rank,
        "|",
        factor.get("label"),
        "|",
        factor.get("direction"),
        "|",
        round(
            float(
                factor.get(
                    "absolute_contribution",
                    0
                )
            ),
            4,
        ),
    )

assert len(factors) > 0
assert assessment["disclaimers"]

print(
    "17G.2 assessment explanation test: PASS"
)

Explanation method: exact_logistic_regression_log_odds_decomposition
Factor count: 5
1 | Conflicting project information | higher | 0.8024
2 | Missing critical project information | higher | 0.6749
3 | Multiple candidate project records | higher | 0.644
4 | Deadline proximity | higher | 0.3422
5 | Project research confidence | higher | 0.2479
17G.2 assessment explanation test: PASS


# Operations-Friendly Explainability UX. 

The key design rule is that the analyst sees why the workflow was prioritized without seeing raw coefficients or log-odds values.
We’ll convert each factor’s absolute contribution into a relative strength score from 0–100, where the strongest factor for that workflow is 100. This is only a visualization of relative explanation strength—not probability, causality, or business severity.
Run this targeted update to dashboard/app.py:

In [35]:
from pathlib import Path

path = Path(
    "dashboard/app.py"
)

text = path.read_text(
    encoding="utf-8"
)

old = '''    st.warning(
        "The model score supports operational review "
        "and prioritization only. It does not determine "
        "legal rights, notice requirements, lien rights, "
        "bond-claim rights, or payment outcomes."
    )'''

new = '''    # -----------------------------------------------------
    # EXPLAINABILITY UX
    # -----------------------------------------------------

    st.markdown(
        "#### Why was this workflow prioritized?"
    )

    explanation = assessment.get(
        "explanation",
        {}
    )

    factors = explanation.get(
        "top_factors",
        []
    )

    if factors:

        max_strength = max(
            float(
                factor.get(
                    "absolute_contribution",
                    0
                )
            )
            for factor in factors
        )

        for rank, factor in enumerate(
            factors,
            start=1,
        ):

            contribution = float(
                factor.get(
                    "absolute_contribution",
                    0
                )
            )

            if max_strength > 0:
                relative_strength = (
                    contribution
                    / max_strength
                )
            else:
                relative_strength = 0.0

            label = factor.get(
                "label",
                "Model factor"
            )

            direction = factor.get(
                "direction",
                "unknown"
            )

            plain_explanation = (
                factor.get(
                    "explanation",
                    ""
                )
            )

            left, right = st.columns(
                [4, 1]
            )

            with left:
                st.markdown(
                    f"**{rank}. {label}**"
                )

            with right:
                if direction == "higher":
                    st.write(
                        "Higher risk"
                    )

                elif direction == "lower":
                    st.write(
                        "Lower risk"
                    )

                else:
                    st.write(
                        "Model factor"
                    )

            st.progress(
                min(
                    max(
                        relative_strength,
                        0.0,
                    ),
                    1.0,
                )
            )

            st.caption(
                plain_explanation
            )

        st.caption(
            "Bar length represents relative explanation "
            "strength within this prediction. It is not a "
            "probability, causal effect, or legal-risk score."
        )

    else:
        st.info(
            "No model explanation is available "
            "for this workflow."
        )


    # -----------------------------------------------------
    # MODEL DISCLAIMERS
    # -----------------------------------------------------

    disclaimers = assessment.get(
        "disclaimers",
        []
    )

    if disclaimers:

        with st.expander(
            "Model explanation notes"
        ):

            for disclaimer in disclaimers:
                st.write(
                    f"• {disclaimer}"
                )


    st.warning(
        "The model score supports operational review "
        "and prioritization only. It does not determine "
        "legal rights, notice requirements, lien rights, "
        "bond-claim rights, or payment outcomes."
    )'''

if old not in text:
    raise RuntimeError(
        "Expected workflow warning block "
        "was not found. No changes made."
    )

text = text.replace(
    old,
    new,
    1,
)

path.write_text(
    text,
    encoding="utf-8",
)

print(
    "17G.3 explainability UX source: PASS"
)

17G.3 explainability UX source: PASS


In [36]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17G.3 dashboard package compile: PASS"
)

17G.3 dashboard package compile: PASS


In [37]:
import subprocess
import os
import time

subprocess.run(
    ["pkill", "-f", "streamlit"],
    capture_output=True,
)

env = os.environ.copy()
env["PYTHONPATH"] = (
    "/home/sagemaker-user"
)

log_file = open(
    "/home/sagemaker-user/streamlit.log",
    "w",
)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "dashboard/app.py",
        "--server.address",
        "0.0.0.0",
        "--server.port",
        "8501",
        "--server.headless",
        "true",
    ],
    cwd="/home/sagemaker-user",
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(5)

print(
    "Streamlit PID:",
    process.pid
)

print(
    "Return code:",
    process.poll()
)

Streamlit PID: 45491
Return code: None


### Human Review & Governance UX


This is an important Senior AI Engineer component. We don't want the workflow to end at:

The analyst should be able to record something like Needs Review / Escalate for Operational Review / Insufficient Information / No Further Attention, while making it explicit that this is an operational review—not an automated legal decision.
We'll build 17H incrementally:
17H.1 Review Decision Contract → 17H.2 Review Service → 17H.3 Review UX → 17H.4 Audit Trail → 17H.5 Governance Acceptance Test.
We should start with the contract rather than immediately adding buttons to Streamlit. That keeps the human-review state governed and testable.

In [38]:
from pathlib import Path
import json

contract = {
    "contract_version": "1.0.0",
    "name": "payment-risk-human-review",
    "purpose": (
        "Capture human operational review of model-prioritized "
        "construction payment-protection workflows."
    ),
    "allowed_dispositions": [
        "NEEDS_FURTHER_REVIEW",
        "ESCALATE_OPERATIONAL_REVIEW",
        "INSUFFICIENT_INFORMATION",
        "NO_FURTHER_ATTENTION"
    ],
    "required_fields": [
        "workflow_id",
        "project_id",
        "model_version",
        "predicted_operational_risk",
        "model_decision",
        "review_disposition",
        "review_note",
        "reviewed_at"
    ],
    "governance": {
        "human_decision_is_separate_from_model_prediction": True,
        "model_prediction_must_not_be_overwritten": True,
        "legal_rights_decision": False,
        "automatic_external_action": False,
        "automatic_notice_filing": False,
        "automatic_lien_action": False,
        "automatic_bond_claim_action": False
    }
}

path = Path(
    "artifacts/dashboard/payment-risk-dashboard/v1/"
    "human-review-contract-v1.json"
)

path.parent.mkdir(
    parents=True,
    exist_ok=True
)

path.write_text(
    json.dumps(
        contract,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "17H.1 human review contract source: PASS"
)

17H.1 human review contract source: PASS


In [39]:
import json
from pathlib import Path

path = Path(
    "artifacts/dashboard/payment-risk-dashboard/v1/"
    "human-review-contract-v1.json"
)

data = json.loads(
    path.read_text(
        encoding="utf-8"
    )
)

assert len(
    data["allowed_dispositions"]
) == 4

assert (
    data["governance"][
        "human_decision_is_separate_from_model_prediction"
    ]
    is True
)

assert (
    data["governance"][
        "model_prediction_must_not_be_overwritten"
    ]
    is True
)

assert (
    data["governance"][
        "legal_rights_decision"
    ]
    is False
)

assert (
    data["governance"][
        "automatic_external_action"
    ]
    is False
)

print(
    "17H.1 human review contract validation: PASS"
)

17H.1 human review contract validation: PASS


In [40]:
from pathlib import Path

review_service_code = '''from datetime import datetime, timezone
from uuid import uuid4


ALLOWED_DISPOSITIONS = {
    "NEEDS_FURTHER_REVIEW",
    "ESCALATE_OPERATIONAL_REVIEW",
    "INSUFFICIENT_INFORMATION",
    "NO_FURTHER_ATTENTION",
}


class HumanReviewService:
    """
    Create governed human-review records.

    Human review is stored separately from the
    model prediction and does not overwrite it.
    """

    @staticmethod
    def create_review(
        assessment: dict,
        review_disposition: str,
        review_note: str,
        reviewer_id: str = "dashboard-analyst",
    ) -> dict:

        if review_disposition not in ALLOWED_DISPOSITIONS:
            raise ValueError(
                "Invalid review disposition: "
                f"{review_disposition}"
            )

        note = (
            review_note.strip()
            if review_note
            else ""
        )

        if not note:
            raise ValueError(
                "Review note is required."
            )

        required_assessment_fields = {
            "workflow_id",
            "project_id",
            "model_version",
            "predicted_operational_risk",
            "model_decision",
        }

        missing = (
            required_assessment_fields
            - set(assessment.keys())
        )

        if missing:
            raise ValueError(
                "Assessment missing required fields: "
                f"{sorted(missing)}"
            )

        return {
            "review_id": (
                "REV-"
                + uuid4().hex[:12].upper()
            ),

            "workflow_id":
                assessment["workflow_id"],

            "project_id":
                assessment["project_id"],

            # Preserve model evidence
            "model_version":
                assessment["model_version"],

            "predicted_operational_risk":
                float(
                    assessment[
                        "predicted_operational_risk"
                    ]
                ),

            "model_decision":
                assessment["model_decision"],

            # Separate human decision
            "review_disposition":
                review_disposition,

            "review_note":
                note,

            "reviewer_id":
                reviewer_id,

            "reviewed_at":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "governance": {
                "decision_type":
                    "HUMAN_OPERATIONAL_REVIEW",

                "model_prediction_overwritten":
                    False,

                "legal_rights_decision":
                    False,

                "external_action_triggered":
                    False,
            },
        }
'''

Path(
    "dashboard/services/human_review_service.py"
).write_text(
    review_service_code,
    encoding="utf-8",
)

print(
    "17H.2 human review service source: PASS"
)

17H.2 human review service source: PASS


In [41]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17H.2 dashboard package compile: PASS"
)

17H.2 dashboard package compile: PASS


In [42]:
import sys
import importlib

importlib.invalidate_caches()

sys.modules.pop(
    "dashboard.services.human_review_service",
    None,
)

from dashboard.services.human_review_service import (
    HumanReviewService,
)

review = HumanReviewService.create_review(
    assessment=assessment,
    review_disposition=(
        "NEEDS_FURTHER_REVIEW"
    ),
    review_note=(
        "Conflicting project information "
        "requires analyst verification."
    ),
    reviewer_id="test-analyst",
)

for key, value in review.items():
    print(
        key,
        "=>",
        value,
    )

review_id => REV-4E56BD643CDD
workflow_id => REC-008999
project_id => PRJ-02155
model_version => logistic-regression-v1
predicted_operational_risk => 0.7583439955818435
model_decision => FLAGGED_BY_MODEL
review_disposition => NEEDS_FURTHER_REVIEW
review_note => Conflicting project information requires analyst verification.
reviewer_id => test-analyst
reviewed_at => 2026-09-14T09:41:34.752372+00:00
governance => {'decision_type': 'HUMAN_OPERATIONAL_REVIEW', 'model_prediction_overwritten': False, 'legal_rights_decision': False, 'external_action_triggered': False}


In [43]:
assert review[
    "review_disposition"
] == "NEEDS_FURTHER_REVIEW"

assert review[
    "model_decision"
] == assessment[
    "model_decision"
]

assert (
    review[
        "predicted_operational_risk"
    ]
    == assessment[
        "predicted_operational_risk"
    ]
)

assert (
    review["governance"][
        "model_prediction_overwritten"
    ]
    is False
)

assert (
    review["governance"][
        "legal_rights_decision"
    ]
    is False
)

assert (
    review["governance"][
        "external_action_triggered"
    ]
    is False
)

print(
    "17H.2 human review service test: PASS"
)

17H.2 human review service test: PASS


# Human Review UX
Now we add the analyst review form directly under the Single Workflow Assessment. The analyst will be able to choose a disposition, enter a mandatory note, and create a governed review record.
We are deliberately not saving it permanently yet. First we prove UI → service → validated review record. Persistent audit storage comes in 17H.4.

In [44]:
from pathlib import Path

path = Path(
    "dashboard/app.py"
)

text = path.read_text(
    encoding="utf-8"
)

# Add service import
old_import = '''from dashboard.services.workflow_assessment_service import (
    WorkflowAssessmentService,
)'''

new_import = '''from dashboard.services.workflow_assessment_service import (
    WorkflowAssessmentService,
)
from dashboard.services.human_review_service import (
    HumanReviewService,
)'''

if old_import not in text:
    raise RuntimeError(
        "WorkflowAssessmentService import block not found."
    )

text = text.replace(
    old_import,
    new_import,
    1,
)


# Insert Human Review UX before governance warning
old_block = '''    st.warning(
        "The model score supports operational review "
        "and prioritization only. It does not determine "
        "legal rights, notice requirements, lien rights, "
        "bond-claim rights, or payment outcomes."
    )'''

new_block = '''    # -----------------------------------------------------
    # HUMAN REVIEW
    # -----------------------------------------------------

    st.markdown(
        "#### Human Operational Review"
    )

    st.caption(
        "Record the analyst's operational disposition. "
        "This review is stored separately from the "
        "model prediction and does not overwrite it."
    )

    disposition_labels = {
        "Needs further review":
            "NEEDS_FURTHER_REVIEW",

        "Escalate for operational review":
            "ESCALATE_OPERATIONAL_REVIEW",

        "Insufficient information":
            "INSUFFICIENT_INFORMATION",

        "No further attention":
            "NO_FURTHER_ATTENTION",
    }

    with st.form(
        key=(
            "human_review_form_"
            + assessment["workflow_id"]
        )
    ):

        disposition_label = st.selectbox(
            "Review disposition",
            options=list(
                disposition_labels.keys()
            ),
        )

        review_note = st.text_area(
            "Review note",
            placeholder=(
                "Document the operational reason "
                "for this review disposition."
            ),
            height=120,
        )

        reviewer_id = st.text_input(
            "Reviewer ID",
            value="dashboard-analyst",
        )

        submit_review = st.form_submit_button(
            "Record Human Review"
        )


    if submit_review:

        try:
            review_record = (
                HumanReviewService
                .create_review(
                    assessment=assessment,

                    review_disposition=(
                        disposition_labels[
                            disposition_label
                        ]
                    ),

                    review_note=review_note,

                    reviewer_id=reviewer_id,
                )
            )

        except ValueError as exc:
            st.error(
                str(exc)
            )

        else:
            st.session_state[
                "latest_human_review"
            ] = review_record

            st.success(
                "Human operational review recorded "
                "in the current dashboard session."
            )


    latest_review = st.session_state.get(
        "latest_human_review"
    )

    if (
        latest_review
        and latest_review.get(
            "workflow_id"
        )
        == assessment["workflow_id"]
    ):

        with st.expander(
            "Latest Human Review",
            expanded=True,
        ):

            review_cols = st.columns(3)

            review_cols[0].metric(
                "Human Disposition",
                latest_review[
                    "review_disposition"
                ],
            )

            review_cols[1].metric(
                "Model Decision",
                latest_review[
                    "model_decision"
                ],
            )

            review_cols[2].metric(
                "Model Risk",
                (
                    f"{latest_review['predicted_operational_risk']:.1%}"
                ),
            )

            st.write(
                "**Review ID:**",
                latest_review[
                    "review_id"
                ],
            )

            st.write(
                "**Reviewer:**",
                latest_review[
                    "reviewer_id"
                ],
            )

            st.write(
                "**Reviewed At:**",
                latest_review[
                    "reviewed_at"
                ],
            )

            st.write(
                "**Review Note:**",
                latest_review[
                    "review_note"
                ],
            )

            st.caption(
                "The human disposition and model "
                "prediction are separate records. "
                "Human review does not rewrite the "
                "historical model prediction."
            )


    st.warning(
        "The model score and human operational review "
        "support workflow prioritization only. Neither "
        "determines legal rights, notice requirements, "
        "lien rights, bond-claim rights, or payment "
        "outcomes. No external action is triggered "
        "automatically."
    )'''

if old_block not in text:
    raise RuntimeError(
        "Expected governance warning block not found. "
        "No changes made."
    )

text = text.replace(
    old_block,
    new_block,
    1,
)

path.write_text(
    text,
    encoding="utf-8",
)

print(
    "17H.3 human review UX source: PASS"
)

17H.3 human review UX source: PASS


In [45]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17H.3 dashboard package compile: PASS"
)

17H.3 dashboard package compile: PASS


In [46]:
from dashboard.services.human_review_service import (
    HumanReviewService,
)

test_review = HumanReviewService.create_review(
    assessment=assessment,
    review_disposition="NEEDS_FURTHER_REVIEW",
    review_note=(
        "Analyst verification required before "
        "operational follow-up."
    ),
    reviewer_id="integration-test",
)

assert (
    test_review[
        "model_decision"
    ]
    == assessment[
        "model_decision"
    ]
)

assert (
    test_review[
        "governance"
    ][
        "model_prediction_overwritten"
    ]
    is False
)

print(
    "17H.2 human review service test: PASS"
)

print(
    "17H.3 service integration: PASS"
)

17H.2 human review service test: PASS
17H.3 service integration: PASS


Now the human review loop is working in the UI, but the review currently lives only in Streamlit session state. If the app restarts, that record disappears. So the next production step is 17H.4 — Persistent Audit Trail.
For this project, I recommend S3 JSON audit records first rather than DynamoDB. Why? Because your current architecture is already heavily AWS + S3 based, the review volume is tiny, and for a portfolio-grade production system this gives us durable, timestamped, append-only records with much less infrastructure. DynamoDB would make sense later if we need high-volume querying, updates, workflow state transitions, or low-latency retrieval.

In [47]:
from pathlib import Path

audit_repository_code = '''import json
from datetime import datetime, timezone

import boto3


class HumanReviewAuditRepository:
    """
    Persist human review records to S3.

    Reviews are written as separate immutable JSON
    objects for auditability.
    """

    def __init__(
        self,
        bucket_name: str = "construction-payment-risk-dev-gk53",
        prefix: str = "artifacts/dashboard/payment-risk-dashboard/v1/human-reviews",
        region_name: str = "ap-southeast-2",
    ):
        self.bucket_name = bucket_name
        self.prefix = prefix.rstrip("/")
        self.s3 = boto3.client(
            "s3",
            region_name=region_name,
        )

    def build_object_key(
        self,
        review: dict,
    ) -> str:

        reviewed_at = review["reviewed_at"]

        reviewed_dt = datetime.fromisoformat(
            reviewed_at.replace(
                "Z",
                "+00:00",
            )
        )

        return (
            f"{self.prefix}/"
            f"{reviewed_dt.year:04d}/"
            f"{reviewed_dt.month:02d}/"
            f"{reviewed_dt.day:02d}/"
            f"{review['workflow_id']}/"
            f"{review['review_id']}.json"
        )

    def save_review(
        self,
        review: dict,
    ) -> dict:

        key = self.build_object_key(
            review
        )

        payload = json.dumps(
            review,
            indent=2,
            sort_keys=True,
        ).encode(
            "utf-8"
        )

        self.s3.put_object(
            Bucket=self.bucket_name,
            Key=key,
            Body=payload,
            ContentType="application/json",
        )

        return {
            "bucket":
                self.bucket_name,

            "key":
                key,

            "review_id":
                review[
                    "review_id"
                ],

            "workflow_id":
                review[
                    "workflow_id"
                ],

            "persisted_at":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }

    def get_review(
        self,
        key: str,
    ) -> dict:

        response = self.s3.get_object(
            Bucket=self.bucket_name,
            Key=key,
        )

        body = (
            response[
                "Body"
            ]
            .read()
            .decode(
                "utf-8"
            )
        )

        return json.loads(
            body
        )
'''

Path(
    "dashboard/repositories/human_review_audit_repository.py"
).write_text(
    audit_repository_code,
    encoding="utf-8",
)

print(
    "17H.4 audit repository source: PASS"
)

17H.4 audit repository source: PASS


In [48]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):

    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17H.4 dashboard package compile: PASS"
)

17H.4 dashboard package compile: PASS


In [49]:
import sys
import importlib

importlib.invalidate_caches()

sys.modules.pop(
    "dashboard.repositories.human_review_audit_repository",
    None,
)

from dashboard.repositories.human_review_audit_repository import (
    HumanReviewAuditRepository,
)

audit_repository = HumanReviewAuditRepository()

audit_result = (
    audit_repository
    .save_review(
        test_review
    )
)

print(
    "Persist result:"
)

for key, value in audit_result.items():
    print(
        key,
        "=>",
        value,
    )

AccessDenied: An error occurred (AccessDenied) when calling the PutObject operation: User: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker is not authorized to perform: s3:PutObject on resource: "arn:aws:s3:::construction-payment-risk-dev-gk53/artifacts/dashboard/payment-risk-dashboard/v1/human-reviews/2026/09/14/REC-008999/REV-F72EB8867381.json" because no identity-based policy allows the s3:PutObject action

In [50]:
import boto3

s3 = boto3.client(
    "s3",
    region_name="ap-southeast-2",
)

test_key = (
    "artifacts/dashboard/"
    "payment-risk-dashboard/v1/"
    "human-reviews/"
    "_permission-test.txt"
)

s3.put_object(
    Bucket="construction-payment-risk-dev-gk53",
    Key=test_key,
    Body=b"permission-test",
)

print(
    "17H.4 S3 human-review write permission: PASS"
)

17H.4 S3 human-review write permission: PASS


In [51]:
s3.delete_object(
    Bucket="construction-payment-risk-dev-gk53",
    Key=test_key,
)

print(
    "Permission test object cleanup: PASS"
)

AccessDenied: An error occurred (AccessDenied) when calling the DeleteObject operation: User: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker is not authorized to perform: s3:DeleteObject on resource: "arn:aws:s3:::construction-payment-risk-dev-gk53/artifacts/dashboard/payment-risk-dashboard/v1/human-reviews/_permission-test.txt" because no identity-based policy allows the s3:DeleteObject action

In [52]:
audit_result = (
    audit_repository
    .save_review(
        test_review
    )
)

print(audit_result)

{'bucket': 'construction-payment-risk-dev-gk53', 'key': 'artifacts/dashboard/payment-risk-dashboard/v1/human-reviews/2026/09/14/REC-008999/REV-F72EB8867381.json', 'review_id': 'REV-F72EB8867381', 'workflow_id': 'REC-008999', 'persisted_at': '2026-09-14T09:48:58.439012+00:00'}


In [53]:
persisted_review = (
    audit_repository
    .get_review(
        audit_result["key"]
    )
)

assert (
    persisted_review["review_id"]
    == test_review["review_id"]
)

assert (
    persisted_review["workflow_id"]
    == test_review["workflow_id"]
)

assert (
    persisted_review["review_disposition"]
    == test_review["review_disposition"]
)

assert (
    persisted_review["model_decision"]
    == test_review["model_decision"]
)

assert (
    persisted_review["governance"][
        "model_prediction_overwritten"
    ]
    is False
)

print(
    "17H.4 persistent audit trail test: PASS"
)

17H.4 persistent audit trail test: PASS


In [54]:
from pathlib import Path

path = Path("dashboard/app.py")
text = path.read_text(encoding="utf-8")

old = '''from dashboard.services.human_review_service import (
    HumanReviewService,
)'''

new = '''from dashboard.services.human_review_service import (
    HumanReviewService,
)
from dashboard.repositories.human_review_audit_repository import (
    HumanReviewAuditRepository,
)'''

if old not in text:
    raise RuntimeError(
        "HumanReviewService import block not found."
    )

text = text.replace(old, new, 1)

path.write_text(
    text,
    encoding="utf-8",
)

print("17H.5 audit repository import: PASS")

17H.5 audit repository import: PASS


In [56]:
from pathlib import Path

path = Path("dashboard/app.py")
text = path.read_text(encoding="utf-8")

old = '''            st.session_state[
                "latest_human_review"
            ] = review_record

            st.success(
                "Human operational review recorded "
                "in the current dashboard session."
            )'''

new = '''            audit_repository = (
                HumanReviewAuditRepository()
            )

            try:
                persistence = (
                    audit_repository.save_review(
                        review_record
                    )
                )

            except Exception as exc:
                st.error(
                    "The review was created but could "
                    "not be persisted to the audit store."
                )
                st.exception(exc)

            else:
                review_record[
                    "audit_location"
                ] = persistence["key"]

                st.session_state[
                    "latest_human_review"
                ] = review_record

                st.success(
                    "Human operational review recorded "
                    "and persisted to the audit trail."
                )'''

if old not in text:
    raise RuntimeError(
        "Expected review session block not found. "
        "No changes made."
    )

text = text.replace(old, new, 1)

path.write_text(
    text,
    encoding="utf-8",
)

print(
    "17H.5 persistent review integration source: PASS"
)

17H.5 persistent review integration source: PASS


In [57]:
import py_compile
from pathlib import Path

for file in Path("dashboard").rglob("*.py"):
    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17H.5 dashboard package compile: PASS"
)

17H.5 dashboard package compile: PASS


In [58]:
import json
from pathlib import Path

contract_path = Path(
    "artifacts/dashboard/payment-risk-dashboard/v1/"
    "human-review-contract-v1.json"
)

contract = json.loads(
    contract_path.read_text(
        encoding="utf-8"
    )
)

assert (
    contract["governance"][
        "human_decision_is_separate_from_model_prediction"
    ]
    is True
)

assert (
    contract["governance"][
        "model_prediction_must_not_be_overwritten"
    ]
    is True
)

assert (
    contract["governance"][
        "legal_rights_decision"
    ]
    is False
)

assert (
    contract["governance"][
        "automatic_external_action"
    ]
    is False
)

assert (
    persisted_review["model_decision"]
    == test_review["model_decision"]
)

assert (
    persisted_review[
        "predicted_operational_risk"
    ]
    == test_review[
        "predicted_operational_risk"
    ]
)

assert (
    persisted_review["governance"][
        "model_prediction_overwritten"
    ]
    is False
)

assert (
    persisted_review["governance"][
        "external_action_triggered"
    ]
    is False
)

print(
    "17H.5 governance acceptance test: PASS"
)

17H.5 governance acceptance test: PASS


# Dashboard Containerization. 
```text
The goal is to package the Streamlit dashboard as its own production container, separate from the Prediction API container.
The architecture becomes:

Browser
   ↓
Streamlit Dashboard Container
   ├── Athena → historical business analytics
   ├── Prediction API → live ML scoring
   └── S3 → human-review audit trail

Prediction API Container
   ↓
Logistic Regression v1

```

In [59]:
from pathlib import Path

dockerfile = '''FROM python:3.12-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PYTHONPATH=/app

COPY requirements-dashboard.txt .

RUN pip install --no-cache-dir \
    -r requirements-dashboard.txt

COPY dashboard ./dashboard
COPY artifacts/dashboard ./artifacts/dashboard

EXPOSE 8501

CMD ["streamlit", "run", "dashboard/app.py", "--server.address=0.0.0.0", "--server.port=8501", "--server.headless=true"]
'''

Path(
    "Dockerfile.dashboard"
).write_text(
    dockerfile,
    encoding="utf-8",
)

print(
    "17I.1 dashboard Dockerfile source: PASS"
)

17I.1 dashboard Dockerfile source: PASS


In [60]:
from pathlib import Path

requirements = '''streamlit
boto3
pandas
requests
'''

Path(
    "requirements-dashboard.txt"
).write_text(
    requirements,
    encoding="utf-8",
)

print(
    "17I.1 dashboard requirements source: PASS"
)

17I.1 dashboard requirements source: PASS


In [61]:
from pathlib import Path

dockerfile = Path(
    "Dockerfile.dashboard"
)

requirements = Path(
    "requirements-dashboard.txt"
)

assert dockerfile.exists()
assert requirements.exists()

docker_text = dockerfile.read_text(
    encoding="utf-8"
)

assert "python:3.12-slim" in docker_text
assert "streamlit" in docker_text
assert "8501" in docker_text
assert "dashboard/app.py" in docker_text

required_packages = {
    "streamlit",
    "boto3",
    "pandas",
    "requests",
}

installed = {
    line.strip()
    for line in requirements.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
}

assert required_packages.issubset(
    installed
)

print(
    "17I.1 container contract validation: PASS"
)

17I.1 container contract validation: PASS


In [62]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):
    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17I.1 dashboard package compile: PASS"
)

17I.1 dashboard package compile: PASS


# Externalize Prediction API URL
Right now the dashboard must not depend on localhost:8000 once it becomes its own container. We’ll make the API endpoint configurable through:

In [63]:
from pathlib import Path

path = Path("dashboard/config.py")
text = path.read_text(encoding="utf-8")

if "PREDICTION_API_BASE_URL" not in text:
    text += '''

# Prediction API runtime configuration
import os

PREDICTION_API_BASE_URL = os.getenv(
    "PREDICTION_API_BASE_URL",
    "http://localhost:8000",
).rstrip("/")

PREDICTION_API_TIMEOUT_SECONDS = int(
    os.getenv(
        "PREDICTION_API_TIMEOUT_SECONDS",
        "10",
    )
)
'''

    path.write_text(
        text,
        encoding="utf-8",
    )

print(
    "17I.2 prediction API runtime config source: PASS"
)

17I.2 prediction API runtime config source: PASS


In [64]:
import importlib

import dashboard.config as dashboard_config

importlib.reload(
    dashboard_config
)

print(
    "Prediction API URL:",
    dashboard_config.PREDICTION_API_BASE_URL,
)

print(
    "Prediction API timeout:",
    dashboard_config.PREDICTION_API_TIMEOUT_SECONDS,
)

assert (
    dashboard_config.PREDICTION_API_BASE_URL
    == "http://localhost:8000"
)

assert (
    dashboard_config.PREDICTION_API_TIMEOUT_SECONDS
    == 10
)

print(
    "17I.2 default runtime configuration: PASS"
)

Prediction API URL: http://localhost:8000
Prediction API timeout: 10
17I.2 default runtime configuration: PASS


In [65]:
import os
import importlib

os.environ[
    "PREDICTION_API_BASE_URL"
] = "http://prediction-api.internal:8000"

os.environ[
    "PREDICTION_API_TIMEOUT_SECONDS"
] = "15"

import dashboard.config as dashboard_config

importlib.reload(
    dashboard_config
)

assert (
    dashboard_config.PREDICTION_API_BASE_URL
    == "http://prediction-api.internal:8000"
)

assert (
    dashboard_config.PREDICTION_API_TIMEOUT_SECONDS
    == 15
)

print(
    "17I.2 environment override: PASS"
)

17I.2 environment override: PASS


In [66]:
os.environ[
    "PREDICTION_API_BASE_URL"
] = "http://localhost:8000"

os.environ[
    "PREDICTION_API_TIMEOUT_SECONDS"
] = "10"

importlib.reload(
    dashboard_config
)

print(
    dashboard_config.PREDICTION_API_BASE_URL
)

http://localhost:8000


In [70]:
PredictionApiClient(
    base_url="http://localhost:8000"
)

In [71]:
from pathlib import Path

for path in Path("dashboard").rglob("*.py"):
    text = path.read_text(
        encoding="utf-8"
    )

    if (
        "PredictionApiClient("
        in text
    ):
        print(
            "\\nFILE:",
            path
        )

        for line_number, line in enumerate(
            text.splitlines(),
            start=1,
        ):
            if (
                "PredictionApiClient"
                in line
                or "localhost:8000"
                in line
            ):
                print(
                    line_number,
                    line
                )

\nFILE: dashboard/services/risk_queue_service.py
2     PredictionApiClient,
29             or PredictionApiClient(


# Wire Prediction Client to Runtime Configuration

In [74]:
from pathlib import Path

path = Path(
    "dashboard/services/risk_queue_service.py"
)

lines = path.read_text(
    encoding="utf-8"
).splitlines()

for i, line in enumerate(
    lines[:60],
    start=1,
):
    print(
        f"{i:03}: {line}"
    )

001: from dashboard.clients.prediction_api_client import (
002:     PredictionApiClient,
003: )
004: from dashboard.repositories.athena_repository import (
005:     AthenaRepository,
006: )
007: 
008: 
009: class RiskQueueService:
010:     """
011:     Build a ranked operational review queue.
012: 
013:     Athena selects candidate workflows.
014:     Prediction API supplies model risk and explanation.
015:     """
016: 
017:     def __init__(
018:         self,
019:         repository=None,
020:         prediction_client=None,
021:     ):
022:         self.repository = (
023:             repository
024:             or AthenaRepository()
025:         )
026: 
027:         self.prediction_client = (
028:             prediction_client
029:             or PredictionApiClient(
030:                 base_url="http://127.0.0.1:8000"
031:             )
032:         )
033: 
034:     @staticmethod
035:     def risk_band(
036:         risk_score: float,
037:     ) -> str:
038:         """
039:    

In [75]:
PredictionApiClient(
    base_url="http://127.0.0.1:8000"
)

In [76]:
from pathlib import Path

path = Path(
    "dashboard/services/risk_queue_service.py"
)

text = path.read_text(
    encoding="utf-8"
)

# --------------------------------------------------
# 1. Add runtime config import
# --------------------------------------------------

old_import = '''from dashboard.clients.prediction_api_client import (
    PredictionApiClient,
)
from dashboard.repositories.athena_repository import (
    AthenaRepository,
)'''

new_import = '''from dashboard.clients.prediction_api_client import (
    PredictionApiClient,
)
from dashboard.config import (
    PREDICTION_API_BASE_URL,
    PREDICTION_API_TIMEOUT_SECONDS,
)
from dashboard.repositories.athena_repository import (
    AthenaRepository,
)'''

if old_import not in text:
    raise RuntimeError(
        "Expected import block not found. "
        "No changes made."
    )

text = text.replace(
    old_import,
    new_import,
    1,
)

# --------------------------------------------------
# 2. Replace hard-coded API client
# --------------------------------------------------

old_client = '''            or PredictionApiClient(
                base_url="http://127.0.0.1:8000"
            )'''

new_client = '''            or PredictionApiClient(
                base_url=PREDICTION_API_BASE_URL,
                timeout_seconds=(
                    PREDICTION_API_TIMEOUT_SECONDS
                ),
            )'''

if old_client not in text:
    raise RuntimeError(
        "Expected hard-coded PredictionApiClient "
        "block not found. No changes made."
    )

text = text.replace(
    old_client,
    new_client,
    1,
)

path.write_text(
    text,
    encoding="utf-8",
)

print(
    "17I.3 runtime-configured prediction client source: PASS"
)

17I.3 runtime-configured prediction client source: PASS


In [77]:
import py_compile
from pathlib import Path

for file in Path(
    "dashboard"
).rglob("*.py"):
    py_compile.compile(
        str(file),
        doraise=True,
    )

print(
    "17I.3 dashboard package compile: PASS"
)

17I.3 dashboard package compile: PASS


In [78]:
from pathlib import Path

text = Path(
    "dashboard/services/risk_queue_service.py"
).read_text(
    encoding="utf-8"
)

assert (
    "http://127.0.0.1:8000"
    not in text
)

assert (
    "PREDICTION_API_BASE_URL"
    in text
)

assert (
    "PREDICTION_API_TIMEOUT_SECONDS"
    in text
)

print(
    "17I.3 hard-coded endpoint removal: PASS"
)

17I.3 hard-coded endpoint removal: PASS


In [79]:
import sys
import importlib

importlib.invalidate_caches()

sys.modules.pop(
    "dashboard.services.risk_queue_service",
    None,
)

from dashboard.services.risk_queue_service import (
    RiskQueueService,
)

risk_queue_service = RiskQueueService()

print(
    "Client base URL:",
    risk_queue_service.prediction_client.base_url
)

print(
    "Client timeout:",
    risk_queue_service.prediction_client.timeout_seconds
)

Client base URL: http://localhost:8000
Client timeout: 10


In [80]:
queue = (
    risk_queue_service
    .build_queue(
        limit=3
    )
)

print(
    "Queue records:",
    len(queue)
)

for item in queue:
    print(
        item.get("workflow_id"),
        item.get("predicted_operational_risk"),
        item.get("model_decision"),
    )

assert len(queue) > 0

successful = [
    item
    for item in queue
    if not item.get("prediction_error")
]

assert len(successful) > 0

print(
    "17I.3 runtime-configured prediction integration: PASS"
)

Queue records: 3
None 0.866567588579171 FLAGGED_BY_MODEL
None 0.7726465864485356 FLAGGED_BY_MODEL
None 0.7583439955818435 FLAGGED_BY_MODEL
17I.3 runtime-configured prediction integration: PASS


# Package the Dashboard Build Context for CodeBuild → ECR.


We’ll mirror the approach that already worked for the Prediction API: create a clean ZIP containing only the files required to build the dashboard image.

In [81]:
from pathlib import Path
import zipfile

project_root = Path("/home/sagemaker-user")

build_dir = (
    project_root
    / "builds"
    / "payment-risk-dashboard"
    / "v1"
)

build_dir.mkdir(
    parents=True,
    exist_ok=True,
)

zip_path = (
    build_dir
    / "payment-risk-dashboard-v1-source.zip"
)

files_to_include = [
    project_root / "Dockerfile.dashboard",
    project_root / "requirements-dashboard.txt",
]

directories_to_include = [
    project_root / "dashboard",
    project_root / "artifacts" / "dashboard",
]

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for file_path in files_to_include:
        if not file_path.exists():
            raise FileNotFoundError(
                file_path
            )

        zf.write(
            file_path,
            arcname=file_path.name,
        )

    for directory in directories_to_include:
        if not directory.exists():
            raise FileNotFoundError(
                directory
            )

        for file_path in directory.rglob("*"):
            if (
                file_path.is_file()
                and "__pycache__"
                not in file_path.parts
                and not file_path.name.endswith(
                    ".pyc"
                )
            ):
                zf.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(
                            project_root
                        )
                    ),
                )

print(
    "17I.4 dashboard build package creation: PASS"
)

print(
    "ZIP:",
    zip_path
)

print(
    "Size:",
    zip_path.stat().st_size,
    "bytes",
)

17I.4 dashboard build package creation: PASS
ZIP: /home/sagemaker-user/builds/payment-risk-dashboard/v1/payment-risk-dashboard-v1-source.zip
Size: 17566 bytes


In [82]:
import zipfile

with zipfile.ZipFile(
    zip_path,
    "r",
) as zf:

    names = zf.namelist()

print(
    "Files in package:",
    len(names)
)

for name in names:
    print(name)

assert (
    "Dockerfile.dashboard"
    in names
)

assert (
    "requirements-dashboard.txt"
    in names
)

assert any(
    name == "dashboard/app.py"
    for name in names
)

assert any(
    name.endswith(
        "human-review-contract-v1.json"
    )
    for name in names
)

assert not any(
    "__pycache__"
    in name
    for name in names
)

assert not any(
    name.endswith(".pyc")
    for name in names
)

print(
    "17I.4 dashboard build package validation: PASS"
)

Files in package: 19
Dockerfile.dashboard
requirements-dashboard.txt
dashboard/__init__.py
dashboard/config.py
dashboard/app.py
dashboard/repositories/__init__.py
dashboard/repositories/athena_repository.py
dashboard/repositories/human_review_audit_repository.py
dashboard/services/__init__.py
dashboard/services/analytics_service.py
dashboard/services/risk_queue_service.py
dashboard/services/workflow_assessment_service.py
dashboard/services/human_review_service.py
dashboard/.ipynb_checkpoints/app-checkpoint.py
dashboard/clients/__init__.py
dashboard/clients/prediction_api_client.py
artifacts/dashboard/payment-risk-dashboard/v1/dashboard-product-contract-v1.json
artifacts/dashboard/payment-risk-dashboard/v1/dashboard-architecture-v1.json
artifacts/dashboard/payment-risk-dashboard/v1/human-review-contract-v1.json
17I.4 dashboard build package validation: PASS


In [84]:
import boto3

s3 = boto3.client(
    "s3",
    region_name="ap-southeast-2",
)

bucket = (
    "construction-payment-risk-dev-gk53"
)

key = (
    "builds/payment-risk-dashboard/v1/"
    "payment-risk-dashboard-v1-source.zip"
)

s3.upload_file(
    str(zip_path),
    bucket,
    key,
)

print(
    "17I.4 dashboard build package S3 upload: PASS"
)

print(
    f"s3://{bucket}/{key}"
)

S3UploadFailedError: Failed to upload /home/sagemaker-user/builds/payment-risk-dashboard/v1/payment-risk-dashboard-v1-source.zip to construction-payment-risk-dev-gk53/builds/payment-risk-dashboard/v1/payment-risk-dashboard-v1-source.zip: An error occurred (AccessDenied) when calling the PutObject operation: User: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker is not authorized to perform: s3:PutObject on resource: "arn:aws:s3:::construction-payment-risk-dev-gk53/builds/payment-risk-dashboard/v1/payment-risk-dashboard-v1-source.zip" because no identity-based policy allows the s3:PutObject action

In [85]:
import boto3

s3 = boto3.client(
    "s3",
    region_name="ap-southeast-2",
)

test_key = (
    "builds/payment-risk-dashboard/v1/"
    "_permission-test.txt"
)

s3.put_object(
    Bucket="construction-payment-risk-dev-gk53",
    Key=test_key,
    Body=b"permission-test",
)

print(
    "17I.4 dashboard build prefix write permission: PASS"
)

17I.4 dashboard build prefix write permission: PASS


In [86]:
bucket = "construction-payment-risk-dev-gk53"

key = (
    "builds/payment-risk-dashboard/v1/"
    "payment-risk-dashboard-v1-source.zip"
)

s3.upload_file(
    str(zip_path),
    bucket,
    key,
)

print(
    "17I.4 dashboard build package S3 upload: PASS"
)

17I.4 dashboard build package S3 upload: PASS


In [87]:
response = s3.head_object(
    Bucket=bucket,
    Key=key,
)

print(
    "ContentLength:",
    response["ContentLength"]
)

print(
    "ETag:",
    response["ETag"]
)

assert response["ContentLength"] > 0

print(
    "17I.4 dashboard build package S3 verification: PASS"
)

ContentLength: 17566
ETag: "d132c34add5f6b8774ec73a64ae8e246"
17I.4 dashboard build package S3 verification: PASS


In [88]:
import boto3

ecr = boto3.client(
    "ecr",
    region_name="ap-southeast-2",
)

response = ecr.describe_repositories(
    repositoryNames=[
        "construction-payment-risk-dashboard-dev"
    ]
)

repo = response["repositories"][0]

print(
    "Repository:",
    repo["repositoryName"]
)

print(
    "URI:",
    repo["repositoryUri"]
)

print(
    "Tag mutability:",
    repo["imageTagMutability"]
)

print(
    "Scan on push:",
    repo[
        "imageScanningConfiguration"
    ]["scanOnPush"]
)

assert (
    repo["repositoryName"]
    == "construction-payment-risk-dashboard-dev"
)

assert (
    repo["imageTagMutability"]
    == "IMMUTABLE"
)

assert (
    repo[
        "imageScanningConfiguration"
    ]["scanOnPush"]
    is True
)

print(
    "17I.5A dashboard ECR repository: PASS"
)

Repository: construction-payment-risk-dashboard-dev
URI: 911797457166.dkr.ecr.ap-southeast-2.amazonaws.com/construction-payment-risk-dashboard-dev
Tag mutability: IMMUTABLE
Scan on push: True
17I.5A dashboard ECR repository: PASS


# Create dashboard buildspec

In [89]:
from pathlib import Path

buildspec = '''version: 0.2

phases:

  pre_build:
    commands:
      - echo Logging in to Amazon ECR...
      - AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
      - AWS_REGION=ap-southeast-2
      - ECR_REPOSITORY=construction-payment-risk-dashboard-dev
      - IMAGE_TAG=dashboard-v1.0.0
      - ECR_URI=$AWS_ACCOUNT_ID.dkr.ecr.$AWS_REGION.amazonaws.com/$ECR_REPOSITORY
      - aws ecr get-login-password --region $AWS_REGION | docker login --username AWS --password-stdin $AWS_ACCOUNT_ID.dkr.ecr.$AWS_REGION.amazonaws.com
      - echo ECR login successful

  build:
    commands:
      - echo Building dashboard Docker image...
      - docker build -f Dockerfile.dashboard -t $ECR_REPOSITORY:$IMAGE_TAG .
      - docker tag $ECR_REPOSITORY:$IMAGE_TAG $ECR_URI:$IMAGE_TAG
      - echo Docker build successful

  post_build:
    commands:
      - echo Pushing dashboard image...
      - docker push $ECR_URI:$IMAGE_TAG
      - printf '{"ImageURI":"%s"}' $ECR_URI:$IMAGE_TAG > imageDetail.json
      - echo Dashboard image push complete

artifacts:
  files:
    - imageDetail.json
'''

Path(
    "/home/sagemaker-user/buildspec-dashboard.yml"
).write_text(
    buildspec,
    encoding="utf-8",
)

print(
    "17I.5C dashboard buildspec source: PASS"
)

17I.5C dashboard buildspec source: PASS


In [90]:
from pathlib import Path

text = Path(
    "/home/sagemaker-user/buildspec-dashboard.yml"
).read_text(
    encoding="utf-8"
)

assert (
    "construction-payment-risk-dashboard-dev"
    in text
)

assert (
    "dashboard-v1.0.0"
    in text
)

assert (
    "docker build -f Dockerfile.dashboard"
    in text
)

assert (
    "docker push"
    in text
)

print(
    "17I.5C dashboard buildspec validation: PASS"
)

17I.5C dashboard buildspec validation: PASS


In [91]:
import boto3

iam = boto3.client("iam")

role = iam.get_role(
    RoleName=(
        "ConstructionPaymentRiskDashboardCodeBuildRole"
    )
)

print(
    "Role:",
    role["Role"]["RoleName"]
)

print(
    "17I.5D CodeBuild service role: PASS"
)

ClientError: An error occurred (AccessDenied) when calling the GetRole operation: User: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker is not authorized to perform: iam:GetRole on resource: role ConstructionPaymentRiskDashboardCodeBuildRole because no identity-based policy allows the iam:GetRole action. Go to https://us-east-1.console.aws.amazon.com/iam/home?region=us-east-1#/authorization-details/f132cgtjz5ytkdkjjskiahaw8 for complete details, or call the GetRequestAuthorizationDetails API with the following authorization id: f132cgtjz5ytkdkjjskiahaw8

In [92]:
from pathlib import Path
import zipfile

project_root = Path("/home/sagemaker-user")

zip_path = (
    project_root
    / "builds"
    / "payment-risk-dashboard"
    / "v1"
    / "payment-risk-dashboard-v1-source.zip"
)

buildspec_path = (
    project_root
    / "buildspec-dashboard.yml"
)

with zipfile.ZipFile(
    zip_path,
    "a",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    if "buildspec-dashboard.yml" not in zf.namelist():
        zf.write(
            buildspec_path,
            arcname="buildspec-dashboard.yml",
        )

print(
    "17I.5D buildspec added to source package: PASS"
)

17I.5D buildspec added to source package: PASS


In [93]:
with zipfile.ZipFile(
    zip_path,
    "r",
) as zf:
    names = zf.namelist()

assert "buildspec-dashboard.yml" in names

print(
    "17I.5D updated package validation: PASS"
)

17I.5D updated package validation: PASS


In [94]:
import boto3

s3 = boto3.client(
    "s3",
    region_name="ap-southeast-2",
)

bucket = "construction-payment-risk-dev-gk53"

key = (
    "builds/payment-risk-dashboard/v1/"
    "payment-risk-dashboard-v1-source.zip"
)

s3.upload_file(
    str(zip_path),
    bucket,
    key,
)

head = s3.head_object(
    Bucket=bucket,
    Key=key,
)

print(
    "Updated ContentLength:",
    head["ContentLength"]
)

print(
    "Updated ETag:",
    head["ETag"]
)

print(
    "17I.5D updated build source upload: PASS"
)

Updated ContentLength: 18160
Updated ETag: "1606bd07486d0dc2501106c7a571bad6"
17I.5D updated build source upload: PASS


In [95]:
import boto3

ecr = boto3.client(
    "ecr",
    region_name="ap-southeast-2",
)

response = ecr.describe_images(
    repositoryName=(
        "construction-payment-risk-dashboard-dev"
    ),
    imageIds=[
        {
            "imageTag": "dashboard-v1.0.0"
        }
    ],
)

image = response["imageDetails"][0]

print(
    "Tags:",
    image.get("imageTags")
)

print(
    "Digest:",
    image["imageDigest"]
)

print(
    "Size:",
    image.get("imageSizeInBytes")
)

print(
    "Pushed at:",
    image.get("imagePushedAt")
)

assert (
    "dashboard-v1.0.0"
    in image.get(
        "imageTags",
        [],
    )
)

assert (
    image["imageDigest"]
    .startswith("sha256:")
)

print(
    "17I.6 dashboard ECR image verification: PASS"
)

ImageNotFoundException: An error occurred (ImageNotFoundException) when calling the DescribeImages operation: The image with imageId {imageDigest:'null', imageTag:'dashboard-v1.0.0'} does not exist within the repository with name 'construction-payment-risk-dashboard-dev' in the registry with id '911797457166'

In [96]:
import boto3

codebuild = boto3.client(
    "codebuild",
    region_name="ap-southeast-2",
)

project_name = (
    "construction-payment-risk-dashboard-build-dev"
)

build_ids = codebuild.list_builds_for_project(
    projectName=project_name,
    sortOrder="DESCENDING",
)["ids"]

print(
    "Build count:",
    len(build_ids)
)

if not build_ids:
    print(
        "NO BUILDS FOUND — start the CodeBuild project first."
    )
else:
    latest_build_id = build_ids[0]

    response = codebuild.batch_get_builds(
        ids=[latest_build_id]
    )

    build = response["builds"][0]

    print(
        "Build ID:",
        build["id"]
    )

    print(
        "Status:",
        build["buildStatus"]
    )

    print(
        "Current phase:",
        build.get("currentPhase")
    )

    print(
        "\nPHASES"
    )

    for phase in build.get(
        "phases",
        []
    ):
        print(
            phase.get("phaseType"),
            "=>",
            phase.get("phaseStatus"),
        )

        for context in phase.get(
            "contexts",
            []
        ):
            print(
                "   MESSAGE:",
                context.get("message")
            )

ClientError: An error occurred (AccessDeniedException) when calling the ListBuildsForProject operation: User: arn:aws:sts::911797457166:assumed-role/ConstructionPaymentRiskSageMakerRole/SageMaker is not authorized to perform: codebuild:ListBuildsForProject on resource: arn:aws:codebuild:ap-southeast-2:911797457166:project/construction-payment-risk-dashboard-build-dev because no identity-based policy allows the codebuild:ListBuildsForProject action

In [97]:
import boto3

ecr = boto3.client(
    "ecr",
    region_name="ap-southeast-2",
)

response = ecr.describe_images(
    repositoryName="construction-payment-risk-dashboard-dev",
    imageIds=[
        {
            "imageTag": "dashboard-v1.0.0"
        }
    ],
)

image = response["imageDetails"][0]

print(
    "Tag:",
    image.get("imageTags")
)

print(
    "Digest:",
    image["imageDigest"]
)

print(
    "Size:",
    image.get("imageSizeInBytes")
)

print(
    "Pushed:",
    image.get("imagePushedAt")
)

assert (
    "dashboard-v1.0.0"
    in image.get("imageTags", [])
)

assert (
    image["imageDigest"]
    .startswith("sha256:")
)

print(
    "17I.6 dashboard ECR image verification: PASS"
)

Tag: ['dashboard-v1.0.0']
Digest: sha256:09e146ca6a4472ec5c4559fb0e16851eaa52f23bf48f21b60b3574a4c2502674
Size: 196557076
Pushed: 2026-09-14 10:28:55.832000+00:00
17I.6 dashboard ECR image verification: PASS


```text
AWS Deployment.

17J.1 — Deployment architecture

Before creating ECS resources, there is one important production decision: the dashboard container must not use localhost:8000 in AWS. In separate ECS tasks, localhost means the dashboard container itself—not the Prediction API.

                    Internet
                       │
                       ▼
             Application Load Balancer
                       │
                 ┌─────┴─────┐
                 │           │
                 ▼           ▼
          Dashboard       Prediction API
          ECS Fargate      ECS Fargate
          Port 8501        Port 8000
                 │           │
                 │           └── LR v1 inference
                 │
                 ├── Athena
                 ├── Glue Catalog
                 └── S3 Human Review Audit

Dashboard environment:
PREDICTION_API_BASE_URL=<reachable API endpoint>

```

# Create ECS cluster

In [98]:
import os
print(os.getcwd())

/home/sagemaker-user


In [99]:
!ls -la /home/sagemaker-user

total 1000
drwx------. 21 sagemaker-user nogroup   4096 Sep 14 10:54 .
drwxrwxrwx.  1 root           root        28 Aug 26 04:12 ..
drwxr-xr-x.  3 sagemaker-user users       20 Sep 11 03:45 .agent
-rw-rw-rw-.  1 nobody         nogroup      5 Sep 11 03:45 .bashrc
drwxr-xr-x.  5 sagemaker-user users       41 Sep 14 06:43 .cache
drwxr-xr-x.  3 sagemaker-user users       20 Sep 11 03:45 .claude
drwxr-xr-x.  3 sagemaker-user users       24 Sep 11 03:48 .config
drwxr-xr-x.  2 sagemaker-user users       62 Sep 11 03:48 .ipynb_checkpoints
drwxr-xr-x.  3 sagemaker-user users       29 Sep 11 03:48 .ipython
drwxr-xr-x.  3 sagemaker-user users       17 Sep 11 03:46 .jupyter
drwxr-xr-x.  5 sagemaker-user users       50 Sep 11 03:45 .kiro
drwxr-xr-x.  3 sagemaker-user users       19 Sep 11 03:45 .local
drwxr-xr-x.  3 sagemaker-user users       19 Sep 11 03:45 .npm
drwxr-xr-x.  3 sagemaker-user users       70 Sep 11 06:10 .pytest_cache
drwxr-xr-x.  2 sagemaker-user users       62 Sep 11 03:45 .sagema

In [100]:
!find /home/sagemaker-user -maxdepth 2 -type d | sort | head -100

/home/sagemaker-user
/home/sagemaker-user/.agent
/home/sagemaker-user/.agent/skills
/home/sagemaker-user/.cache
/home/sagemaker-user/.cache/YAPF
/home/sagemaker-user/.cache/jedi
/home/sagemaker-user/.cache/pip
/home/sagemaker-user/.claude
/home/sagemaker-user/.claude/skills
/home/sagemaker-user/.config
/home/sagemaker-user/.config/matplotlib
/home/sagemaker-user/.ipynb_checkpoints
/home/sagemaker-user/.ipython
/home/sagemaker-user/.ipython/profile_default
/home/sagemaker-user/.jupyter
/home/sagemaker-user/.jupyter/lab
/home/sagemaker-user/.kiro
/home/sagemaker-user/.kiro/agents
/home/sagemaker-user/.kiro/settings
/home/sagemaker-user/.kiro/skills
/home/sagemaker-user/.local
/home/sagemaker-user/.local/share
/home/sagemaker-user/.npm
/home/sagemaker-user/.npm/_logs
/home/sagemaker-user/.pytest_cache
/home/sagemaker-user/.pytest_cache/v
/home/sagemaker-user/.sagemaker_sql_editor_api_cache
/home/sagemaker-user/.streamlit
/home/sagemaker-user/.virtual_documents
/home/sagemaker-user/app
/ho